Dataset upload

In [ ]:
pip install numpy pandas matplotlib scikit-learn

In [ ]:
import pickle

# Load QR code images
with open("C:\\Users\\LENOVO\\Desktop\\FYP\\QuishingDataset\\qr_codes_29.pickle", 'rb') as f:
    qr_codes = pickle.load(f)

# Load labels
with open("C:\\Users\\LENOVO\\Desktop\\FYP\\QuishingDataset\\qr_codes_29_labels.pickle", 'rb') as f:
    labels = pickle.load(f)

print(type(qr_codes), qr_codes.shape)
print(type(labels), labels.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Select 10 random samples
indices = np.random.choice(len(qr_codes), 10, replace=False)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(qr_codes[indices[i]], cmap="gray")
    ax.set_title(f"Label: {labels[indices[i]]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

print("Unique labels:", np.unique(labels))

unique, counts = np.unique(labels, return_counts=True)

for u, c in zip(unique, counts):
    print(f"Label {u}: {c}")

QR codes decodes

In [ ]:
import cv2
import numpy as np
import pandas as pd
detector = cv2.QRCodeDetector()

qr = qr_codes[0]

# 尝试两种黑白方向
if qr.max() <= 1:
    img1 = (qr * 255).astype(np.uint8)
    img2 = ((1 - qr) * 255).astype(np.uint8)
else:
    img1 = qr.astype(np.uint8)
    img2 = 255 - img1

for idx, img in enumerate([img1, img2], start=1):
    img_border = cv2.copyMakeBorder(
        img,
        4, 4, 4, 4,
        cv2.BORDER_CONSTANT,
        value=255
    )

    img_large = cv2.resize(
        img_border,
        None,
        fx=10,
        fy=10,
        interpolation=cv2.INTER_NEAREST
    )

    data, points, straight_qrcode = detector.detectAndDecode(img_large)

    print(f"Attempt {idx}:")
    print("Decoded:", data)

In [ ]:
import cv2
import numpy as np
import pandas as pd

detector = cv2.QRCodeDetector()

decoded_urls = []

for i, qr in enumerate(qr_codes):
    # Based on your successful Attempt 1
    if qr.max() <= 1:
        img = (qr * 255).astype(np.uint8)
    else:
        img = qr.astype(np.uint8)

    # Add quiet zone
    img = cv2.copyMakeBorder(
        img,
        4, 4, 4, 4,
        cv2.BORDER_CONSTANT,
        value=255
    )

    # Enlarge QR
    img = cv2.resize(
        img,
        None,
        fx=10,
        fy=10,
        interpolation=cv2.INTER_NEAREST
    )

    data, points, _ = detector.detectAndDecode(img)

    decoded_urls.append(data if data else None)

print("Total QR codes:", len(decoded_urls))
print("Successfully decoded:", sum(x is not None for x in decoded_urls))
print("Failed:", sum(x is None for x in decoded_urls))

In [ ]:
df = pd.DataFrame({
    "url": decoded_urls,
    "label": labels
})

df.head(10)

Invalid URL Detection

In [ ]:
print(df["label"].value_counts())
print()
print("Missing URLs:", df["url"].isna().sum())
print("Decoded rate:", df["url"].notna().mean())

In [ ]:
df_valid = df.dropna(subset=["url"]).copy()

print("Valid samples:", len(df_valid))
print(df_valid["label"].value_counts())

In [ ]:
df.to_csv("qr_decoding_all.csv", index=False)
df_valid.to_csv("qr_decoded_valid.csv", index=False)

In [ ]:
df_valid["url"].duplicated().sum()

In [ ]:
df["decode_success"] = df["url"].notna().astype(int)

pd.crosstab(df["label"], df["decode_success"])

In [ ]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df["label"], df["decode_success"])

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p)

In [ ]:
decode_rate_by_label = df.groupby("label")["decode_success"].mean()

print(decode_rate_by_label)

In [ ]:
import numpy as np

n = table.to_numpy().sum()

cramers_v = np.sqrt(chi2 / n)

print("Cramer's V:", cramers_v)

QR decoding success differed significantly between the two classes (\(\chi^2=12.33, p<0.001\)); however, the effect size was very small (Cramér's \(V\approx0.035\)), suggesting limited practical association between class membership and decoding failure.

In [ ]:
from urllib.parse import urlparse, urlunparse

def normalize_url(url):
    url = str(url).strip()

    # Add temporary scheme for parsing
    parse_url = url
    if not url.lower().startswith(("http://", "https://")):
        parse_url = "http://" + url

    parsed = urlparse(parse_url)

    # Normalize hostname
    hostname = parsed.hostname.lower() if parsed.hostname else ""

    # Remove www.
    if hostname.startswith("www."):
        hostname = hostname[4:]

    # Preserve port if one exists
    try:
        port = parsed.port
    except ValueError:
        port = None

    netloc = hostname

    if port is not None:
        netloc += f":{port}"

    # Lower-case scheme
    scheme = parsed.scheme.lower()

    # Keep path/query because they can carry phishing information
    normalized = urlunparse(
        (
            scheme,
            netloc,
            parsed.path,
            "",
            parsed.query,
            ""
        )
    )

    return normalized

In [ ]:
df_valid["normalized_url"] = df_valid["url"].apply(normalize_url)

df_valid[
    ["url", "normalized_url", "label"]
].head(10)

In [ ]:
print(
    "Normalized URL duplicates:",
    df_valid["normalized_url"].duplicated().sum()
)

In [ ]:
conflict_check = (
    df_valid
    .groupby("normalized_url")["label"]
    .nunique()
)

conflicts = conflict_check[conflict_check > 1]

print("Conflicting normalized URLs:", len(conflicts))

In [ ]:
df_valid_clean = (
    df_valid
    .drop_duplicates(subset=["normalized_url"])
    .copy()
)

print("Before:", len(df_valid))
print("After:", len(df_valid_clean))

In [ ]:
print(df_valid_clean["label"].value_counts())

Data Audit

In [ ]:
%pip install tldextract

In [ ]:
import tldextract
from urllib.parse import urlparse

def get_domain(url):

    parsed_url = url

    if not url.lower().startswith(
        ("http://", "https://")
    ):
        parsed_url = "http://" + url

    hostname = urlparse(parsed_url).hostname

    if hostname is None:
        return None

    ext = tldextract.extract(hostname)

    if ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    else:
        return ext.domain

In [ ]:
df_valid_clean["domain"] = (
    df_valid_clean["url"]
    .apply(get_domain)
)

In [ ]:
print(
    "Unique URLs:",
    df_valid_clean["normalized_url"].nunique()
)

print(
    "Unique domains:",
    df_valid_clean["domain"].nunique()
)

In [ ]:
print(
    df_valid_clean["domain"]
    .value_counts()
    .head(20)
)

In [ ]:
domain_label_count = (
    df_valid_clean
    .groupby("domain")["label"]
    .nunique()
)

mixed_domains = domain_label_count[
    domain_label_count > 1
]

print(
    "Domains appearing in both labels:",
    len(mixed_domains)
)

In [ ]:
mixed_domain_names = mixed_domains.index.tolist()

mixed_detail = (
    df_valid_clean[
        df_valid_clean["domain"].isin(mixed_domain_names)
    ]
    .groupby(["domain", "label"])
    .size()
    .unstack(fill_value=0)
)

mixed_detail.columns = [
    f"label_{c}" for c in mixed_detail.columns
]

mixed_detail["total"] = mixed_detail.sum(axis=1)

mixed_detail.sort_values(
    "total",
    ascending=False
)

In [ ]:
for domain in mixed_domain_names:
    print("=" * 80)
    print("DOMAIN:", domain)

    display(
        df_valid_clean[
            df_valid_clean["domain"] == domain
        ][
            ["url", "normalized_url", "label"]
        ].head(20)
    )

In [ ]:
from urllib.parse import urlparse

def extract_hostname(url):
    parse_url = url

    if not url.lower().startswith(("http://", "https://")):
        parse_url = "http://" + url

    parsed = urlparse(parse_url)

    hostname = parsed.hostname

    if hostname is None:
        return None

    hostname = hostname.lower()

    if hostname.startswith("www."):
        hostname = hostname[4:]

    return hostname


df_valid_clean["hostname"] = (
    df_valid_clean["url"]
    .apply(extract_hostname)
)

In [ ]:
print(df_valid_clean.columns.tolist())

In [ ]:
df_valid_clean[
    ["url", "hostname", "domain", "label"]
].head(10)

In [ ]:
print("Unique hostnames:", df_valid_clean["hostname"].nunique())

hostname_label_count = (
    df_valid_clean
    .groupby("hostname")["label"]
    .nunique()
)

mixed_hostnames = hostname_label_count[
    hostname_label_count > 1
]

print(
    "Hostnames appearing in both labels:",
    len(mixed_hostnames)
)

In [ ]:
print(
    df_valid_clean["hostname"]
    .value_counts()
    .head(20)
)

In [ ]:
mixed_hostname_names = mixed_hostnames.index.tolist()

mixed_hostname_detail = (
    df_valid_clean[
        df_valid_clean["hostname"].isin(mixed_hostname_names)
    ]
    .groupby(["hostname", "label"])
    .size()
    .unstack(fill_value=0)
)

print(mixed_hostname_detail)

In [ ]:
for h in mixed_hostname_names:
    print("=" * 80)
    print("HOSTNAME:", h)

    display(
        df_valid_clean[
            df_valid_clean["hostname"] == h
        ][["url", "normalized_url", "label"]]
    )

In [ ]:
host_counts = df_valid_clean["hostname"].value_counts()

print("Total unique hostnames:", len(host_counts))

print(
    "Hostnames with >=2 URLs:",
    (host_counts >= 2).sum()
)

print(
    "Hostnames with >=5 URLs:",
    (host_counts >= 5).sum()
)

print(
    "Hostnames with >=10 URLs:",
    (host_counts >= 10).sum()
)

print(
    "Hostnames with >=50 URLs:",
    (host_counts >= 50).sum()
)

print(
    "Largest hostname group:",
    host_counts.max()
)

In [ ]:
for k in [1, 5, 10, 20]:
    share = host_counts.head(k).sum() / len(df_valid_clean)

    print(
        f"Top-{k} hostname share: "
        f"{share:.4f} ({share*100:.2f}%)"
    )

In [ ]:
for label in sorted(df_valid_clean["label"].unique()):

    temp = df_valid_clean[
        df_valid_clean["label"] == label
    ]

    print(
        f"Label {label}: "
        f"{len(temp)} URLs, "
        f"{temp['hostname'].nunique()} unique hostnames, "
        f"{len(temp)/temp['hostname'].nunique():.2f} URLs/hostname"
    )

In [ ]:
df_valid_clean.to_csv(
    "qr_dataset_audited_v1.csv",
    index=False
)

In [ ]:
audit_summary = {
    "Original QR samples": 9987,
    "Successfully decoded": 9720,
    "Decode rate": 9720 / 9987,
    "Final unique URLs": len(df_valid_clean),
    "Label 0": (df_valid_clean["label"] == 0).sum(),
    "Label 1": (df_valid_clean["label"] == 1).sum(),
    "Unique hostnames": df_valid_clean["hostname"].nunique(),
    "Unique domains": df_valid_clean["domain"].nunique(),
    "Mixed hostnames": len(mixed_hostnames),
    "Largest hostname group": host_counts.max(),
    "Top-10 hostname share": host_counts.head(10).sum() / len(df_valid_clean)
}

pd.DataFrame(
    audit_summary.items(),
    columns=["Statistic", "Value"]
)

Standardized QR Generation

In [ ]:
%pip install qrcode[pil]

In [ ]:
import qrcode
from qrcode.constants import ERROR_CORRECT_M
import numpy as np

print("qrcode imported successfully")

In [ ]:
#testing if the library are working
def generate_standardized_qr(url):
    qr = qrcode.QRCode(
        version=None,
        error_correction=ERROR_CORRECT_M,
        box_size=1,
        border=4
    )

    qr.add_data(url)
    qr.make(fit=True)

    matrix = np.array(
        qr.get_matrix(),
        dtype=np.uint8
    )

    return matrix, qr.version


test_matrix, test_version = generate_standardized_qr(
    df_valid_clean.iloc[0]["url"]
)

print("QR version:", test_version)
print("Matrix shape:", test_matrix.shape)
print("Unique values:", np.unique(test_matrix))

In [ ]:
standard_qrs = []
standard_versions = []
standard_sizes = []

for i, url in enumerate(df_valid_clean["url"]):
    matrix, version = generate_standardized_qr(url)

    standard_qrs.append(matrix)
    standard_versions.append(version)
    standard_sizes.append(matrix.shape[0])

    if (i + 1) % 1000 == 0:
        print(f"Generated {i + 1} QR codes")

print("Done:", len(standard_qrs))

In [ ]:
df_valid_clean["qr_version"] = standard_versions
df_valid_clean["qr_matrix_size"] = standard_sizes

In [ ]:
print(df_valid_clean["qr_version"].value_counts().sort_index())

In [ ]:
print(
    pd.crosstab(
        df_valid_clean["qr_version"],
        df_valid_clean["label"]
    )
)

In [ ]:
df_valid_clean.to_csv(
    "qr_dataset_standardized_metadata.csv",
    index=False
)

In [ ]:
import pickle

with open("standardized_qr_matrices.pkl", "wb") as f:
    pickle.dump(standard_qrs, f)

In [ ]:
print("Samples:", len(df_valid_clean))
print("QR matrices:", len(standard_qrs))
print("Versions:", len(standard_versions))
print("Labels:", len(df_valid_clean["label"]))

Preliminary inspection suggests that QR version distribution differs across the two labels. Formal statistical testing will be performed after the full feature set is extracted.

Lexical Features

In [ ]:
import re
from urllib.parse import urlparse
import pandas as pd
import numpy as np

SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]

In [ ]:
# Basic extractions
def extract_lexical_features(url):

    raw_url = str(url).strip()
    lower_url = raw_url.lower()

    scheme_present = int(
        lower_url.startswith(("http://", "https://"))
    )

    uses_https = int(
        lower_url.startswith("https://")
    )

    parse_url = raw_url

    if not scheme_present:
        parse_url = "http://" + raw_url

    parsed = urlparse(parse_url)

    hostname = parsed.hostname or ""
    path = parsed.path or ""
    query = parsed.query or ""
    fragment = parsed.fragment or ""

    url_length = len(raw_url)
    domain_length = len(hostname)
    path_length = len(path)
    query_length = len(query)

    num_dots = raw_url.count(".")
    num_hyphens = raw_url.count("-")
    num_digits = sum(c.isdigit() for c in raw_url)

    num_special_chars = sum(
        not c.isalnum()
        for c in raw_url
    )

    digit_ratio = (
        num_digits / url_length
        if url_length > 0
        else 0
    )

    special_char_ratio = (
        num_special_chars / url_length
        if url_length > 0
        else 0
    )

    parts = [
        p for p in hostname.split(".")
        if p
    ]

    num_subdomains = max(
        len(parts) - 2,
        0
    )

    contains_ip = int(
        bool(
            re.fullmatch(
                r"(?:\d{1,3}\.){3}\d{1,3}",
                hostname
            )
        )
    )

    contains_at = int("@" in raw_url)

    try:
        contains_port = int(parsed.port is not None)
    except ValueError:
        contains_port = 0

    suspicious_keyword_count = sum(
        keyword in lower_url
        for keyword in SUSPICIOUS_KEYWORDS
    )

    return {
        "url_length": url_length,
        "hostname_length": len(hostname),
        "path_length": path_length,
        "query_length": query_length,

        "num_dots": num_dots,
        "num_hyphens": num_hyphens,
        "num_digits": num_digits,
        "num_special_chars": num_special_chars,

        "digit_ratio": digit_ratio,
        "special_char_ratio": special_char_ratio,

        "num_subdomains": num_subdomains,

        "has_query": int(bool(query)),
        "has_fragment": int(bool(fragment)),

        "scheme_present": scheme_present,
        "uses_https": uses_https,

        "contains_ip": contains_ip,
        "contains_at": contains_at,
        "contains_port": contains_port,

        "suspicious_keyword_count":
            suspicious_keyword_count
    }

In [ ]:
lexical_features = (
    df_valid_clean["url"]
    .apply(extract_lexical_features)
)

lexical_df = pd.DataFrame(
    lexical_features.tolist(),
    index=df_valid_clean.index
)

print(lexical_df.shape)
lexical_df.head()

In [ ]:
# Quanlity checking
print("Missing values:")
print(lexical_df.isna().sum())

In [ ]:
print("Infinite values:")
print(
    np.isinf(
        lexical_df.select_dtypes(include=np.number)
    ).sum()
)

In [ ]:
lexical_df.describe().T

In [ ]:
# zero variance checking
zero_variance_lexical = [
    col
    for col in lexical_df.columns
    if lexical_df[col].nunique() <= 1
]

print(
    "Zero-variance lexical features:",
    zero_variance_lexical
)

In [ ]:
df_features = pd.concat(
    [
        df_valid_clean.reset_index(drop=True),
        lexical_df.reset_index(drop=True)
    ],
    axis=1
)

print(df_features.shape)
df_features.head()

In [ ]:
df_features.to_csv(
    "qr_dataset_with_lexical_features.csv",
    index=False
)

Qiut three with zero variances 

In [ ]:
import re
import tldextract
from urllib.parse import urlparse

SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]


def extract_lexical_features(url):

    raw_url = str(url).strip()
    lower_url = raw_url.lower()

    scheme_present = int(
        lower_url.startswith(("http://", "https://"))
    )

    parse_url = raw_url

    if not scheme_present:
        parse_url = "http://" + raw_url

    parsed = urlparse(parse_url)

    hostname = parsed.hostname or ""
    path = parsed.path or ""
    query = parsed.query or ""
    fragment = parsed.fragment or ""

    url_length = len(raw_url)

    num_digits = sum(
        c.isdigit()
        for c in raw_url
    )

    num_special_chars = sum(
        not c.isalnum()
        for c in raw_url
    )

    # Correct subdomain calculation
    ext = tldextract.extract(hostname)

    if ext.subdomain:
        num_subdomains = len(
            [x for x in ext.subdomain.split(".") if x]
        )
    else:
        num_subdomains = 0

    suspicious_keyword_count = sum(
        keyword in lower_url
        for keyword in SUSPICIOUS_KEYWORDS
    )

    return {
        "url_length": url_length,
        "hostname_length": len(hostname), #fixed
        "path_length": len(path),
        "query_length": len(query),

        "num_dots": raw_url.count("."),
        "num_hyphens": raw_url.count("-"),
        "num_digits": num_digits,
        "num_special_chars": num_special_chars,

        "digit_ratio":
            num_digits / max(url_length, 1),

        "special_char_ratio":
            num_special_chars / max(url_length, 1),

        "num_subdomains": num_subdomains,

        "has_query": int(bool(query)),
        "has_fragment": int(bool(fragment)),

        "contains_at": int("@" in raw_url),

        "contains_port": int(
            parsed.port is not None
        ),

        "suspicious_keyword_count":
            suspicious_keyword_count
    }

In [ ]:
lexical_features = (
    df_valid_clean["url"]
    .apply(extract_lexical_features)
)

lexical_df = pd.DataFrame(
    lexical_features.tolist(),
    index=df_valid_clean.index
)

In [ ]:
print("Shape:", lexical_df.shape)

print(
    "Zero variance:",
    [
        c for c in lexical_df.columns
        if lexical_df[c].nunique() <= 1
    ]
)

Structural Features

In [ ]:
# 1. Remove QR quiet-zone border
def remove_border(matrix, border=4):
    if border == 0:
        return matrix

    return matrix[
        border:-border,
        border:-border
    ]

In [ ]:
def transition_features(matrix):
    m = remove_border(matrix)

    horizontal = (m[:, 1:] != m[:, :-1])
    vertical = (m[1:, :] != m[:-1, :])

    h_density = horizontal.mean()
    v_density = vertical.mean()

    total_transitions = (
        horizontal.sum()
        + vertical.sum()
    )

    total_pairs = (
        horizontal.size
        + vertical.size
    )

    overall_density = (
        total_transitions / total_pairs
    )

    return (
        h_density,
        v_density,
        overall_density
    )

In [ ]:
def get_runs(arr):
    runs = []

    if len(arr) == 0:
        return runs

    current = arr[0]
    length = 1

    for value in arr[1:]:
        if value == current:
            length += 1
        else:
            runs.append(length)
            current = value
            length = 1

    runs.append(length)

    return runs

In [ ]:
def run_length_features(matrix):
    m = remove_border(matrix)

    horizontal_runs = []
    vertical_runs = []

    for row in m:
        horizontal_runs.extend(
            get_runs(row)
        )

    for col in m.T:
        vertical_runs.extend(
            get_runs(col)
        )

    return {
        "mean_horizontal_run":
            np.mean(horizontal_runs),

        "std_horizontal_run":
            np.std(horizontal_runs),

        "max_horizontal_run":
            np.max(horizontal_runs),

        "mean_vertical_run":
            np.mean(vertical_runs),

        "std_vertical_run":
            np.std(vertical_runs),

        "max_vertical_run":
            np.max(vertical_runs)
    }

In [ ]:
from collections import Counter

def pattern_2x2_features(matrix):
    m = remove_border(matrix)

    patterns = []

    for i in range(m.shape[0] - 1):
        for j in range(m.shape[1] - 1):

            block = m[
                i:i+2,
                j:j+2
            ]

            pattern = tuple(
                block.flatten()
            )

            patterns.append(pattern)

    counts = Counter(patterns)

    total = len(patterns)

    probs = np.array(
        list(counts.values()),
        dtype=float
    ) / total

    entropy = -np.sum(
        probs * np.log2(probs)
    )

    diversity = len(counts) / 16

    return entropy, diversity


In [ ]:
def extract_structural_features(
    matrix,
    qr_version
):
    m = remove_border(matrix)

    h_t, v_t, overall_t = (
        transition_features(matrix)
    )

    run_features = (
        run_length_features(matrix)
    )

    entropy_2x2, diversity_2x2 = (
        pattern_2x2_features(matrix)
    )

    features = {
        "qr_version":
            qr_version,

        "qr_matrix_size":
            m.shape[0],

        "black_ratio":
            np.mean(m == 1),

        "horizontal_transition_density":
            h_t,

        "vertical_transition_density":
            v_t,

        "overall_transition_density":
            overall_t,

        "pattern_2x2_entropy":
            entropy_2x2,

        "pattern_2x2_diversity":
            diversity_2x2
    }

    features.update(run_features)

    return features

In [ ]:
structural_features = []

for matrix, version in zip(
    standard_qrs,
    standard_versions
):
    features = extract_structural_features(
        matrix,
        version
    )

    structural_features.append(features)

structural_df = pd.DataFrame(
    structural_features
)

print(structural_df.shape)
structural_df.head()

In [ ]:
print("Missing values:")
print(structural_df.isna().sum())

print(
    "Zero variance:",
    [
        c for c in structural_df.columns
        if structural_df[c].nunique() <= 1
    ]
)

structural_df.describe().T

In [ ]:
structural_df = structural_df.drop(
    columns=["pattern_2x2_diversity"]
)

In [ ]:
print(structural_df.shape)

print(
    [
        c for c in structural_df.columns
        if structural_df[c].nunique() <= 1
    ]
)

In [ ]:
# Keep original QR index for later sensitivity analysis
if "original_index" not in df_valid_clean.columns:
    df_valid_clean["original_index"] = df_valid_clean.index

metadata_cols = [
    "original_index",
    "url",
    "normalized_url",
    "hostname",
    "domain",
    "label"
]

In [ ]:
df_all_features = pd.concat(
    [
        df_valid_clean[metadata_cols].reset_index(drop=True),
        lexical_df.reset_index(drop=True),
        structural_df.reset_index(drop=True)
    ],
    axis=1
)

print("Shape:", df_all_features.shape)
print(df_all_features.columns.tolist())

In [ ]:
duplicate_columns = (
    df_all_features.columns[
        df_all_features.columns.duplicated()
    ].tolist()
)

print("Duplicate columns:", duplicate_columns)

In [ ]:
lexical_cols = lexical_df.columns.tolist()
structural_cols = structural_df.columns.tolist()

feature_cols = lexical_cols + structural_cols

print("Lexical features:", len(lexical_cols))
print("Structural features:", len(structural_cols))
print("Total candidate features:", len(feature_cols))

In [ ]:
print(
    "Duplicate feature names:",
    len(feature_cols) != len(set(feature_cols))
)

In [ ]:
print("Samples:", len(df_all_features))
print("Columns:", df_all_features.shape[1])

print(
    "Missing feature values:",
    df_all_features[feature_cols]
    .isna()
    .sum()
    .sum()
)

print(
    "Infinite feature values:",
    np.isinf(
        df_all_features[feature_cols]
    ).sum().sum()
)

print(
    "Class distribution:"
)

print(
    df_all_features["label"]
    .value_counts()
)

In [ ]:
df_all_features.to_csv(
    "qr_full_feature_dataset_v2.csv",
    index=False
)

Statistical Analysis

In [ ]:
binary_features = [
    "has_query",
    "has_fragment",
    "contains_at",
    "contains_port"
]

continuous_features = [
    c for c in feature_cols
    if c not in binary_features
]

print("Continuous/count features:", len(continuous_features))
print("Binary features:", len(binary_features))

In [ ]:
# Continuous Variables Class-wise summary
summary_rows = []

for feature in continuous_features:
    for label in [0, 1]:
        x = df_all_features.loc[
            df_all_features["label"] == label,
            feature
        ]

        summary_rows.append({
            "feature": feature,
            "label": label,
            "n": len(x),
            "mean": x.mean(),
            "std": x.std(),
            "median": x.median(),
            "q1": x.quantile(0.25),
            "q3": x.quantile(0.75)
        })

descriptive_df = pd.DataFrame(summary_rows)

descriptive_df.head(10)

In [ ]:
# Mann–Whitney U
from scipy.stats import mannwhitneyu

mw_results = []

for feature in continuous_features:

    x0 = df_all_features.loc[
        df_all_features["label"] == 0,
        feature
    ].values

    x1 = df_all_features.loc[
        df_all_features["label"] == 1,
        feature
    ].values

    u_stat, p_value = mannwhitneyu(
        x0,
        x1,
        alternative="two-sided"
    )

    mw_results.append({
        "feature": feature,
        "U": u_stat,
        "p_value": p_value,
        "median_label0": np.median(x0),
        "median_label1": np.median(x1)
    })

mw_df = pd.DataFrame(mw_results)

mw_df.sort_values("p_value").head(20)

In [ ]:
# Cliff’s delta
def cliffs_delta(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    greater = 0
    lower = 0

    for xi in x:
        greater += np.sum(xi > y)
        lower += np.sum(xi < y)

    return (greater - lower) / (len(x) * len(y))

In [ ]:
mw_df["cliffs_delta"] = 0.0

for i, row in mw_df.iterrows():

    feature = row["feature"]

    x0 = df_all_features.loc[
        df_all_features["label"] == 0,
        feature
    ]

    x1 = df_all_features.loc[
        df_all_features["label"] == 1,
        feature
    ]

    n0 = len(x0)
    n1 = len(x1)

    U = row["U"]

    mw_df.loc[i, "cliffs_delta"] = (
        2 * U / (n0 * n1) - 1
    )

In [ ]:
def cliff_effect_size(delta):

    d = abs(delta)

    if d < 0.147:
        return "negligible"
    elif d < 0.33:
        return "small"
    elif d < 0.474:
        return "medium"
    else:
        return "large"


mw_df["effect_size"] = (
    mw_df["cliffs_delta"]
    .apply(cliff_effect_size)
)

mw_df.sort_values(
    "cliffs_delta",
    key=lambda x: abs(x),
    ascending=False
).head(25)

In [ ]:
# Binary Feature Treatment
binary_features = [
    "has_query",
    "has_fragment",
    "contains_at",
    "contains_port"
]

In [ ]:
from scipy.stats import fisher_exact

binary_results = []

for feature in binary_features:

    table = pd.crosstab(
        df_all_features["label"],
        df_all_features[feature]
    )

    # Make sure both 0 and 1 columns exist
    table = table.reindex(
        index=[0, 1],
        columns=[0, 1],
        fill_value=0
    )

    odds_ratio, p_value = fisher_exact(
        table.values
    )

    binary_results.append({
        "feature": feature,

        "label0_feature0": table.loc[0, 0],
        "label0_feature1": table.loc[0, 1],

        "label1_feature0": table.loc[1, 0],
        "label1_feature1": table.loc[1, 1],

        "odds_ratio": odds_ratio,
        "p_value": p_value
    })


binary_df = pd.DataFrame(binary_results)

binary_df

In [ ]:
binary_results = []

for feature in binary_features:

    table = pd.crosstab(
        df_all_features["label"],
        df_all_features[feature]
    ).reindex(
        index=[0, 1],
        columns=[0, 1],
        fill_value=0
    )

    a = table.loc[1, 1]   # label1, feature1
    b = table.loc[1, 0]   # label1, feature0
    c = table.loc[0, 1]   # label0, feature1
    d = table.loc[0, 0]   # label0, feature0

    # Fisher exact p-value
    _, p_value = fisher_exact(
        [[a, b],
         [c, d]]
    )

    # Haldane-Anscombe correction if zeros appear
    if min(a, b, c, d) == 0:
        a += 0.5
        b += 0.5
        c += 0.5
        d += 0.5

    odds_ratio = (
        (a / b) /
        (c / d)
    )

    binary_results.append({
        "feature": feature,
        "label0_present": table.loc[0, 1],
        "label1_present": table.loc[1, 1],
        "odds_ratio_label1_vs_label0": odds_ratio,
        "p_value": p_value
    })

binary_df = pd.DataFrame(binary_results)

binary_df

In [ ]:
continuous_p = mw_df[
    ["feature", "p_value"]
].copy()

continuous_p["feature_type"] = "continuous"

binary_p = binary_df[
    ["feature", "p_value"]
].copy()

binary_p["feature_type"] = "binary"

all_pvalues = pd.concat(
    [continuous_p, binary_p],
    ignore_index=True
)

print(all_pvalues.shape)

In [ ]:
# BH-FDR correction
from statsmodels.stats.multitest import multipletests

# Combine all 29 p-values
continuous_p = mw_df[
    ["feature", "p_value"]
].copy()

continuous_p["feature_type"] = "continuous"

binary_p = binary_df[
    ["feature", "p_value"]
].copy()

binary_p["feature_type"] = "binary"

all_pvalues = pd.concat(
    [continuous_p, binary_p],
    ignore_index=True
)

# BH-FDR correction
reject, p_adj, _, _ = multipletests(
    all_pvalues["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

all_pvalues["p_fdr"] = p_adj
all_pvalues["significant_fdr"] = reject

print(all_pvalues.columns)
all_pvalues.head()

In [ ]:
mw_df = mw_df.merge(
    all_pvalues[
        all_pvalues["feature_type"] == "continuous"
    ][
        ["feature", "p_fdr", "significant_fdr"]
    ],
    on="feature",
    how="left"
)

In [ ]:
binary_df = binary_df.merge(
    all_pvalues[
        all_pvalues["feature_type"] == "binary"
    ][
        ["feature", "p_fdr", "significant_fdr"]
    ],
    on="feature",
    how="left"
)

In [ ]:
print(mw_df.columns)
print(binary_df.columns)

In [ ]:
print(mw_df.columns.tolist())

In [ ]:
mw_final = mw_df[
    [
        "feature",
        "median_label0",
        "median_label1",
        "cliffs_delta",
        "effect_size",
        "p_value",
        "p_fdr",
        "significant_fdr"
    ]
].copy()

mw_final = mw_final.sort_values(
    "cliffs_delta",
    key=lambda x: abs(x),
    ascending=False
)

mw_final

In [ ]:
binary_final = binary_df[
    [
        "feature",
        "label0_present",
        "label1_present",
        "odds_ratio_label1_vs_label0",
        "p_value",
        "p_fdr",
        "significant_fdr"
    ]
].copy()

binary_final

In [ ]:
mw_final.to_csv(
    "continuous_statistical_results.csv",
    index=False
)

binary_final.to_csv(
    "binary_statistical_results.csv",
    index=False
)

In [ ]:
# Spearman correlation + redundancy analysis
corr_matrix = (
    df_all_features[feature_cols]
    .corr(method="spearman")
)

corr_matrix.round(3)

In [ ]:
# seeking high correlations pairs
high_corr_pairs = []

for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):

        f1 = feature_cols[i]
        f2 = feature_cols[j]

        rho = corr_matrix.loc[f1, f2]

        if abs(rho) >= 0.85:
            high_corr_pairs.append({
                "feature_1": f1,
                "feature_2": f2,
                "spearman_rho": rho
            })

high_corr_df = pd.DataFrame(high_corr_pairs)

high_corr_df = high_corr_df.sort_values(
    "spearman_rho",
    key=lambda x: abs(x),
    ascending=False
)

high_corr_df

In [ ]:
features_to_remove = [
    "qr_matrix_size"
]

In [ ]:
redundant_features = [
    "qr_matrix_size",
    "has_query",
    "mean_horizontal_run",
    "mean_vertical_run"
]

In [ ]:
# redundancy-filtered candidate set
reduced_feature_cols = [
    f for f in feature_cols
    if f not in redundant_features
]

print("Before:", len(feature_cols))
print("After:", len(reduced_feature_cols))

In [ ]:
check_features = [
    "num_digits",
    "digit_ratio",
    "url_length",
    "num_special_chars",
    "qr_version",
    "pattern_2x2_entropy"
]

mw_df[
    mw_df["feature"].isin(check_features)
][
    [
        "feature",
        "median_label0",
        "median_label1",
        "cliffs_delta",
        "effect_size",
        "p_fdr"
    ]
].sort_values(
    "cliffs_delta",
    key=lambda x: abs(x),
    ascending=False
)

In [ ]:
redundant_features = [
    "qr_matrix_size",
    "has_query",
    "mean_horizontal_run",
    "mean_vertical_run",
    "digit_ratio"
]

reduced_feature_cols = [
    f for f in feature_cols
    if f not in redundant_features
]

print("Original candidate features:", len(feature_cols))
print("After redundancy filtering:", len(reduced_feature_cols))

print(reduced_feature_cols)

Hostname-grouped Train/Test Split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = df_all_features[reduced_feature_cols]
y = df_all_features["label"]
groups = df_all_features["hostname"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_dev = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_dev = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_dev = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Development samples:", len(X_dev))
print("Test samples:", len(X_test))

print("\nDevelopment class distribution:")
print(y_dev.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())

In [ ]:
train_hosts = set(groups_dev)
test_hosts = set(groups_test)

overlap = train_hosts.intersection(test_hosts)

print("Hostname overlap:", len(overlap))

In [ ]:
print(
    "Development positive rate:",
    y_dev.mean()
)

print(
    "Test positive rate:",
    y_test.mean()
)

In [ ]:
split_df = pd.DataFrame({
    "index": df_all_features.index,
    "split": "development"
})

split_df.loc[test_idx, "split"] = "test"

split_df.to_csv(
    "hostname_grouped_split_v1.csv",
    index=False
)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_fold_idx, val_fold_idx) in enumerate(
    sgkf.split(X_dev, y_dev, groups=groups_dev),
    start=1
):

    y_train_fold = y_dev.iloc[train_fold_idx]
    y_val_fold = y_dev.iloc[val_fold_idx]

    train_groups = set(
        groups_dev.iloc[train_fold_idx]
    )

    val_groups = set(
        groups_dev.iloc[val_fold_idx]
    )

    print(f"Fold {fold}")
    print("Train:", len(train_fold_idx))
    print("Validation:", len(val_fold_idx))
    print(
        "Train positive rate:",
        y_train_fold.mean()
    )
    print(
        "Validation positive rate:",
        y_val_fold.mean()
    )
    print(
        "Hostname overlap:",
        len(train_groups & val_groups)
    )
    print("-" * 40)

Feature selection was performed exclusively within the development partition under hostname-grouped cross-validation.

L1 Logistic Regression stability selection

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

In [ ]:
selection_records = []

for fold, (train_fold_idx, val_fold_idx) in enumerate(
    sgkf.split(X_dev, y_dev, groups=groups_dev),
    start=1
):

    X_train_fold = X_dev.iloc[train_fold_idx]
    y_train_fold = y_dev.iloc[train_fold_idx]

    # Standardize using training fold only
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train_fold
    )

    # L1 Logistic Regression
    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=1.0,
        max_iter=5000,
        random_state=42
    )

    model.fit(
        X_train_scaled,
        y_train_fold
    )

    coefficients = model.coef_[0]

    for feature, coef in zip(
        reduced_feature_cols,
        coefficients
    ):
        selection_records.append({
            "fold": fold,
            "feature": feature,
            "coefficient": coef,
            "selected": int(abs(coef) > 1e-8)
        })

In [ ]:
selection_df = pd.DataFrame(
    selection_records
)

selection_df.head()

In [ ]:
stability_df = (
    selection_df
    .groupby("feature")
    .agg(
        selected_folds=("selected", "sum"),
        mean_abs_coefficient=(
            "coefficient",
            lambda x: np.mean(np.abs(x))
        )
    )
    .reset_index()
)

stability_df["selection_stability"] = (
    stability_df["selected_folds"] / 5
)

stability_df = stability_df.sort_values(
    [
        "selection_stability",
        "mean_abs_coefficient"
    ],
    ascending=False
)

stability_df

In [ ]:
stable_features = stability_df.loc[
    stability_df["selection_stability"] >= 0.8,
    "feature"
].tolist()

print("Stable features:")
print(stable_features)

print(
    "Number of stable features:",
    len(stable_features)
)

In [ ]:
# adjust C value
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    matthews_corrcoef
)

C_grid = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0
]

tuning_results = []

for C in C_grid:

    fold_f1 = []
    fold_auc = []
    fold_mcc = []
    fold_n_features = []

    for train_fold_idx, val_fold_idx in sgkf.split(
        X_dev,
        y_dev,
        groups=groups_dev
    ):

        X_train_fold = X_dev.iloc[train_fold_idx]
        X_val_fold = X_dev.iloc[val_fold_idx]

        y_train_fold = y_dev.iloc[train_fold_idx]
        y_val_fold = y_dev.iloc[val_fold_idx]

        # Fit scaler ONLY on training fold
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(
            X_train_fold
        )

        X_val_scaled = scaler.transform(
            X_val_fold
        )

        model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=C,
            max_iter=5000,
            random_state=42
        )

        model.fit(
            X_train_scaled,
            y_train_fold
        )

        y_pred = model.predict(
            X_val_scaled
        )

        y_prob = model.predict_proba(
            X_val_scaled
        )[:, 1]

        f1 = f1_score(
            y_val_fold,
            y_pred
        )

        auc = roc_auc_score(
            y_val_fold,
            y_prob
        )

        mcc = matthews_corrcoef(
            y_val_fold,
            y_pred
        )

        n_selected = np.sum(
            np.abs(model.coef_[0]) > 1e-8
        )

        fold_f1.append(f1)
        fold_auc.append(auc)
        fold_mcc.append(mcc)
        fold_n_features.append(
            n_selected
        )

    tuning_results.append({
        "C": C,

        "mean_f1":
            np.mean(fold_f1),

        "std_f1":
            np.std(fold_f1),

        "mean_auc":
            np.mean(fold_auc),

        "mean_mcc":
            np.mean(fold_mcc),

        "mean_selected_features":
            np.mean(fold_n_features),

        "min_selected_features":
            np.min(fold_n_features),

        "max_selected_features":
            np.max(fold_n_features)
    })

In [ ]:
tuning_df = pd.DataFrame(
    tuning_results
)

tuning_df

In [ ]:
best_idx = tuning_df["mean_f1"].idxmax()

best_mean = tuning_df.loc[
    best_idx,
    "mean_f1"
]

best_se = (
    tuning_df.loc[best_idx, "std_f1"]
    / np.sqrt(5)
)

threshold = (
    best_mean - best_se
)

candidate_models = tuning_df[
    tuning_df["mean_f1"] >= threshold
]

chosen_row = candidate_models.sort_values(
    [
        "mean_selected_features",
        "C"
    ]
).iloc[0]

chosen_C = chosen_row["C"]

print("Best mean F1:", best_mean)
print("One-SE-like threshold:", threshold)
print("Chosen C:", chosen_C)

chosen_row

Choose C=0.01

In [ ]:
selection_records_final = []

for fold, (train_fold_idx, val_fold_idx) in enumerate(
    sgkf.split(X_dev, y_dev, groups=groups_dev),
    start=1
):

    X_train_fold = X_dev.iloc[train_fold_idx]
    y_train_fold = y_dev.iloc[train_fold_idx]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train_fold
    )

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=chosen_C,
        max_iter=5000,
        random_state=42
    )

    model.fit(
        X_train_scaled,
        y_train_fold
    )

    coefficients = model.coef_[0]

    for feature, coef in zip(
        reduced_feature_cols,
        coefficients
    ):
        selection_records_final.append({
            "fold": fold,
            "feature": feature,
            "coefficient": coef,
            "selected": int(abs(coef) > 1e-8)
        })

In [ ]:
selection_final_df = pd.DataFrame(
    selection_records_final
)

stability_final_df = (
    selection_final_df
    .groupby("feature")
    .agg(
        selected_folds=("selected", "sum"),
        mean_abs_coefficient=(
            "coefficient",
            lambda x: np.mean(np.abs(x))
        )
    )
    .reset_index()
)

stability_final_df["selection_stability"] = (
    stability_final_df["selected_folds"] / 5
)

stability_final_df = stability_final_df.sort_values(
    [
        "selection_stability",
        "mean_abs_coefficient"
    ],
    ascending=False
)

stability_final_df

In [ ]:
stable_features_final = stability_final_df.loc[
    stability_final_df["selection_stability"] >= 0.8,
    "feature"
].tolist()

print("Final stable features:")
print(stable_features_final)

print(
    "Number of final stable features:",
    len(stable_features_final)
)

In [ ]:
dev_corr = X_dev.corr(method="spearman")
dev_high_corr_pairs = []

for i in range(len(reduced_feature_cols)):
    for j in range(i + 1, len(reduced_feature_cols)):

        f1 = reduced_feature_cols[i]
        f2 = reduced_feature_cols[j]

        rho = dev_corr.loc[f1, f2]

        if abs(rho) >= 0.85:
            dev_high_corr_pairs.append({
                "feature_1": f1,
                "feature_2": f2,
                "spearman_rho": rho
            })

dev_high_corr_df = pd.DataFrame(dev_high_corr_pairs)

if len(dev_high_corr_df) > 0:
    dev_high_corr_df = dev_high_corr_df.sort_values(
        "spearman_rho",
        key=lambda x: abs(x),
        ascending=False
    )

dev_high_corr_df

ML Baseline

In [ ]:
lexical_reduced = [
    f for f in lexical_cols
    if f in reduced_feature_cols
]

structural_reduced = [
    f for f in structural_cols
    if f in reduced_feature_cols
]

combined_reduced = reduced_feature_cols

selected_features = stable_features_final

In [ ]:
print("Lexical:", len(lexical_reduced))
print("Structural:", len(structural_reduced))
print("Combined:", len(combined_reduced))
print("Selected:", len(selected_features))

In [ ]:
feature_sets = {
    "Lexical": lexical_reduced,
    "Structural": structural_reduced,
    "Combined": combined_reduced,
    "Selected": selected_features
}

In [ ]:
%pip install xgboost

In [ ]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "LR": LogisticRegression(
        max_iter=5000,
        random_state=42
    ),

    "DT": DecisionTreeClassifier(
        random_state=42
    ),

    "RF": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "XGB": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "LR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "DT": DecisionTreeClassifier(
        random_state=42
    ),

    "RF": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "XGB": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
from sklearn.metrics import (
    f1_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef
)

baseline_results = []

for feature_set_name, feature_list in feature_sets.items():

    print("=" * 70)
    print("Feature set:", feature_set_name)

    X_current = X_dev[feature_list]

    for model_name, model in models.items():

        fold_f1 = []
        fold_recall = []
        fold_auc = []
        fold_mcc = []

        for fold, (train_fold_idx, val_fold_idx) in enumerate(
            sgkf.split(
                X_current,
                y_dev,
                groups=groups_dev
            ),
            start=1
        ):

            X_train_fold = X_current.iloc[train_fold_idx]
            X_val_fold = X_current.iloc[val_fold_idx]

            y_train_fold = y_dev.iloc[train_fold_idx]
            y_val_fold = y_dev.iloc[val_fold_idx]

            model.fit(
                X_train_fold,
                y_train_fold
            )

            y_pred = model.predict(
                X_val_fold
            )

            if hasattr(model, "predict_proba"):
                y_prob = model.predict_proba(
                    X_val_fold
                )[:, 1]
            else:
                y_prob = model.decision_function(
                    X_val_fold
                )

            fold_f1.append(
                f1_score(
                    y_val_fold,
                    y_pred
                )
            )

            fold_recall.append(
                recall_score(
                    y_val_fold,
                    y_pred
                )
            )

            fold_auc.append(
                roc_auc_score(
                    y_val_fold,
                    y_prob
                )
            )

            fold_mcc.append(
                matthews_corrcoef(
                    y_val_fold,
                    y_pred
                )
            )

        baseline_results.append({
            "feature_set": feature_set_name,
            "model": model_name,

            "mean_f1": np.mean(fold_f1),
            "std_f1": np.std(fold_f1),

            "mean_recall": np.mean(fold_recall),
            "std_recall": np.std(fold_recall),

            "mean_auc": np.mean(fold_auc),
            "std_auc": np.std(fold_auc),

            "mean_mcc": np.mean(fold_mcc),
            "std_mcc": np.std(fold_mcc)
        })

        print(
            model_name,
            "F1:",
            round(np.mean(fold_f1), 4),
            "AUC:",
            round(np.mean(fold_auc), 4)
        )

In [ ]:
baseline_df = pd.DataFrame(
    baseline_results
)

baseline_df

In [ ]:
structural_intrinsic = [
    f for f in structural_reduced
    if f != "qr_version"
]

lexical_plus_intrinsic = (
    lexical_reduced
    + [
        f for f in structural_intrinsic
        if f not in lexical_reduced
    ]
)

In [ ]:
feature_sets_intrinsic = {
    "Structural_intrinsic":
        structural_intrinsic,

    "Lexical_plus_intrinsic":
        lexical_plus_intrinsic
}

In [ ]:
intrinsic_results = []

for feature_set_name, feature_list in feature_sets_intrinsic.items():

    print("=" * 70)
    print("Feature set:", feature_set_name)

    X_current = X_dev[feature_list]

    for model_name, model in models.items():

        fold_f1 = []
        fold_recall = []
        fold_auc = []
        fold_mcc = []

        for fold, (train_fold_idx, val_fold_idx) in enumerate(
            sgkf.split(
                X_current,
                y_dev,
                groups=groups_dev
            ),
            start=1
        ):

            X_train_fold = X_current.iloc[train_fold_idx]
            X_val_fold = X_current.iloc[val_fold_idx]

            y_train_fold = y_dev.iloc[train_fold_idx]
            y_val_fold = y_dev.iloc[val_fold_idx]

            model.fit(
                X_train_fold,
                y_train_fold
            )

            y_pred = model.predict(
                X_val_fold
            )

            y_prob = model.predict_proba(
                X_val_fold
            )[:, 1]

            fold_f1.append(
                f1_score(
                    y_val_fold,
                    y_pred
                )
            )

            fold_recall.append(
                recall_score(
                    y_val_fold,
                    y_pred
                )
            )

            fold_auc.append(
                roc_auc_score(
                    y_val_fold,
                    y_prob
                )
            )

            fold_mcc.append(
                matthews_corrcoef(
                    y_val_fold,
                    y_pred
                )
            )

        intrinsic_results.append({
            "feature_set": feature_set_name,
            "model": model_name,

            "mean_f1": np.mean(fold_f1),
            "std_f1": np.std(fold_f1),

            "mean_recall": np.mean(fold_recall),
            "std_recall": np.std(fold_recall),

            "mean_auc": np.mean(fold_auc),
            "std_auc": np.std(fold_auc),

            "mean_mcc": np.mean(fold_mcc),
            "std_mcc": np.std(fold_mcc)
        })

        print(
            model_name,
            "F1:",
            round(np.mean(fold_f1), 4),
            "AUC:",
            round(np.mean(fold_auc), 4)
        )

In [ ]:
intrinsic_df = pd.DataFrame(
    intrinsic_results
)

intrinsic_df

In [ ]:
all_baseline_df = pd.concat(
    [
        baseline_df,
        intrinsic_df
    ],
    ignore_index=True
)

all_baseline_df = all_baseline_df.sort_values(
    ["mean_f1", "mean_auc"],
    ascending=False
)

all_baseline_df[
    [
        "feature_set",
        "model",
        "mean_f1",
        "std_f1",
        "mean_recall",
        "mean_auc",
        "mean_mcc"
    ]
]

In [ ]:
all_baseline_df.to_csv(
    "baseline_all_feature_sets_grouped_cv.csv",
    index=False
)

Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

xgb_param_grid = {
    "n_estimators": [200, 300, 400, 600],
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3]
}

In [ ]:
import time
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

start = time.time()

model_test = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

train_fold_idx, val_fold_idx = next(
    sgkf.split(
        X_dev[lexical_reduced],
        y_dev,
        groups=groups_dev
    )
)

model_test.fit(
    X_dev[lexical_reduced].iloc[train_fold_idx],
    y_dev.iloc[train_fold_idx]
)

pred = model_test.predict(
    X_dev[lexical_reduced].iloc[val_fold_idx]
)

print(
    "F1:",
    f1_score(
        y_dev.iloc[val_fold_idx],
        pred
    )
)

print(
    "Time:",
    time.time() - start,
    "seconds"
)

In [ ]:
# XGBoost tuning - Lexical features

xgb_candidates = [
    {
        "n_estimators": 200,
        "max_depth": 3,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 400,
        "max_depth": 5,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 1.0,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.9,
        "colsample_bytree": 0.9
    },
    {
        "n_estimators": 400,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 1.0
    },
    {
        "n_estimators": 300,
        "max_depth": 7,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.9
    }
]

In [ ]:
from sklearn.metrics import (
    f1_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef
)

xgb_lexical_tuning_results = []

for candidate_id, params in enumerate(
    xgb_candidates,
    start=1
):

    print("=" * 60)
    print(
        f"Candidate {candidate_id}/"
        f"{len(xgb_candidates)}"
    )
    print(params)

    fold_f1 = []
    fold_recall = []
    fold_auc = []
    fold_mcc = []

    for fold, (tr_idx, va_idx) in enumerate(
        sgkf.split(
            X_dev[lexical_reduced],
            y_dev,
            groups=groups_dev
        ),
        start=1
    ):

        model = XGBClassifier(
            **params,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_dev[lexical_reduced].iloc[tr_idx],
            y_dev.iloc[tr_idx]
        )

        pred = model.predict(
            X_dev[lexical_reduced].iloc[va_idx]
        )

        prob = model.predict_proba(
            X_dev[lexical_reduced].iloc[va_idx]
        )[:, 1]

        f1 = f1_score(
            y_dev.iloc[va_idx],
            pred
        )

        recall = recall_score(
            y_dev.iloc[va_idx],
            pred
        )

        auc = roc_auc_score(
            y_dev.iloc[va_idx],
            prob
        )

        mcc = matthews_corrcoef(
            y_dev.iloc[va_idx],
            pred
        )

        fold_f1.append(f1)
        fold_recall.append(recall)
        fold_auc.append(auc)
        fold_mcc.append(mcc)

        print(
            f" Fold {fold}: "
            f"F1={f1:.4f}, "
            f"AUC={auc:.4f}"
        )

    xgb_lexical_tuning_results.append({
        **params,

        "mean_f1": np.mean(fold_f1),
        "std_f1": np.std(fold_f1),

        "mean_recall": np.mean(fold_recall),

        "mean_auc": np.mean(fold_auc),

        "mean_mcc": np.mean(fold_mcc)
    })

    print(
        " Mean F1:",
        round(np.mean(fold_f1), 4)
    )

In [ ]:
xgb_lexical_tuning_df = pd.DataFrame(
    xgb_lexical_tuning_results
).sort_values(
    ["mean_f1", "mean_auc"],
    ascending=False
)

xgb_lexical_tuning_df

In [ ]:
best_xgb_lexical_row = (
    xgb_lexical_tuning_df.iloc[0]
)

best_xgb_lexical_row

In [ ]:
xgb_lexical_tuning_df.to_csv(
    "xgb_lexical_manual_tuning.csv",
    index=False
)

In [ ]:
# Save best XGB + Lexical result


best_xgb_lexical_params = {
    "n_estimators": int(best_xgb_lexical_row["n_estimators"]),
    "max_depth": int(best_xgb_lexical_row["max_depth"]),
    "learning_rate": float(best_xgb_lexical_row["learning_rate"]),
    "subsample": float(best_xgb_lexical_row["subsample"]),
    "colsample_bytree": float(
        best_xgb_lexical_row["colsample_bytree"]
    )
}

print("Best XGB + Lexical params:")
print(best_xgb_lexical_params)

In [ ]:
# XGBoost tuning - Lexical + Intrinsic

xgb_intrinsic_tuning_results = []

for candidate_id, params in enumerate(
    xgb_candidates,
    start=1
):

    print("=" * 60)
    print(
        f"Candidate {candidate_id}/"
        f"{len(xgb_candidates)}"
    )
    print(params)

    fold_f1 = []
    fold_recall = []
    fold_auc = []
    fold_mcc = []

    for fold, (tr_idx, va_idx) in enumerate(
        sgkf.split(
            X_dev[lexical_plus_intrinsic],
            y_dev,
            groups=groups_dev
        ),
        start=1
    ):

        model = XGBClassifier(
            **params,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_dev[
                lexical_plus_intrinsic
            ].iloc[tr_idx],
            y_dev.iloc[tr_idx]
        )

        pred = model.predict(
            X_dev[
                lexical_plus_intrinsic
            ].iloc[va_idx]
        )

        prob = model.predict_proba(
            X_dev[
                lexical_plus_intrinsic
            ].iloc[va_idx]
        )[:, 1]

        f1 = f1_score(
            y_dev.iloc[va_idx],
            pred
        )

        recall = recall_score(
            y_dev.iloc[va_idx],
            pred
        )

        auc = roc_auc_score(
            y_dev.iloc[va_idx],
            prob
        )

        mcc = matthews_corrcoef(
            y_dev.iloc[va_idx],
            pred
        )

        fold_f1.append(f1)
        fold_recall.append(recall)
        fold_auc.append(auc)
        fold_mcc.append(mcc)

        print(
            f" Fold {fold}: "
            f"F1={f1:.4f}, "
            f"AUC={auc:.4f}"
        )

    xgb_intrinsic_tuning_results.append({
        **params,

        "mean_f1":
            np.mean(fold_f1),

        "std_f1":
            np.std(fold_f1),

        "mean_recall":
            np.mean(fold_recall),

        "mean_auc":
            np.mean(fold_auc),

        "mean_mcc":
            np.mean(fold_mcc)
    })

    print(
        " Mean F1:",
        round(
            np.mean(fold_f1),
            4
        )
    )

In [ ]:
xgb_intrinsic_tuning_df = pd.DataFrame(
    xgb_intrinsic_tuning_results
).sort_values(
    ["mean_f1", "mean_auc"],
    ascending=False
)

xgb_intrinsic_tuning_df

In [ ]:
best_xgb_intrinsic_row = (
    xgb_intrinsic_tuning_df.iloc[0]
)

best_xgb_intrinsic_row

In [ ]:
best_xgb_intrinsic_params = {
    "n_estimators": int(
        best_xgb_intrinsic_row[
            "n_estimators"
        ]
    ),

    "max_depth": int(
        best_xgb_intrinsic_row[
            "max_depth"
        ]
    ),

    "learning_rate": float(
        best_xgb_intrinsic_row[
            "learning_rate"
        ]
    ),

    "subsample": float(
        best_xgb_intrinsic_row[
            "subsample"
        ]
    ),

    "colsample_bytree": float(
        best_xgb_intrinsic_row[
            "colsample_bytree"
        ]
    )
}

print(
    "Best XGB + Lexical_plus_intrinsic params:"
)
print(best_xgb_intrinsic_params)

In [ ]:
xgb_intrinsic_tuning_df.to_csv(
    "xgb_lexical_plus_intrinsic_tuning.csv",
    index=False
)

In [ ]:
# Random Forest candidate settings

rf_candidates = [
    {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "n_estimators": 500,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "n_estimators": 500,
        "max_depth": 20,
        "min_samples_split": 5,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "n_estimators": 500,
        "max_depth": 15,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    },

    {
        "n_estimators": 800,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    },

    {
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "log2"
    },

    {
        "n_estimators": 800,
        "max_depth": None,
        "min_samples_split": 5,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    }
]

In [ ]:
# Random Forest tuning - Selected features

rf_selected_tuning_results = []

for candidate_id, params in enumerate(
    rf_candidates,
    start=1
):

    print("=" * 60)
    print(
        f"Candidate {candidate_id}/"
        f"{len(rf_candidates)}"
    )
    print(params)

    fold_f1 = []
    fold_recall = []
    fold_auc = []
    fold_mcc = []

    for fold, (tr_idx, va_idx) in enumerate(
        sgkf.split(
            X_dev[selected_features],
            y_dev,
            groups=groups_dev
        ),
        start=1
    ):

        model = RandomForestClassifier(
            **params,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_dev[
                selected_features
            ].iloc[tr_idx],
            y_dev.iloc[tr_idx]
        )

        pred = model.predict(
            X_dev[
                selected_features
            ].iloc[va_idx]
        )

        prob = model.predict_proba(
            X_dev[
                selected_features
            ].iloc[va_idx]
        )[:, 1]

        f1 = f1_score(
            y_dev.iloc[va_idx],
            pred
        )

        recall = recall_score(
            y_dev.iloc[va_idx],
            pred
        )

        auc = roc_auc_score(
            y_dev.iloc[va_idx],
            prob
        )

        mcc = matthews_corrcoef(
            y_dev.iloc[va_idx],
            pred
        )

        fold_f1.append(f1)
        fold_recall.append(recall)
        fold_auc.append(auc)
        fold_mcc.append(mcc)

        print(
            f" Fold {fold}: "
            f"F1={f1:.4f}, "
            f"AUC={auc:.4f}"
        )

    rf_selected_tuning_results.append({
        **params,

        "mean_f1":
            np.mean(fold_f1),

        "std_f1":
            np.std(fold_f1),

        "mean_recall":
            np.mean(fold_recall),

        "mean_auc":
            np.mean(fold_auc),

        "mean_mcc":
            np.mean(fold_mcc)
    })

    print(
        " Mean F1:",
        round(
            np.mean(fold_f1),
            4
        )
    )

In [ ]:
rf_selected_tuning_df = pd.DataFrame(
    rf_selected_tuning_results
).sort_values(
    ["mean_f1", "mean_auc"],
    ascending=False
)

rf_selected_tuning_df

In [ ]:
best_rf_selected_row = (
    rf_selected_tuning_df.iloc[0]
)

best_rf_selected_row

In [ ]:
rf_selected_tuning_df.to_csv(
    "rf_selected_tuning.csv",
    index=False
)

In [ ]:
# Random Forest tuning - Lexical + Intrinsic

rf_intrinsic_tuning_results = []

for candidate_id, params in enumerate(
    rf_candidates,
    start=1
):

    print("=" * 60)
    print(
        f"Candidate {candidate_id}/"
        f"{len(rf_candidates)}"
    )
    print(params)

    fold_f1 = []
    fold_recall = []
    fold_auc = []
    fold_mcc = []

    for fold, (tr_idx, va_idx) in enumerate(
        sgkf.split(
            X_dev[lexical_plus_intrinsic],
            y_dev,
            groups=groups_dev
        ),
        start=1
    ):

        model = RandomForestClassifier(
            **params,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_dev[
                lexical_plus_intrinsic
            ].iloc[tr_idx],
            y_dev.iloc[tr_idx]
        )

        pred = model.predict(
            X_dev[
                lexical_plus_intrinsic
            ].iloc[va_idx]
        )

        prob = model.predict_proba(
            X_dev[
                lexical_plus_intrinsic
            ].iloc[va_idx]
        )[:, 1]

        f1 = f1_score(
            y_dev.iloc[va_idx],
            pred
        )

        recall = recall_score(
            y_dev.iloc[va_idx],
            pred
        )

        auc = roc_auc_score(
            y_dev.iloc[va_idx],
            prob
        )

        mcc = matthews_corrcoef(
            y_dev.iloc[va_idx],
            pred
        )

        fold_f1.append(f1)
        fold_recall.append(recall)
        fold_auc.append(auc)
        fold_mcc.append(mcc)

        print(
            f" Fold {fold}: "
            f"F1={f1:.4f}, "
            f"AUC={auc:.4f}"
        )

    rf_intrinsic_tuning_results.append({
        **params,

        "mean_f1":
            np.mean(fold_f1),

        "std_f1":
            np.std(fold_f1),

        "mean_recall":
            np.mean(fold_recall),

        "mean_auc":
            np.mean(fold_auc),

        "mean_mcc":
            np.mean(fold_mcc)
    })

    print(
        " Mean F1:",
        round(
            np.mean(fold_f1),
            4
        )
    )

In [ ]:
rf_intrinsic_tuning_df = pd.DataFrame(
    rf_intrinsic_tuning_results
).sort_values(
    ["mean_f1", "mean_auc"],
    ascending=False
)

rf_intrinsic_tuning_df

In [ ]:
best_rf_intrinsic_row = (
    rf_intrinsic_tuning_df.iloc[0]
)

best_rf_intrinsic_row

In [ ]:
rf_intrinsic_tuning_df.to_csv(
    "rf_lexical_plus_intrinsic_tuning.csv",
    index=False
)

Model Selection

In [ ]:
# ==========================================
# Compare tuned candidate models
# ==========================================

tuned_summary = pd.DataFrame([
    {
        "model":
            "XGB",

        "feature_set":
            "Lexical",

        "mean_f1":
            best_xgb_lexical_row[
                "mean_f1"
            ],

        "std_f1":
            best_xgb_lexical_row[
                "std_f1"
            ],

        "mean_recall":
            best_xgb_lexical_row[
                "mean_recall"
            ],

        "mean_auc":
            best_xgb_lexical_row[
                "mean_auc"
            ],

        "mean_mcc":
            best_xgb_lexical_row[
                "mean_mcc"
            ]
    },

    {
        "model":
            "XGB",

        "feature_set":
            "Lexical_plus_intrinsic",

        "mean_f1":
            best_xgb_intrinsic_row[
                "mean_f1"
            ],

        "std_f1":
            best_xgb_intrinsic_row[
                "std_f1"
            ],

        "mean_recall":
            best_xgb_intrinsic_row[
                "mean_recall"
            ],

        "mean_auc":
            best_xgb_intrinsic_row[
                "mean_auc"
            ],

        "mean_mcc":
            best_xgb_intrinsic_row[
                "mean_mcc"
            ]
    },

    {
        "model":
            "RF",

        "feature_set":
            "Selected",

        "mean_f1":
            best_rf_selected_row[
                "mean_f1"
            ],

        "std_f1":
            best_rf_selected_row[
                "std_f1"
            ],

        "mean_recall":
            best_rf_selected_row[
                "mean_recall"
            ],

        "mean_auc":
            best_rf_selected_row[
                "mean_auc"
            ],

        "mean_mcc":
            best_rf_selected_row[
                "mean_mcc"
            ]
    },

    {
        "model":
            "RF",

        "feature_set":
            "Lexical_plus_intrinsic",

        "mean_f1":
            best_rf_intrinsic_row[
                "mean_f1"
            ],

        "std_f1":
            best_rf_intrinsic_row[
                "std_f1"
            ],

        "mean_recall":
            best_rf_intrinsic_row[
                "mean_recall"
            ],

        "mean_auc":
            best_rf_intrinsic_row[
                "mean_auc"
            ],

        "mean_mcc":
            best_rf_intrinsic_row[
                "mean_mcc"
            ]
    }
])

tuned_summary = tuned_summary.sort_values(
    ["mean_f1", "mean_auc"],
    ascending=False
)

tuned_summary

In [ ]:
tuned_summary.to_csv(
    "tuned_model_summary.csv",
    index=False
)

In [ ]:
print(best_xgb_lexical_params)

In [ ]:
from scipy.stats import wilcoxon

comparison_records = []

feature_sets_compare = {
    "Lexical": lexical_reduced,
    "Lexical_plus_intrinsic": lexical_plus_intrinsic
}

params_compare = {
    "Lexical": best_xgb_lexical_params,
    "Lexical_plus_intrinsic": best_xgb_intrinsic_params
}

for feature_set_name, features in feature_sets_compare.items():

    fold_f1 = []
    fold_auc = []
    fold_mcc = []

    for fold, (tr_idx, va_idx) in enumerate(
        sgkf.split(
            X_dev[features],
            y_dev,
            groups=groups_dev
        ),
        start=1
    ):

        model = XGBClassifier(
            **params_compare[feature_set_name],
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_dev[features].iloc[tr_idx],
            y_dev.iloc[tr_idx]
        )

        pred = model.predict(
            X_dev[features].iloc[va_idx]
        )

        prob = model.predict_proba(
            X_dev[features].iloc[va_idx]
        )[:, 1]

        f1 = f1_score(
            y_dev.iloc[va_idx],
            pred
        )

        auc = roc_auc_score(
            y_dev.iloc[va_idx],
            prob
        )

        mcc = matthews_corrcoef(
            y_dev.iloc[va_idx],
            pred
        )

        comparison_records.append({
            "feature_set": feature_set_name,
            "fold": fold,
            "f1": f1,
            "auc": auc,
            "mcc": mcc
        })

In [ ]:
comparison_df = pd.DataFrame(
    comparison_records
)

comparison_df

In [ ]:
f1_pivot = comparison_df.pivot(
    index="fold",
    columns="feature_set",
    values="f1"
)

f1_pivot

In [ ]:
f1_pivot["difference"] = (
    f1_pivot["Lexical"]
    - f1_pivot["Lexical_plus_intrinsic"]
)

f1_pivot

In [ ]:
# Wilcoxon paired test
stat, p = wilcoxon(
    f1_pivot["Lexical"],
    f1_pivot["Lexical_plus_intrinsic"],
    alternative="two-sided"
)

print("Wilcoxon statistic:", stat)
print("p-value:", p)

In [ ]:
print(
    "Mean F1 difference:",
    f1_pivot["difference"].mean()
)

print(
    "Folds where Lexical wins:",
    (
        f1_pivot["difference"] > 0
    ).sum(),
    "/ 5"
)

In [ ]:
from sklearn.inspection import permutation_importance

perm_records = []

for fold, (tr_idx, va_idx) in enumerate(
    sgkf.split(
        X_dev[lexical_reduced],
        y_dev,
        groups=groups_dev
    ),
    start=1
):

    print(f"Fold {fold}/5")

    model = XGBClassifier(
        **best_xgb_lexical_params,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    )

    model.fit(
        X_dev[lexical_reduced].iloc[tr_idx],
        y_dev.iloc[tr_idx]
    )

    perm = permutation_importance(
        model,
        X_dev[lexical_reduced].iloc[va_idx],
        y_dev.iloc[va_idx],
        scoring="f1",
        n_repeats=10,
        random_state=42,
        n_jobs=1
    )

    for feature, imp_mean, imp_std in zip(
        lexical_reduced,
        perm.importances_mean,
        perm.importances_std
    ):
        perm_records.append({
            "fold": fold,
            "feature": feature,
            "importance_mean": imp_mean,
            "importance_std": imp_std
        })

In [ ]:
perm_df = pd.DataFrame(perm_records)

perm_summary = (
    perm_df
    .groupby("feature")
    .agg(
        mean_importance=("importance_mean", "mean"),
        std_importance=("importance_mean", "std")
    )
    .reset_index()
    .sort_values(
        "mean_importance",
        ascending=False
    )
)

perm_summary

In [ ]:
final_model = XGBClassifier(
    **best_xgb_lexical_params,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)


final_model.fit(
    X_dev[lexical_reduced],
    y_dev
)

In [ ]:
test_pred = final_model.predict(
    X_test[lexical_reduced]
)

test_prob = final_model.predict_proba(
    X_test[lexical_reduced]
)[:,1]

In [ ]:
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    recall_score,
    precision_score,
    matthews_corrcoef,
    confusion_matrix
)


print(
    "F1:",
    f1_score(y_test,test_pred)
)

print(
    "Recall:",
    recall_score(y_test,test_pred)
)

print(
    "Precision:",
    precision_score(y_test,test_pred)
)

print(
    "AUC:",
    roc_auc_score(y_test,test_prob)
)

print(
    "MCC:",
    matthews_corrcoef(y_test,test_pred)
)


print(
    confusion_matrix(
        y_test,
        test_pred
    )
)

In [ ]:
import joblib

joblib.dump(
    final_model,
    "final_xgb_lexical_model.pkl"
)

In [ ]:
import json

with open(
    "final_features.json",
    "w"
) as f:
    json.dump(
        lexical_reduced,
        f
    )

Ablation Study

In [ ]:
# Random Split vs Hostname-Grouped Split
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold
)

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score
)

from xgboost import XGBClassifier

In [ ]:
# Random CV: ignores hostname
random_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Leakage-free CV: keeps hostname separated
grouped_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
random_overlap_records = []

for fold, (tr_idx, va_idx) in enumerate(
    random_cv.split(
        X_dev[lexical_reduced],
        y_dev
    ),
    start=1
):

    train_hosts = set(
        groups_dev.iloc[tr_idx]
    )

    val_hosts = set(
        groups_dev.iloc[va_idx]
    )

    overlap_hosts = (
        train_hosts.intersection(
            val_hosts
        )
    )

    random_overlap_records.append({
        "fold": fold,
        "train_samples": len(tr_idx),
        "val_samples": len(va_idx),
        "train_unique_hosts": len(train_hosts),
        "val_unique_hosts": len(val_hosts),
        "overlapping_hosts": len(overlap_hosts),
        "overlap_rate_val_hosts":
            len(overlap_hosts) /
            len(val_hosts)
    })


random_overlap_df = pd.DataFrame(
    random_overlap_records
)

random_overlap_df

In [ ]:
grouped_overlap_records = []

for fold, (tr_idx, va_idx) in enumerate(
    grouped_cv.split(
        X_dev[lexical_reduced],
        y_dev,
        groups=groups_dev
    ),
    start=1
):

    train_hosts = set(
        groups_dev.iloc[tr_idx]
    )

    val_hosts = set(
        groups_dev.iloc[va_idx]
    )

    overlap_hosts = (
        train_hosts.intersection(
            val_hosts
        )
    )

    grouped_overlap_records.append({
        "fold": fold,
        "train_samples": len(tr_idx),
        "val_samples": len(va_idx),
        "train_unique_hosts": len(train_hosts),
        "val_unique_hosts": len(val_hosts),
        "overlapping_hosts": len(overlap_hosts),
        "overlap_rate_val_hosts":
            len(overlap_hosts) /
            len(val_hosts)
    })


grouped_overlap_df = pd.DataFrame(
    grouped_overlap_records
)

grouped_overlap_df

In [ ]:
def evaluate_split_strategy(
    split_name,
    cv,
    use_groups
):

    records = []

    X_current = X_dev[
        lexical_reduced
    ]

    if use_groups:
        split_iterator = cv.split(
            X_current,
            y_dev,
            groups=groups_dev
        )
    else:
        split_iterator = cv.split(
            X_current,
            y_dev
        )

    for fold, (tr_idx, va_idx) in enumerate(
        split_iterator,
        start=1
    ):

        model = XGBClassifier(
            **best_xgb_lexical_params,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

        X_train = X_current.iloc[
            tr_idx
        ]

        X_val = X_current.iloc[
            va_idx
        ]

        y_train = y_dev.iloc[
            tr_idx
        ]

        y_val = y_dev.iloc[
            va_idx
        ]

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_val
        )

        prob = model.predict_proba(
            X_val
        )[:, 1]

        f1 = f1_score(
            y_val,
            pred
        )

        precision = precision_score(
            y_val,
            pred
        )

        recall = recall_score(
            y_val,
            pred
        )

        auc = roc_auc_score(
            y_val,
            prob
        )

        mcc = matthews_corrcoef(
            y_val,
            pred
        )

        accuracy = accuracy_score(
            y_val,
            pred
        )

        # hostname overlap
        train_hosts = set(
            groups_dev.iloc[tr_idx]
        )

        val_hosts = set(
            groups_dev.iloc[va_idx]
        )

        host_overlap = len(
            train_hosts.intersection(
                val_hosts
            )
        )

        records.append({
            "split_strategy":
                split_name,

            "fold":
                fold,

            "f1":
                f1,

            "precision":
                precision,

            "recall":
                recall,

            "auc":
                auc,

            "mcc":
                mcc,

            "accuracy":
                accuracy,

            "hostname_overlap":
                host_overlap
        })

        print(
            f"{split_name} | "
            f"Fold {fold}: "
            f"F1={f1:.4f}, "
            f"AUC={auc:.4f}, "
            f"MCC={mcc:.4f}, "
            f"Host overlap={host_overlap}"
        )

    return pd.DataFrame(
        records
    )

In [ ]:
random_results = evaluate_split_strategy(
    split_name="Random",
    cv=random_cv,
    use_groups=False
)

In [ ]:
grouped_results = evaluate_split_strategy(
    split_name="Hostname_Grouped",
    cv=grouped_cv,
    use_groups=True
)

In [ ]:
split_ablation_results = pd.concat(
    [
        random_results,
        grouped_results
    ],
    ignore_index=True
)

split_ablation_results

In [ ]:
split_ablation_summary = (
    split_ablation_results
    .groupby(
        "split_strategy"
    )
    .agg(
        mean_f1=("f1", "mean"),
        std_f1=("f1", "std"),

        mean_precision=(
            "precision",
            "mean"
        ),

        mean_recall=(
            "recall",
            "mean"
        ),

        mean_auc=(
            "auc",
            "mean"
        ),

        std_auc=(
            "auc",
            "std"
        ),

        mean_mcc=(
            "mcc",
            "mean"
        ),

        std_mcc=(
            "mcc",
            "std"
        ),

        mean_accuracy=(
            "accuracy",
            "mean"
        ),

        mean_hostname_overlap=(
            "hostname_overlap",
            "mean"
        )
    )
    .reset_index()
)

split_ablation_summary

In [ ]:
random_row = (
    split_ablation_summary[
        split_ablation_summary[
            "split_strategy"
        ] == "Random"
    ]
    .iloc[0]
)

grouped_row = (
    split_ablation_summary[
        split_ablation_summary[
            "split_strategy"
        ] == "Hostname_Grouped"
    ]
    .iloc[0]
)


f1_inflation = (
    random_row["mean_f1"]
    -
    grouped_row["mean_f1"]
)

auc_inflation = (
    random_row["mean_auc"]
    -
    grouped_row["mean_auc"]
)

mcc_inflation = (
    random_row["mean_mcc"]
    -
    grouped_row["mean_mcc"]
)


print(
    "F1 inflation from random split:",
    f1_inflation
)

print(
    "AUC inflation from random split:",
    auc_inflation
)

print(
    "MCC inflation from random split:",
    mcc_inflation
)

In [ ]:
print(
    "Relative F1 inflation (%):",
    100
    * f1_inflation
    / grouped_row["mean_f1"]
)

In [ ]:
print(
    "Average overlapping hostnames "
    "under random CV:",
    random_results[
        "hostname_overlap"
    ].mean()
)

print(
    "Average overlapping hostnames "
    "under grouped CV:",
    grouped_results[
        "hostname_overlap"
    ].mean()
)

In [ ]:
random_results.to_csv(
    "ablation_random_split_fold_results.csv",
    index=False
)

grouped_results.to_csv(
    "ablation_grouped_split_fold_results.csv",
    index=False
)

split_ablation_summary.to_csv(
    "ablation_split_strategy_summary.csv",
    index=False
)

random_overlap_df.to_csv(
    "random_split_hostname_overlap.csv",
    index=False
)

# Proposed Confidence-Gated Method

In [ ]:
# Environment / variable check
required_vars = [
    "X_dev",
    "y_dev",
    "groups_dev",
    "X_test",
    "y_test",
    "lexical_reduced",
    "structural_intrinsic",
    "lexical_plus_intrinsic",
    "best_xgb_lexical_params"
]

for var in required_vars:
    print(
        f"{var}:",
        "OK" if var in globals() else "MISSING"
    )

In [ ]:
print("Development samples:", len(X_dev))
print("Test samples:", len(X_test))

print(
    "Lexical features:",
    len(lexical_reduced)
)

print(
    "Intrinsic structural features:",
    len(structural_intrinsic)
)

print(
    "Fusion features:",
    len(lexical_plus_intrinsic)
)

print(
    "Development groups:",
    groups_dev.nunique()
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedGroupKFold
)

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score,
    confusion_matrix
)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

In [ ]:
N_JOBS = 1
RANDOM_STATE = 42
def make_stage1_model():
    return XGBClassifier(
        **best_xgb_lexical_params,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    )

In [ ]:
if "best_xgb_intrinsic_params" in globals():

    stage2_params = (
        best_xgb_intrinsic_params.copy()
    )

    print(
        "Using previously tuned "
        "Lexical+Intrinsic XGB parameters."
    )

else:

    stage2_params = {
        "n_estimators": 400,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 1.0
    }

    print(
        "Using fixed Stage-2 parameters."
    )


print(stage2_params)

In [ ]:
def make_stage2_model():

    return XGBClassifier(
        **stage2_params,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    )

In [ ]:
def generate_grouped_oof_probabilities(
    X,
    y,
    groups,
    feature_cols,
    n_splits=5,
    random_state=42
):

    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    oof_prob = np.zeros(
        len(X),
        dtype=float
    )

    oof_pred = np.zeros(
        len(X),
        dtype=int
    )

    fold_id = np.zeros(
        len(X),
        dtype=int
    )

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X[feature_cols],
            y,
            groups=groups
        ),
        start=1
    ):

        model = make_stage1_model()

        model.fit(
            X.iloc[tr_idx][feature_cols],
            y.iloc[tr_idx]
        )

        prob = model.predict_proba(
            X.iloc[va_idx][feature_cols]
        )[:, 1]

        pred = (
            prob >= 0.5
        ).astype(int)

        oof_prob[va_idx] = prob
        oof_pred[va_idx] = pred
        fold_id[va_idx] = fold

        print(
            f"Fold {fold}: "
            f"n={len(va_idx)}, "
            f"F1={f1_score(y.iloc[va_idx], pred):.4f}"
        )

    return (
        oof_prob,
        oof_pred,
        fold_id
    )

In [ ]:
(
    lexical_oof_prob,
    lexical_oof_pred,
    lexical_oof_fold
) = generate_grouped_oof_probabilities(
    X=X_dev,
    y=y_dev,
    groups=groups_dev,
    feature_cols=lexical_reduced,
    n_splits=5,
    random_state=42
)

In [ ]:
oof_lexical_metrics = {
    "F1":
        f1_score(
            y_dev,
            lexical_oof_pred
        ),

    "Precision":
        precision_score(
            y_dev,
            lexical_oof_pred
        ),

    "Recall":
        recall_score(
            y_dev,
            lexical_oof_pred
        ),

    "AUC":
        roc_auc_score(
            y_dev,
            lexical_oof_prob
        ),

    "MCC":
        matthews_corrcoef(
            y_dev,
            lexical_oof_pred
        ),

    "Accuracy":
        accuracy_score(
            y_dev,
            lexical_oof_pred
        )
}

pd.Series(
    oof_lexical_metrics
)

In [ ]:
lexical_uncertainty = np.abs(
    lexical_oof_prob - 0.5
)

confidence_df = pd.DataFrame({
    "true_label":
        y_dev.to_numpy(),

    "probability":
        lexical_oof_prob,

    "prediction":
        lexical_oof_pred,

    "uncertainty_distance":
        lexical_uncertainty,

    "correct":
        (
            lexical_oof_pred
            ==
            y_dev.to_numpy()
        ).astype(int),

    "fold":
        lexical_oof_fold
})

confidence_df.head()

In [ ]:
confidence_df[
    "confidence"
] = np.abs(
    confidence_df["probability"] - 0.5
)

confidence_df[
    "confidence_bin"
] = pd.cut(
    confidence_df["confidence"],
    bins=[
        0.0,
        0.05,
        0.10,
        0.20,
        0.30,
        0.40,
        0.50
    ],
    include_lowest=True
)

confidence_analysis = (
    confidence_df
    .groupby(
        "confidence_bin",
        observed=True
    )
    .agg(
        samples=("correct", "size"),
        correct=("correct", "sum"),
        mean_prob=("probability", "mean")
    )
    .reset_index()
)

confidence_analysis[
    "error_rate"
] = (
    1
    -
    confidence_analysis["correct"]
    /
    confidence_analysis["samples"]
)

confidence_analysis[
    "sample_share"
] = (
    confidence_analysis["samples"]
    /
    len(confidence_df)
)

confidence_analysis

In [ ]:
candidate_taus = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40
]

In [ ]:
gate_diagnostic_records = []

lexical_errors = (
    lexical_oof_pred
    !=
    y_dev.to_numpy()
)

total_errors = lexical_errors.sum()

for tau in candidate_taus:

    ambiguous = (
        np.abs(
            lexical_oof_prob - 0.5
        )
        < tau
    )

    ambiguous_n = ambiguous.sum()

    errors_inside = (
        lexical_errors
        &
        ambiguous
    ).sum()

    if ambiguous_n > 0:
        ambiguous_error_rate = (
            errors_inside
            /
            ambiguous_n
        )
    else:
        ambiguous_error_rate = np.nan

    if total_errors > 0:
        error_capture_rate = (
            errors_inside
            /
            total_errors
        )
    else:
        error_capture_rate = np.nan

    gate_diagnostic_records.append({
        "tau":
            tau,

        "ambiguous_samples":
            ambiguous_n,

        "invocation_rate":
            ambiguous_n
            /
            len(y_dev),

        "errors_captured":
            errors_inside,

        "error_capture_rate":
            error_capture_rate,

        "ambiguous_error_rate":
            ambiguous_error_rate
    })


gate_diagnostic_df = pd.DataFrame(
    gate_diagnostic_records
)

gate_diagnostic_df

In [ ]:
# ============================================================
# Visualization 1:
# Structural Invocation Rate vs Lexical Error Capture Rate
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

plot_df = gate_diagnostic_df.copy()

x = plot_df["invocation_rate"].values
y = plot_df["error_capture_rate"].values
taus = plot_df["tau"].values

plt.figure(figsize=(7, 6))

# Proposed confidence gate
plt.plot(
    x,
    y,
    marker="o",
    linewidth=2,
    label="Confidence-gated routing"
)

# Random-routing reference line: y = x
max_axis = max(x.max(), y.max()) * 1.05

plt.plot(
    [0, max_axis],
    [0, max_axis],
    linestyle="--",
    linewidth=1.5,
    label="Random routing reference"
)

# Annotate each tau
for xi, yi, tau in zip(x, y, taus):
    plt.annotate(
        f"τ={tau:.2f}",
        (xi, yi),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=9
    )

plt.xlabel("Structural Invocation Rate")
plt.ylabel("Lexical Error Capture Rate")
plt.title("Confidence Gate Concentrates Computation on Error-Prone Samples")

plt.xlim(0, max_axis)
plt.ylim(0, max_axis)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Visualization 2:
# Invocation Rate and Error Capture Rate across tau
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    plot_df["tau"],
    plot_df["invocation_rate"],
    marker="o",
    linewidth=2,
    label="Structural invocation rate"
)

plt.plot(
    plot_df["tau"],
    plot_df["error_capture_rate"],
    marker="o",
    linewidth=2,
    label="Lexical error capture rate"
)

plt.xlabel("Gating Threshold τ")
plt.ylabel("Rate")
plt.title("Effect of Gating Threshold on Computation and Error Coverage")

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
def build_stage2_features(
    X,
    stage1_prob
):

    stage2_X = (
        X[
            lexical_plus_intrinsic
        ]
        .copy()
    )

    stage2_X[
        "stage1_probability"
    ] = np.asarray(
        stage1_prob
    )

    return stage2_X

In [ ]:
def evaluate_gated_method_grouped_cv(
    X,
    y,
    groups,
    candidate_taus,
    outer_splits=5,
    inner_splits=4,
    random_state=42
):

    outer_cv = StratifiedGroupKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=random_state
    )

    all_records = []

    for outer_fold, (train_idx, val_idx) in enumerate(
        outer_cv.split(
            X[lexical_reduced],
            y,
            groups=groups
        ),
        start=1
    ):

        print("\n" + "=" * 70)
        print(f"Outer Fold {outer_fold}")
        print("=" * 70)

        X_train = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_val = y.iloc[val_idx].copy()

        groups_train = groups.iloc[train_idx].copy()

        # ------------------------------------------------
        # 1. Outer Stage-1 lexical model
        # ------------------------------------------------
        stage1_outer = make_stage1_model()

        stage1_outer.fit(
            X_train[lexical_reduced],
            y_train
        )

        val_stage1_prob = (
            stage1_outer
            .predict_proba(
                X_val[lexical_reduced]
            )[:, 1]
        )

        val_stage1_pred = (
            val_stage1_prob >= 0.5
        ).astype(int)

        # ------------------------------------------------
        # 2. Inner grouped OOF probabilities
        #    for outer-training samples
        # ------------------------------------------------
        inner_cv = StratifiedGroupKFold(
            n_splits=inner_splits,
            shuffle=True,
            random_state=random_state + outer_fold
        )

        train_oof_prob = np.zeros(
            len(X_train)
        )

        for inner_tr, inner_va in inner_cv.split(
            X_train[lexical_reduced],
            y_train,
            groups=groups_train
        ):

            inner_model = make_stage1_model()

            inner_model.fit(
                X_train.iloc[
                    inner_tr
                ][lexical_reduced],
                y_train.iloc[
                    inner_tr
                ]
            )

            inner_prob = (
                inner_model
                .predict_proba(
                    X_train.iloc[
                        inner_va
                    ][lexical_reduced]
                )[:, 1]
            )

            train_oof_prob[
                inner_va
            ] = inner_prob

        # ------------------------------------------------
        # 3. Build Stage-2 inputs
        # ------------------------------------------------
        train_stage2_X = build_stage2_features(
            X_train,
            train_oof_prob
        )

        val_stage2_X = build_stage2_features(
            X_val,
            val_stage1_prob
        )

        # ------------------------------------------------
        # 4. Global fusion baseline
        # ------------------------------------------------
        global_model = make_stage2_model()

        global_model.fit(
            X_train[
                lexical_plus_intrinsic
            ],
            y_train
        )

        global_prob = (
            global_model.predict_proba(
                X_val[
                    lexical_plus_intrinsic
                ]
            )[:, 1]
        )

        global_pred = (
            global_prob >= 0.5
        ).astype(int)

        # ------------------------------------------------
        # 5. Evaluate each gate threshold
        # ------------------------------------------------
        for tau in candidate_taus:

            ambiguous_train = (
                np.abs(
                    train_oof_prob - 0.5
                )
                < tau
            )

            ambiguous_val = (
                np.abs(
                    val_stage1_prob - 0.5
                )
                < tau
            )

            n_amb_train = (
                ambiguous_train.sum()
            )

            n_amb_val = (
                ambiguous_val.sum()
            )

            train_classes = np.unique(
                y_train.to_numpy()[
                    ambiguous_train
                ]
            )

            valid_stage2 = (
                n_amb_train >= 20
                and
                len(train_classes) == 2
            )

            # Default: lexical predictions
            gated_prob = (
                val_stage1_prob.copy()
            )

            gated_pred = (
                val_stage1_pred.copy()
            )

            if valid_stage2:

                stage2_model = (
                    make_stage2_model()
                )

                stage2_model.fit(
                    train_stage2_X.loc[
                        ambiguous_train
                    ],
                    y_train.loc[
                        ambiguous_train
                    ]
                )

                if n_amb_val > 0:

                    stage2_prob = (
                        stage2_model
                        .predict_proba(
                            val_stage2_X.loc[
                                ambiguous_val
                            ]
                        )[:, 1]
                    )

                    stage2_pred = (
                        stage2_prob >= 0.5
                    ).astype(int)

                    gated_prob[
                        ambiguous_val
                    ] = stage2_prob

                    gated_pred[
                        ambiguous_val
                    ] = stage2_pred

            # ------------------------------------------------
            # Error capture
            # ------------------------------------------------
            lexical_errors = (
                val_stage1_pred
                !=
                y_val.to_numpy()
            )

            total_lexical_errors = (
                lexical_errors.sum()
            )

            errors_inside_gate = (
                lexical_errors
                &
                ambiguous_val
            ).sum()

            error_capture_rate = (
                errors_inside_gate
                /
                total_lexical_errors
                if total_lexical_errors > 0
                else np.nan
            )

            # ------------------------------------------------
            # Decision changes
            # ------------------------------------------------
            changed = (
                gated_pred
                !=
                val_stage1_pred
            )

            corrections = (
                changed
                &
                (
                    gated_pred
                    ==
                    y_val.to_numpy()
                )
            ).sum()

            harms = (
                changed
                &
                (
                    val_stage1_pred
                    ==
                    y_val.to_numpy()
                )
                &
                (
                    gated_pred
                    !=
                    y_val.to_numpy()
                )
            ).sum()

            # ------------------------------------------------
            # Save result
            # ------------------------------------------------
            all_records.append({
                "outer_fold":
                    outer_fold,

                "tau":
                    tau,

                "stage2_valid":
                    valid_stage2,

                "ambiguous_train_n":
                    n_amb_train,

                "ambiguous_val_n":
                    n_amb_val,

                "invocation_rate":
                    n_amb_val
                    /
                    len(X_val),

                "error_capture_rate":
                    error_capture_rate,

                "changed_decisions":
                    changed.sum(),

                "corrections":
                    corrections,

                "harms":
                    harms,

                "lexical_f1":
                    f1_score(
                        y_val,
                        val_stage1_pred
                    ),

                "lexical_auc":
                    roc_auc_score(
                        y_val,
                        val_stage1_prob
                    ),

                "lexical_mcc":
                    matthews_corrcoef(
                        y_val,
                        val_stage1_pred
                    ),

                "global_fusion_f1":
                    f1_score(
                        y_val,
                        global_pred
                    ),

                "global_fusion_auc":
                    roc_auc_score(
                        y_val,
                        global_prob
                    ),

                "global_fusion_mcc":
                    matthews_corrcoef(
                        y_val,
                        global_pred
                    ),

                "gated_f1":
                    f1_score(
                        y_val,
                        gated_pred
                    ),

                "gated_precision":
                    precision_score(
                        y_val,
                        gated_pred
                    ),

                "gated_recall":
                    recall_score(
                        y_val,
                        gated_pred
                    ),

                "gated_auc":
                    roc_auc_score(
                        y_val,
                        gated_prob
                    ),

                "gated_mcc":
                    matthews_corrcoef(
                        y_val,
                        gated_pred
                    ),

                "gated_accuracy":
                    accuracy_score(
                        y_val,
                        gated_pred
                    )
            })

            print(
                f"tau={tau:.2f} | "
                f"invoke={n_amb_val/len(X_val):.3f} | "
                f"Lex F1={f1_score(y_val, val_stage1_pred):.4f} | "
                f"Gated F1={f1_score(y_val, gated_pred):.4f} | "
                f"correct={corrections} | "
                f"harm={harms}"
            )

    return pd.DataFrame(
        all_records
    )

In [ ]:
gated_cv_results = (
    evaluate_gated_method_grouped_cv(
        X=X_dev,
        y=y_dev,
        groups=groups_dev,
        candidate_taus=candidate_taus,
        outer_splits=5,
        inner_splits=4,
        random_state=42
    )
)

In [ ]:
gated_cv_results.head(20)

In [ ]:
gated_cv_results[
    [
        "tau",
        "stage2_valid"
    ]
].value_counts()

In [ ]:
gated_tau_summary = (
    gated_cv_results
    .groupby("tau")
    .agg(

        mean_gated_f1=(
            "gated_f1",
            "mean"
        ),

        std_gated_f1=(
            "gated_f1",
            "std"
        ),

        mean_gated_auc=(
            "gated_auc",
            "mean"
        ),

        mean_gated_mcc=(
            "gated_mcc",
            "mean"
        ),

        mean_gated_precision=(
            "gated_precision",
            "mean"
        ),

        mean_gated_recall=(
            "gated_recall",
            "mean"
        ),

        mean_invocation_rate=(
            "invocation_rate",
            "mean"
        ),

        std_invocation_rate=(
            "invocation_rate",
            "std"
        ),

        mean_error_capture_rate=(
            "error_capture_rate",
            "mean"
        ),

        total_corrections=(
            "corrections",
            "sum"
        ),

        total_harms=(
            "harms",
            "sum"
        ),

        mean_lexical_f1=(
            "lexical_f1",
            "mean"
        ),

        mean_global_fusion_f1=(
            "global_fusion_f1",
            "mean"
        )

    )
    .reset_index()
)

In [ ]:
gated_tau_summary[
    "delta_f1_vs_lexical"
] = (
    gated_tau_summary[
        "mean_gated_f1"
    ]
    -
    gated_tau_summary[
        "mean_lexical_f1"
    ]
)

gated_tau_summary[
    "delta_f1_vs_global_fusion"
] = (
    gated_tau_summary[
        "mean_gated_f1"
    ]
    -
    gated_tau_summary[
        "mean_global_fusion_f1"
    ]
)

gated_tau_summary[
    "net_corrections"
] = (
    gated_tau_summary[
        "total_corrections"
    ]
    -
    gated_tau_summary[
        "total_harms"
    ]
)

gated_tau_summary[
    [
        "tau",
        "mean_gated_f1",
        "std_gated_f1",
        "delta_f1_vs_lexical",
        "delta_f1_vs_global_fusion",
        "mean_invocation_rate",
        "mean_error_capture_rate",
        "total_corrections",
        "total_harms",
        "net_corrections"
    ]
]

In [ ]:
plt.figure(figsize=(7, 6))

plt.plot(
    gated_tau_summary[
        "mean_invocation_rate"
    ],
    gated_tau_summary[
        "mean_gated_f1"
    ],
    marker="o",
    linewidth=2
)

for _, row in gated_tau_summary.iterrows():
    plt.annotate(
        f"τ={row['tau']:.2f}",
        (
            row[
                "mean_invocation_rate"
            ],
            row[
                "mean_gated_f1"
            ]
        ),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=9
    )

baseline_f1 = (
    gated_tau_summary[
        "mean_lexical_f1"
    ].mean()
)

plt.axhline(
    baseline_f1,
    linestyle="--",
    linewidth=1.5,
    label="Lexical-only baseline"
)

plt.xlabel(
    "Structural Invocation Rate"
)

plt.ylabel(
    "Mean Grouped-CV F1"
)

plt.title(
    "Performance–Computation Trade-off of Confidence-Gated Fusion"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
best_tau_row = (
    gated_tau_summary
    .sort_values(
        by=[
            "mean_gated_f1",
            "mean_invocation_rate"
        ],
        ascending=[
            False,
            True
        ]
    )
    .iloc[0]
)

best_tau = float(
    best_tau_row["tau"]
)

print(
    "Selected tau:",
    best_tau
)

print(
    "CV gated F1:",
    best_tau_row[
        "mean_gated_f1"
    ]
)

print(
    "Invocation rate:",
    best_tau_row[
        "mean_invocation_rate"
    ]
)

print(
    "Delta F1 vs lexical:",
    best_tau_row[
        "delta_f1_vs_lexical"
    ]
)

In [ ]:
selected_tau_cv = (
    gated_cv_results[
        gated_cv_results[
            "tau"
        ] == best_tau
    ]
    .copy()
)

selected_tau_cv[
    [
        "outer_fold",
        "lexical_f1",
        "global_fusion_f1",
        "gated_f1",
        "gated_auc",
        "gated_mcc",
        "invocation_rate",
        "corrections",
        "harms"
    ]
]

In [ ]:
gated_wins = (
    selected_tau_cv[
        "gated_f1"
    ]
    >
    selected_tau_cv[
        "lexical_f1"
    ]
).sum()

gated_ties = (
    selected_tau_cv[
        "gated_f1"
    ]
    ==
    selected_tau_cv[
        "lexical_f1"
    ]
).sum()

gated_losses = (
    selected_tau_cv[
        "gated_f1"
    ]
    <
    selected_tau_cv[
        "lexical_f1"
    ]
).sum()

print(
    "Gated wins:",
    gated_wins
)

print(
    "Ties:",
    gated_ties
)

print(
    "Losses:",
    gated_losses
)

In [ ]:
confidence_analysis.to_csv(
    "gated_confidence_analysis.csv",
    index=False
)

gate_diagnostic_df.to_csv(
    "gated_threshold_diagnostics.csv",
    index=False
)

gated_cv_results.to_csv(
    "gated_nested_grouped_cv_results.csv",
    index=False
)

gated_tau_summary.to_csv(
    "gated_threshold_summary.csv",
    index=False
)

selected_tau_cv.to_csv(
    "gated_selected_threshold_fold_results.csv",
    index=False
)

# Benefit-Aware Adaptive Routing

The confidence-gated experiment showed that low-confidence lexical samples
contain a disproportionately large fraction of prediction errors. However,
routing these samples directly to the structural-assisted classifier did not
improve overall performance.

This indicates that lexical uncertainty and structural usefulness are not
equivalent. A sample can be difficult for the lexical classifier without
necessarily containing complementary information in its QR structure.

We therefore replace uncertainty-based routing with benefit-aware routing.
Rather than asking whether a lexical prediction is uncertain, the router
learns whether invoking the lexical–structural fusion branch is likely to
correct an error made by the lexical model.

The routing target is defined as:

z_i = 1 if the lexical model is wrong and the fusion model is correct,
z_i = 0 otherwise.

The router uses only inexpensive lexical information and Stage-1 prediction
statistics so that structural features do not need to be computed before
the routing decision.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix
)

RANDOM_STATE = 42

In [ ]:
def make_lexical_model():
    return make_stage1_model()


def make_fusion_model():
    return make_stage2_model()

In [ ]:
def generate_dual_oof_predictions(
    X,
    y,
    groups,
    n_splits=5,
    random_state=42
):

    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    lexical_prob = np.zeros(len(X))
    fusion_prob = np.zeros(len(X))

    lexical_pred = np.zeros(len(X), dtype=int)
    fusion_pred = np.zeros(len(X), dtype=int)

    fold_ids = np.zeros(len(X), dtype=int)

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X[lexical_reduced],
            y,
            groups=groups
        ),
        start=1
    ):

        # -------------------------
        # Lexical model
        # -------------------------

        lex_model = make_lexical_model()

        lex_model.fit(
            X.iloc[tr_idx][lexical_reduced],
            y.iloc[tr_idx]
        )

        lex_prob_fold = (
            lex_model.predict_proba(
                X.iloc[va_idx][lexical_reduced]
            )[:, 1]
        )

        lex_pred_fold = (
            lex_prob_fold >= 0.5
        ).astype(int)

        # -------------------------
        # Global fusion model
        # -------------------------

        fusion_model = make_fusion_model()

        fusion_model.fit(
            X.iloc[tr_idx][lexical_plus_intrinsic],
            y.iloc[tr_idx]
        )

        fusion_prob_fold = (
            fusion_model.predict_proba(
                X.iloc[va_idx][lexical_plus_intrinsic]
            )[:, 1]
        )

        fusion_pred_fold = (
            fusion_prob_fold >= 0.5
        ).astype(int)

        lexical_prob[va_idx] = lex_prob_fold
        lexical_pred[va_idx] = lex_pred_fold

        fusion_prob[va_idx] = fusion_prob_fold
        fusion_pred[va_idx] = fusion_pred_fold

        fold_ids[va_idx] = fold

        print(
            f"Fold {fold}: "
            f"Lexical F1={f1_score(y.iloc[va_idx], lex_pred_fold):.4f} | "
            f"Fusion F1={f1_score(y.iloc[va_idx], fusion_pred_fold):.4f}"
        )

    return {
        "lexical_prob": lexical_prob,
        "lexical_pred": lexical_pred,

        "fusion_prob": fusion_prob,
        "fusion_pred": fusion_pred,

        "fold": fold_ids
    }

In [ ]:
dual_oof = generate_dual_oof_predictions(
    X=X_dev,
    y=y_dev,
    groups=groups_dev,
    n_splits=5,
    random_state=42
)

In [ ]:
y_true_dev = y_dev.to_numpy()

lex_correct = (
    dual_oof["lexical_pred"]
    ==
    y_true_dev
)

fusion_correct = (
    dual_oof["fusion_pred"]
    ==
    y_true_dev
)

benefit_mask = (
    (~lex_correct)
    &
    fusion_correct
)

harm_mask = (
    lex_correct
    &
    (~fusion_correct)
)

both_correct = (
    lex_correct
    &
    fusion_correct
)

both_wrong = (
    (~lex_correct)
    &
    (~fusion_correct)
)

In [ ]:
routing_outcome_summary = pd.Series({
    "Benefit: Lexical wrong, Fusion correct":
        benefit_mask.sum(),

    "Harm: Lexical correct, Fusion wrong":
        harm_mask.sum(),

    "Both correct":
        both_correct.sum(),

    "Both wrong":
        both_wrong.sum()
})

routing_outcome_summary

In [ ]:
routing_outcome_rates = (
    routing_outcome_summary
    /
    len(y_dev)
)

routing_outcome_rates

In [ ]:
oracle_pred = (
    dual_oof["lexical_pred"].copy()
)

oracle_pred[
    benefit_mask
] = dual_oof["fusion_pred"][
    benefit_mask
]

In [ ]:
lexical_oof_f1 = f1_score(
    y_true_dev,
    dual_oof["lexical_pred"]
)

fusion_oof_f1 = f1_score(
    y_true_dev,
    dual_oof["fusion_pred"]
)

oracle_f1 = f1_score(
    y_true_dev,
    oracle_pred
)

oracle_mcc = matthews_corrcoef(
    y_true_dev,
    oracle_pred
)

oracle_accuracy = accuracy_score(
    y_true_dev,
    oracle_pred
)

print(
    "Lexical OOF F1:",
    lexical_oof_f1
)

print(
    "Global Fusion OOF F1:",
    fusion_oof_f1
)

print(
    "Oracle Routing F1:",
    oracle_f1
)

print(
    "Oracle potential gain:",
    oracle_f1 - lexical_oof_f1
)

print(
    "Oracle MCC:",
    oracle_mcc
)

print(
    "Oracle Accuracy:",
    oracle_accuracy
)

In [ ]:
benefit_rate = (
    benefit_mask.mean()
)

harm_rate = (
    harm_mask.mean()
)

print(
    "Benefit cases:",
    benefit_mask.sum()
)

print(
    "Benefit rate:",
    benefit_rate
)

print(
    "Harm cases:",
    harm_mask.sum()
)

print(
    "Harm rate:",
    harm_rate
)

In [ ]:
router_target = (
    benefit_mask.astype(int)
)

print(
    pd.Series(
        router_target
    ).value_counts()
)

print()

print(
    pd.Series(
        router_target
    ).value_counts(
        normalize=True
    )
)

In [ ]:
def build_router_features(
    X,
    lexical_prob
):

    router_X = (
        X[lexical_reduced]
        .copy()
    )

    p = np.asarray(
        lexical_prob
    )

    p_safe = np.clip(
        p,
        1e-8,
        1 - 1e-8
    )

    router_X[
        "stage1_probability"
    ] = p

    router_X[
        "stage1_confidence"
    ] = np.abs(
        p - 0.5
    )

    router_X[
        "stage1_entropy"
    ] = (
        -p_safe * np.log(p_safe)
        -
        (1 - p_safe)
        * np.log(1 - p_safe)
    )

    router_X[
        "stage1_prediction"
    ] = (
        p >= 0.5
    ).astype(int)

    return router_X

In [ ]:
router_X_oof = build_router_features(
    X_dev,
    dual_oof["lexical_prob"]
)

router_X_oof.shape

In [ ]:
router_X_oof.head()

In [ ]:
def make_router_model():

    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),

        (
            "classifier",
            LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=2000,
                random_state=42
            )
        )
    ])

In [ ]:
router_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

router_oof_prob = np.zeros(
    len(X_dev)
)

for fold, (tr_idx, va_idx) in enumerate(
    router_cv.split(
        router_X_oof,
        router_target,
        groups=groups_dev
    ),
    start=1
):

    router = make_router_model()

    router.fit(
        router_X_oof.iloc[tr_idx],
        router_target[tr_idx]
    )

    router_oof_prob[
        va_idx
    ] = (
        router.predict_proba(
            router_X_oof.iloc[
                va_idx
            ]
        )[:, 1]
    )

In [ ]:
router_auc = roc_auc_score(
    router_target,
    router_oof_prob
)

router_ap = average_precision_score(
    router_target,
    router_oof_prob
)

print(
    "Router ROC-AUC:",
    router_auc
)

print(
    "Router Average Precision:",
    router_ap
)

print(
    "Positive base rate:",
    router_target.mean()
)

In [ ]:
router_diag_df = pd.DataFrame({
    "benefit":
        router_target,

    "benefit_probability":
        router_oof_prob
})

router_diag_df[
    "probability_bin"
] = pd.qcut(
    router_diag_df[
        "benefit_probability"
    ],
    q=10,
    duplicates="drop"
)

router_decile = (
    router_diag_df
    .groupby(
        "probability_bin",
        observed=True
    )
    .agg(
        samples=(
            "benefit",
            "size"
        ),

        actual_benefit_rate=(
            "benefit",
            "mean"
        ),

        mean_predicted_probability=(
            "benefit_probability",
            "mean"
        )
    )
    .reset_index()
)

router_decile

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    range(
        1,
        len(router_decile) + 1
    ),
    router_decile[
        "actual_benefit_rate"
    ],
    marker="o"
)

plt.axhline(
    router_target.mean(),
    linestyle="--",
    label="Overall benefit rate"
)

plt.xlabel(
    "Router Probability Decile"
)

plt.ylabel(
    "Observed Structural Benefit Rate"
)

plt.title(
    "Can the Router Identify Samples That Benefit from Structural Fusion?"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
candidate_gammas = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

In [ ]:
def evaluate_benefit_router_grouped_cv(
    X,
    y,
    groups,
    candidate_gammas,
    outer_splits=5,
    inner_splits=4,
    random_state=42
):

    outer_cv = StratifiedGroupKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=random_state
    )

    records = []

    for outer_fold, (
        tr_idx,
        va_idx
    ) in enumerate(

        outer_cv.split(
            X[lexical_reduced],
            y,
            groups=groups
        ),

        start=1
    ):

        print(
            "\n"
            + "=" * 75
        )

        print(
            f"Outer Fold {outer_fold}"
        )

        print(
            "=" * 75
        )

        # --------------------------------------
        # Outer train / validation
        # --------------------------------------

        X_train = (
            X.iloc[tr_idx]
            .copy()
        )

        X_val = (
            X.iloc[va_idx]
            .copy()
        )

        y_train = (
            y.iloc[tr_idx]
            .copy()
        )

        y_val = (
            y.iloc[va_idx]
            .copy()
        )

        groups_train = (
            groups.iloc[tr_idx]
            .copy()
        )

        # ======================================
        # A. Inner OOF predictions
        # ======================================

        inner_cv = StratifiedGroupKFold(
            n_splits=inner_splits,
            shuffle=True,
            random_state=(
                random_state
                + outer_fold
            )
        )

        inner_lex_prob = np.zeros(
            len(X_train)
        )

        inner_lex_pred = np.zeros(
            len(X_train),
            dtype=int
        )

        inner_fusion_prob = np.zeros(
            len(X_train)
        )

        inner_fusion_pred = np.zeros(
            len(X_train),
            dtype=int
        )

        for inner_tr, inner_va in inner_cv.split(

            X_train[
                lexical_reduced
            ],

            y_train,

            groups=groups_train
        ):

            # Lexical
            lex_model_inner = (
                make_lexical_model()
            )

            lex_model_inner.fit(

                X_train.iloc[
                    inner_tr
                ][
                    lexical_reduced
                ],

                y_train.iloc[
                    inner_tr
                ]
            )

            lex_prob = (
                lex_model_inner
                .predict_proba(

                    X_train.iloc[
                        inner_va
                    ][
                        lexical_reduced
                    ]
                )[:, 1]
            )

            lex_pred = (
                lex_prob >= 0.5
            ).astype(int)

            # Fusion
            fusion_model_inner = (
                make_fusion_model()
            )

            fusion_model_inner.fit(

                X_train.iloc[
                    inner_tr
                ][
                    lexical_plus_intrinsic
                ],

                y_train.iloc[
                    inner_tr
                ]
            )

            fusion_prob = (
                fusion_model_inner
                .predict_proba(

                    X_train.iloc[
                        inner_va
                    ][
                        lexical_plus_intrinsic
                    ]
                )[:, 1]
            )

            fusion_pred = (
                fusion_prob >= 0.5
            ).astype(int)

            inner_lex_prob[
                inner_va
            ] = lex_prob

            inner_lex_pred[
                inner_va
            ] = lex_pred

            inner_fusion_prob[
                inner_va
            ] = fusion_prob

            inner_fusion_pred[
                inner_va
            ] = fusion_pred

        # ======================================
        # B. Construct benefit labels
        # ======================================

        y_train_np = (
            y_train.to_numpy()
        )

        inner_lex_correct = (
            inner_lex_pred
            ==
            y_train_np
        )

        inner_fusion_correct = (
            inner_fusion_pred
            ==
            y_train_np
        )

        benefit_train = (
            (~inner_lex_correct)
            &
            inner_fusion_correct
        ).astype(int)

        benefit_count = (
            benefit_train.sum()
        )

        print(
            "Benefit training cases:",
            benefit_count,
            "/",
            len(benefit_train)
        )

        # ======================================
        # C. Train benefit router
        # ======================================

        router_train_X = (
            build_router_features(
                X_train,
                inner_lex_prob
            )
        )

        router_valid = (
            benefit_count >= 10
            and
            len(
                np.unique(
                    benefit_train
                )
            ) == 2
        )

        if router_valid:

            router_model = (
                make_router_model()
            )

            router_model.fit(
                router_train_X,
                benefit_train
            )

        # ======================================
        # D. Train outer lexical model
        # ======================================

        lexical_outer = (
            make_lexical_model()
        )

        lexical_outer.fit(

            X_train[
                lexical_reduced
            ],

            y_train
        )

        val_lex_prob = (
            lexical_outer
            .predict_proba(

                X_val[
                    lexical_reduced
                ]
            )[:, 1]
        )

        val_lex_pred = (
            val_lex_prob >= 0.5
        ).astype(int)

        # ======================================
        # E. Train outer fusion model
        # ======================================

        fusion_outer = (
            make_fusion_model()
        )

        fusion_outer.fit(

            X_train[
                lexical_plus_intrinsic
            ],

            y_train
        )

        val_fusion_prob = (
            fusion_outer
            .predict_proba(

                X_val[
                    lexical_plus_intrinsic
                ]
            )[:, 1]
        )

        val_fusion_pred = (
            val_fusion_prob >= 0.5
        ).astype(int)

        # ======================================
        # F. Actual validation benefit/harm
        # ======================================

        y_val_np = (
            y_val.to_numpy()
        )

        lex_correct_val = (
            val_lex_pred
            ==
            y_val_np
        )

        fusion_correct_val = (
            val_fusion_pred
            ==
            y_val_np
        )

        actual_benefit = (
            (~lex_correct_val)
            &
            fusion_correct_val
        )

        actual_harm = (
            lex_correct_val
            &
            (~fusion_correct_val)
        )

        # ======================================
        # G. Router validation probabilities
        # ======================================

        router_val_X = (
            build_router_features(
                X_val,
                val_lex_prob
            )
        )

        if router_valid:

            benefit_prob = (
                router_model
                .predict_proba(
                    router_val_X
                )[:, 1]
            )

        else:

            benefit_prob = (
                np.zeros(
                    len(X_val)
                )
            )

        if (
            router_valid
            and
            len(
                np.unique(
                    actual_benefit.astype(int)
                )
            ) == 2
        ):

            router_auc = (
                roc_auc_score(
                    actual_benefit.astype(int),
                    benefit_prob
                )
            )

            router_ap = (
                average_precision_score(
                    actual_benefit.astype(int),
                    benefit_prob
                )
            )

        else:

            router_auc = np.nan
            router_ap = np.nan

        # ======================================
        # H. Evaluate routing thresholds
        # ======================================

        for gamma in candidate_gammas:

            routed = (
                benefit_prob
                >=
                gamma
            )

            routed_n = (
                routed.sum()
            )

            # Start from lexical
            routed_pred = (
                val_lex_pred.copy()
            )

            routed_prob = (
                val_lex_prob.copy()
            )

            # Invoke fusion only for routed cases
            routed_pred[
                routed
            ] = val_fusion_pred[
                routed
            ]

            routed_prob[
                routed
            ] = val_fusion_prob[
                routed
            ]

            # ---------------------------------
            # Routing quality
            # ---------------------------------

            benefits_routed = (
                actual_benefit
                &
                routed
            ).sum()

            harms_routed = (
                actual_harm
                &
                routed
            ).sum()

            total_benefits = (
                actual_benefit.sum()
            )

            total_harms = (
                actual_harm.sum()
            )

            benefit_capture_rate = (
                benefits_routed
                /
                total_benefits

                if total_benefits > 0

                else np.nan
            )

            routing_precision = (
                benefits_routed
                /
                routed_n

                if routed_n > 0

                else np.nan
            )

            harm_invocation_rate = (
                harms_routed
                /
                routed_n

                if routed_n > 0

                else np.nan
            )

            # ---------------------------------
            # Net prediction effects
            # ---------------------------------

            changed = (
                routed_pred
                !=
                val_lex_pred
            )

            corrections = (
                changed
                &
                (
                    routed_pred
                    ==
                    y_val_np
                )
            ).sum()

            harms = (
                changed
                &
                (
                    val_lex_pred
                    ==
                    y_val_np
                )
                &
                (
                    routed_pred
                    !=
                    y_val_np
                )
            ).sum()

            records.append({

                "outer_fold":
                    outer_fold,

                "gamma":
                    gamma,

                "router_valid":
                    router_valid,

                "benefit_train_n":
                    benefit_count,

                "actual_benefit_n":
                    total_benefits,

                "actual_harm_n":
                    total_harms,

                "router_auc":
                    router_auc,

                "router_ap":
                    router_ap,

                "invocation_rate":
                    routed_n
                    /
                    len(X_val),

                "routing_precision":
                    routing_precision,

                "benefit_capture_rate":
                    benefit_capture_rate,

                "harm_invocation_rate":
                    harm_invocation_rate,

                "corrections":
                    corrections,

                "harms":
                    harms,

                "net_corrections":
                    corrections
                    -
                    harms,

                # Lexical
                "lexical_f1":
                    f1_score(
                        y_val,
                        val_lex_pred
                    ),

                "lexical_auc":
                    roc_auc_score(
                        y_val,
                        val_lex_prob
                    ),

                "lexical_mcc":
                    matthews_corrcoef(
                        y_val,
                        val_lex_pred
                    ),

                # Global fusion
                "fusion_f1":
                    f1_score(
                        y_val,
                        val_fusion_pred
                    ),

                "fusion_auc":
                    roc_auc_score(
                        y_val,
                        val_fusion_prob
                    ),

                "fusion_mcc":
                    matthews_corrcoef(
                        y_val,
                        val_fusion_pred
                    ),

                # Benefit routing
                "routed_f1":
                    f1_score(
                        y_val,
                        routed_pred
                    ),

                "routed_precision":
                    precision_score(
                        y_val,
                        routed_pred
                    ),

                "routed_recall":
                    recall_score(
                        y_val,
                        routed_pred
                    ),

                "routed_auc":
                    roc_auc_score(
                        y_val,
                        routed_prob
                    ),

                "routed_mcc":
                    matthews_corrcoef(
                        y_val,
                        routed_pred
                    ),

                "routed_accuracy":
                    accuracy_score(
                        y_val,
                        routed_pred
                    )
            })

            print(
                f"γ={gamma:.2f} | "
                f"invoke={routed_n/len(X_val):.3f} | "
                f"Lex={f1_score(y_val,val_lex_pred):.4f} | "
                f"Route={f1_score(y_val,routed_pred):.4f} | "
                f"+{corrections}/-{harms}"
            )

    return pd.DataFrame(
        records
    )

In [ ]:
benefit_router_results = (
    evaluate_benefit_router_grouped_cv(

        X=X_dev,

        y=y_dev,

        groups=groups_dev,

        candidate_gammas=
            candidate_gammas,

        outer_splits=5,

        inner_splits=4,

        random_state=42
    )
)

In [ ]:
benefit_router_results[
    [
        "outer_fold",
        "router_auc",
        "router_ap",
        "benefit_train_n"
    ]
].drop_duplicates()

In [ ]:
print(
    "Mean Router AUC:",
    benefit_router_results[
        "router_auc"
    ].mean()
)

print(
    "Mean Router AP:",
    benefit_router_results[
        "router_ap"
    ].mean()
)

In [ ]:
benefit_gamma_summary = (
    benefit_router_results
    .groupby("gamma")
    .agg(

        mean_routed_f1=(
            "routed_f1",
            "mean"
        ),

        std_routed_f1=(
            "routed_f1",
            "std"
        ),

        mean_routed_auc=(
            "routed_auc",
            "mean"
        ),

        mean_routed_mcc=(
            "routed_mcc",
            "mean"
        ),

        mean_invocation_rate=(
            "invocation_rate",
            "mean"
        ),

        mean_routing_precision=(
            "routing_precision",
            "mean"
        ),

        mean_benefit_capture_rate=(
            "benefit_capture_rate",
            "mean"
        ),

        mean_harm_invocation_rate=(
            "harm_invocation_rate",
            "mean"
        ),

        total_corrections=(
            "corrections",
            "sum"
        ),

        total_harms=(
            "harms",
            "sum"
        ),

        total_net_corrections=(
            "net_corrections",
            "sum"
        ),

        mean_lexical_f1=(
            "lexical_f1",
            "mean"
        ),

        mean_fusion_f1=(
            "fusion_f1",
            "mean"
        ),

        mean_router_auc=(
            "router_auc",
            "mean"
        ),

        mean_router_ap=(
            "router_ap",
            "mean"
        )
    )
    .reset_index()
)

In [ ]:
benefit_gamma_summary[
    "delta_f1_vs_lexical"
] = (
    benefit_gamma_summary[
        "mean_routed_f1"
    ]
    -
    benefit_gamma_summary[
        "mean_lexical_f1"
    ]
)

benefit_gamma_summary[
    "delta_f1_vs_global_fusion"
] = (
    benefit_gamma_summary[
        "mean_routed_f1"
    ]
    -
    benefit_gamma_summary[
        "mean_fusion_f1"
    ]
)

In [ ]:
benefit_gamma_summary[
    [
        "gamma",
        "mean_routed_f1",
        "std_routed_f1",
        "delta_f1_vs_lexical",
        "delta_f1_vs_global_fusion",
        "mean_invocation_rate",
        "mean_routing_precision",
        "mean_benefit_capture_rate",
        "total_corrections",
        "total_harms",
        "total_net_corrections",
        "mean_router_auc",
        "mean_router_ap"
    ]
]

In [ ]:
plt.figure(
    figsize=(7, 6)
)

plt.plot(
    benefit_gamma_summary[
        "mean_invocation_rate"
    ],

    benefit_gamma_summary[
        "mean_routed_f1"
    ],

    marker="o",
    linewidth=2
)

for _, row in benefit_gamma_summary.iterrows():

    plt.annotate(
        f"γ={row['gamma']:.2f}",

        (
            row[
                "mean_invocation_rate"
            ],

            row[
                "mean_routed_f1"
            ]
        ),

        xytext=(6, 6),
        textcoords="offset points",
        fontsize=9
    )

lexical_reference = (
    benefit_gamma_summary[
        "mean_lexical_f1"
    ].mean()
)

fusion_reference = (
    benefit_gamma_summary[
        "mean_fusion_f1"
    ].mean()
)

plt.axhline(
    lexical_reference,
    linestyle="--",
    label="Lexical-only"
)

plt.axhline(
    fusion_reference,
    linestyle=":",
    label="Global fusion"
)

plt.xlabel(
    "Structural Invocation Rate"
)

plt.ylabel(
    "Mean Grouped-CV F1"
)

plt.title(
    "Benefit-Aware Routing: Performance–Computation Trade-off"
)

plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(7, 6)
)

plt.plot(
    benefit_gamma_summary[
        "mean_invocation_rate"
    ],

    benefit_gamma_summary[
        "mean_benefit_capture_rate"
    ],

    marker="o",
    linewidth=2,
    label="Benefit capture"
)

max_axis = max(
    benefit_gamma_summary[
        "mean_invocation_rate"
    ].max(),

    benefit_gamma_summary[
        "mean_benefit_capture_rate"
    ].max()
)

plt.plot(
    [0, max_axis],
    [0, max_axis],
    linestyle="--",
    label="Random routing reference"
)

plt.xlabel(
    "Structural Invocation Rate"
)

plt.ylabel(
    "Benefit Case Capture Rate"
)

plt.title(
    "Does the Learned Router Concentrate Structural Benefit?"
)

plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
best_gamma_row = (
    benefit_gamma_summary
    .sort_values(
        by=[
            "mean_routed_f1",
            "mean_invocation_rate"
        ],
        ascending=[
            False,
            True
        ]
    )
    .iloc[0]
)

best_gamma = float(
    best_gamma_row[
        "gamma"
    ]
)

print(
    "Selected gamma:",
    best_gamma
)

print(
    "Routed F1:",
    best_gamma_row[
        "mean_routed_f1"
    ]
)

print(
    "Lexical F1:",
    best_gamma_row[
        "mean_lexical_f1"
    ]
)

print(
    "Delta F1:",
    best_gamma_row[
        "delta_f1_vs_lexical"
    ]
)

print(
    "Invocation rate:",
    best_gamma_row[
        "mean_invocation_rate"
    ]
)

print(
    "Net corrections:",
    best_gamma_row[
        "total_net_corrections"
    ]
)

In [ ]:
selected_gamma_cv = (
    benefit_router_results[
        benefit_router_results[
            "gamma"
        ]
        ==
        best_gamma
    ]
    .copy()
)

In [ ]:
selected_gamma_cv[
    [
        "outer_fold",
        "lexical_f1",
        "fusion_f1",
        "routed_f1",
        "invocation_rate",
        "routing_precision",
        "benefit_capture_rate",
        "corrections",
        "harms",
        "net_corrections",
        "router_auc",
        "router_ap"
    ]
]

In [ ]:
routing_wins = (
    selected_gamma_cv[
        "routed_f1"
    ]
    >
    selected_gamma_cv[
        "lexical_f1"
    ]
).sum()

routing_ties = (
    selected_gamma_cv[
        "routed_f1"
    ]
    ==
    selected_gamma_cv[
        "lexical_f1"
    ]
).sum()

routing_losses = (
    selected_gamma_cv[
        "routed_f1"
    ]
    <
    selected_gamma_cv[
        "lexical_f1"
    ]
).sum()

print(
    "Routing wins:",
    routing_wins
)

print(
    "Ties:",
    routing_ties
)

print(
    "Losses:",
    routing_losses
)

In [ ]:
routing_outcome_summary.to_csv(
    "benefit_routing_outcome_summary.csv"
)

router_decile.to_csv(
    "benefit_router_probability_deciles.csv",
    index=False
)

benefit_router_results.to_csv(
    "benefit_router_nested_grouped_cv.csv",
    index=False
)

benefit_gamma_summary.to_csv(
    "benefit_router_gamma_summary.csv",
    index=False
)

selected_gamma_cv.to_csv(
    "benefit_router_selected_gamma_folds.csv",
    index=False
)

# Phase 11C — Utility-Aware Adaptive Routing

The benefit-aware router showed that samples for which structural fusion can
correct lexical errors are highly predictable. However, benefit-only routing
still degraded overall performance because it ignored the possibility that
fusion may also overturn correct lexical decisions.

We therefore formulate routing as an expected-utility decision.

For each sample, three outcomes are possible:

- Benefit (+1): lexical prediction is wrong and fusion prediction is correct.
- Harm (-1): lexical prediction is correct and fusion prediction is wrong.
- Neutral (0): both models are simultaneously correct or simultaneously wrong.

Two lightweight lexical routers estimate:

P(Benefit | x)

and

P(Harm | x).

The structural branch is invoked only when its expected routing utility

U(x) = P(Benefit | x) - P(Harm | x)

exceeds a routing threshold gamma.

This formulation explicitly balances the expected gain from structural fusion
against the risk of replacing a correct lexical prediction.

In [ ]:
required_11c = [
    "dual_oof",
    "benefit_mask",
    "harm_mask",
    "X_dev",
    "y_dev",
    "groups_dev",
    "lexical_reduced",
    "lexical_plus_intrinsic"
]

for var in required_11c:
    print(
        var,
        "OK" if var in globals() else "MISSING"
    )

In [ ]:
benefit_target = benefit_mask.astype(int)
harm_target = harm_mask.astype(int)

print("Benefit cases:", benefit_target.sum())
print("Benefit rate:", benefit_target.mean())

print()

print("Harm cases:", harm_target.sum())
print("Harm rate:", harm_target.mean())

In [ ]:
def build_utility_router_features(
    X,
    lexical_prob
):

    router_X = (
        X[lexical_reduced]
        .copy()
    )

    p = np.asarray(
        lexical_prob
    )

    p_safe = np.clip(
        p,
        1e-8,
        1 - 1e-8
    )

    router_X[
        "stage1_probability"
    ] = p

    router_X[
        "stage1_confidence"
    ] = np.abs(
        p - 0.5
    )

    router_X[
        "stage1_entropy"
    ] = (
        -p_safe * np.log(p_safe)
        -
        (1 - p_safe)
        * np.log(1 - p_safe)
    )

    router_X[
        "stage1_prediction"
    ] = (
        p >= 0.5
    ).astype(int)

    return router_X

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def make_utility_router():

    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),

        (
            "classifier",
            LogisticRegression(
                C=1.0,
                max_iter=3000,
                random_state=42
            )
        )
    ])

In [ ]:
utility_router_X = (
    build_utility_router_features(
        X_dev,
        dual_oof["lexical_prob"]
    )
)

In [ ]:
utility_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
benefit_oof_prob = np.zeros(
    len(X_dev)
)

for fold, (tr_idx, va_idx) in enumerate(
    utility_cv.split(
        utility_router_X,
        benefit_target,
        groups=groups_dev
    ),
    start=1
):

    model = make_utility_router()

    model.fit(
        utility_router_X.iloc[
            tr_idx
        ],
        benefit_target[
            tr_idx
        ]
    )

    benefit_oof_prob[
        va_idx
    ] = (
        model.predict_proba(
            utility_router_X.iloc[
                va_idx
            ]
        )[:, 1]
    )

In [ ]:
utility_cv_harm = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

harm_oof_prob = np.zeros(
    len(X_dev)
)

for fold, (tr_idx, va_idx) in enumerate(
    utility_cv_harm.split(
        utility_router_X,
        harm_target,
        groups=groups_dev
    ),
    start=1
):

    model = make_utility_router()

    model.fit(
        utility_router_X.iloc[
            tr_idx
        ],
        harm_target[
            tr_idx
        ]
    )

    harm_oof_prob[
        va_idx
    ] = (
        model.predict_proba(
            utility_router_X.iloc[
                va_idx
            ]
        )[:, 1]
    )

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

benefit_auc = roc_auc_score(
    benefit_target,
    benefit_oof_prob
)

benefit_ap = average_precision_score(
    benefit_target,
    benefit_oof_prob
)

harm_auc = roc_auc_score(
    harm_target,
    harm_oof_prob
)

harm_ap = average_precision_score(
    harm_target,
    harm_oof_prob
)

print(
    "Benefit Router AUC:",
    benefit_auc
)

print(
    "Benefit Router AP:",
    benefit_ap
)

print(
    "Benefit base rate:",
    benefit_target.mean()
)

print()

print(
    "Harm Router AUC:",
    harm_auc
)

print(
    "Harm Router AP:",
    harm_ap
)

print(
    "Harm base rate:",
    harm_target.mean()
)

In [ ]:
utility_oof = (
    benefit_oof_prob
    -
    harm_oof_prob
)

In [ ]:
pd.Series(
    utility_oof
).describe()

In [ ]:
utility_diagnostic = pd.DataFrame({
    "utility":
        utility_oof,

    "benefit":
        benefit_target,

    "harm":
        harm_target
})

In [ ]:
print(
    "Mean utility — Benefit cases:",
    utility_diagnostic.loc[
        utility_diagnostic[
            "benefit"
        ] == 1,
        "utility"
    ].mean()
)

print(
    "Mean utility — Harm cases:",
    utility_diagnostic.loc[
        utility_diagnostic[
            "harm"
        ] == 1,
        "utility"
    ].mean()
)

print(
    "Mean utility — All samples:",
    utility_diagnostic[
        "utility"
    ].mean()
)

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.hist(
    utility_oof[
        benefit_target == 1
    ],
    bins=25,
    alpha=0.6,
    density=True,
    label="Benefit"
)

plt.hist(
    utility_oof[
        harm_target == 1
    ],
    bins=25,
    alpha=0.6,
    density=True,
    label="Harm"
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel(
    "Estimated Routing Utility"
)

plt.ylabel(
    "Density"
)

plt.title(
    "Estimated Utility for Beneficial and Harmful Structural Interventions"
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
candidate_utility_thresholds = [
    -0.02,
    0.00,
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.075,
    0.10
]

In [ ]:
def evaluate_utility_routing_grouped_cv(
    X,
    y,
    groups,
    candidate_thresholds,
    outer_splits=5,
    inner_splits=4,
    random_state=42
):

    outer_cv = StratifiedGroupKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=random_state
    )

    records = []

    for outer_fold, (
        outer_tr,
        outer_va
    ) in enumerate(

        outer_cv.split(
            X[lexical_reduced],
            y,
            groups=groups
        ),

        start=1
    ):

        print(
            "\n"
            + "=" * 75
        )

        print(
            f"Outer Fold {outer_fold}"
        )

        print(
            "=" * 75
        )

        X_train = (
            X.iloc[
                outer_tr
            ].copy()
        )

        X_val = (
            X.iloc[
                outer_va
            ].copy()
        )

        y_train = (
            y.iloc[
                outer_tr
            ].copy()
        )

        y_val = (
            y.iloc[
                outer_va
            ].copy()
        )

        groups_train = (
            groups.iloc[
                outer_tr
            ].copy()
        )

        # =====================================
        # A. Inner OOF lexical + fusion
        # =====================================

        inner_cv = (
            StratifiedGroupKFold(
                n_splits=inner_splits,
                shuffle=True,
                random_state=(
                    random_state
                    + outer_fold
                )
            )
        )

        inner_lex_prob = np.zeros(
            len(X_train)
        )

        inner_lex_pred = np.zeros(
            len(X_train),
            dtype=int
        )

        inner_fusion_prob = np.zeros(
            len(X_train)
        )

        inner_fusion_pred = np.zeros(
            len(X_train),
            dtype=int
        )

        for inner_tr, inner_va in inner_cv.split(

            X_train[
                lexical_reduced
            ],

            y_train,

            groups=groups_train
        ):

            # Lexical
            lex_inner = (
                make_lexical_model()
            )

            lex_inner.fit(
                X_train.iloc[
                    inner_tr
                ][
                    lexical_reduced
                ],
                y_train.iloc[
                    inner_tr
                ]
            )

            lex_prob = (
                lex_inner.predict_proba(
                    X_train.iloc[
                        inner_va
                    ][
                        lexical_reduced
                    ]
                )[:, 1]
            )

            lex_pred = (
                lex_prob >= 0.5
            ).astype(int)

            # Fusion
            fusion_inner = (
                make_fusion_model()
            )

            fusion_inner.fit(
                X_train.iloc[
                    inner_tr
                ][
                    lexical_plus_intrinsic
                ],
                y_train.iloc[
                    inner_tr
                ]
            )

            fusion_prob = (
                fusion_inner.predict_proba(
                    X_train.iloc[
                        inner_va
                    ][
                        lexical_plus_intrinsic
                    ]
                )[:, 1]
            )

            fusion_pred = (
                fusion_prob >= 0.5
            ).astype(int)

            inner_lex_prob[
                inner_va
            ] = lex_prob

            inner_lex_pred[
                inner_va
            ] = lex_pred

            inner_fusion_prob[
                inner_va
            ] = fusion_prob

            inner_fusion_pred[
                inner_va
            ] = fusion_pred

        # =====================================
        # B. Inner benefit / harm labels
        # =====================================

        y_train_np = (
            y_train.to_numpy()
        )

        lex_correct = (
            inner_lex_pred
            ==
            y_train_np
        )

        fusion_correct = (
            inner_fusion_pred
            ==
            y_train_np
        )

        benefit_train = (
            (~lex_correct)
            &
            fusion_correct
        ).astype(int)

        harm_train = (
            lex_correct
            &
            (~fusion_correct)
        ).astype(int)

        print(
            "Benefit/Harm training cases:",
            benefit_train.sum(),
            "/",
            harm_train.sum()
        )

        # =====================================
        # C. Router training features
        # =====================================

        router_train_X = (
            build_utility_router_features(
                X_train,
                inner_lex_prob
            )
        )

        benefit_valid = (
            benefit_train.sum() >= 10
            and
            len(
                np.unique(
                    benefit_train
                )
            ) == 2
        )

        harm_valid = (
            harm_train.sum() >= 10
            and
            len(
                np.unique(
                    harm_train
                )
            ) == 2
        )

        router_valid = (
            benefit_valid
            and
            harm_valid
        )

        if router_valid:

            benefit_router = (
                make_utility_router()
            )

            harm_router = (
                make_utility_router()
            )

            benefit_router.fit(
                router_train_X,
                benefit_train
            )

            harm_router.fit(
                router_train_X,
                harm_train
            )

        # =====================================
        # D. Outer lexical model
        # =====================================

        lexical_outer = (
            make_lexical_model()
        )

        lexical_outer.fit(
            X_train[
                lexical_reduced
            ],
            y_train
        )

        val_lex_prob = (
            lexical_outer
            .predict_proba(
                X_val[
                    lexical_reduced
                ]
            )[:, 1]
        )

        val_lex_pred = (
            val_lex_prob >= 0.5
        ).astype(int)

        # =====================================
        # E. Outer fusion model
        # =====================================

        fusion_outer = (
            make_fusion_model()
        )

        fusion_outer.fit(
            X_train[
                lexical_plus_intrinsic
            ],
            y_train
        )

        val_fusion_prob = (
            fusion_outer
            .predict_proba(
                X_val[
                    lexical_plus_intrinsic
                ]
            )[:, 1]
        )

        val_fusion_pred = (
            val_fusion_prob >= 0.5
        ).astype(int)

        # =====================================
        # F. Validation router features
        # =====================================

        router_val_X = (
            build_utility_router_features(
                X_val,
                val_lex_prob
            )
        )

        if router_valid:

            val_benefit_prob = (
                benefit_router
                .predict_proba(
                    router_val_X
                )[:, 1]
            )

            val_harm_prob = (
                harm_router
                .predict_proba(
                    router_val_X
                )[:, 1]
            )

            val_utility = (
                val_benefit_prob
                -
                val_harm_prob
            )

        else:

            val_benefit_prob = (
                np.zeros(
                    len(X_val)
                )
            )

            val_harm_prob = (
                np.zeros(
                    len(X_val)
                )
            )

            val_utility = (
                np.zeros(
                    len(X_val)
                )
            )

        # =====================================
        # G. Actual validation benefit/harm
        # =====================================

        y_val_np = (
            y_val.to_numpy()
        )

        val_lex_correct = (
            val_lex_pred
            ==
            y_val_np
        )

        val_fusion_correct = (
            val_fusion_pred
            ==
            y_val_np
        )

        actual_benefit = (
            (~val_lex_correct)
            &
            val_fusion_correct
        )

        actual_harm = (
            val_lex_correct
            &
            (~val_fusion_correct)
        )

        # diagnostic router AUC/AP
        if (
            len(
                np.unique(
                    actual_benefit.astype(int)
                )
            )
            == 2
        ):

            benefit_auc = (
                roc_auc_score(
                    actual_benefit.astype(int),
                    val_benefit_prob
                )
            )

            benefit_ap = (
                average_precision_score(
                    actual_benefit.astype(int),
                    val_benefit_prob
                )
            )

        else:

            benefit_auc = np.nan
            benefit_ap = np.nan

        if (
            len(
                np.unique(
                    actual_harm.astype(int)
                )
            )
            == 2
        ):

            harm_auc = (
                roc_auc_score(
                    actual_harm.astype(int),
                    val_harm_prob
                )
            )

            harm_ap = (
                average_precision_score(
                    actual_harm.astype(int),
                    val_harm_prob
                )
            )

        else:

            harm_auc = np.nan
            harm_ap = np.nan

        # =====================================
        # H. Evaluate utility thresholds
        # =====================================

        for threshold in (
            candidate_thresholds
        ):

            routed = (
                val_utility
                >
                threshold
            )

            routed_n = (
                routed.sum()
            )

            routed_pred = (
                val_lex_pred.copy()
            )

            routed_prob = (
                val_lex_prob.copy()
            )

            routed_pred[
                routed
            ] = (
                val_fusion_pred[
                    routed
                ]
            )

            routed_prob[
                routed
            ] = (
                val_fusion_prob[
                    routed
                ]
            )

            # ------------------------------
            # Routing outcomes
            # ------------------------------

            benefits_routed = (
                actual_benefit
                &
                routed
            ).sum()

            harms_routed = (
                actual_harm
                &
                routed
            ).sum()

            total_benefits = (
                actual_benefit.sum()
            )

            total_harms = (
                actual_harm.sum()
            )

            benefit_capture_rate = (
                benefits_routed
                /
                total_benefits

                if total_benefits > 0

                else np.nan
            )

            harm_capture_rate = (
                harms_routed
                /
                total_harms

                if total_harms > 0

                else np.nan
            )

            routing_precision = (
                benefits_routed
                /
                routed_n

                if routed_n > 0

                else np.nan
            )

            routing_harm_rate = (
                harms_routed
                /
                routed_n

                if routed_n > 0

                else np.nan
            )

            # ------------------------------
            # Real decision corrections
            # ------------------------------

            changed = (
                routed_pred
                !=
                val_lex_pred
            )

            corrections = (
                changed
                &
                (
                    routed_pred
                    ==
                    y_val_np
                )
            ).sum()

            harms = (
                changed
                &
                (
                    val_lex_pred
                    ==
                    y_val_np
                )
                &
                (
                    routed_pred
                    !=
                    y_val_np
                )
            ).sum()

            records.append({

                "outer_fold":
                    outer_fold,

                "utility_threshold":
                    threshold,

                "router_valid":
                    router_valid,

                "benefit_train_n":
                    benefit_train.sum(),

                "harm_train_n":
                    harm_train.sum(),

                "actual_benefit_n":
                    total_benefits,

                "actual_harm_n":
                    total_harms,

                "benefit_router_auc":
                    benefit_auc,

                "benefit_router_ap":
                    benefit_ap,

                "harm_router_auc":
                    harm_auc,

                "harm_router_ap":
                    harm_ap,

                "invocation_rate":
                    routed_n
                    /
                    len(X_val),

                "benefit_capture_rate":
                    benefit_capture_rate,

                "harm_capture_rate":
                    harm_capture_rate,

                "routing_precision":
                    routing_precision,

                "routing_harm_rate":
                    routing_harm_rate,

                "corrections":
                    corrections,

                "harms":
                    harms,

                "net_corrections":
                    corrections
                    -
                    harms,

                # lexical baseline
                "lexical_f1":
                    f1_score(
                        y_val,
                        val_lex_pred
                    ),

                "lexical_auc":
                    roc_auc_score(
                        y_val,
                        val_lex_prob
                    ),

                "lexical_mcc":
                    matthews_corrcoef(
                        y_val,
                        val_lex_pred
                    ),

                # global fusion
                "fusion_f1":
                    f1_score(
                        y_val,
                        val_fusion_pred
                    ),

                "fusion_auc":
                    roc_auc_score(
                        y_val,
                        val_fusion_prob
                    ),

                "fusion_mcc":
                    matthews_corrcoef(
                        y_val,
                        val_fusion_pred
                    ),

                # utility routing
                "utility_f1":
                    f1_score(
                        y_val,
                        routed_pred
                    ),

                "utility_precision":
                    precision_score(
                        y_val,
                        routed_pred
                    ),

                "utility_recall":
                    recall_score(
                        y_val,
                        routed_pred
                    ),

                "utility_auc":
                    roc_auc_score(
                        y_val,
                        routed_prob
                    ),

                "utility_mcc":
                    matthews_corrcoef(
                        y_val,
                        routed_pred
                    ),

                "utility_accuracy":
                    accuracy_score(
                        y_val,
                        routed_pred
                    )
            })

            print(
                f"u>{threshold:.3f} | "
                f"invoke={routed_n/len(X_val):.3f} | "
                f"Lex={f1_score(y_val,val_lex_pred):.4f} | "
                f"Utility={f1_score(y_val,routed_pred):.4f} | "
                f"+{corrections}/-{harms}"
            )

    return pd.DataFrame(
        records
    )

In [ ]:
utility_router_results = (
    evaluate_utility_routing_grouped_cv(

        X=X_dev,

        y=y_dev,

        groups=groups_dev,

        candidate_thresholds=
            candidate_utility_thresholds,

        outer_splits=5,

        inner_splits=4,

        random_state=42
    )
)

In [ ]:
utility_router_results[
    [
        "outer_fold",
        "benefit_router_auc",
        "benefit_router_ap",
        "harm_router_auc",
        "harm_router_ap",
        "benefit_train_n",
        "harm_train_n"
    ]
].drop_duplicates()

In [ ]:
print(
    "Mean Benefit Router AUC:",
    utility_router_results[
        "benefit_router_auc"
    ].mean()
)

print(
    "Mean Benefit Router AP:",
    utility_router_results[
        "benefit_router_ap"
    ].mean()
)

print()

print(
    "Mean Harm Router AUC:",
    utility_router_results[
        "harm_router_auc"
    ].mean()
)

print(
    "Mean Harm Router AP:",
    utility_router_results[
        "harm_router_ap"
    ].mean()
)

In [ ]:
utility_threshold_summary = (
    utility_router_results
    .groupby(
        "utility_threshold"
    )
    .agg(

        mean_utility_f1=(
            "utility_f1",
            "mean"
        ),

        std_utility_f1=(
            "utility_f1",
            "std"
        ),

        mean_utility_auc=(
            "utility_auc",
            "mean"
        ),

        mean_utility_mcc=(
            "utility_mcc",
            "mean"
        ),

        mean_invocation_rate=(
            "invocation_rate",
            "mean"
        ),

        mean_benefit_capture_rate=(
            "benefit_capture_rate",
            "mean"
        ),

        mean_harm_capture_rate=(
            "harm_capture_rate",
            "mean"
        ),

        mean_routing_precision=(
            "routing_precision",
            "mean"
        ),

        mean_routing_harm_rate=(
            "routing_harm_rate",
            "mean"
        ),

        total_corrections=(
            "corrections",
            "sum"
        ),

        total_harms=(
            "harms",
            "sum"
        ),

        total_net_corrections=(
            "net_corrections",
            "sum"
        ),

        mean_lexical_f1=(
            "lexical_f1",
            "mean"
        ),

        mean_fusion_f1=(
            "fusion_f1",
            "mean"
        ),

        mean_benefit_router_auc=(
            "benefit_router_auc",
            "mean"
        ),

        mean_harm_router_auc=(
            "harm_router_auc",
            "mean"
        )

    )
    .reset_index()
)

In [ ]:
utility_threshold_summary[
    "delta_f1_vs_lexical"
] = (
    utility_threshold_summary[
        "mean_utility_f1"
    ]
    -
    utility_threshold_summary[
        "mean_lexical_f1"
    ]
)

utility_threshold_summary[
    "delta_f1_vs_fusion"
] = (
    utility_threshold_summary[
        "mean_utility_f1"
    ]
    -
    utility_threshold_summary[
        "mean_fusion_f1"
    ]
)

In [ ]:
utility_threshold_summary[
    [
        "utility_threshold",
        "mean_utility_f1",
        "std_utility_f1",
        "delta_f1_vs_lexical",
        "delta_f1_vs_fusion",
        "mean_invocation_rate",
        "mean_benefit_capture_rate",
        "mean_harm_capture_rate",
        "total_corrections",
        "total_harms",
        "total_net_corrections"
    ]
]

In [ ]:
plt.figure(
    figsize=(7, 6)
)

plt.plot(
    utility_threshold_summary[
        "mean_invocation_rate"
    ],

    utility_threshold_summary[
        "mean_utility_f1"
    ],

    marker="o",
    linewidth=2
)

for _, row in (
    utility_threshold_summary
    .iterrows()
):

    plt.annotate(
        f"u>{row['utility_threshold']:.3f}",

        (
            row[
                "mean_invocation_rate"
            ],

            row[
                "mean_utility_f1"
            ]
        ),

        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

lex_f1_ref = (
    utility_threshold_summary[
        "mean_lexical_f1"
    ].mean()
)

fusion_f1_ref = (
    utility_threshold_summary[
        "mean_fusion_f1"
    ].mean()
)

plt.axhline(
    lex_f1_ref,
    linestyle="--",
    label="Lexical-only"
)

plt.axhline(
    fusion_f1_ref,
    linestyle=":",
    label="Global fusion"
)

plt.xlabel(
    "Structural Invocation Rate"
)

plt.ylabel(
    "Mean Grouped-CV F1"
)

plt.title(
    "Utility-Aware Routing: Performance–Computation Trade-off"
)

plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(7, 6)
)

plt.plot(
    utility_threshold_summary[
        "mean_invocation_rate"
    ],

    utility_threshold_summary[
        "mean_benefit_capture_rate"
    ],

    marker="o",
    linewidth=2,
    label="Benefit capture"
)

plt.plot(
    utility_threshold_summary[
        "mean_invocation_rate"
    ],

    utility_threshold_summary[
        "mean_harm_capture_rate"
    ],

    marker="o",
    linewidth=2,
    label="Harm capture"
)

plt.xlabel(
    "Structural Invocation Rate"
)

plt.ylabel(
    "Capture Rate"
)

plt.title(
    "Benefit–Harm Trade-off of Utility-Aware Routing"
)

plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
best_utility_row = (
    utility_threshold_summary
    .sort_values(
        by=[
            "mean_utility_f1",
            "mean_invocation_rate"
        ],
        ascending=[
            False,
            True
        ]
    )
    .iloc[0]
)

best_utility_threshold = float(
    best_utility_row[
        "utility_threshold"
    ]
)

print(
    "Selected utility threshold:",
    best_utility_threshold
)

print(
    "Utility-routed F1:",
    best_utility_row[
        "mean_utility_f1"
    ]
)

print(
    "Lexical F1:",
    best_utility_row[
        "mean_lexical_f1"
    ]
)

print(
    "Delta F1:",
    best_utility_row[
        "delta_f1_vs_lexical"
    ]
)

print(
    "Invocation rate:",
    best_utility_row[
        "mean_invocation_rate"
    ]
)

print(
    "Net corrections:",
    best_utility_row[
        "total_net_corrections"
    ]
)

In [ ]:
selected_utility_cv = (
    utility_router_results[
        utility_router_results[
            "utility_threshold"
        ]
        ==
        best_utility_threshold
    ]
    .copy()
)

In [ ]:
selected_utility_cv[
    [
        "outer_fold",
        "lexical_f1",
        "fusion_f1",
        "utility_f1",
        "invocation_rate",
        "benefit_capture_rate",
        "harm_capture_rate",
        "corrections",
        "harms",
        "net_corrections",
        "benefit_router_auc",
        "harm_router_auc"
    ]
]

In [ ]:
utility_wins = (
    selected_utility_cv[
        "utility_f1"
    ]
    >
    selected_utility_cv[
        "lexical_f1"
    ]
).sum()

utility_ties = (
    selected_utility_cv[
        "utility_f1"
    ]
    ==
    selected_utility_cv[
        "lexical_f1"
    ]
).sum()

utility_losses = (
    selected_utility_cv[
        "utility_f1"
    ]
    <
    selected_utility_cv[
        "lexical_f1"
    ]
).sum()

print(
    "Utility routing wins:",
    utility_wins
)

print(
    "Ties:",
    utility_ties
)

print(
    "Losses:",
    utility_losses
)

In [ ]:
utility_diagnostic.to_csv(
    "utility_router_diagnostic.csv",
    index=False
)

utility_router_results.to_csv(
    "utility_router_nested_grouped_cv.csv",
    index=False
)

utility_threshold_summary.to_csv(
    "utility_router_threshold_summary.csv",
    index=False
)

selected_utility_cv.to_csv(
    "utility_router_selected_threshold_folds.csv",
    index=False
)

# Phase 12 — Lightweight Model Design via Knowledge Distillation

The preceding experiments showed that intrinsic QR structural descriptors do
not provide sufficiently stable complementary information beyond lexical URL
features. Consequently, the final detector should avoid unnecessary structural
computation.

However, the strongest lexical baseline is still a relatively large XGBoost
model. We therefore investigate whether its decision function can be compressed
into a smaller student classifier.

The full lexical XGBoost model is treated as the teacher. A compact student
model is trained using both the true phishing labels and the teacher's soft
probability outputs.

For sample i, the student training target is defined as

    y_soft = alpha * y + (1 - alpha) * p_teacher

where alpha controls the trade-off between the ground-truth label and teacher
knowledge.

The objective is not merely to maximize classification performance, but to
identify a compact model that preserves most of the teacher's predictive
performance while substantially reducing model complexity and deployment cost.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import time
import os
import joblib

from sklearn.model_selection import (
    StratifiedGroupKFold
)

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score
)

from xgboost import (
    XGBClassifier,
    XGBRegressor
)

RANDOM_STATE = 42
N_JOBS = 1

In [ ]:
RANDOM_STATE = 42
N_JOBS = 1

In [ ]:
def make_teacher_model():

    return XGBClassifier(
        **best_xgb_lexical_params,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    )

In [ ]:
student_configs = [

    {
        "name": "XGB_25_D2",
        "n_estimators": 25,
        "max_depth": 2
    },

    {
        "name": "XGB_50_D2",
        "n_estimators": 50,
        "max_depth": 2
    },

    {
        "name": "XGB_50_D3",
        "n_estimators": 50,
        "max_depth": 3
    },

    {
        "name": "XGB_100_D2",
        "n_estimators": 100,
        "max_depth": 2
    },

    {
        "name": "XGB_100_D3",
        "n_estimators": 100,
        "max_depth": 3
    }
]

In [ ]:
def make_distilled_student(config):

    return XGBRegressor(
        n_estimators=config[
            "n_estimators"
        ],

        max_depth=config[
            "max_depth"
        ],

        learning_rate=0.05,

        subsample=0.9,

        colsample_bytree=1.0,

        objective="reg:squarederror",

        random_state=RANDOM_STATE,

        n_jobs=N_JOBS
    )

In [ ]:
candidate_alphas = [
    0.00,
    0.25,
    0.50,
    0.75
]

In [ ]:
def evaluate_lightweight_students(
    X,
    y,
    groups,
    student_configs,
    alphas,
    outer_splits=5,
    inner_splits=4,
    random_state=42
):

    outer_cv = StratifiedGroupKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=random_state
    )

    records = []

    for fold, (
        tr_idx,
        va_idx
    ) in enumerate(

        outer_cv.split(
            X[lexical_reduced],
            y,
            groups=groups
        ),

        start=1
    ):

        print(
            "\n"
            + "=" * 75
        )

        print(
            f"Outer Fold {fold}"
        )

        print(
            "=" * 75
        )

        # ----------------------------------------------------
        # Outer train / validation
        # ----------------------------------------------------

        X_train = (
            X.iloc[tr_idx]
            [lexical_reduced]
            .reset_index(drop=True)
            .copy()
        )

        X_val = (
            X.iloc[va_idx]
            [lexical_reduced]
            .reset_index(drop=True)
            .copy()
        )

        y_train = (
            y.iloc[tr_idx]
            .reset_index(drop=True)
            .copy()
        )

        y_val = (
            y.iloc[va_idx]
            .reset_index(drop=True)
            .copy()
        )

        groups_train = (
            groups.iloc[tr_idx]
            .reset_index(drop=True)
            .copy()
        )

        # ====================================================
        # STEP A
        # Inner grouped OOF Teacher probabilities
        # ====================================================

        inner_cv = StratifiedGroupKFold(
            n_splits=inner_splits,
            shuffle=True,
            random_state=(
                random_state
                +
                fold
            )
        )

        teacher_train_oof_prob = (
            np.zeros(
                len(X_train),
                dtype=float
            )
        )

        for inner_tr, inner_va in (
            inner_cv.split(
                X_train,
                y_train,
                groups=groups_train
            )
        ):

            inner_teacher = (
                make_teacher_model()
            )

            inner_teacher.fit(
                X_train.iloc[
                    inner_tr
                ],
                y_train.iloc[
                    inner_tr
                ]
            )

            inner_prob = (
                inner_teacher
                .predict_proba(
                    X_train.iloc[
                        inner_va
                    ]
                )[:, 1]
            )

            teacher_train_oof_prob[
                inner_va
            ] = inner_prob

        # ====================================================
        # STEP B
        # Full Outer Teacher
        # ====================================================

        teacher = (
            make_teacher_model()
        )

        teacher.fit(
            X_train,
            y_train
        )

        teacher_val_prob = (
            teacher.predict_proba(
                X_val
            )[:, 1]
        )

        teacher_val_pred = (
            teacher_val_prob
            >=
            0.5
        ).astype(int)

        teacher_f1 = (
            f1_score(
                y_val,
                teacher_val_pred
            )
        )

        teacher_auc = (
            roc_auc_score(
                y_val,
                teacher_val_prob
            )
        )

        teacher_mcc = (
            matthews_corrcoef(
                y_val,
                teacher_val_pred
            )
        )

        print(
            f"Teacher | "
            f"F1={teacher_f1:.4f} | "
            f"AUC={teacher_auc:.4f}"
        )

        # ====================================================
        # STEP C
        # Each compact architecture
        # ====================================================

        for config in student_configs:

            # =================================================
            # C1. Ordinary compact classifier
            # =================================================

            plain_student = (
                make_plain_student_classifier(
                    config
                )
            )

            plain_student.fit(
                X_train,
                y_train
            )

            plain_prob = (
                plain_student
                .predict_proba(
                    X_val
                )[:, 1]
            )

            plain_pred = (
                plain_prob
                >=
                0.5
            ).astype(int)

            plain_f1 = (
                f1_score(
                    y_val,
                    plain_pred
                )
            )

            records.append({

                "fold":
                    fold,

                "student":
                    config["name"],

                "training_type":
                    "Plain",

                "alpha":
                    np.nan,

                "n_estimators":
                    config[
                        "n_estimators"
                    ],

                "max_depth":
                    config[
                        "max_depth"
                    ],

                # Teacher
                "teacher_f1":
                    teacher_f1,

                "teacher_auc":
                    teacher_auc,

                "teacher_mcc":
                    teacher_mcc,

                # Student
                "student_f1":
                    plain_f1,

                "student_precision":
                    precision_score(
                        y_val,
                        plain_pred,
                        zero_division=0
                    ),

                "student_recall":
                    recall_score(
                        y_val,
                        plain_pred,
                        zero_division=0
                    ),

                "student_auc":
                    roc_auc_score(
                        y_val,
                        plain_prob
                    ),

                "student_mcc":
                    matthews_corrcoef(
                        y_val,
                        plain_pred
                    ),

                "student_accuracy":
                    accuracy_score(
                        y_val,
                        plain_pred
                    )
            })

            print(
                f"{config['name']} | "
                f"Plain | "
                f"F1={plain_f1:.4f}"
            )

            # =================================================
            # C2. Distilled students
            # =================================================

            for alpha in alphas:

                soft_target = (
                    alpha
                    *
                    y_train.to_numpy()
                    +
                    (
                        1
                        -
                        alpha
                    )
                    *
                    teacher_train_oof_prob
                )

                distilled_student = (
                    make_distilled_student(
                        config
                    )
                )

                distilled_student.fit(
                    X_train,
                    soft_target
                )

                # Regressor output
                distilled_prob = (
                    distilled_student.predict(
                        X_val
                    )
                )

                # Safety: constrain to [0,1]
                distilled_prob = np.clip(
                    distilled_prob,
                    0.0,
                    1.0
                )

                distilled_pred = (
                    distilled_prob
                    >=
                    0.5
                ).astype(int)

                distilled_f1 = (
                    f1_score(
                        y_val,
                        distilled_pred
                    )
                )

                records.append({

                    "fold":
                        fold,

                    "student":
                        config["name"],

                    "training_type":
                        "Distilled",

                    "alpha":
                        alpha,

                    "n_estimators":
                        config[
                            "n_estimators"
                        ],

                    "max_depth":
                        config[
                            "max_depth"
                        ],

                    # Teacher
                    "teacher_f1":
                        teacher_f1,

                    "teacher_auc":
                        teacher_auc,

                    "teacher_mcc":
                        teacher_mcc,

                    # Student
                    "student_f1":
                        distilled_f1,

                    "student_precision":
                        precision_score(
                            y_val,
                            distilled_pred,
                            zero_division=0
                        ),

                    "student_recall":
                        recall_score(
                            y_val,
                            distilled_pred,
                            zero_division=0
                        ),

                    "student_auc":
                        roc_auc_score(
                            y_val,
                            distilled_prob
                        ),

                    "student_mcc":
                        matthews_corrcoef(
                            y_val,
                            distilled_pred
                        ),

                    "student_accuracy":
                        accuracy_score(
                            y_val,
                            distilled_pred
                        )
                })

                print(
                    f"{config['name']} | "
                    f"Distilled α={alpha:.2f} | "
                    f"F1={distilled_f1:.4f}"
                )

    return pd.DataFrame(
        records
    )

In [ ]:
# ============================================================
# FIX — Plain compact XGBoost student classifier
# ============================================================

from xgboost import XGBClassifier

def make_plain_student_classifier(config):
    """
    Build an ordinary compact XGBoost classifier trained
    directly on the hard class labels.

    This is the non-distilled baseline for each student
    architecture.
    """

    model = XGBClassifier(
        n_estimators=int(config["n_estimators"]),
        max_depth=int(config["max_depth"]),

        # Keep the same basic optimization settings used
        # throughout the lightweight-model experiment
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=1.0,

        objective="binary:logistic",
        eval_metric="logloss",

        random_state=RANDOM_STATE,
        n_jobs=1
    )

    return model


print("make_plain_student_classifier defined successfully.")

In [ ]:
print(
    "make_plain_student_classifier:",
    "FOUND"
    if "make_plain_student_classifier" in globals()
    else "NOT FOUND"
)

print(
    "make_distilled_student:",
    "FOUND"
    if "make_distilled_student" in globals()
    else "NOT FOUND"
)

print(
    "evaluate_lightweight_students:",
    "FOUND"
    if "evaluate_lightweight_students" in globals()
    else "NOT FOUND"
)

In [ ]:
lightweight_results = (
    evaluate_lightweight_students(
        X=X_dev,
        y=y_dev,
        groups=groups_dev,
        student_configs=
            student_configs,
        alphas=
            candidate_alphas,
        outer_splits=5,
        inner_splits=4,
        random_state=42
    )
)

In [ ]:
lightweight_results.head(20)

In [ ]:
lightweight_results.isna().sum()

In [ ]:
plain_summary = (
    lightweight_results[
        lightweight_results[
            "training_type"
        ] == "Plain"
    ]
    .groupby(
        [
            "student",
            "n_estimators",
            "max_depth"
        ]
    )
    .agg(

        mean_f1=(
            "student_f1",
            "mean"
        ),

        std_f1=(
            "student_f1",
            "std"
        ),

        mean_auc=(
            "student_auc",
            "mean"
        ),

        std_auc=(
            "student_auc",
            "std"
        ),

        mean_mcc=(
            "student_mcc",
            "mean"
        ),

        mean_precision=(
            "student_precision",
            "mean"
        ),

        mean_recall=(
            "student_recall",
            "mean"
        ),

        teacher_f1=(
            "teacher_f1",
            "mean"
        )

    )
    .reset_index()
)

plain_summary[
    "f1_gap_to_teacher"
] = (
    plain_summary[
        "mean_f1"
    ]
    -
    plain_summary[
        "teacher_f1"
    ]
)

plain_summary.sort_values(
    "mean_f1",
    ascending=False
)

In [ ]:
distilled_summary = (
    lightweight_results[
        lightweight_results[
            "training_type"
        ] == "Distilled"
    ]
    .groupby(
        [
            "student",
            "n_estimators",
            "max_depth",
            "alpha"
        ]
    )
    .agg(

        mean_f1=(
            "student_f1",
            "mean"
        ),

        std_f1=(
            "student_f1",
            "std"
        ),

        mean_auc=(
            "student_auc",
            "mean"
        ),

        std_auc=(
            "student_auc",
            "std"
        ),

        mean_mcc=(
            "student_mcc",
            "mean"
        ),

        mean_precision=(
            "student_precision",
            "mean"
        ),

        mean_recall=(
            "student_recall",
            "mean"
        ),

        teacher_f1=(
            "teacher_f1",
            "mean"
        )

    )
    .reset_index()
)

distilled_summary[
    "f1_gap_to_teacher"
] = (
    distilled_summary[
        "mean_f1"
    ]
    -
    distilled_summary[
        "teacher_f1"
    ]
)

distilled_summary.sort_values(
    "mean_f1",
    ascending=False
).head(20)

In [ ]:
best_distilled_by_student = (
    distilled_summary
    .sort_values(
        "mean_f1",
        ascending=False
    )
    .groupby(
        "student",
        as_index=False
    )
    .first()
)

best_distilled_by_student[
    [
        "student",
        "n_estimators",
        "max_depth",
        "alpha",
        "mean_f1",
        "std_f1",
        "mean_auc",
        "mean_mcc",
        "f1_gap_to_teacher"
    ]
]

In [ ]:
plain_vs_distilled = (
    plain_summary[
        [
            "student",
            "n_estimators",
            "max_depth",
            "mean_f1",
            "std_f1",
            "mean_auc",
            "mean_mcc"
        ]
    ]
    .rename(
        columns={
            "mean_f1":
                "plain_f1",

            "std_f1":
                "plain_std_f1",

            "mean_auc":
                "plain_auc",

            "mean_mcc":
                "plain_mcc"
        }
    )
    .merge(

        best_distilled_by_student[
            [
                "student",
                "alpha",
                "mean_f1",
                "std_f1",
                "mean_auc",
                "mean_mcc"
            ]
        ]
        .rename(
            columns={
                "alpha":
                    "best_alpha",

                "mean_f1":
                    "distilled_f1",

                "std_f1":
                    "distilled_std_f1",

                "mean_auc":
                    "distilled_auc",

                "mean_mcc":
                    "distilled_mcc"
            }
        ),

        on="student",
        how="left"
    )
)

In [ ]:
plain_vs_distilled[
    "distillation_gain"
] = (
    plain_vs_distilled[
        "distilled_f1"
    ]
    -
    plain_vs_distilled[
        "plain_f1"
    ]
)

plain_vs_distilled.sort_values(
    "distillation_gain",
    ascending=False
)

In [ ]:
teacher_mean_f1 = (
    lightweight_results[
        [
            "fold",
            "teacher_f1"
        ]
    ]
    .drop_duplicates()
    ["teacher_f1"]
    .mean()
)

teacher_mean_auc = (
    lightweight_results[
        [
            "fold",
            "teacher_auc"
        ]
    ]
    .drop_duplicates()
    ["teacher_auc"]
    .mean()
)

print(
    "Teacher mean F1:",
    teacher_mean_f1
)

print(
    "Teacher mean AUC:",
    teacher_mean_auc
)

In [ ]:
plain_candidates = (
    plain_summary[
        [
            "student",
            "n_estimators",
            "max_depth",
            "mean_f1",
            "std_f1",
            "mean_auc",
            "mean_mcc"
        ]
    ]
    .copy()
)

plain_candidates[
    "training_type"
] = "Plain"

plain_candidates[
    "alpha"
] = np.nan

In [ ]:
distilled_candidates = (
    best_distilled_by_student[
        [
            "student",
            "n_estimators",
            "max_depth",
            "alpha",
            "mean_f1",
            "std_f1",
            "mean_auc",
            "mean_mcc"
        ]
    ]
    .copy()
)

distilled_candidates[
    "training_type"
] = "Distilled"

In [ ]:
all_student_candidates = pd.concat(
    [
        plain_candidates,
        distilled_candidates
    ],
    ignore_index=True
)

In [ ]:
all_student_candidates[
    "complexity_proxy"
] = (
    all_student_candidates[
        "n_estimators"
    ]
    *
    (
        2
        **
        all_student_candidates[
            "max_depth"
        ]
    )
)

all_student_candidates[
    "f1_gap_to_teacher"
] = (
    all_student_candidates[
        "mean_f1"
    ]
    -
    teacher_mean_f1
)

all_student_candidates.sort_values(
    "mean_f1",
    ascending=False
)

In [ ]:
F1_TOLERANCE = 0.005

acceptable_students = (
    all_student_candidates[
        all_student_candidates[
            "mean_f1"
        ]
        >=
        teacher_mean_f1
        -
        F1_TOLERANCE
    ]
    .copy()
)

acceptable_students.sort_values(
    [
        "complexity_proxy",
        "mean_f1"
    ],
    ascending=[
        True,
        False
    ]
)

In [ ]:
if len(
    acceptable_students
) > 0:

    selected_student_row = (
        acceptable_students
        .sort_values(
            by=[
                "complexity_proxy",
                "mean_f1"
            ],
            ascending=[
                True,
                False
            ]
        )
        .iloc[0]
    )

else:

    print(
        "No student satisfies "
        "the predefined F1 tolerance."
    )

    selected_student_row = (
        all_student_candidates
        .sort_values(
            "mean_f1",
            ascending=False
        )
        .iloc[0]
    )

In [ ]:
print(
    "Selected lightweight student:"
)

print(
    selected_student_row
)

In [ ]:
# ============================================================
# Build final selected lightweight student model
# ============================================================

selected_name = selected_student_row["student"]
selected_training_type = selected_student_row["training_type"]

selected_config = {
    "name": selected_name,
    "n_estimators": int(
        selected_student_row["n_estimators"]
    ),
    "max_depth": int(
        selected_student_row["max_depth"]
    )
}

print("Selected config:")
print(selected_config)

print("Training type:")
print(selected_training_type)

In [ ]:
# ============================================================
# Train final student on full development set
# ============================================================

if selected_training_type == "Plain":

    final_student = (
        make_plain_student_classifier(
            selected_config
        )
    )

    final_student.fit(
        X_dev[lexical_reduced],
        y_dev
    )

    print(
        "Final student trained as Plain XGBClassifier."
    )

else:

    selected_alpha = float(
        selected_student_row["alpha"]
    )

    # Train teacher on full development data
    full_teacher = make_teacher_model()

    full_teacher.fit(
        X_dev[lexical_reduced],
        y_dev
    )

    teacher_prob_dev = (
        full_teacher.predict_proba(
            X_dev[lexical_reduced]
        )[:, 1]
    )

    soft_target = (
        selected_alpha
        *
        y_dev.to_numpy()
        +
        (
            1 - selected_alpha
        )
        *
        teacher_prob_dev
    )

    final_student = (
        make_distilled_student(
            selected_config
        )
    )

    final_student.fit(
        X_dev[lexical_reduced],
        soft_target
    )

    print(
        f"Final student trained as Distilled model, "
        f"alpha={selected_alpha:.2f}"
    )

In [ ]:
print(final_student)

In [ ]:
import joblib

joblib.dump(
    final_student,
    "current_compact_student_xgb100_d2.pkl"
)

print("Student model saved.")

In [ ]:
# ============================================================
# Phase 12B — Medium-Complexity Student Search
# ============================================================

medium_student_configs = [

    {
        "name": "XGB_150_D2",
        "n_estimators": 150,
        "max_depth": 2
    },

    {
        "name": "XGB_150_D3",
        "n_estimators": 150,
        "max_depth": 3
    },

    {
        "name": "XGB_200_D2",
        "n_estimators": 200,
        "max_depth": 2
    },

    {
        "name": "XGB_200_D3",
        "n_estimators": 200,
        "max_depth": 3
    },

    {
        "name": "XGB_250_D2",
        "n_estimators": 250,
        "max_depth": 2
    },

    {
        "name": "XGB_250_D3",
        "n_estimators": 250,
        "max_depth": 3
    },

    {
        "name": "XGB_300_D2",
        "n_estimators": 300,
        "max_depth": 2
    },

    {
        "name": "XGB_300_D3",
        "n_estimators": 300,
        "max_depth": 3
    }
]

len(medium_student_configs)

In [ ]:
def evaluate_plain_compact_students(
    X,
    y,
    groups,
    configs,
    n_splits=5,
    random_state=42
):

    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    records = []

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X[lexical_reduced],
            y,
            groups=groups
        ),
        start=1
    ):

        print("\n" + "=" * 70)
        print(f"Fold {fold}")
        print("=" * 70)

        X_train = (
            X.iloc[tr_idx]
            [lexical_reduced]
            .copy()
        )

        X_val = (
            X.iloc[va_idx]
            [lexical_reduced]
            .copy()
        )

        y_train = (
            y.iloc[tr_idx]
            .copy()
        )

        y_val = (
            y.iloc[va_idx]
            .copy()
        )

        # --------------------------------------------
        # Teacher
        # --------------------------------------------

        teacher = make_teacher_model()

        teacher.fit(
            X_train,
            y_train
        )

        teacher_prob = (
            teacher.predict_proba(
                X_val
            )[:, 1]
        )

        teacher_pred = (
            teacher_prob >= 0.5
        ).astype(int)

        teacher_f1 = f1_score(
            y_val,
            teacher_pred
        )

        teacher_auc = roc_auc_score(
            y_val,
            teacher_prob
        )

        teacher_mcc = matthews_corrcoef(
            y_val,
            teacher_pred
        )

        print(
            f"Teacher | "
            f"F1={teacher_f1:.4f} | "
            f"AUC={teacher_auc:.4f}"
        )

        # --------------------------------------------
        # Compact models
        # --------------------------------------------

        for config in configs:

            model = (
                make_plain_student_classifier(
                    config
                )
            )

            model.fit(
                X_train,
                y_train
            )

            prob = (
                model.predict_proba(
                    X_val
                )[:, 1]
            )

            pred = (
                prob >= 0.5
            ).astype(int)

            current_f1 = f1_score(
                y_val,
                pred
            )

            records.append({

                "fold":
                    fold,

                "student":
                    config["name"],

                "n_estimators":
                    config["n_estimators"],

                "max_depth":
                    config["max_depth"],

                "student_f1":
                    current_f1,

                "student_precision":
                    precision_score(
                        y_val,
                        pred,
                        zero_division=0
                    ),

                "student_recall":
                    recall_score(
                        y_val,
                        pred,
                        zero_division=0
                    ),

                "student_auc":
                    roc_auc_score(
                        y_val,
                        prob
                    ),

                "student_mcc":
                    matthews_corrcoef(
                        y_val,
                        pred
                    ),

                "student_accuracy":
                    accuracy_score(
                        y_val,
                        pred
                    ),

                "teacher_f1":
                    teacher_f1,

                "teacher_auc":
                    teacher_auc,

                "teacher_mcc":
                    teacher_mcc
            })

            print(
                f"{config['name']} | "
                f"F1={current_f1:.4f}"
            )

    return pd.DataFrame(records)

In [ ]:
  medium_results = (
    evaluate_plain_compact_students(
        X=X_dev,
        y=y_dev,
        groups=groups_dev,
        configs=medium_student_configs,
        n_splits=5,
        random_state=42
    )
)

In [ ]:
medium_summary = (
    medium_results
    .groupby(
        [
            "student",
            "n_estimators",
            "max_depth"
        ]
    )
    .agg(

        mean_f1=(
            "student_f1",
            "mean"
        ),

        std_f1=(
            "student_f1",
            "std"
        ),

        mean_auc=(
            "student_auc",
            "mean"
        ),

        std_auc=(
            "student_auc",
            "std"
        ),

        mean_mcc=(
            "student_mcc",
            "mean"
        ),

        mean_precision=(
            "student_precision",
            "mean"
        ),

        mean_recall=(
            "student_recall",
            "mean"
        ),

        teacher_f1=(
            "teacher_f1",
            "mean"
        ),

        teacher_auc=(
            "teacher_auc",
            "mean"
        )

    )
    .reset_index()
)

In [ ]:
medium_summary[
    "f1_gap_to_teacher"
] = (
    medium_summary[
        "mean_f1"
    ]
    -
    medium_summary[
        "teacher_f1"
    ]
)

In [ ]:
medium_summary[
    "complexity_proxy"
] = (
    medium_summary[
        "n_estimators"
    ]
    *
    (
        2
        **
        medium_summary[
            "max_depth"
        ]
    )
)

In [ ]:
medium_summary.sort_values(
    "mean_f1",
    ascending=False
)

In [ ]:
teacher_medium_f1 = (
    medium_summary[
        "teacher_f1"
    ].mean()
)

F1_TOLERANCE = 0.005

print(
    "Teacher mean F1:",
    teacher_medium_f1
)

print(
    "Minimum acceptable student F1:",
    teacher_medium_f1
    -
    F1_TOLERANCE
)

In [ ]:
acceptable_medium = (
    medium_summary[
        medium_summary[
            "mean_f1"
        ]
        >=
        teacher_medium_f1
        -
        F1_TOLERANCE
    ]
    .copy()
)

acceptable_medium.sort_values(
    by=[
        "complexity_proxy",
        "mean_f1"
    ],
    ascending=[
        True,
        False
    ]
)

In [ ]:
if len(acceptable_medium) > 0:

    selected_medium_row = (
        acceptable_medium
        .sort_values(
            by=[
                "complexity_proxy",
                "mean_f1"
            ],
            ascending=[
                True,
                False
            ]
        )
        .iloc[0]
    )

    medium_status = "ACCEPTABLE"

else:

    selected_medium_row = (
        medium_summary
        .sort_values(
            "mean_f1",
            ascending=False
        )
        .iloc[0]
    )

    medium_status = (
        "NO MODEL WITHIN TOLERANCE"
    )

In [ ]:
print(
    "Status:",
    medium_status
)

print()

print(
    "Selected candidate:"
)

print(
    selected_medium_row
)

In [ ]:
plt.figure(
    figsize=(8, 6)
)

plt.scatter(
    medium_summary[
        "complexity_proxy"
    ],
    medium_summary[
        "mean_f1"
    ],
    s=80
)

for _, row in (
    medium_summary
    .iterrows()
):

    plt.annotate(
        row["student"],
        (
            row[
                "complexity_proxy"
            ],
            row[
                "mean_f1"
            ]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.axhline(
    teacher_medium_f1,
    linestyle="--",
    label="Teacher"
)

plt.axhline(
    teacher_medium_f1
    -
    F1_TOLERANCE,
    linestyle=":",
    label="Acceptance threshold"
)

plt.xlabel(
    "Complexity Proxy "
    "(Trees × 2^Depth)"
)

plt.ylabel(
    "Mean Grouped-CV F1"
)

plt.title(
    "Performance–Complexity Trade-off "
    "for Medium Compact XGBoost Models"
)

plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(8, 5)
)

for depth in sorted(
    medium_summary[
        "max_depth"
    ].unique()
):

    sub = (
        medium_summary[
            medium_summary[
                "max_depth"
            ]
            ==
            depth
        ]
        .sort_values(
            "n_estimators"
        )
    )

    plt.plot(
        sub[
            "n_estimators"
        ],
        sub[
            "mean_f1"
        ],
        marker="o",
        linewidth=2,
        label=f"Depth {depth}"
    )

plt.axhline(
    teacher_medium_f1,
    linestyle="--",
    label="Teacher"
)

plt.axhline(
    teacher_medium_f1
    -
    F1_TOLERANCE,
    linestyle=":",
    label="Acceptance threshold"
)

plt.xlabel(
    "Number of Trees"
)

plt.ylabel(
    "Mean Grouped-CV F1"
)

plt.title(
    "Effect of Student Capacity on Predictive Performance"
)

plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
selected_medium_name = (
    selected_medium_row[
        "student"
    ]
)

selected_medium_cv = (
    medium_results[
        medium_results[
            "student"
        ]
        ==
        selected_medium_name
    ]
    .copy()
)

selected_medium_cv[
    [
        "fold",
        "student_f1",
        "teacher_f1",
        "student_auc",
        "teacher_auc",
        "student_mcc",
        "teacher_mcc"
    ]
]

In [ ]:
medium_wins = (
    selected_medium_cv[
        "student_f1"
    ]
    >
    selected_medium_cv[
        "teacher_f1"
    ]
).sum()

medium_ties = (
    selected_medium_cv[
        "student_f1"
    ]
    ==
    selected_medium_cv[
        "teacher_f1"
    ]
).sum()

medium_losses = (
    selected_medium_cv[
        "student_f1"
    ]
    <
    selected_medium_cv[
        "teacher_f1"
    ]
).sum()

print(
    "Student wins:",
    medium_wins
)

print(
    "Ties:",
    medium_ties
)

print(
    "Losses:",
    medium_losses
)

In [ ]:
old_student_record = pd.DataFrame([
    {
        "student":
            "XGB_100_D2",

        "n_estimators":
            100,

        "max_depth":
            2,

        "mean_f1":
            0.870788,

        "complexity_proxy":
            100 * (2 ** 2)
    }
])

In [ ]:
capacity_progression = pd.concat(
    [
        old_student_record,
        medium_summary[
            [
                "student",
                "n_estimators",
                "max_depth",
                "mean_f1",
                "complexity_proxy"
            ]
        ]
    ],
    ignore_index=True
)

capacity_progression.sort_values(
    "complexity_proxy"
)

In [ ]:
if medium_status == "ACCEPTABLE":

    final_compact_config = {

        "name":
            selected_medium_row[
                "student"
            ],

        "n_estimators":
            int(
                selected_medium_row[
                    "n_estimators"
                ]
            ),

        "max_depth":
            int(
                selected_medium_row[
                    "max_depth"
                ]
            )
    }

    final_compact_student = (
        make_plain_student_classifier(
            final_compact_config
        )
    )

    final_compact_student.fit(
        X_dev[
            lexical_reduced
        ],
        y_dev
    )

    print(
        "Final compact student trained."
    )

    print(
        final_compact_config
    )

else:

    print(
        "No final compact student frozen yet."
    )

In [ ]:
best_compact_config = {
    "name": "XGB_300_D3",
    "n_estimators": 300,
    "max_depth": 3
}

best_compact_candidate = (
    make_plain_student_classifier(
        best_compact_config
    )
)

best_compact_candidate.fit(
    X_dev[lexical_reduced],
    y_dev
)

print(
    "Best compact candidate trained:"
)

print(
    best_compact_config
)

In [ ]:
full_teacher_phase12 = (
    make_teacher_model()
)

full_teacher_phase12.fit(
    X_dev[lexical_reduced],
    y_dev
)

In [ ]:
import joblib
import os

joblib.dump(
    full_teacher_phase12,
    "phase12_teacher_xgb.pkl"
)

joblib.dump(
    best_compact_candidate,
    "phase12_compact_xgb300_d3.pkl"
)

In [ ]:
teacher_size_kb = (
    os.path.getsize(
        "phase12_teacher_xgb.pkl"
    )
    / 1024
)

compact_size_kb = (
    os.path.getsize(
        "phase12_compact_xgb300_d3.pkl"
    )
    / 1024
)

size_reduction = (
    1
    -
    compact_size_kb
    /
    teacher_size_kb
)

print(
    "Teacher size (KB):",
    teacher_size_kb
)

print(
    "Compact model size (KB):",
    compact_size_kb
)

print(
    "Size reduction (%):",
    100 * size_reduction
)

In [ ]:
import time
import numpy as np

def measure_latency(
    model,
    X,
    repeats=200
):

    # warm-up
    for _ in range(10):
        model.predict_proba(X)

    times = []

    for _ in range(repeats):

        start = time.perf_counter()

        model.predict_proba(X)

        end = time.perf_counter()

        times.append(
            end - start
        )

    times = np.asarray(times)

    return {
        "mean_total_ms":
            times.mean() * 1000,

        "std_total_ms":
            times.std() * 1000,

        "mean_ms_per_sample":
            (
                times.mean()
                * 1000
                /
                len(X)
            )
    }

In [ ]:
rng = np.random.default_rng(42)

latency_idx = rng.choice(
    len(X_dev),
    size=min(
        1000,
        len(X_dev)
    ),
    replace=False
)

X_latency = (
    X_dev.iloc[
        latency_idx
    ][
        lexical_reduced
    ]
)

In [ ]:
teacher_latency = (
    measure_latency(
        full_teacher_phase12,
        X_latency,
        repeats=200
    )
)

teacher_latency

In [ ]:
compact_latency = (
    measure_latency(
        best_compact_candidate,
        X_latency,
        repeats=200
    )
)

compact_latency

In [ ]:
latency_reduction = (
    1
    -
    compact_latency[
        "mean_ms_per_sample"
    ]
    /
    teacher_latency[
        "mean_ms_per_sample"
    ]
)

print(
    "Latency reduction (%):",
    100 * latency_reduction
)

In [ ]:
compact_tradeoff = pd.DataFrame([
    {
        "model":
            "Teacher_XGB",

        "cv_f1":
            0.888811,

        "cv_auc":
            0.955430,

        "complexity_proxy":
            6400,

        "model_size_kb":
            teacher_size_kb,

        "latency_ms_per_sample":
            teacher_latency[
                "mean_ms_per_sample"
            ]
    },

    {
        "model":
            "Compact_XGB_300_D3",

        "cv_f1":
            0.882395,

        "cv_auc":
            0.953352,

        "complexity_proxy":
            2400,

        "model_size_kb":
            compact_size_kb,

        "latency_ms_per_sample":
            compact_latency[
                "mean_ms_per_sample"
            ]
    }
])

compact_tradeoff

In [ ]:
f1_retention = (
    0.882395
    /
    0.888811
)

print(
    "F1 retention (%):",
    100 * f1_retention
)

print(
    "Complexity-proxy reduction (%):",
    100 * (
        1 - 2400 / 6400
    )
)

print(
    "Model-size reduction (%):",
    100 * size_reduction
)

print(
    "Latency reduction (%):",
    100 * latency_reduction
)

## Phase 12C — Feature-Level Lightweight Design

The previous model-level compression experiment showed that a compact
XGB-300-D3 configuration retained 99.28% of the teacher F1 while reducing
model size by 42.5%. However, its measured inference-latency reduction was
relatively modest.

We therefore investigate feature-level compression.

Rather than reducing model capacity, this experiment preserves the full
teacher architecture and progressively removes low-utility lexical features.
To avoid feature-selection leakage, feature importance is estimated only
within the training portion of each outer grouped cross-validation fold.

Candidate feature budgets are 14, 12, 10, 8, and 6 features.

The objective is to determine whether a smaller lexical representation can
preserve predictive performance while reducing feature-extraction and model
complexity.

In [ ]:
# ============================================================
# Phase 12C — Feature-Level Lightweight Design
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score
)

RANDOM_STATE = 42
N_JOBS = 1

In [ ]:
print(
    "Current lexical feature count:",
    len(lexical_reduced)
)

print()

for i, feature in enumerate(
    lexical_reduced,
    start=1
):
    print(i, feature)

In [ ]:
feature_budgets = [
    len(lexical_reduced),
    12,
    10,
    8,
    6
]

# 去掉可能超过实际 feature 数的情况
feature_budgets = sorted(
    set(
        k
        for k in feature_budgets
        if k <= len(lexical_reduced)
    ),
    reverse=True
)

print(
    "Feature budgets:",
    feature_budgets
)

In [ ]:
def get_inner_grouped_feature_ranking(
    X,
    y,
    groups,
    feature_cols,
    n_splits=4,
    n_repeats=5,
    random_state=42
):

    inner_cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    importance_records = []

    for inner_fold, (
        tr_idx,
        va_idx
    ) in enumerate(
        inner_cv.split(
            X[feature_cols],
            y,
            groups=groups
        ),
        start=1
    ):

        model = make_teacher_model()

        model.fit(
            X.iloc[tr_idx][feature_cols],
            y.iloc[tr_idx]
        )

        perm = permutation_importance(
            model,
            X.iloc[va_idx][feature_cols],
            y.iloc[va_idx],
            scoring="f1",
            n_repeats=n_repeats,
            random_state=(
                random_state
                +
                inner_fold
            ),
            n_jobs=1
        )

        for feature, importance in zip(
            feature_cols,
            perm.importances_mean
        ):

            importance_records.append({
                "inner_fold":
                    inner_fold,

                "feature":
                    feature,

                "importance":
                    importance
            })

    importance_df = pd.DataFrame(
        importance_records
    )

    ranking_df = (
        importance_df
        .groupby("feature")
        .agg(
            mean_importance=(
                "importance",
                "mean"
            ),

            std_importance=(
                "importance",
                "std"
            )
        )
        .reset_index()
        .sort_values(
            "mean_importance",
            ascending=False
        )
        .reset_index(drop=True)
    )

    ranking_df[
        "rank"
    ] = (
        np.arange(
            1,
            len(ranking_df) + 1
        )
    )

    return ranking_df

In [ ]:
def evaluate_feature_budgets_nested_cv(
    X,
    y,
    groups,
    feature_cols,
    feature_budgets,
    outer_splits=5,
    inner_splits=4,
    permutation_repeats=5,
    random_state=42
):

    outer_cv = StratifiedGroupKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=random_state
    )

    records = []
    ranking_records = []

    for outer_fold, (
        tr_idx,
        va_idx
    ) in enumerate(
        outer_cv.split(
            X[feature_cols],
            y,
            groups=groups
        ),
        start=1
    ):

        print(
            "\n"
            + "=" * 75
        )

        print(
            f"Outer Fold {outer_fold}"
        )

        print(
            "=" * 75
        )

        X_train = (
            X.iloc[tr_idx]
            .reset_index(drop=True)
            .copy()
        )

        X_val = (
            X.iloc[va_idx]
            .reset_index(drop=True)
            .copy()
        )

        y_train = (
            y.iloc[tr_idx]
            .reset_index(drop=True)
            .copy()
        )

        y_val = (
            y.iloc[va_idx]
            .reset_index(drop=True)
            .copy()
        )

        groups_train = (
            groups.iloc[tr_idx]
            .reset_index(drop=True)
            .copy()
        )

        # ====================================================
        # Inner-CV feature ranking
        # ====================================================

        ranking_df = (
            get_inner_grouped_feature_ranking(
                X=X_train,
                y=y_train,
                groups=groups_train,
                feature_cols=feature_cols,
                n_splits=inner_splits,
                n_repeats=
                    permutation_repeats,
                random_state=(
                    random_state
                    +
                    outer_fold
                )
            )
        )

        ranking_df[
            "outer_fold"
        ] = outer_fold

        ranking_records.append(
            ranking_df
        )

        ranked_features = (
            ranking_df[
                "feature"
            ].tolist()
        )

        print(
            "Top features:"
        )

        print(
            ranked_features
        )

        # ====================================================
        # Evaluate each feature budget
        # ====================================================

        for k in feature_budgets:

            selected_features = (
                ranked_features[:k]
            )

            model = (
                make_teacher_model()
            )

            model.fit(
                X_train[
                    selected_features
                ],
                y_train
            )

            prob = (
                model.predict_proba(
                    X_val[
                        selected_features
                    ]
                )[:, 1]
            )

            pred = (
                prob >= 0.5
            ).astype(int)

            current_f1 = (
                f1_score(
                    y_val,
                    pred
                )
            )

            records.append({

                "outer_fold":
                    outer_fold,

                "feature_count":
                    k,

                "selected_features":
                    "|".join(
                        selected_features
                    ),

                "f1":
                    current_f1,

                "precision":
                    precision_score(
                        y_val,
                        pred,
                        zero_division=0
                    ),

                "recall":
                    recall_score(
                        y_val,
                        pred,
                        zero_division=0
                    ),

                "auc":
                    roc_auc_score(
                        y_val,
                        prob
                    ),

                "mcc":
                    matthews_corrcoef(
                        y_val,
                        pred
                    ),

                "accuracy":
                    accuracy_score(
                        y_val,
                        pred
                    )
            })

            print(
                f"Top-{k:2d} | "
                f"F1={current_f1:.4f}"
            )

    results_df = pd.DataFrame(
        records
    )

    rankings_df = pd.concat(
        ranking_records,
        ignore_index=True
    )

    return (
        results_df,
        rankings_df
    )

In [ ]:
(
    feature_budget_results,
    nested_feature_rankings
) = (
    evaluate_feature_budgets_nested_cv(
        X=X_dev,
        y=y_dev,
        groups=groups_dev,
        feature_cols=
            lexical_reduced,
        feature_budgets=
            feature_budgets,
        outer_splits=5,
        inner_splits=4,
        permutation_repeats=5,
        random_state=42
    )
)

In [ ]:
feature_budget_summary = (
    feature_budget_results
    .groupby(
        "feature_count"
    )
    .agg(

        mean_f1=(
            "f1",
            "mean"
        ),

        std_f1=(
            "f1",
            "std"
        ),

        mean_auc=(
            "auc",
            "mean"
        ),

        std_auc=(
            "auc",
            "std"
        ),

        mean_mcc=(
            "mcc",
            "mean"
        ),

        mean_precision=(
            "precision",
            "mean"
        ),

        mean_recall=(
            "recall",
            "mean"
        ),

        mean_accuracy=(
            "accuracy",
            "mean"
        )

    )
    .reset_index()
    .sort_values(
        "feature_count",
        ascending=False
    )
)

feature_budget_summary

In [ ]:
full_feature_count = (
    max(feature_budgets)
)

full_feature_row = (
    feature_budget_summary[
        feature_budget_summary[
            "feature_count"
        ]
        ==
        full_feature_count
    ]
    .iloc[0]
)

full_feature_f1 = (
    full_feature_row[
        "mean_f1"
    ]
)

print(
    "Full lexical feature count:",
    full_feature_count
)

print(
    "Full-feature F1:",
    full_feature_f1
)

In [ ]:
feature_budget_summary[
    "f1_gap_vs_full"
] = (
    feature_budget_summary[
        "mean_f1"
    ]
    -
    full_feature_f1
)

feature_budget_summary[
    "feature_reduction_rate"
] = (
    1
    -
    feature_budget_summary[
        "feature_count"
    ]
    /
    full_feature_count
)

In [ ]:
feature_budget_summary[
    [
        "feature_count",
        "mean_f1",
        "std_f1",
        "f1_gap_vs_full",
        "mean_auc",
        "mean_mcc",
        "feature_reduction_rate"
    ]
]

In [ ]:
FEATURE_F1_TOLERANCE = 0.005

acceptable_feature_sets = (
    feature_budget_summary[
        feature_budget_summary[
            "mean_f1"
        ]
        >=
        full_feature_f1
        -
        FEATURE_F1_TOLERANCE
    ]
    .copy()
)

acceptable_feature_sets

In [ ]:
if len(
    acceptable_feature_sets
) > 0:

    selected_feature_budget_row = (
        acceptable_feature_sets
        .sort_values(
            by=[
                "feature_count",
                "mean_f1"
            ],
            ascending=[
                True,
                False
            ]
        )
        .iloc[0]
    )

    feature_budget_status = (
        "ACCEPTABLE"
    )

else:

    selected_feature_budget_row = (
        feature_budget_summary
        .sort_values(
            "mean_f1",
            ascending=False
        )
        .iloc[0]
    )

    feature_budget_status = (
        "NO REDUCED SET WITHIN TOLERANCE"
    )

In [ ]:
plot_budget = (
    feature_budget_summary
    .sort_values(
        "feature_count"
    )
)

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    plot_budget[
        "feature_count"
    ],
    plot_budget[
        "mean_f1"
    ],
    marker="o",
    linewidth=2
)

plt.axhline(
    full_feature_f1,
    linestyle="--",
    label="Full lexical model"
)

plt.axhline(
    full_feature_f1
    -
    FEATURE_F1_TOLERANCE,
    linestyle=":",
    label="Acceptance threshold"
)

for _, row in (
    plot_budget.iterrows()
):

    plt.annotate(
        f"{int(row['feature_count'])}",
        (
            row[
                "feature_count"
            ],
            row[
                "mean_f1"
            ]
        ),

        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel(
    "Number of Lexical Features"
)

plt.ylabel(
    "Mean Grouped-CV F1"
)

plt.title(
    "Feature-Level Performance–Complexity Trade-off"
)

plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
feature_ranking_stability = (
    nested_feature_rankings
    .groupby(
        "feature"
    )
    .agg(

        mean_rank=(
            "rank",
            "mean"
        ),

        std_rank=(
            "rank",
            "std"
        ),

        mean_importance=(
            "mean_importance",
            "mean"
        ),

        std_importance_across_outer=(
            "mean_importance",
            "std"
        )

    )
    .reset_index()
    .sort_values(
        "mean_rank"
    )
)

feature_ranking_stability

In [ ]:
selected_k = int(
    selected_feature_budget_row[
        "feature_count"
    ]
)

selection_frequency_records = []

for outer_fold in sorted(
    nested_feature_rankings[
        "outer_fold"
    ].unique()
):

    fold_rank = (
        nested_feature_rankings[
            nested_feature_rankings[
                "outer_fold"
            ]
            ==
            outer_fold
        ]
        .sort_values(
            "rank"
        )
    )

    top_k_features = set(
        fold_rank
        .head(
            selected_k
        )[
            "feature"
        ]
    )

    for feature in lexical_reduced:

        selection_frequency_records.append({
            "outer_fold":
                outer_fold,

            "feature":
                feature,

            "selected":
                int(
                    feature
                    in
                    top_k_features
                )
        })

In [ ]:
feature_selection_frequency = (
    pd.DataFrame(
        selection_frequency_records
    )
    .groupby(
        "feature"
    )[
        "selected"
    ]
    .agg(
        [
            "sum",
            "mean"
        ]
    )
    .reset_index()
    .rename(
        columns={
            "sum":
                "selected_folds",

            "mean":
                "selection_frequency"
        }
    )
    .sort_values(
        [
            "selected_folds",
            "feature"
        ],
        ascending=[
            False,
            True
        ]
    )
)

feature_selection_frequency

In [ ]:
final_dev_ranking = (
    get_inner_grouped_feature_ranking(
        X=X_dev.reset_index(
            drop=True
        ),
        y=y_dev.reset_index(
            drop=True
        ),
        groups=groups_dev.reset_index(
            drop=True
        ),
        feature_cols=
            lexical_reduced,
        n_splits=5,
        n_repeats=10,
        random_state=42
    )
)

final_dev_ranking

In [ ]:
final_reduced_lexical_features = (
    final_dev_ranking
    .head(
        selected_k
    )[
        "feature"
    ]
    .tolist()
)

print(
    "Selected feature count:",
    len(
        final_reduced_lexical_features
    )
)

print()

print(
    "Selected features:"
)

for i, feature in enumerate(
    final_reduced_lexical_features,
    start=1
):
    print(
        i,
        feature
    )

In [ ]:
if (
    feature_budget_status
    ==
    "ACCEPTABLE"
):

    reduced_feature_teacher = (
        make_teacher_model()
    )

    reduced_feature_teacher.fit(
        X_dev[
            final_reduced_lexical_features
        ],
        y_dev
    )

    print(
        "Reduced-feature full XGB trained."
    )

else:

    print(
        "Reduced feature set has not "
        "passed the predefined criterion."
    )

In [ ]:
import joblib
import os

if (
    feature_budget_status
    ==
    "ACCEPTABLE"
):

    joblib.dump(
        reduced_feature_teacher,
        "phase12C_reduced_feature_xgb.pkl"
    )

    reduced_feature_model_size_kb = (
        os.path.getsize(
            "phase12C_reduced_feature_xgb.pkl"
        )
        /
        1024
    )

    print(
        "Reduced-feature model size (KB):",
        reduced_feature_model_size_kb
    )

In [ ]:
# Check reduced-feature model prediction

reduced_prob = (
    reduced_feature_teacher
    .predict_proba(
        X_dev[
            final_reduced_lexical_features
        ]
    )[:, 1]
)

reduced_pred = (
    reduced_prob >= 0.5
).astype(int)

print("Prediction completed successfully.")
print("Number of predictions:", len(reduced_pred))

In [ ]:
# ============================================================
# Final selected reduced lexical feature set
# ============================================================

print(
    "Selected feature budget:",
    selected_k
)

print(
    "Original lexical feature count:",
    len(lexical_reduced)
)

print(
    "Feature reduction:",
    len(lexical_reduced) - selected_k
)

print(
    "Feature reduction rate (%):",
    100 * (
        1
        -
        selected_k
        /
        len(lexical_reduced)
    )
)

print("\nFinal selected features:")

for i, feature in enumerate(
    final_reduced_lexical_features,
    start=1
):
    print(
        f"{i:2d}. {feature}"
    )

In [ ]:
final_dev_ranking[
    [
        "rank",
        "feature",
        "mean_importance",
        "std_importance"
    ]
]

In [ ]:
feature_budget_summary[
    [
        "feature_count",
        "mean_f1",
        "std_f1",
        "f1_gap_vs_full",
        "mean_auc",
        "mean_mcc",
        "feature_reduction_rate"
    ]
].sort_values(
    "feature_count",
    ascending=False
)

In [ ]:
print(
    "Full-feature F1:",
    full_feature_f1
)

print(
    "Selected reduced-feature F1:",
    selected_feature_budget_row[
        "mean_f1"
    ]
)

print(
    "F1 gap:",
    selected_feature_budget_row[
        "f1_gap_vs_full"
    ]
)

print(
    "Selected feature count:",
    selected_k
)

print(
    "Status:",
    feature_budget_status
)

In [ ]:
selected_feature_budget_cv = (
    feature_budget_results[
        feature_budget_results[
            "feature_count"
        ]
        ==
        selected_k
    ]
    .copy()
)

selected_feature_budget_cv[
    [
        "outer_fold",
        "feature_count",
        "f1",
        "precision",
        "recall",
        "auc",
        "mcc"
    ]
]

In [ ]:
full_feature_cv = (
    feature_budget_results[
        feature_budget_results[
            "feature_count"
        ]
        ==
        full_feature_count
    ]
    .copy()
)

In [ ]:
feature_fold_comparison = (
    full_feature_cv[
        [
            "outer_fold",
            "f1",
            "auc",
            "mcc"
        ]
    ]
    .rename(
        columns={
            "f1": "full_f1",
            "auc": "full_auc",
            "mcc": "full_mcc"
        }
    )
    .merge(
        selected_feature_budget_cv[
            [
                "outer_fold",
                "f1",
                "auc",
                "mcc"
            ]
        ].rename(
            columns={
                "f1": "reduced_f1",
                "auc": "reduced_auc",
                "mcc": "reduced_mcc"
            }
        ),
        on="outer_fold"
    )
)

In [ ]:
feature_fold_comparison[
    "delta_f1"
] = (
    feature_fold_comparison[
        "reduced_f1"
    ]
    -
    feature_fold_comparison[
        "full_f1"
    ]
)

feature_fold_comparison[
    "delta_auc"
] = (
    feature_fold_comparison[
        "reduced_auc"
    ]
    -
    feature_fold_comparison[
        "full_auc"
    ]
)

feature_fold_comparison

In [ ]:
feature_wins = (
    feature_fold_comparison[
        "delta_f1"
    ] > 0
).sum()

feature_ties = (
    feature_fold_comparison[
        "delta_f1"
    ] == 0
).sum()

feature_losses = (
    feature_fold_comparison[
        "delta_f1"
    ] < 0
).sum()

print(
    "Reduced-feature wins:",
    feature_wins
)

print(
    "Ties:",
    feature_ties
)

print(
    "Losses:",
    feature_losses
)

In [ ]:
plot_budget = (
    feature_budget_summary
    .sort_values(
        "feature_count"
    )
)

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    plot_budget[
        "feature_count"
    ],
    plot_budget[
        "mean_f1"
    ],
    marker="o",
    linewidth=2
)

plt.axhline(
    full_feature_f1,
    linestyle="--",
    label="Full lexical features"
)

plt.axhline(
    full_feature_f1
    -
    FEATURE_F1_TOLERANCE,
    linestyle=":",
    label="Acceptance threshold"
)

for _, row in plot_budget.iterrows():

    plt.annotate(
        f"{int(row['feature_count'])}",
        (
            row["feature_count"],
            row["mean_f1"]
        ),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel(
    "Number of Lexical Features"
)

plt.ylabel(
    "Mean Grouped-CV F1"
)

plt.title(
    "Feature-Level Performance–Complexity Trade-off"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
ranking_plot = (
    final_dev_ranking
    .sort_values(
        "mean_importance",
        ascending=True
    )
)

plt.figure(
    figsize=(8, 6)
)

plt.barh(
    ranking_plot[
        "feature"
    ],
    ranking_plot[
        "mean_importance"
    ]
)

plt.xlabel(
    "Mean Grouped Permutation Importance (F1 decrease)"
)

plt.ylabel(
    "Lexical Feature"
)

plt.title(
    "Development-Only Lexical Feature Ranking"
)

plt.tight_layout()

plt.show()

In [ ]:
import joblib
import json

joblib.dump(
    reduced_feature_teacher,
    "phase12C_reduced_feature_teacher.pkl"
)

with open(
    "phase12C_reduced_feature_list.json",
    "w"
) as f:

    json.dump(
        {
            "feature_count":
                len(
                    final_reduced_lexical_features
                ),

            "features":
                final_reduced_lexical_features
        },
        f,
        indent=4
    )

print(
    "Reduced-feature model and feature list saved."
)

In [ ]:
feature_budget_results.to_csv(
    "phase12C_feature_budget_fold_results.csv",
    index=False
)

feature_budget_summary.to_csv(
    "phase12C_feature_budget_summary.csv",
    index=False
)

nested_feature_rankings.to_csv(
    "phase12C_nested_feature_rankings.csv",
    index=False
)

feature_ranking_stability.to_csv(
    "phase12C_feature_ranking_stability.csv",
    index=False
)

feature_selection_frequency.to_csv(
    "phase12C_feature_selection_frequency.csv",
    index=False
)

final_dev_ranking.to_csv(
    "phase12C_final_development_feature_ranking.csv",
    index=False
)

feature_fold_comparison.to_csv(
    "phase12C_full_vs_reduced_fold_comparison.csv",
    index=False
)

In [ ]:
# ============================================================
# Phase 12D — Joint Feature + Model Compression
# ============================================================

joint_configs = [

    {
        "name":
            "FullFeatures_Teacher",

        "features":
            lexical_reduced,

        "model_type":
            "teacher"
    },

    {
        "name":
            "FullFeatures_Compact",

        "features":
            lexical_reduced,

        "model_type":
            "compact"
    },

    {
        "name":
            "ReducedFeatures_Teacher",

        "features":
            final_reduced_lexical_features,

        "model_type":
            "teacher"
    },

    {
        "name":
            "ReducedFeatures_Compact",

        "features":
            final_reduced_lexical_features,

        "model_type":
            "compact"
    }
]

In [ ]:
def make_compact_xgb():

    return XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=1.0,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    )

In [ ]:
def evaluate_joint_lightweight_configs(
    X,
    y,
    groups,
    configs,
    n_splits=5,
    random_state=42
):

    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    records = []

    for fold, (
        tr_idx,
        va_idx
    ) in enumerate(

        cv.split(
            X[lexical_reduced],
            y,
            groups=groups
        ),

        start=1
    ):

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"Fold {fold}"
        )

        print(
            "=" * 70
        )

        for config in configs:

            feature_set = (
                config[
                    "features"
                ]
            )

            if (
                config[
                    "model_type"
                ]
                ==
                "teacher"
            ):

                model = (
                    make_teacher_model()
                )

            else:

                model = (
                    make_compact_xgb()
                )

            model.fit(
                X.iloc[
                    tr_idx
                ][
                    feature_set
                ],

                y.iloc[
                    tr_idx
                ]
            )

            prob = (
                model.predict_proba(
                    X.iloc[
                        va_idx
                    ][
                        feature_set
                    ]
                )[:, 1]
            )

            pred = (
                prob >= 0.5
            ).astype(int)

            records.append({

                "fold":
                    fold,

                "configuration":
                    config[
                        "name"
                    ],

                "feature_count":
                    len(
                        feature_set
                    ),

                "model_type":
                    config[
                        "model_type"
                    ],

                "f1":
                    f1_score(
                        y.iloc[
                            va_idx
                        ],
                        pred
                    ),

                "precision":
                    precision_score(
                        y.iloc[
                            va_idx
                        ],
                        pred,
                        zero_division=0
                    ),

                "recall":
                    recall_score(
                        y.iloc[
                            va_idx
                        ],
                        pred,
                        zero_division=0
                    ),

                "auc":
                    roc_auc_score(
                        y.iloc[
                            va_idx
                        ],
                        prob
                    ),

                "mcc":
                    matthews_corrcoef(
                        y.iloc[
                            va_idx
                        ],
                        pred
                    ),

                "accuracy":
                    accuracy_score(
                        y.iloc[
                            va_idx
                        ],
                        pred
                    )
            })

            print(
                f"{config['name']} | "
                f"F1="
                f"{f1_score(y.iloc[va_idx], pred):.4f}"
            )

    return pd.DataFrame(
        records
    )

In [ ]:
joint_lightweight_results = (
    evaluate_joint_lightweight_configs(
        X=X_dev,
        y=y_dev,
        groups=groups_dev,
        configs=joint_configs,
        n_splits=5,
        random_state=42
    )
)

In [ ]:
joint_lightweight_summary = (
    joint_lightweight_results
    .groupby(
        [
            "configuration",
            "feature_count",
            "model_type"
        ]
    )
    .agg(

        mean_f1=(
            "f1",
            "mean"
        ),

        std_f1=(
            "f1",
            "std"
        ),

        mean_auc=(
            "auc",
            "mean"
        ),

        mean_mcc=(
            "mcc",
            "mean"
        ),

        mean_precision=(
            "precision",
            "mean"
        ),

        mean_recall=(
            "recall",
            "mean"
        )

    )
    .reset_index()
)

joint_lightweight_summary.sort_values(
    "mean_f1",
    ascending=False
)

In [ ]:
print("Final 8 features:")

for i, feature in enumerate(
    final_reduced_lexical_features,
    start=1
):
    print(i, feature)

In [ ]:
# ============================================================
# Phase 12E — Feature Extraction Efficiency Evaluation
# ============================================================

dev_urls = (
    df_all_features
    .iloc[train_idx]
    ["url"]
    .reset_index(drop=True)
)

print("Development URLs:", len(dev_urls))
print(dev_urls.head())

In [ ]:
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import time

In [ ]:
SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]

In [ ]:
def safe_parse_url(url):

    url = str(url).strip()

    # Parse URLs without scheme safely
    if "://" not in url:
        parsed = urlparse("//" + url)
    else:
        parsed = urlparse(url)

    hostname = (
        parsed.hostname
        if parsed.hostname is not None
        else ""
    )

    hostname = hostname.lower()

    # Match original preprocessing
    if hostname.startswith("www."):
        hostname = hostname[4:]

    path = (
        parsed.path
        if parsed.path is not None
        else ""
    )

    query = (
        parsed.query
        if parsed.query is not None
        else ""
    )

    fragment = (
        parsed.fragment
        if parsed.fragment is not None
        else ""
    )

    return (
        url,
        parsed,
        hostname,
        path,
        query,
        fragment
    )

In [ ]:
def extract_full_lexical_14(url):

    (
        url,
        parsed,
        hostname,
        path,
        query,
        fragment
    ) = safe_parse_url(url)

    url_lower = url.lower()

    # ------------------------------------------------
    # Length features
    # ------------------------------------------------

    url_length = len(url)

    hostname_length = len(
        hostname
    )

    path_length = len(
        path
    )

    query_length = len(
        query
    )

    # ------------------------------------------------
    # Character-count features
    # ------------------------------------------------

    num_dots = url.count(".")

    num_hyphens = url.count("-")

    num_digits = sum(
        char.isdigit()
        for char in url
    )

    num_special_chars = sum(
        not char.isalnum()
        for char in url
    )

    special_char_ratio = (
        num_special_chars
        /
        url_length
        if url_length > 0
        else 0.0
    )

    # ------------------------------------------------
    # Subdomain count
    # ------------------------------------------------

    hostname_parts = [
        p
        for p in hostname.split(".")
        if p
    ]

    num_subdomains = max(
        len(hostname_parts) - 2,
        0
    )

    # ------------------------------------------------
    # Binary indicators
    # ------------------------------------------------

    has_fragment = int(
        len(fragment) > 0
    )

    contains_at = int(
        "@" in url
    )

    try:
        contains_port = int(
            parsed.port is not None
        )

    except ValueError:
        contains_port = 0

    # ------------------------------------------------
    # Suspicious keyword count
    # ------------------------------------------------

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    return {

        "url_length":
            url_length,

        "hostname_length":
            hostname_length,

        "path_length":
            path_length,

        "query_length":
            query_length,

        "num_dots":
            num_dots,

        "num_hyphens":
            num_hyphens,

        "num_digits":
            num_digits,

        "num_special_chars":
            num_special_chars,

        "special_char_ratio":
            special_char_ratio,

        "num_subdomains":
            num_subdomains,

        "has_fragment":
            has_fragment,

        "contains_at":
            contains_at,

        "contains_port":
            contains_port,

        "suspicious_keyword_count":
            suspicious_keyword_count
    }

In [ ]:
def extract_reduced_lexical_8(url):

    (
        url,
        parsed,
        hostname,
        path,
        query,
        fragment
    ) = safe_parse_url(url)

    url_lower = url.lower()

    # ------------------------------------------------
    # 1. num_subdomains
    # ------------------------------------------------

    hostname_parts = [
        p
        for p in hostname.split(".")
        if p
    ]

    num_subdomains = max(
        len(hostname_parts) - 2,
        0
    )

    # ------------------------------------------------
    # 2. suspicious_keyword_count
    # ------------------------------------------------

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    # ------------------------------------------------
    # 3. num_digits
    # ------------------------------------------------

    num_digits = sum(
        char.isdigit()
        for char in url
    )

    # ------------------------------------------------
    # 4. path_length
    # ------------------------------------------------

    path_length = len(
        path
    )

    # ------------------------------------------------
    # 5. hostname_length
    # ------------------------------------------------

    hostname_length = len(
        hostname
    )

    # ------------------------------------------------
    # 6. url_length
    # ------------------------------------------------

    url_length = len(
        url
    )

    # ------------------------------------------------
    # 7. num_dots
    # ------------------------------------------------

    num_dots = url.count(
        "."
    )

    # ------------------------------------------------
    # 8. special_char_ratio
    # ------------------------------------------------

    num_special_chars = sum(
        not char.isalnum()
        for char in url
    )

    special_char_ratio = (
        num_special_chars
        /
        url_length
        if url_length > 0
        else 0.0
    )

    return {

        "num_subdomains":
            num_subdomains,

        "suspicious_keyword_count":
            suspicious_keyword_count,

        "num_digits":
            num_digits,

        "path_length":
            path_length,

        "hostname_length":
            hostname_length,

        "url_length":
            url_length,

        "num_dots":
            num_dots,

        "special_char_ratio":
            special_char_ratio
    }

In [ ]:
idx = 100

print("URL:")
print(
    dev_urls.iloc[idx]
)

print()

print(
    "Old num_subdomains:",
    X_dev.iloc[idx][
        "num_subdomains"
    ]
)

print(
    "New num_subdomains:",
    extract_reduced_lexical_8(
        dev_urls.iloc[idx]
    )[
        "num_subdomains"
    ]
)

In [ ]:
consistency_records = []

for idx in range(
    len(dev_urls)
):

    url = (
        dev_urls.iloc[idx]
    )

    extracted = (
        extract_reduced_lexical_8(
            url
        )
    )

    for feature in (
        final_reduced_lexical_features
    ):

        old_value = (
            X_dev.iloc[idx][
                feature
            ]
        )

        new_value = (
            extracted[
                feature
            ]
        )

        difference = (
            float(old_value)
            -
            float(new_value)
        )

        consistency_records.append({

            "sample":
                idx,

            "feature":
                feature,

            "old_value":
                old_value,

            "new_value":
                new_value,

            "difference":
                difference
        })

In [ ]:
full_consistency_df = pd.DataFrame(
    consistency_records
)

In [ ]:
max_abs_difference = (
    full_consistency_df[
        "difference"
    ]
    .abs()
    .max()
)

print(
    "Maximum absolute difference:",
    max_abs_difference
)

In [ ]:
mismatch_df = (
    full_consistency_df[
        full_consistency_df[
            "difference"
        ].abs()
        >
        1e-12
    ]
    .copy()
)

print(
    "Mismatched feature values:",
    len(mismatch_df)
)

print(
    "Affected samples:",
    mismatch_df[
        "sample"
    ].nunique()
)

In [ ]:
mismatch_df[
    "feature"
].value_counts()

In [ ]:
mismatch_feature_summary = (
    mismatch_df
    .groupby("feature")
    .agg(
        mismatch_count=(
            "difference",
            "size"
        ),
        mean_abs_difference=(
            "difference",
            lambda x: x.abs().mean()
        ),
        max_abs_difference=(
            "difference",
            lambda x: x.abs().max()
        )
    )
    .sort_values(
        "mismatch_count",
        ascending=False
    )
)

mismatch_feature_summary

In [ ]:
def extract_reduced_lexical_8_from_row(url, hostname):

    url = str(url).strip()

    hostname = (
        ""
        if pd.isna(hostname)
        else str(hostname).strip().lower()
    )

    # Parse only for path
    if "://" not in url:
        parsed = urlparse("//" + url)
    else:
        parsed = urlparse(url)

    path = (
        parsed.path
        if parsed.path is not None
        else ""
    )

    url_lower = url.lower()

    # 1. num_subdomains
    hostname_parts = [
        p
        for p in hostname.split(".")
        if p
    ]

    num_subdomains = max(
        len(hostname_parts) - 2,
        0
    )

    # 2. suspicious_keyword_count
    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    # 3. num_digits
    num_digits = sum(
        char.isdigit()
        for char in url
    )

    # 4. path_length
    path_length = len(path)

    # 5. hostname_length
    hostname_length = len(hostname)

    # 6. url_length
    url_length = len(url)

    # 7. num_dots
    num_dots = url.count(".")

    # 8. special_char_ratio
    num_special_chars = sum(
        not char.isalnum()
        for char in url
    )

    special_char_ratio = (
        num_special_chars / url_length
        if url_length > 0
        else 0.0
    )

    return {
        "num_subdomains": num_subdomains,
        "suspicious_keyword_count": suspicious_keyword_count,
        "num_digits": num_digits,
        "path_length": path_length,
        "hostname_length": hostname_length,
        "url_length": url_length,
        "num_dots": num_dots,
        "special_char_ratio": special_char_ratio
    }

In [ ]:
dev_meta = (
    df_all_features
    .iloc[train_idx]
    [["url", "hostname"]]
    .reset_index(drop=True)
)

print(dev_meta.head())

In [ ]:
consistency_records = []

for idx in range(len(dev_meta)):

    url = dev_meta.iloc[idx]["url"]
    hostname = dev_meta.iloc[idx]["hostname"]

    extracted = extract_reduced_lexical_8_from_row(
        url,
        hostname
    )

    for feature in final_reduced_lexical_features:

        old_value = X_dev.iloc[idx][feature]
        new_value = extracted[feature]

        consistency_records.append({
            "sample": idx,
            "feature": feature,
            "old_value": old_value,
            "new_value": new_value,
            "difference": (
                float(old_value)
                -
                float(new_value)
            )
        })

full_consistency_df_v2 = pd.DataFrame(
    consistency_records
)

mismatch_df_v2 = (
    full_consistency_df_v2[
        full_consistency_df_v2[
            "difference"
        ].abs() > 1e-12
    ]
    .copy()
)

print(
    "Maximum absolute difference:",
    full_consistency_df_v2[
        "difference"
    ].abs().max()
)

print(
    "Mismatched feature values:",
    len(mismatch_df_v2)
)

print(
    "Affected samples:",
    mismatch_df_v2[
        "sample"
    ].nunique()
)

print()

print(
    mismatch_df_v2[
        "feature"
    ].value_counts()
)

In [ ]:
path_mismatches = (
    mismatch_df_v2[
        mismatch_df_v2[
            "feature"
        ] == "path_length"
    ]
)

path_mismatches

In [ ]:
for idx in (
    path_mismatches[
        "sample"
    ].unique()
):

    row = (
        df_all_features
        .iloc[train_idx]
        .reset_index(drop=True)
        .iloc[idx]
    )

    print("=" * 60)
    print("sample:", idx)
    print("url:", row["url"])
    print("normalized_url:", row["normalized_url"])
    print("stored path_length:", X_dev.iloc[idx]["path_length"])

In [ ]:
# ============================================================
# Phase 12E new — Feature Extraction Efficiency Evaluation
# ============================================================

import re
import time
import numpy as np
import pandas as pd

from urllib.parse import urlparse

In [ ]:
dev_meta = (
    df_all_features
    .iloc[train_idx]
    [["url", "hostname"]]
    .reset_index(drop=True)
    .copy()
)

print("Development samples:", len(dev_meta))

dev_meta.head()

In [ ]:
SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]

In [ ]:
def safe_parse_url(url):

    url = str(url).strip()

    # A real URL scheme must occur at the START
    has_scheme = bool(
        re.match(
            r"^[A-Za-z][A-Za-z0-9+.-]*://",
            url
        )
    )

    if has_scheme:
        parsed = urlparse(url)
    else:
        parsed = urlparse("//" + url)

    parsed_hostname = (
        parsed.hostname
        if parsed.hostname is not None
        else ""
    )

    path = (
        parsed.path
        if parsed.path is not None
        else ""
    )

    query = (
        parsed.query
        if parsed.query is not None
        else ""
    )

    fragment = (
        parsed.fragment
        if parsed.fragment is not None
        else ""
    )

    return (
        url,
        parsed,
        parsed_hostname,
        path,
        query,
        fragment
    )

In [ ]:
def extract_full_lexical_14_from_row(
    url,
    hostname
):

    url = str(url).strip()

    hostname = (
        ""
        if pd.isna(hostname)
        else str(hostname).strip().lower()
    )

    (
        url,
        parsed,
        parsed_hostname,
        path,
        query,
        fragment
    ) = safe_parse_url(url)

    url_lower = url.lower()

    # ------------------------------------------------
    # Length features
    # ------------------------------------------------

    url_length = len(url)

    hostname_length = len(hostname)

    path_length = len(path)

    query_length = len(query)

    # ------------------------------------------------
    # Character-count features
    # ------------------------------------------------

    num_dots = url.count(".")

    num_hyphens = url.count("-")

    num_digits = sum(
        char.isdigit()
        for char in url
    )

    num_special_chars = sum(
        not char.isalnum()
        for char in url
    )

    special_char_ratio = (
        num_special_chars / url_length
        if url_length > 0
        else 0.0
    )

    # ------------------------------------------------
    # Subdomain count
    # ------------------------------------------------

    hostname_parts = [
        p
        for p in hostname.split(".")
        if p
    ]

    num_subdomains = max(
        len(hostname_parts) - 2,
        0
    )

    # ------------------------------------------------
    # Binary indicators
    # ------------------------------------------------

    has_fragment = int(
        len(fragment) > 0
    )

    contains_at = int(
        "@" in url
    )

    try:

        contains_port = int(
            parsed.port is not None
        )

    except ValueError:

        contains_port = 0

    # ------------------------------------------------
    # Suspicious keyword count
    # ------------------------------------------------

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    return {

        "url_length":
            url_length,

        "hostname_length":
            hostname_length,

        "path_length":
            path_length,

        "query_length":
            query_length,

        "num_dots":
            num_dots,

        "num_hyphens":
            num_hyphens,

        "num_digits":
            num_digits,

        "num_special_chars":
            num_special_chars,

        "special_char_ratio":
            special_char_ratio,

        "num_subdomains":
            num_subdomains,

        "has_fragment":
            has_fragment,

        "contains_at":
            contains_at,

        "contains_port":
            contains_port,

        "suspicious_keyword_count":
            suspicious_keyword_count
    }

In [ ]:
consistency_records = []

for idx in range(
    len(dev_meta)
):

    url = (
        dev_meta.iloc[idx][
            "url"
        ]
    )

    hostname = (
        dev_meta.iloc[idx][
            "hostname"
        ]
    )

    extracted = (
        extract_reduced_lexical_8_from_row(
            url,
            hostname
        )
    )

    for feature in (
        final_reduced_lexical_features
    ):

        old_value = (
            X_dev.iloc[idx][
                feature
            ]
        )

        new_value = (
            extracted[
                feature
            ]
        )

        difference = (
            float(old_value)
            -
            float(new_value)
        )

        consistency_records.append({

            "sample":
                idx,

            "feature":
                feature,

            "old_value":
                old_value,

            "new_value":
                new_value,

            "difference":
                difference
        })


final_consistency_df = (
    pd.DataFrame(
        consistency_records
    )
)

In [ ]:
final_mismatch_df = (
    final_consistency_df[
        final_consistency_df[
            "difference"
        ].abs()
        >
        1e-12
    ]
    .copy()
)

In [ ]:
print(
    "Maximum absolute difference:",
    final_consistency_df[
        "difference"
    ].abs().max()
)

print(
    "Mismatched feature values:",
    len(
        final_mismatch_df
    )
)

print(
    "Affected samples:",
    final_mismatch_df[
        "sample"
    ].nunique()
)

In [ ]:
final_mismatch_df[
    "feature"
].value_counts()

In [ ]:
sample_meta = (
    dev_meta.iloc[:100]
)

full_sample_records = [
    extract_full_lexical_14_from_row(
        row["url"],
        row["hostname"]
    )
    for _, row in sample_meta.iterrows()
]

reduced_sample_records = [
    extract_reduced_lexical_8_from_row(
        row["url"],
        row["hostname"]
    )
    for _, row in sample_meta.iterrows()
]

full_sample_df = (
    pd.DataFrame(
        full_sample_records
    )
)

reduced_sample_df = (
    pd.DataFrame(
        reduced_sample_records
    )
)

print(
    "Full feature shape:",
    full_sample_df.shape
)

print(
    "Reduced feature shape:",
    reduced_sample_df.shape
)

In [ ]:
rng = np.random.default_rng(
    42
)

benchmark_idx = rng.choice(
    len(dev_meta),

    size=min(
        2000,
        len(dev_meta)
    ),

    replace=False
)

benchmark_meta = (
    dev_meta.iloc[
        benchmark_idx
    ]
    .reset_index(drop=True)
)

print(
    "Benchmark samples:",
    len(benchmark_meta)
)

In [ ]:
def measure_feature_extraction_latency(
    meta_df,
    extractor,
    repeats=50
):

    # ------------------------------------------------
    # Warm-up
    # ------------------------------------------------

    for _ in range(3):

        for _, row in (
            meta_df.iloc[:100]
            .iterrows()
        ):

            extractor(
                row["url"],
                row["hostname"]
            )

    # ------------------------------------------------
    # Measurement
    # ------------------------------------------------

    times = []

    for _ in range(
        repeats
    ):

        start = (
            time.perf_counter()
        )

        for _, row in (
            meta_df.iterrows()
        ):

            extractor(
                row["url"],
                row["hostname"]
            )

        end = (
            time.perf_counter()
        )

        times.append(
            end - start
        )

    times = np.asarray(
        times
    )

    return {

        "mean_total_ms":
            times.mean()
            * 1000,

        "std_total_ms":
            times.std()
            * 1000,

        "median_total_ms":
            np.median(
                times
            )
            * 1000,

        "mean_ms_per_url":
            times.mean()
            * 1000
            /
            len(meta_df),

        "median_ms_per_url":
            np.median(
                times
            )
            * 1000
            /
            len(meta_df)
    }

In [ ]:
full_extraction_latency = (
    measure_feature_extraction_latency(
        benchmark_meta,
        extract_full_lexical_14_from_row,
        repeats=50
    )
)

full_extraction_latency

In [ ]:
reduced_extraction_latency = (
    measure_feature_extraction_latency(
        benchmark_meta,
        extract_reduced_lexical_8_from_row,
        repeats=50
    )
)

reduced_extraction_latency

In [ ]:
feature_latency_reduction = (
    1
    -
    reduced_extraction_latency[
        "mean_ms_per_url"
    ]
    /
    full_extraction_latency[
        "mean_ms_per_url"
    ]
)

feature_speedup = (
    full_extraction_latency[
        "mean_ms_per_url"
    ]
    /
    reduced_extraction_latency[
        "mean_ms_per_url"
    ]
)

In [ ]:
print(
    "Full extraction time "
    "(ms/URL):",
    full_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Reduced extraction time "
    "(ms/URL):",
    reduced_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Extraction-time reduction (%):",
    100
    *
    feature_latency_reduction
)

print(
    "Extraction speedup:",
    feature_speedup,
    "x"
)

In [ ]:
full_feature_row = (
    feature_budget_summary[
        feature_budget_summary[
            "feature_count"
        ]
        ==
        14
    ]
    .iloc[0]
)

reduced_feature_row = (
    feature_budget_summary[
        feature_budget_summary[
            "feature_count"
        ]
        ==
        8
    ]
    .iloc[0]
)

In [ ]:
feature_efficiency_summary = (
    pd.DataFrame([
        {
            "representation":
                "Full_14_Features",

            "feature_count":
                14,

            "mean_cv_f1":
                full_feature_row[
                    "mean_f1"
                ],

            "mean_cv_auc":
                full_feature_row[
                    "mean_auc"
                ],

            "mean_cv_mcc":
                full_feature_row[
                    "mean_mcc"
                ],

            "extraction_ms_per_url":
                full_extraction_latency[
                    "mean_ms_per_url"
                ]
        },

        {
            "representation":
                "Reduced_8_Features",

            "feature_count":
                8,

            "mean_cv_f1":
                reduced_feature_row[
                    "mean_f1"
                ],

            "mean_cv_auc":
                reduced_feature_row[
                    "mean_auc"
                ],

            "mean_cv_mcc":
                reduced_feature_row[
                    "mean_mcc"
                ],

            "extraction_ms_per_url":
                reduced_extraction_latency[
                    "mean_ms_per_url"
                ]
        }
    ])
)

feature_efficiency_summary

In [ ]:
feature_count_reduction = (
    1
    -
    8 / 14
)

f1_retention_feature = (
    reduced_feature_row[
        "mean_f1"
    ]
    /
    full_feature_row[
        "mean_f1"
    ]
)

auc_difference = (
    reduced_feature_row[
        "mean_auc"
    ]
    -
    full_feature_row[
        "mean_auc"
    ]
)

In [ ]:
print(
    "Feature-count reduction (%):",
    100
    *
    feature_count_reduction
)

print(
    "F1 retention (%):",
    100
    *
    f1_retention_feature
)

print(
    "AUC difference:",
    auc_difference
)

print(
    "Extraction-time reduction (%):",
    100
    *
    feature_latency_reduction
)

print(
    "Extraction speedup:",
    feature_speedup,
    "x"
)

In [ ]:
plt.figure(
    figsize=(7, 5)
)

plt.scatter(
    feature_efficiency_summary[
        "extraction_ms_per_url"
    ],

    feature_efficiency_summary[
        "mean_cv_f1"
    ],

    s=120
)

for _, row in (
    feature_efficiency_summary
    .iterrows()
):

    plt.annotate(
        row[
            "representation"
        ],

        (
            row[
                "extraction_ms_per_url"
            ],

            row[
                "mean_cv_f1"
            ]
        ),

        xytext=(7, 7),
        textcoords="offset points"
    )

plt.xlabel(
    "Feature Extraction Time (ms/URL)"
)

plt.ylabel(
    "Mean Grouped-CV F1"
)

plt.title(
    "Performance–Feature Extraction Trade-off"
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

In [ ]:
feature_efficiency_summary.to_csv(
    "phase12E_feature_efficiency_summary.csv",
    index=False
)

final_consistency_df.to_csv(
    "phase12E_final_consistency_check.csv",
    index=False
)

final_mismatch_df.to_csv(
    "phase12E_final_mismatches.csv",
    index=False
)

In [ ]:
print(
    "=" * 60
)

print(
    "PHASE 12E SUMMARY"
)

print(
    "=" * 60
)

print(
    "Consistency max difference:",
    final_consistency_df[
        "difference"
    ].abs().max()
)

print(
    "Mismatch count:",
    len(
        final_mismatch_df
    )
)

print(
    "Feature count:",
    "14 -> 8"
)

print(
    "Feature reduction (%):",
    100
    *
    feature_count_reduction
)

print(
    "F1 retention (%):",
    100
    *
    f1_retention_feature
)

print(
    "Full extraction ms/URL:",
    full_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Reduced extraction ms/URL:",
    reduced_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Extraction reduction (%):",
    100
    *
    feature_latency_reduction
)

print(
    "Speedup:",
    feature_speedup,
    "x"
)

In [ ]:
import re
from urllib.parse import urlparse

def safe_parse_url(url):

    url = str(url).strip()

    # Only treat scheme as valid if it is at the beginning
    has_scheme = bool(
        re.match(
            r"^[A-Za-z][A-Za-z0-9+.-]*://",
            url
        )
    )

    if has_scheme:
        parsed = urlparse(url)
    else:
        parsed = urlparse("//" + url)

    parsed_hostname = (
        parsed.hostname
        if parsed.hostname is not None
        else ""
    )

    path = (
        parsed.path
        if parsed.path is not None
        else ""
    )

    query = (
        parsed.query
        if parsed.query is not None
        else ""
    )

    fragment = (
        parsed.fragment
        if parsed.fragment is not None
        else ""
    )

    return (
        url,
        parsed,
        parsed_hostname,
        path,
        query,
        fragment
    )

In [ ]:
def extract_reduced_lexical_8_from_row(
    url,
    hostname
):

    url = str(url).strip()

    hostname = (
        ""
        if pd.isna(hostname)
        else str(hostname).strip().lower()
    )

    (
        url,
        parsed,
        parsed_hostname,
        path,
        query,
        fragment
    ) = safe_parse_url(url)

    url_lower = url.lower()

    # 1. num_subdomains
    hostname_parts = [
        p
        for p in hostname.split(".")
        if p
    ]

    num_subdomains = max(
        len(hostname_parts) - 2,
        0
    )

    # 2. suspicious_keyword_count
    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    # 3. num_digits
    num_digits = sum(
        char.isdigit()
        for char in url
    )

    # 4. path_length
    path_length = len(path)

    # 5. hostname_length
    hostname_length = len(hostname)

    # 6. url_length
    url_length = len(url)

    # 7. num_dots
    num_dots = url.count(".")

    # 8. special_char_ratio
    num_special_chars = sum(
        not char.isalnum()
        for char in url
    )

    special_char_ratio = (
        num_special_chars / url_length
        if url_length > 0
        else 0.0
    )

    return {
        "num_subdomains":
            num_subdomains,

        "suspicious_keyword_count":
            suspicious_keyword_count,

        "num_digits":
            num_digits,

        "path_length":
            path_length,

        "hostname_length":
            hostname_length,

        "url_length":
            url_length,

        "num_dots":
            num_dots,

        "special_char_ratio":
            special_char_ratio
    }

In [ ]:
test_indices = [
    1711,
    4555,
    4712
]

dev_meta = (
    df_all_features
    .iloc[train_idx]
    [["url", "hostname"]]
    .reset_index(drop=True)
)

for idx in test_indices:

    url = dev_meta.iloc[idx]["url"]
    hostname = dev_meta.iloc[idx]["hostname"]

    extracted = (
        extract_reduced_lexical_8_from_row(
            url,
            hostname
        )
    )

    print("=" * 60)
    print("sample:", idx)

    print(
        "stored path_length:",
        X_dev.iloc[idx]["path_length"]
    )

    print(
        "new path_length:",
        extracted["path_length"]
    )

    print(
        "stored hostname_length:",
        X_dev.iloc[idx]["hostname_length"]
    )

    print(
        "new hostname_length:",
        extracted["hostname_length"]
    )

    print(
        "stored num_subdomains:",
        X_dev.iloc[idx]["num_subdomains"]
    )

    print(
        "new num_subdomains:",
        extracted["num_subdomains"]
    )

In [ ]:
for var_name in [
    "final_consistency_df",
    "final_mismatch_df",
    "consistency_records"
]:
    if var_name in globals():
        del globals()[var_name]

print("Old consistency variables cleared.")

In [ ]:
consistency_records = []

for idx in range(
    len(dev_meta)
):

    url = (
        dev_meta.iloc[idx][
            "url"
        ]
    )

    hostname = (
        dev_meta.iloc[idx][
            "hostname"
        ]
    )

    extracted = (
        extract_reduced_lexical_8_from_row(
            url,
            hostname
        )
    )

    for feature in (
        final_reduced_lexical_features
    ):

        old_value = float(
            X_dev.iloc[idx][
                feature
            ]
        )

        new_value = float(
            extracted[
                feature
            ]
        )

        consistency_records.append({

            "sample":
                idx,

            "feature":
                feature,

            "old_value":
                old_value,

            "new_value":
                new_value,

            "difference":
                old_value
                -
                new_value
        })


final_consistency_df = pd.DataFrame(
    consistency_records
)

final_mismatch_df = (
    final_consistency_df[
        final_consistency_df[
            "difference"
        ].abs()
        >
        1e-12
    ]
    .copy()
)

In [ ]:
print(
    "Maximum absolute difference:",
    final_consistency_df[
        "difference"
    ].abs().max()
)

print(
    "Mismatch count:",
    len(final_mismatch_df)
)

print(
    "Affected samples:",
    final_mismatch_df[
        "sample"
    ].nunique()
)

if len(final_mismatch_df) > 0:

    print("\nMismatch features:")

    print(
        final_mismatch_df[
            "feature"
        ].value_counts()
    )

In [ ]:
idx = 100

url = dev_meta.iloc[idx]["url"]
stored_hostname = dev_meta.iloc[idx]["hostname"]

(
    _,
    _,
    parsed_hostname,
    path,
    _,
    _
) = safe_parse_url(url)

print("URL:", url)
print("Parsed hostname:", parsed_hostname)
print("Stored hostname:", stored_hostname)

print()

print(
    "Old hostname_length:",
    X_dev.iloc[idx]["hostname_length"]
)

print(
    "len(parsed hostname):",
    len(parsed_hostname)
)

print(
    "len(stored hostname):",
    len(stored_hostname)
)

print()

print(
    "Old num_subdomains:",
    X_dev.iloc[idx]["num_subdomains"]
)

In [ ]:
def extract_reduced_lexical_8_from_row(
    url,
    hostname
):

    url = str(url).strip()

    (
        url,
        parsed,
        parsed_hostname,
        path,
        query,
        fragment
    ) = safe_parse_url(url)

    parsed_hostname = (
        ""
        if parsed_hostname is None
        else str(parsed_hostname).lower()
    )

    url_lower = url.lower()

    # ========================================================
    # 1. num_subdomains
    #
    # Original logic:
    # parse hostname from original URL,
    # remove leading "www." only for subdomain counting
    # ========================================================

    hostname_for_subdomains = (
        parsed_hostname
    )

    if hostname_for_subdomains.startswith(
        "www."
    ):
        hostname_for_subdomains = (
            hostname_for_subdomains[4:]
        )

    hostname_parts = [
        p
        for p in hostname_for_subdomains.split(".")
        if p
    ]

    num_subdomains = max(
        len(hostname_parts) - 2,
        0
    )

    # ========================================================
    # 2. suspicious_keyword_count
    # ========================================================

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    # ========================================================
    # 3. num_digits
    # ========================================================

    num_digits = sum(
        char.isdigit()
        for char in url
    )

    # ========================================================
    # 4. path_length
    # ========================================================

    path_length = len(path)

    # ========================================================
    # 5. hostname_length
    #
    # Keep original parsed hostname, including www.
    # ========================================================

    hostname_length = len(
        parsed_hostname
    )

    # ========================================================
    # 6. url_length
    # ========================================================

    url_length = len(url)

    # ========================================================
    # 7. num_dots
    # ========================================================

    num_dots = url.count(".")

    # ========================================================
    # 8. special_char_ratio
    # ========================================================

    num_special_chars = sum(
        not char.isalnum()
        for char in url
    )

    special_char_ratio = (
        num_special_chars / url_length
        if url_length > 0
        else 0.0
    )

    return {

        "num_subdomains":
            num_subdomains,

        "suspicious_keyword_count":
            suspicious_keyword_count,

        "num_digits":
            num_digits,

        "path_length":
            path_length,

        "hostname_length":
            hostname_length,

        "url_length":
            url_length,

        "num_dots":
            num_dots,

        "special_char_ratio":
            special_char_ratio
    }

In [ ]:
subdomain_mismatch_samples = (
    final_mismatch_df[
        final_mismatch_df[
            "feature"
        ] == "num_subdomains"
    ]["sample"]
    .head(10)
    .astype(int)
    .tolist()
)

for idx in subdomain_mismatch_samples:

    url = (
        dev_meta.iloc[idx][
            "url"
        ]
    )

    hostname = (
        dev_meta.iloc[idx][
            "hostname"
        ]
    )

    (
        _,
        _,
        parsed_hostname,
        _,
        _,
        _
    ) = safe_parse_url(url)

    extracted = (
        extract_reduced_lexical_8_from_row(
            url,
            hostname
        )
    )

    print("=" * 70)

    print(
        "sample:",
        idx
    )

    print(
        "URL:",
        url
    )

    print(
        "parsed hostname:",
        parsed_hostname
    )

    print(
        "stored hostname:",
        hostname
    )

    print(
        "OLD num_subdomains:",
        X_dev.iloc[idx][
            "num_subdomains"
        ]
    )

    print(
        "NEW num_subdomains:",
        extracted[
            "num_subdomains"
        ]
    )

In [ ]:
for var_name in [
    "final_consistency_df",
    "final_mismatch_df",
    "consistency_records"
]:
    if var_name in globals():
        del globals()[var_name]

print(
    "Old consistency results cleared."
)

In [ ]:
consistency_records = []

for idx in range(
    len(dev_meta)
):

    url = (
        dev_meta.iloc[idx][
            "url"
        ]
    )

    hostname = (
        dev_meta.iloc[idx][
            "hostname"
        ]
    )

    extracted = (
        extract_reduced_lexical_8_from_row(
            url,
            hostname
        )
    )

    for feature in (
        final_reduced_lexical_features
    ):

        old_value = float(
            X_dev.iloc[idx][
                feature
            ]
        )

        new_value = float(
            extracted[
                feature
            ]
        )

        consistency_records.append({

            "sample":
                idx,

            "feature":
                feature,

            "old_value":
                old_value,

            "new_value":
                new_value,

            "difference":
                old_value
                -
                new_value
        })


final_consistency_df = (
    pd.DataFrame(
        consistency_records
    )
)

final_mismatch_df = (
    final_consistency_df[
        final_consistency_df[
            "difference"
        ].abs()
        >
        1e-12
    ]
    .copy()
)

In [ ]:
print(
    "Maximum absolute difference:",
    final_consistency_df[
        "difference"
    ].abs().max()
)

print(
    "Mismatch count:",
    len(final_mismatch_df)
)

print(
    "Affected samples:",
    final_mismatch_df[
        "sample"
    ].nunique()
)

if len(final_mismatch_df) > 0:

    print(
        "\nRemaining mismatch features:"
    )

    print(
        final_mismatch_df[
            "feature"
        ].value_counts()
    )

In [ ]:
# ============================================================
# Diagnose the ORIGINAL num_subdomains definition
# ============================================================

diagnostic_records = []

mismatch_samples = (
    final_mismatch_df["sample"]
    .unique()
    .astype(int)
)

for idx in mismatch_samples:

    url = dev_meta.iloc[idx]["url"]
    stored_hostname = str(
        dev_meta.iloc[idx]["hostname"]
    )

    (
        _,
        _,
        parsed_hostname,
        _,
        _,
        _
    ) = safe_parse_url(url)

    parsed_hostname = (
        ""
        if parsed_hostname is None
        else str(parsed_hostname).lower()
    )

    stored_hostname = stored_hostname.lower()

    parsed_no_www = (
        parsed_hostname[4:]
        if parsed_hostname.startswith("www.")
        else parsed_hostname
    )

    stored_no_www = (
        stored_hostname[4:]
        if stored_hostname.startswith("www.")
        else stored_hostname
    )

    old_value = int(
        X_dev.iloc[idx]["num_subdomains"]
    )

    # Candidate definitions
    candidates = {

        # number of labels minus 2
        "parsed_parts_minus_2":
            max(len(parsed_hostname.split(".")) - 2, 0),

        "parsed_no_www_parts_minus_2":
            max(len(parsed_no_www.split(".")) - 2, 0),

        "stored_parts_minus_2":
            max(len(stored_hostname.split(".")) - 2, 0),

        "stored_no_www_parts_minus_2":
            max(len(stored_no_www.split(".")) - 2, 0),

        # dot-based alternatives
        "parsed_dots_minus_1":
            max(parsed_hostname.count(".") - 1, 0),

        "parsed_no_www_dots_minus_1":
            max(parsed_no_www.count(".") - 1, 0),

        "stored_dots_minus_1":
            max(stored_hostname.count(".") - 1, 0),

        "stored_no_www_dots_minus_1":
            max(stored_no_www.count(".") - 1, 0),

        # sometimes people simply count dots
        "parsed_dot_count":
            parsed_hostname.count("."),

        "stored_dot_count":
            stored_hostname.count(".")
    }

    record = {
        "sample": idx,
        "url": url,
        "parsed_hostname": parsed_hostname,
        "stored_hostname": stored_hostname,
        "old_num_subdomains": old_value
    }

    for name, value in candidates.items():
        record[name] = value

    diagnostic_records.append(record)


subdomain_diagnostic = pd.DataFrame(
    diagnostic_records
)

In [ ]:
candidate_cols = [
    col
    for col in subdomain_diagnostic.columns
    if col not in [
        "sample",
        "url",
        "parsed_hostname",
        "stored_hostname",
        "old_num_subdomains"
    ]
]

match_summary = []

for col in candidate_cols:

    matches = (
        subdomain_diagnostic[col]
        ==
        subdomain_diagnostic[
            "old_num_subdomains"
        ]
    ).sum()

    match_summary.append({
        "candidate_formula": col,
        "matches": matches,
        "total": len(subdomain_diagnostic),
        "match_rate": (
            matches
            /
            len(subdomain_diagnostic)
        )
    })


match_summary_df = (
    pd.DataFrame(match_summary)
    .sort_values(
        "match_rate",
        ascending=False
    )
)

match_summary_df

In [ ]:
subdomain_diagnostic[
    [
        "url",
        "parsed_hostname",
        "stored_hostname",
        "old_num_subdomains"
    ]
    +
    candidate_cols
].head(20)

In [ ]:
# ============================================================
# Verify num_subdomains using stored registered domain
# ============================================================

subdomain_domain_check = []

dev_meta_v3 = (
    df_all_features
    .iloc[train_idx]
    [["url", "hostname", "domain"]]
    .reset_index(drop=True)
    .copy()
)

for idx in range(len(dev_meta_v3)):

    url = dev_meta_v3.iloc[idx]["url"]

    stored_domain = (
        ""
        if pd.isna(dev_meta_v3.iloc[idx]["domain"])
        else str(
            dev_meta_v3.iloc[idx]["domain"]
        ).strip().lower()
    )

    (
        _,
        _,
        parsed_hostname,
        _,
        _,
        _
    ) = safe_parse_url(url)

    parsed_hostname = (
        ""
        if parsed_hostname is None
        else str(parsed_hostname).lower()
    )

    hostname_parts = [
        x
        for x in parsed_hostname.split(".")
        if x
    ]

    domain_parts = [
        x
        for x in stored_domain.split(".")
        if x
    ]

    predicted_num_subdomains = max(
        len(hostname_parts)
        -
        len(domain_parts),
        0
    )

    old_num_subdomains = int(
        X_dev.iloc[idx][
            "num_subdomains"
        ]
    )

    subdomain_domain_check.append({
        "sample": idx,
        "url": url,
        "parsed_hostname": parsed_hostname,
        "domain": stored_domain,
        "old_num_subdomains":
            old_num_subdomains,
        "predicted_num_subdomains":
            predicted_num_subdomains,
        "difference":
            old_num_subdomains
            -
            predicted_num_subdomains
    })


subdomain_domain_check = pd.DataFrame(
    subdomain_domain_check
)

print(
    "Exact matches:",
    (
        subdomain_domain_check[
            "difference"
        ] == 0
    ).sum(),
    "/",
    len(subdomain_domain_check)
)

print(
    "Match rate:",
    (
        subdomain_domain_check[
            "difference"
        ] == 0
    ).mean()
)

print(
    "Maximum difference:",
    subdomain_domain_check[
        "difference"
    ].abs().max()
)

In [ ]:
def extract_reduced_lexical_8_from_row(
    url,
    hostname,
    domain
):

    url = str(url).strip()

    domain = (
        ""
        if pd.isna(domain)
        else str(domain).strip().lower()
    )

    (
        url,
        parsed,
        parsed_hostname,
        path,
        query,
        fragment
    ) = safe_parse_url(url)

    parsed_hostname = (
        ""
        if parsed_hostname is None
        else str(parsed_hostname).lower()
    )

    url_lower = url.lower()

    # 1. num_subdomains
    hostname_parts = [
        x
        for x in parsed_hostname.split(".")
        if x
    ]

    domain_parts = [
        x
        for x in domain.split(".")
        if x
    ]

    num_subdomains = max(
        len(hostname_parts)
        -
        len(domain_parts),
        0
    )

    # 2. suspicious_keyword_count
    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    # 3. num_digits
    num_digits = sum(
        char.isdigit()
        for char in url
    )

    # 4. path_length
    path_length = len(path)

    # 5. hostname_length
    hostname_length = len(
        parsed_hostname
    )

    # 6. url_length
    url_length = len(url)

    # 7. num_dots
    num_dots = url.count(".")

    # 8. special_char_ratio
    num_special_chars = sum(
        not char.isalnum()
        for char in url
    )

    special_char_ratio = (
        num_special_chars / url_length
        if url_length > 0
        else 0.0
    )

    return {
        "num_subdomains":
            num_subdomains,

        "suspicious_keyword_count":
            suspicious_keyword_count,

        "num_digits":
            num_digits,

        "path_length":
            path_length,

        "hostname_length":
            hostname_length,

        "url_length":
            url_length,

        "num_dots":
            num_dots,

        "special_char_ratio":
            special_char_ratio
    }

In [ ]:
dev_meta = (
    df_all_features
    .iloc[train_idx]
    [
        [
            "url",
            "hostname",
            "domain"
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

In [ ]:
consistency_records = []

for idx in range(len(dev_meta)):

    row = dev_meta.iloc[idx]

    extracted = (
        extract_reduced_lexical_8_from_row(
            row["url"],
            row["hostname"],
            row["domain"]
        )
    )

    for feature in (
        final_reduced_lexical_features
    ):

        old_value = float(
            X_dev.iloc[idx][feature]
        )

        new_value = float(
            extracted[feature]
        )

        consistency_records.append({
            "sample":
                idx,

            "feature":
                feature,

            "old_value":
                old_value,

            "new_value":
                new_value,

            "difference":
                old_value - new_value
        })


final_consistency_df = pd.DataFrame(
    consistency_records
)

final_mismatch_df = (
    final_consistency_df[
        final_consistency_df[
            "difference"
        ].abs() > 1e-12
    ]
    .copy()
)

In [ ]:
print(
    "Maximum absolute difference:",
    final_consistency_df[
        "difference"
    ].abs().max()
)

print(
    "Mismatch count:",
    len(final_mismatch_df)
)

print(
    "Affected samples:",
    final_mismatch_df[
        "sample"
    ].nunique()
)

In [ ]:
def extract_full_lexical_14_from_row(
    url,
    hostname,
    domain
):

    url = str(url).strip()

    domain = (
        ""
        if pd.isna(domain)
        else str(domain).strip().lower()
    )

    (
        url,
        parsed,
        parsed_hostname,
        path,
        query,
        fragment
    ) = safe_parse_url(url)

    parsed_hostname = (
        ""
        if parsed_hostname is None
        else str(parsed_hostname).lower()
    )

    url_lower = url.lower()

    url_length = len(url)

    hostname_length = len(
        parsed_hostname
    )

    path_length = len(path)

    query_length = len(query)

    num_dots = url.count(".")

    num_hyphens = url.count("-")

    num_digits = sum(
        char.isdigit()
        for char in url
    )

    num_special_chars = sum(
        not char.isalnum()
        for char in url
    )

    special_char_ratio = (
        num_special_chars / url_length
        if url_length > 0
        else 0.0
    )

    # registered-domain-aware subdomain count
    hostname_parts = [
        x
        for x in parsed_hostname.split(".")
        if x
    ]

    domain_parts = [
        x
        for x in domain.split(".")
        if x
    ]

    num_subdomains = max(
        len(hostname_parts)
        -
        len(domain_parts),
        0
    )

    has_fragment = int(
        len(fragment) > 0
    )

    contains_at = int(
        "@" in url
    )

    try:
        contains_port = int(
            parsed.port is not None
        )
    except ValueError:
        contains_port = 0

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    return {
        "url_length":
            url_length,

        "hostname_length":
            hostname_length,

        "path_length":
            path_length,

        "query_length":
            query_length,

        "num_dots":
            num_dots,

        "num_hyphens":
            num_hyphens,

        "num_digits":
            num_digits,

        "num_special_chars":
            num_special_chars,

        "special_char_ratio":
            special_char_ratio,

        "num_subdomains":
            num_subdomains,

        "has_fragment":
            has_fragment,

        "contains_at":
            contains_at,

        "contains_port":
            contains_port,

        "suspicious_keyword_count":
            suspicious_keyword_count
    }

In [ ]:
rng = np.random.default_rng(42)

benchmark_idx = rng.choice(
    len(dev_meta),
    size=min(
        2000,
        len(dev_meta)
    ),
    replace=False
)

benchmark_meta = (
    dev_meta.iloc[
        benchmark_idx
    ]
    .reset_index(drop=True)
)

print(
    "Benchmark samples:",
    len(benchmark_meta)
)

In [ ]:
def measure_feature_extraction_latency(
    meta_df,
    extractor,
    repeats=50
):

    records = list(
        meta_df[
            [
                "url",
                "hostname",
                "domain"
            ]
        ].itertuples(
            index=False,
            name=None
        )
    )

    # Warm-up
    for _ in range(3):

        for url, hostname, domain in (
            records[:100]
        ):

            extractor(
                url,
                hostname,
                domain
            )

    times = []

    for _ in range(repeats):

        start = time.perf_counter()

        for url, hostname, domain in records:

            extractor(
                url,
                hostname,
                domain
            )

        end = time.perf_counter()

        times.append(
            end - start
        )

    times = np.asarray(times)

    return {
        "mean_total_ms":
            times.mean() * 1000,

        "std_total_ms":
            times.std() * 1000,

        "median_total_ms":
            np.median(times) * 1000,

        "mean_ms_per_url":
            times.mean()
            * 1000
            / len(records),

        "median_ms_per_url":
            np.median(times)
            * 1000
            / len(records)
    }

In [ ]:
full_extraction_latency = (
    measure_feature_extraction_latency(
        benchmark_meta,
        extract_full_lexical_14_from_row,
        repeats=50
    )
)

full_extraction_latency

In [ ]:
reduced_extraction_latency = (
    measure_feature_extraction_latency(
        benchmark_meta,
        extract_reduced_lexical_8_from_row,
        repeats=50
    )
)

reduced_extraction_latency

In [ ]:
feature_latency_reduction = (
    1
    -
    reduced_extraction_latency[
        "mean_ms_per_url"
    ]
    /
    full_extraction_latency[
        "mean_ms_per_url"
    ]
)

feature_speedup = (
    full_extraction_latency[
        "mean_ms_per_url"
    ]
    /
    reduced_extraction_latency[
        "mean_ms_per_url"
    ]
)

In [ ]:
print(
    "Full extraction ms/URL:",
    full_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Reduced extraction ms/URL:",
    reduced_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Extraction-time reduction (%):",
    100 * feature_latency_reduction
)

print(
    "Extraction speedup:",
    feature_speedup,
    "x"
)

In [ ]:
full_feature_row = (
    feature_budget_summary[
        feature_budget_summary[
            "feature_count"
        ] == 14
    ]
    .iloc[0]
)

reduced_feature_row = (
    feature_budget_summary[
        feature_budget_summary[
            "feature_count"
        ] == 8
    ]
    .iloc[0]
)

In [ ]:
feature_count_reduction = (
    1 - 8 / 14
)

f1_retention_feature = (
    reduced_feature_row[
        "mean_f1"
    ]
    /
    full_feature_row[
        "mean_f1"
    ]
)

f1_difference = (
    reduced_feature_row[
        "mean_f1"
    ]
    -
    full_feature_row[
        "mean_f1"
    ]
)

auc_difference = (
    reduced_feature_row[
        "mean_auc"
    ]
    -
    full_feature_row[
        "mean_auc"
    ]
)

In [ ]:
print("=" * 65)
print("PHASE 12E — FINAL SUMMARY")
print("=" * 65)

print(
    "Consistency max difference:",
    final_consistency_df[
        "difference"
    ].abs().max()
)

print(
    "Mismatch count:",
    len(final_mismatch_df)
)

print()

print(
    "Feature count:",
    "14 -> 8"
)

print(
    "Feature-count reduction (%):",
    100 * feature_count_reduction
)

print()

print(
    "Full-feature CV F1:",
    full_feature_row["mean_f1"]
)

print(
    "Reduced-feature CV F1:",
    reduced_feature_row["mean_f1"]
)

print(
    "F1 difference:",
    f1_difference
)

print(
    "F1 retention (%):",
    100 * f1_retention_feature
)

print(
    "AUC difference:",
    auc_difference
)

print()

print(
    "Full extraction ms/URL:",
    full_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Reduced extraction ms/URL:",
    reduced_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Extraction-time reduction (%):",
    100 * feature_latency_reduction
)

print(
    "Extraction speedup:",
    feature_speedup,
    "x"
)

In [ ]:
feature_efficiency_summary = pd.DataFrame([
    {
        "representation":
            "Full lexical",

        "feature_count":
            14,

        "mean_cv_f1":
            full_feature_row[
                "mean_f1"
            ],

        "mean_cv_auc":
            full_feature_row[
                "mean_auc"
            ],

        "mean_cv_mcc":
            full_feature_row[
                "mean_mcc"
            ],

        "extraction_ms_per_url":
            full_extraction_latency[
                "mean_ms_per_url"
            ]
    },

    {
        "representation":
            "Reduced lexical",

        "feature_count":
            8,

        "mean_cv_f1":
            reduced_feature_row[
                "mean_f1"
            ],

        "mean_cv_auc":
            reduced_feature_row[
                "mean_auc"
            ],

        "mean_cv_mcc":
            reduced_feature_row[
                "mean_mcc"
            ],

        "extraction_ms_per_url":
            reduced_extraction_latency[
                "mean_ms_per_url"
            ]
    }
])

feature_efficiency_summary

In [ ]:
feature_efficiency_summary.to_csv(
    "phase12E_final_feature_efficiency_summary.csv",
    index=False
)

final_consistency_df.to_csv(
    "phase12E_final_consistency_check.csv",
    index=False
)

In [ ]:
# ============================================================
# Phase 12F — Final Lightweight Pareto Analysis
# ============================================================

import os
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
full_features = list(
    lexical_reduced
)

reduced_features = list(
    final_reduced_lexical_features
)

print(
    "Full feature count:",
    len(full_features)
)

print(
    "Reduced feature count:",
    len(reduced_features)
)

print(
    "\nReduced features:"
)

for i, feature in enumerate(
    reduced_features,
    start=1
):
    print(
        i,
        feature
    )

In [ ]:
def make_final_compact_xgb():

    return XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=1.0,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    )

In [ ]:
# A — Full features + Teacher
model_A = make_teacher_model()

model_A.fit(
    X_dev[
        full_features
    ],
    y_dev
)


# B — Reduced features + Teacher
model_B = make_teacher_model()

model_B.fit(
    X_dev[
        reduced_features
    ],
    y_dev
)


# C — Full features + Compact
model_C = make_final_compact_xgb()

model_C.fit(
    X_dev[
        full_features
    ],
    y_dev
)


# D — Reduced features + Compact
model_D = make_final_compact_xgb()

model_D.fit(
    X_dev[
        reduced_features
    ],
    y_dev
)

print(
    "All four development models trained."
)

In [ ]:
joblib.dump(
    model_A,
    "phase12F_A_full_teacher.pkl"
)

joblib.dump(
    model_B,
    "phase12F_B_reduced_teacher.pkl"
)

joblib.dump(
    model_C,
    "phase12F_C_full_compact.pkl"
)

joblib.dump(
    model_D,
    "phase12F_D_reduced_compact.pkl"
)

print(
    "All four models saved."
)

In [ ]:
model_files = {
    "A_Full_Teacher":
        "phase12F_A_full_teacher.pkl",

    "B_Reduced_Teacher":
        "phase12F_B_reduced_teacher.pkl",

    "C_Full_Compact":
        "phase12F_C_full_compact.pkl",

    "D_Reduced_Compact":
        "phase12F_D_reduced_compact.pkl"
}

model_sizes_kb = {
    name:
        os.path.getsize(path)
        /
        1024

    for name, path
    in model_files.items()
}

model_sizes_kb

In [ ]:
def measure_prediction_latency(
    model,
    X,
    repeats=200
):

    # warm-up
    for _ in range(10):
        model.predict_proba(
            X
        )

    times = []

    for _ in range(
        repeats
    ):

        start = (
            time.perf_counter()
        )

        model.predict_proba(
            X
        )

        end = (
            time.perf_counter()
        )

        times.append(
            end - start
        )

    times = np.asarray(
        times
    )

    return {

        "mean_total_ms":
            times.mean()
            *
            1000,

        "std_total_ms":
            times.std()
            *
            1000,

        "mean_ms_per_sample":
            (
                times.mean()
                *
                1000
                /
                len(X)
            )
    }

In [ ]:
rng = np.random.default_rng(
    42
)

latency_idx = rng.choice(
    len(X_dev),

    size=min(
        1000,
        len(X_dev)
    ),

    replace=False
)

In [ ]:
latency_A = (
    measure_prediction_latency(
        model_A,
        X_dev.iloc[
            latency_idx
        ][
            full_features
        ],
        repeats=200
    )
)

latency_B = (
    measure_prediction_latency(
        model_B,
        X_dev.iloc[
            latency_idx
        ][
            reduced_features
        ],
        repeats=200
    )
)

latency_C = (
    measure_prediction_latency(
        model_C,
        X_dev.iloc[
            latency_idx
        ][
            full_features
        ],
        repeats=200
    )
)

latency_D = (
    measure_prediction_latency(
        model_D,
        X_dev.iloc[
            latency_idx
        ][
            reduced_features
        ],
        repeats=200
    )
)

In [ ]:
print(
    "A:",
    latency_A
)

print(
    "B:",
    latency_B
)

print(
    "C:",
    latency_C
)

print(
    "D:",
    latency_D
)

In [ ]:
joint_lightweight_summary

In [ ]:
joint_perf = (
    joint_lightweight_summary
    .set_index(
        "configuration"
    )
)

In [ ]:
final_pareto_table = pd.DataFrame([
    {
        "configuration":
            "A_Full_Teacher",

        "feature_count":
            14,

        "model_type":
            "Teacher",

        "mean_cv_f1":
            joint_perf.loc[
                "FullFeatures_Teacher",
                "mean_f1"
            ],

        "mean_cv_auc":
            joint_perf.loc[
                "FullFeatures_Teacher",
                "mean_auc"
            ],

        "mean_cv_mcc":
            joint_perf.loc[
                "FullFeatures_Teacher",
                "mean_mcc"
            ],

        "feature_extraction_ms":
            full_extraction_latency[
                "mean_ms_per_url"
            ],

        "model_size_kb":
            model_sizes_kb[
                "A_Full_Teacher"
            ],

        "inference_ms_per_sample":
            latency_A[
                "mean_ms_per_sample"
            ]
    },

    {
        "configuration":
            "B_Reduced_Teacher",

        "feature_count":
            8,

        "model_type":
            "Teacher",

        "mean_cv_f1":
            joint_perf.loc[
                "ReducedFeatures_Teacher",
                "mean_f1"
            ],

        "mean_cv_auc":
            joint_perf.loc[
                "ReducedFeatures_Teacher",
                "mean_auc"
            ],

        "mean_cv_mcc":
            joint_perf.loc[
                "ReducedFeatures_Teacher",
                "mean_mcc"
            ],

        "feature_extraction_ms":
            reduced_extraction_latency[
                "mean_ms_per_url"
            ],

        "model_size_kb":
            model_sizes_kb[
                "B_Reduced_Teacher"
            ],

        "inference_ms_per_sample":
            latency_B[
                "mean_ms_per_sample"
            ]
    },

    {
        "configuration":
            "C_Full_Compact",

        "feature_count":
            14,

        "model_type":
            "Compact",

        "mean_cv_f1":
            joint_perf.loc[
                "FullFeatures_Compact",
                "mean_f1"
            ],

        "mean_cv_auc":
            joint_perf.loc[
                "FullFeatures_Compact",
                "mean_auc"
            ],

        "mean_cv_mcc":
            joint_perf.loc[
                "FullFeatures_Compact",
                "mean_mcc"
            ],

        "feature_extraction_ms":
            full_extraction_latency[
                "mean_ms_per_url"
            ],

        "model_size_kb":
            model_sizes_kb[
                "C_Full_Compact"
            ],

        "inference_ms_per_sample":
            latency_C[
                "mean_ms_per_sample"
            ]
    },

    {
        "configuration":
            "D_Reduced_Compact",

        "feature_count":
            8,

        "model_type":
            "Compact",

        "mean_cv_f1":
            joint_perf.loc[
                "ReducedFeatures_Compact",
                "mean_f1"
            ],

        "mean_cv_auc":
            joint_perf.loc[
                "ReducedFeatures_Compact",
                "mean_auc"
            ],

        "mean_cv_mcc":
            joint_perf.loc[
                "ReducedFeatures_Compact",
                "mean_mcc"
            ],

        "feature_extraction_ms":
            reduced_extraction_latency[
                "mean_ms_per_url"
            ],

        "model_size_kb":
            model_sizes_kb[
                "D_Reduced_Compact"
            ],

        "inference_ms_per_sample":
            latency_D[
                "mean_ms_per_sample"
            ]
    }
])

final_pareto_table

In [ ]:
final_pareto_table[
    "total_latency_ms"
] = (
    final_pareto_table[
        "feature_extraction_ms"
    ]
    +
    final_pareto_table[
        "inference_ms_per_sample"
    ]
)

final_pareto_table

In [ ]:
reference_row = (
    final_pareto_table[
        final_pareto_table[
            "configuration"
        ]
        ==
        "A_Full_Teacher"
    ]
    .iloc[0]
)

In [ ]:
final_pareto_table[
    "f1_retention"
] = (
    final_pareto_table[
        "mean_cv_f1"
    ]
    /
    reference_row[
        "mean_cv_f1"
    ]
)

final_pareto_table[
    "feature_reduction"
] = (
    1
    -
    final_pareto_table[
        "feature_count"
    ]
    /
    reference_row[
        "feature_count"
    ]
)

final_pareto_table[
    "model_size_reduction"
] = (
    1
    -
    final_pareto_table[
        "model_size_kb"
    ]
    /
    reference_row[
        "model_size_kb"
    ]
)

final_pareto_table[
    "total_latency_reduction"
] = (
    1
    -
    final_pareto_table[
        "total_latency_ms"
    ]
    /
    reference_row[
        "total_latency_ms"
    ]
)

In [ ]:
final_pareto_table[
    [
        "configuration",
        "mean_cv_f1",
        "f1_retention",
        "feature_count",
        "feature_reduction",
        "model_size_kb",
        "model_size_reduction",
        "total_latency_ms",
        "total_latency_reduction"
    ]
]

In [ ]:
def is_dominated(
    row,
    df
):

    for _, other in df.iterrows():

        if (
            other["configuration"]
            ==
            row["configuration"]
        ):
            continue

        no_worse = (

            other[
                "mean_cv_f1"
            ]
            >=
            row[
                "mean_cv_f1"
            ]

            and

            other[
                "total_latency_ms"
            ]
            <=
            row[
                "total_latency_ms"
            ]

            and

            other[
                "model_size_kb"
            ]
            <=
            row[
                "model_size_kb"
            ]
        )

        strictly_better = (

            other[
                "mean_cv_f1"
            ]
            >
            row[
                "mean_cv_f1"
            ]

            or

            other[
                "total_latency_ms"
            ]
            <
            row[
                "total_latency_ms"
            ]

            or

            other[
                "model_size_kb"
            ]
            <
            row[
                "model_size_kb"
            ]
        )

        if (
            no_worse
            and
            strictly_better
        ):
            return True

    return False

In [ ]:
final_pareto_table[
    "pareto_optimal"
] = (
    final_pareto_table
    .apply(
        lambda row:
            not is_dominated(
                row,
                final_pareto_table
            ),
        axis=1
    )
)

final_pareto_table

In [ ]:
pareto_models = (
    final_pareto_table[
        final_pareto_table[
            "pareto_optimal"
        ]
    ]
    .copy()
)

pareto_models

In [ ]:
FINAL_F1_TOLERANCE = 0.005

minimum_acceptable_f1 = (
    reference_row[
        "mean_cv_f1"
    ]
    -
    FINAL_F1_TOLERANCE
)

print(
    "Minimum acceptable F1:",
    minimum_acceptable_f1
)

In [ ]:
acceptable_final_models = (
    final_pareto_table[
        final_pareto_table[
            "mean_cv_f1"
        ]
        >=
        minimum_acceptable_f1
    ]
    .copy()
)

acceptable_final_models[
    [
        "configuration",
        "mean_cv_f1",
        "feature_count",
        "model_size_kb",
        "total_latency_ms",
        "pareto_optimal"
    ]
]

In [ ]:
candidate_pool = (
    acceptable_final_models[
        acceptable_final_models[
            "pareto_optimal"
        ]
    ]
    .copy()
)

if len(candidate_pool) == 0:

    candidate_pool = (
        acceptable_final_models
        .copy()
    )

In [ ]:
final_selected_row = (
    candidate_pool
    .sort_values(
        by=[
            "feature_count",
            "total_latency_ms",
            "model_size_kb",
            "mean_cv_f1"
        ],
        ascending=[
            True,
            True,
            True,
            False
        ]
    )
    .iloc[0]
)

In [ ]:
print(
    "FINAL LIGHTWEIGHT CANDIDATE"
)

print(
    "=" * 60
)

print(
    final_selected_row
)

In [ ]:
plt.figure(
    figsize=(8, 6)
)

for _, row in (
    final_pareto_table
    .iterrows()
):

    marker_size = (
        80
        if row[
            "pareto_optimal"
        ]
        else
        50
    )

    plt.scatter(
        row[
            "total_latency_ms"
        ],

        row[
            "mean_cv_f1"
        ],

        s=marker_size
    )

    plt.annotate(
        row[
            "configuration"
        ],

        (
            row[
                "total_latency_ms"
            ],
            row[
                "mean_cv_f1"
            ]
        ),

        xytext=(6, 6),
        textcoords="offset points"
    )

plt.xlabel(
    "Total Latency (ms/sample)"
)

plt.ylabel(
    "Mean Grouped-CV F1"
)

plt.title(
    "Accuracy–Latency Pareto Analysis"
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(
    figsize=(8, 6)
)

for _, row in (
    final_pareto_table
    .iterrows()
):

    plt.scatter(
        row[
            "model_size_kb"
        ],

        row[
            "mean_cv_f1"
        ],

        s=80
    )

    plt.annotate(
        row[
            "configuration"
        ],

        (
            row[
                "model_size_kb"
            ],
            row[
                "mean_cv_f1"
            ]
        ),

        xytext=(6, 6),
        textcoords="offset points"
    )

plt.xlabel(
    "Serialized Model Size (KB)"
)

plt.ylabel(
    "Mean Grouped-CV F1"
)

plt.title(
    "Accuracy–Model Size Trade-off"
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

In [ ]:
selected_configuration = (
    final_selected_row[
        "configuration"
    ]
)

print(
    "Selected configuration:",
    selected_configuration
)

In [ ]:
if (
    selected_configuration
    ==
    "A_Full_Teacher"
):

    final_lightweight_model = (
        model_A
    )

    final_lightweight_features = (
        full_features
    )


elif (
    selected_configuration
    ==
    "B_Reduced_Teacher"
):

    final_lightweight_model = (
        model_B
    )

    final_lightweight_features = (
        reduced_features
    )


elif (
    selected_configuration
    ==
    "C_Full_Compact"
):

    final_lightweight_model = (
        model_C
    )

    final_lightweight_features = (
        full_features
    )


elif (
    selected_configuration
    ==
    "D_Reduced_Compact"
):

    final_lightweight_model = (
        model_D
    )

    final_lightweight_features = (
        reduced_features
    )

In [ ]:
with open(
    "FINAL_lightweight_features.json",
    "w"
) as f:

    json.dump(
        {
            "configuration":
                selected_configuration,

            "feature_count":
                len(
                    final_lightweight_features
                ),

            "features":
                final_lightweight_features,

            "selection_rule":
                "Pareto-optimal configuration within "
                "0.005 F1 tolerance of full teacher"
        },
        f,
        indent=4
    )

In [ ]:
final_pareto_table.to_csv(
    "phase12F_final_pareto_table.csv",
    index=False
)

pareto_models.to_csv(
    "phase12F_pareto_optimal_models.csv",
    index=False
)

In [ ]:
print("=" * 70)

print(
    "PHASE 12 — FINAL LIGHTWEIGHT SUMMARY"
)

print("=" * 70)

print(
    "Selected configuration:",
    selected_configuration
)

print(
    "Feature count:",
    len(
        final_lightweight_features
    )
)

print(
    "Selected features:"
)

for feature in (
    final_lightweight_features
):

    print(
        " -",
        feature
    )

print()

print(
    "Mean CV F1:",
    final_selected_row[
        "mean_cv_f1"
    ]
)

print(
    "Mean CV AUC:",
    final_selected_row[
        "mean_cv_auc"
    ]
)

print(
    "F1 retention (%):",
    100
    *
    final_selected_row[
        "f1_retention"
    ]
)

print(
    "Feature reduction (%):",
    100
    *
    final_selected_row[
        "feature_reduction"
    ]
)

print(
    "Model size reduction (%):",
    100
    *
    final_selected_row[
        "model_size_reduction"
    ]
)

print(
    "Total latency reduction (%):",
    100
    *
    final_selected_row[
        "total_latency_reduction"
    ]
)

print(
    "Pareto optimal:",
    final_selected_row[
        "pareto_optimal"
    ]
)

Significance Test

In [ ]:
# ============================================================
# Phase 13 — Statistical Significance & Non-Inferiority Tests
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score
)

from scipy.stats import binomtest

In [ ]:
full_features = list(
    lexical_reduced
)

reduced_features = list(
    final_reduced_lexical_features
)

print("Full features:", len(full_features))
print("Reduced features:", len(reduced_features))

In [ ]:
def generate_paired_grouped_oof_predictions(
    X,
    y,
    groups,
    full_features,
    reduced_features,
    n_splits=5,
    random_state=42
):

    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    records = []

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X,
            y,
            groups=groups
        ),
        start=1
    ):

        print("=" * 70)
        print(f"Fold {fold}")
        print("=" * 70)

        # ----------------------------------------------------
        # Full 14-feature model
        # ----------------------------------------------------

        full_model = make_teacher_model()

        full_model.fit(
            X.iloc[tr_idx][
                full_features
            ],
            y.iloc[tr_idx]
        )

        full_prob = (
            full_model.predict_proba(
                X.iloc[va_idx][
                    full_features
                ]
            )[:, 1]
        )

        full_pred = (
            full_prob >= 0.5
        ).astype(int)

        # ----------------------------------------------------
        # Reduced 8-feature model
        # ----------------------------------------------------

        reduced_model = make_teacher_model()

        reduced_model.fit(
            X.iloc[tr_idx][
                reduced_features
            ],
            y.iloc[tr_idx]
        )

        reduced_prob = (
            reduced_model.predict_proba(
                X.iloc[va_idx][
                    reduced_features
                ]
            )[:, 1]
        )

        reduced_pred = (
            reduced_prob >= 0.5
        ).astype(int)

        # ----------------------------------------------------
        # Group leakage check
        # ----------------------------------------------------

        train_groups = set(
            groups.iloc[tr_idx]
        )

        val_groups = set(
            groups.iloc[va_idx]
        )

        overlap = len(
            train_groups
            &
            val_groups
        )

        print(
            "Hostname overlap:",
            overlap
        )

        # ----------------------------------------------------
        # Store sample-level paired predictions
        # ----------------------------------------------------

        for local_i, original_idx in enumerate(
            va_idx
        ):

            records.append({

                "original_dev_index":
                    int(original_idx),

                "fold":
                    fold,

                "hostname":
                    groups.iloc[
                        original_idx
                    ],

                "y_true":
                    int(
                        y.iloc[
                            original_idx
                        ]
                    ),

                "full_prob":
                    float(
                        full_prob[
                            local_i
                        ]
                    ),

                "full_pred":
                    int(
                        full_pred[
                            local_i
                        ]
                    ),

                "reduced_prob":
                    float(
                        reduced_prob[
                            local_i
                        ]
                    ),

                "reduced_pred":
                    int(
                        reduced_pred[
                            local_i
                        ]
                    )
            })

    return pd.DataFrame(
        records
    )

In [ ]:
paired_oof = (
    generate_paired_grouped_oof_predictions(
        X=X_dev.reset_index(drop=True),
        y=y_dev.reset_index(drop=True),
        groups=groups_dev.reset_index(drop=True),
        full_features=full_features,
        reduced_features=reduced_features,
        n_splits=5,
        random_state=42
    )
)

print(
    "OOF samples:",
    len(paired_oof)
)

paired_oof.head()

In [ ]:
print(
    "Unique samples:",
    paired_oof[
        "original_dev_index"
    ].nunique()
)

print(
    "Total samples:",
    len(paired_oof)
)

print(
    "Duplicated OOF samples:",
    paired_oof[
        "original_dev_index"
    ].duplicated().sum()
)

In [ ]:
paired_oof.groupby(
    "fold"
).size()

In [ ]:
full_oof_f1 = f1_score(
    paired_oof["y_true"],
    paired_oof["full_pred"]
)

full_oof_auc = roc_auc_score(
    paired_oof["y_true"],
    paired_oof["full_prob"]
)

full_oof_mcc = matthews_corrcoef(
    paired_oof["y_true"],
    paired_oof["full_pred"]
)

full_oof_accuracy = accuracy_score(
    paired_oof["y_true"],
    paired_oof["full_pred"]
)

In [ ]:
reduced_oof_f1 = f1_score(
    paired_oof["y_true"],
    paired_oof["reduced_pred"]
)

reduced_oof_auc = roc_auc_score(
    paired_oof["y_true"],
    paired_oof["reduced_prob"]
)

reduced_oof_mcc = matthews_corrcoef(
    paired_oof["y_true"],
    paired_oof["reduced_pred"]
)

reduced_oof_accuracy = accuracy_score(
    paired_oof["y_true"],
    paired_oof["reduced_pred"]
)

In [ ]:
oof_performance = pd.DataFrame([
    {
        "model":
            "Full_14",

        "F1":
            full_oof_f1,

        "AUROC":
            full_oof_auc,

        "MCC":
            full_oof_mcc,

        "Accuracy":
            full_oof_accuracy
    },

    {
        "model":
            "Reduced_8",

        "F1":
            reduced_oof_f1,

        "AUROC":
            reduced_oof_auc,

        "MCC":
            reduced_oof_mcc,

        "Accuracy":
            reduced_oof_accuracy
    }
])

oof_performance

In [ ]:
paired_oof[
    "full_correct"
] = (
    paired_oof[
        "full_pred"
    ]
    ==
    paired_oof[
        "y_true"
    ]
)

paired_oof[
    "reduced_correct"
] = (
    paired_oof[
        "reduced_pred"
    ]
    ==
    paired_oof[
        "y_true"
    ]
)

In [ ]:
both_correct = (
    paired_oof[
        "full_correct"
    ]
    &
    paired_oof[
        "reduced_correct"
    ]
).sum()

both_wrong = (
    ~paired_oof[
        "full_correct"
    ]
    &
    ~paired_oof[
        "reduced_correct"
    ]
).sum()

full_only_correct = (
    paired_oof[
        "full_correct"
    ]
    &
    ~paired_oof[
        "reduced_correct"
    ]
).sum()

reduced_only_correct = (
    ~paired_oof[
        "full_correct"
    ]
    &
    paired_oof[
        "reduced_correct"
    ]
).sum()

In [ ]:
paired_error_table = pd.Series({

    "Both correct":
        both_correct,

    "Both wrong":
        both_wrong,

    "Full correct, Reduced wrong":
        full_only_correct,

    "Reduced correct, Full wrong":
        reduced_only_correct
})

paired_error_table

In [ ]:
b = int(
    full_only_correct
)

c = int(
    reduced_only_correct
)

discordant = (
    b + c
)

print(
    "Full-only correct:",
    b
)

print(
    "Reduced-only correct:",
    c
)

print(
    "Discordant predictions:",
    discordant
)

In [ ]:
if discordant > 0:

    mcnemar_test = binomtest(
        k=min(b, c),
        n=discordant,
        p=0.5,
        alternative="two-sided"
    )

    mcnemar_p = (
        mcnemar_test.pvalue
    )

else:

    mcnemar_p = 1.0


print(
    "Exact McNemar p-value:",
    mcnemar_p
)

In [ ]:
def calculate_metrics_from_df(df):

    y_true = (
        df["y_true"]
        .to_numpy()
    )

    full_pred = (
        df["full_pred"]
        .to_numpy()
    )

    full_prob = (
        df["full_prob"]
        .to_numpy()
    )

    reduced_pred = (
        df["reduced_pred"]
        .to_numpy()
    )

    reduced_prob = (
        df["reduced_prob"]
        .to_numpy()
    )

    result = {

        "full_f1":
            f1_score(
                y_true,
                full_pred
            ),

        "reduced_f1":
            f1_score(
                y_true,
                reduced_pred
            ),

        "full_auc":
            roc_auc_score(
                y_true,
                full_prob
            ),

        "reduced_auc":
            roc_auc_score(
                y_true,
                reduced_prob
            ),

        "full_mcc":
            matthews_corrcoef(
                y_true,
                full_pred
            ),

        "reduced_mcc":
            matthews_corrcoef(
                y_true,
                reduced_pred
            )
    }

    result[
        "delta_f1"
    ] = (
        result[
            "reduced_f1"
        ]
        -
        result[
            "full_f1"
        ]
    )

    result[
        "delta_auc"
    ] = (
        result[
            "reduced_auc"
        ]
        -
        result[
            "full_auc"
        ]
    )

    result[
        "delta_mcc"
    ] = (
        result[
            "reduced_mcc"
        ]
        -
        result[
            "full_mcc"
        ]
    )

    return result

In [ ]:
def hostname_cluster_bootstrap(
    paired_df,
    n_bootstrap=5000,
    random_state=42
):

    rng = np.random.default_rng(
        random_state
    )

    hostname_groups = {
        hostname:
            group.copy()

        for hostname, group
        in paired_df.groupby(
            "hostname"
        )
    }

    unique_hosts = np.array(
        list(
            hostname_groups.keys()
        ),
        dtype=object
    )

    bootstrap_records = []

    valid_iterations = 0

    while (
        valid_iterations
        <
        n_bootstrap
    ):

        sampled_hosts = rng.choice(
            unique_hosts,
            size=len(
                unique_hosts
            ),
            replace=True
        )

        sampled_frames = [
            hostname_groups[
                host
            ]
            for host in sampled_hosts
        ]

        boot_df = pd.concat(
            sampled_frames,
            ignore_index=True
        )

        # AUROC requires both classes
        if (
            boot_df[
                "y_true"
            ].nunique()
            <
            2
        ):
            continue

        metrics = (
            calculate_metrics_from_df(
                boot_df
            )
        )

        bootstrap_records.append(
            metrics
        )

        valid_iterations += 1

        if (
            valid_iterations
            %
            1000
            ==
            0
        ):
            print(
                "Completed:",
                valid_iterations
            )

    return pd.DataFrame(
        bootstrap_records
    )

In [ ]:
bootstrap_results = (
    hostname_cluster_bootstrap(
        paired_oof,
        n_bootstrap=5000,
        random_state=42
    )
)

bootstrap_results.head()

In [ ]:
def bootstrap_ci(
    series,
    confidence=0.95
):

    alpha = (
        1 - confidence
    )

    lower = np.quantile(
        series,
        alpha / 2
    )

    upper = np.quantile(
        series,
        1 - alpha / 2
    )

    return (
        lower,
        upper
    )

In [ ]:
delta_f1_ci = bootstrap_ci(
    bootstrap_results[
        "delta_f1"
    ]
)

delta_auc_ci = bootstrap_ci(
    bootstrap_results[
        "delta_auc"
    ]
)

delta_mcc_ci = bootstrap_ci(
    bootstrap_results[
        "delta_mcc"
    ]
)

In [ ]:
print(
    "Observed ΔF1:",
    reduced_oof_f1
    -
    full_oof_f1
)

print(
    "95% CI for ΔF1:",
    delta_f1_ci
)

print()

print(
    "Observed ΔAUROC:",
    reduced_oof_auc
    -
    full_oof_auc
)

print(
    "95% CI for ΔAUROC:",
    delta_auc_ci
)

print()

print(
    "Observed ΔMCC:",
    reduced_oof_mcc
    -
    full_oof_mcc
)

print(
    "95% CI for ΔMCC:",
    delta_mcc_ci
)

In [ ]:
NON_INFERIORITY_MARGIN = (
    0.005
)

delta_f1_lower = (
    delta_f1_ci[0]
)

delta_f1_upper = (
    delta_f1_ci[1]
)

In [ ]:
non_inferior = (
    delta_f1_lower
    >
    -NON_INFERIORITY_MARGIN
)

print(
    "Non-inferiority margin:",
    NON_INFERIORITY_MARGIN
)

print(
    "Lower 95% CI of ΔF1:",
    delta_f1_lower
)

print(
    "Non-inferior:",
    non_inferior
)

In [ ]:
prob_worse_than_margin = (
    bootstrap_results["delta_f1"]
    <= -0.005
).mean()

prob_within_margin = (
    bootstrap_results["delta_f1"]
    > -0.005
).mean()

print(
    "P(ΔF1 <= -0.005):",
    prob_worse_than_margin
)

print(
    "P(ΔF1 > -0.005):",
    prob_within_margin
)

In [ ]:
# ============================================================
# Build fold-level paired comparison
# ============================================================

fold_records = []

for fold_id, df_fold in paired_oof.groupby("fold"):

    full_f1 = f1_score(
        df_fold["y_true"],
        df_fold["full_pred"]
    )

    reduced_f1 = f1_score(
        df_fold["y_true"],
        df_fold["reduced_pred"]
    )

    full_auc = roc_auc_score(
        df_fold["y_true"],
        df_fold["full_prob"]
    )

    reduced_auc = roc_auc_score(
        df_fold["y_true"],
        df_fold["reduced_prob"]
    )

    fold_records.append({
        "fold": fold_id,
        "full_f1": full_f1,
        "reduced_f1": reduced_f1,
        "delta_f1": reduced_f1 - full_f1,
        "full_auc": full_auc,
        "reduced_auc": reduced_auc,
        "delta_auc": reduced_auc - full_auc
    })

fold_comparison = pd.DataFrame(
    fold_records
)

fold_comparison

In [ ]:
from scipy.stats import wilcoxon

wilcoxon_f1 = wilcoxon(
    fold_comparison["reduced_f1"],
    fold_comparison["full_f1"],
    alternative="two-sided"
)

print(
    "Wilcoxon statistic:",
    wilcoxon_f1.statistic
)

print(
    "Wilcoxon p-value:",
    wilcoxon_f1.pvalue
)

In [ ]:
print(
    "Reduced wins:",
    (fold_comparison["delta_f1"] > 0).sum()
)

print(
    "Ties:",
    (fold_comparison["delta_f1"] == 0).sum()
)

print(
    "Reduced losses:",
    (fold_comparison["delta_f1"] < 0).sum()
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.hist(
    bootstrap_results["delta_f1"],
    bins=40,
    alpha=0.8
)

plt.axvline(
    0,
    linestyle="--",
    label="No difference"
)

plt.axvline(
    -NON_INFERIORITY_MARGIN,
    linestyle=":",
    label="NI margin = -0.005"
)

plt.axvline(
    delta_f1_ci[0],
    linestyle="-.",
    label="95% CI lower bound"
)

plt.axvline(
    delta_f1_ci[1],
    linestyle="-.",
    label="95% CI upper bound"
)

plt.xlabel(
    "ΔF1 = Reduced − Full"
)

plt.ylabel(
    "Bootstrap Frequency"
)

plt.title(
    "Hostname-Cluster Bootstrap Distribution of ΔF1"
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    fold_comparison["fold"],
    fold_comparison["full_f1"],
    marker="o",
    label="Full 14-feature"
)

plt.plot(
    fold_comparison["fold"],
    fold_comparison["reduced_f1"],
    marker="o",
    label="Reduced 8-feature"
)

plt.xlabel("Grouped-CV Fold")
plt.ylabel("F1")
plt.title(
    "Fold-Level F1 Comparison"
)

plt.xticks(
    fold_comparison["fold"]
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
significance_summary = pd.DataFrame([
    {
        "comparison":
            "Reduced_8_vs_Full_14",

        "full_oof_f1":
            full_oof_f1,

        "reduced_oof_f1":
            reduced_oof_f1,

        "delta_f1":
            reduced_oof_f1
            -
            full_oof_f1,

        "delta_f1_ci_lower":
            delta_f1_ci[0],

        "delta_f1_ci_upper":
            delta_f1_ci[1],

        "full_oof_auc":
            full_oof_auc,

        "reduced_oof_auc":
            reduced_oof_auc,

        "delta_auc":
            reduced_oof_auc
            -
            full_oof_auc,

        "delta_auc_ci_lower":
            delta_auc_ci[0],

        "delta_auc_ci_upper":
            delta_auc_ci[1],

        "full_oof_mcc":
            full_oof_mcc,

        "reduced_oof_mcc":
            reduced_oof_mcc,

        "delta_mcc":
            reduced_oof_mcc
            -
            full_oof_mcc,

        "delta_mcc_ci_lower":
            delta_mcc_ci[0],

        "delta_mcc_ci_upper":
            delta_mcc_ci[1],

        "mcnemar_p":
            mcnemar_p,

        "wilcoxon_p":
            wilcoxon_f1.pvalue,

        "reduced_fold_wins":
            int(
                (
                    fold_comparison[
                        "delta_f1"
                    ] > 0
                ).sum()
            ),

        "reduced_fold_losses":
            int(
                (
                    fold_comparison[
                        "delta_f1"
                    ] < 0
                ).sum()
            ),

        "noninferiority_margin":
            NON_INFERIORITY_MARGIN,

        "formal_noninferiority":
            non_inferior,

        "bootstrap_prob_within_margin":
            prob_within_margin,

        "bootstrap_prob_worse_than_margin":
            prob_worse_than_margin
    }
])

significance_summary

In [ ]:
print("=" * 75)
print("PHASE 13 — FINAL STATISTICAL SUMMARY")
print("=" * 75)

print(
    "Full OOF F1:",
    full_oof_f1
)

print(
    "Reduced OOF F1:",
    reduced_oof_f1
)

print(
    "Observed ΔF1:",
    reduced_oof_f1
    -
    full_oof_f1
)

print(
    "95% CI for ΔF1:",
    delta_f1_ci
)

print()

print(
    "McNemar p-value:",
    mcnemar_p
)

print(
    "Wilcoxon p-value:",
    wilcoxon_f1.pvalue
)

print(
    "Reduced fold wins:",
    (
        fold_comparison[
            "delta_f1"
        ] > 0
    ).sum()
)

print(
    "Reduced fold losses:",
    (
        fold_comparison[
            "delta_f1"
        ] < 0
    ).sum()
)

print()

print(
    "Non-inferiority margin:",
    NON_INFERIORITY_MARGIN
)

print(
    "Formal non-inferiority established:",
    non_inferior
)

print(
    "Bootstrap P(ΔF1 > -0.005):",
    prob_within_margin
)

print()

if non_inferior:

    print(
        "Conclusion: "
        "The reduced model satisfies the "
        "predefined non-inferiority criterion."
    )

else:

    print(
        "Conclusion: "
        "Formal non-inferiority was not established. "
        "However, the observed F1 loss was small and "
        "paired-error tests did not detect a significant "
        "difference."
    )

In [ ]:
paired_oof.to_csv(
    "phase13_paired_grouped_oof_predictions.csv",
    index=False
)

bootstrap_results.to_csv(
    "phase13_hostname_cluster_bootstrap.csv",
    index=False
)

fold_comparison.to_csv(
    "phase13_fold_paired_comparison.csv",
    index=False
)

significance_summary.to_csv(
    "phase13_final_significance_summary.csv",
    index=False
)

print(
    "Phase 13 results saved."
)

In [ ]:
# ============================================================
# Phase 14 — Robustness Evaluation
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score
)

In [ ]:
full_features = list(
    lexical_reduced
)

reduced_features = list(
    final_reduced_lexical_features
)

print(
    "Full feature count:",
    len(full_features)
)

print(
    "Reduced feature count:",
    len(reduced_features)
)

In [ ]:
ROBUSTNESS_SEEDS = [
    11,
    23,
    37,
    42,
    57,
    71,
    89,
    103,
    131,
    157
]

N_SPLITS = 5

print(
    "Number of seeds:",
    len(ROBUSTNESS_SEEDS)
)

print(
    "Total robustness folds:",
    len(ROBUSTNESS_SEEDS)
    *
    N_SPLITS
)

In [ ]:
def evaluate_robustness_repeated_grouped_cv(
    X,
    y,
    groups,
    full_features,
    reduced_features,
    seeds,
    n_splits=5
):

    records = []

    for seed in seeds:

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"Random seed: {seed}"
        )

        print(
            "=" * 70
        )

        cv = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed
        )

        for fold, (
            tr_idx,
            va_idx
        ) in enumerate(
            cv.split(
                X,
                y,
                groups=groups
            ),
            start=1
        ):

            # ------------------------------------------------
            # Leakage check
            # ------------------------------------------------

            train_groups = set(
                groups.iloc[
                    tr_idx
                ]
            )

            val_groups = set(
                groups.iloc[
                    va_idx
                ]
            )

            overlap = len(
                train_groups
                &
                val_groups
            )

            if overlap != 0:
                raise RuntimeError(
                    f"Hostname leakage detected "
                    f"at seed={seed}, fold={fold}"
                )

            # ------------------------------------------------
            # Full 14-feature model
            # ------------------------------------------------

            full_model = (
                make_teacher_model()
            )

            full_model.fit(
                X.iloc[
                    tr_idx
                ][
                    full_features
                ],
                y.iloc[
                    tr_idx
                ]
            )

            full_prob = (
                full_model.predict_proba(
                    X.iloc[
                        va_idx
                    ][
                        full_features
                    ]
                )[:, 1]
            )

            full_pred = (
                full_prob >= 0.5
            ).astype(int)

            # ------------------------------------------------
            # Reduced 8-feature model
            # ------------------------------------------------

            reduced_model = (
                make_teacher_model()
            )

            reduced_model.fit(
                X.iloc[
                    tr_idx
                ][
                    reduced_features
                ],
                y.iloc[
                    tr_idx
                ]
            )

            reduced_prob = (
                reduced_model.predict_proba(
                    X.iloc[
                        va_idx
                    ][
                        reduced_features
                    ]
                )[:, 1]
            )

            reduced_pred = (
                reduced_prob >= 0.5
            ).astype(int)

            # ------------------------------------------------
            # Metrics
            # ------------------------------------------------

            full_f1 = f1_score(
                y.iloc[
                    va_idx
                ],
                full_pred
            )

            reduced_f1 = f1_score(
                y.iloc[
                    va_idx
                ],
                reduced_pred
            )

            full_auc = roc_auc_score(
                y.iloc[
                    va_idx
                ],
                full_prob
            )

            reduced_auc = roc_auc_score(
                y.iloc[
                    va_idx
                ],
                reduced_prob
            )

            full_mcc = matthews_corrcoef(
                y.iloc[
                    va_idx
                ],
                full_pred
            )

            reduced_mcc = matthews_corrcoef(
                y.iloc[
                    va_idx
                ],
                reduced_pred
            )

            full_precision = precision_score(
                y.iloc[
                    va_idx
                ],
                full_pred,
                zero_division=0
            )

            reduced_precision = precision_score(
                y.iloc[
                    va_idx
                ],
                reduced_pred,
                zero_division=0
            )

            full_recall = recall_score(
                y.iloc[
                    va_idx
                ],
                full_pred,
                zero_division=0
            )

            reduced_recall = recall_score(
                y.iloc[
                    va_idx
                ],
                reduced_pred,
                zero_division=0
            )

            records.append({

                "seed":
                    seed,

                "fold":
                    fold,

                "hostname_overlap":
                    overlap,

                "n_val":
                    len(
                        va_idx
                    ),

                "full_f1":
                    full_f1,

                "reduced_f1":
                    reduced_f1,

                "delta_f1":
                    reduced_f1
                    -
                    full_f1,

                "full_auc":
                    full_auc,

                "reduced_auc":
                    reduced_auc,

                "delta_auc":
                    reduced_auc
                    -
                    full_auc,

                "full_mcc":
                    full_mcc,

                "reduced_mcc":
                    reduced_mcc,

                "delta_mcc":
                    reduced_mcc
                    -
                    full_mcc,

                "full_precision":
                    full_precision,

                "reduced_precision":
                    reduced_precision,

                "full_recall":
                    full_recall,

                "reduced_recall":
                    reduced_recall
            })

            print(
                f"Fold {fold} | "
                f"Full F1={full_f1:.4f} | "
                f"Reduced F1={reduced_f1:.4f} | "
                f"Δ={reduced_f1-full_f1:+.4f}"
            )

    return pd.DataFrame(
        records
    )

In [ ]:
robustness_results = (
    evaluate_robustness_repeated_grouped_cv(
        X=X_dev.reset_index(drop=True),
        y=y_dev.reset_index(drop=True),
        groups=groups_dev.reset_index(drop=True),
        full_features=full_features,
        reduced_features=reduced_features,
        seeds=ROBUSTNESS_SEEDS,
        n_splits=N_SPLITS
    )
)

print(
    "Total robustness evaluations:",
    len(
        robustness_results
    )
)

robustness_results.head()

In [ ]:
robustness_summary = pd.DataFrame([
    {
        "model":
            "Full_14",

        "mean_f1":
            robustness_results[
                "full_f1"
            ].mean(),

        "std_f1":
            robustness_results[
                "full_f1"
            ].std(),

        "min_f1":
            robustness_results[
                "full_f1"
            ].min(),

        "max_f1":
            robustness_results[
                "full_f1"
            ].max(),

        "mean_auc":
            robustness_results[
                "full_auc"
            ].mean(),

        "std_auc":
            robustness_results[
                "full_auc"
            ].std(),

        "mean_mcc":
            robustness_results[
                "full_mcc"
            ].mean(),

        "std_mcc":
            robustness_results[
                "full_mcc"
            ].std()
    },

    {
        "model":
            "Reduced_8",

        "mean_f1":
            robustness_results[
                "reduced_f1"
            ].mean(),

        "std_f1":
            robustness_results[
                "reduced_f1"
            ].std(),

        "min_f1":
            robustness_results[
                "reduced_f1"
            ].min(),

        "max_f1":
            robustness_results[
                "reduced_f1"
            ].max(),

        "mean_auc":
            robustness_results[
                "reduced_auc"
            ].mean(),

        "std_auc":
            robustness_results[
                "reduced_auc"
            ].std(),

        "mean_mcc":
            robustness_results[
                "reduced_mcc"
            ].mean(),

        "std_mcc":
            robustness_results[
                "reduced_mcc"
            ].std()
    }
])

robustness_summary

In [ ]:
reduced_wins = (
    robustness_results[
        "delta_f1"
    ] > 0
).sum()

ties = (
    robustness_results[
        "delta_f1"
    ] == 0
).sum()

reduced_losses = (
    robustness_results[
        "delta_f1"
    ] < 0
).sum()

print(
    "Reduced wins:",
    reduced_wins
)

print(
    "Ties:",
    ties
)

print(
    "Reduced losses:",
    reduced_losses
)

print(
    "Reduced win rate:",
    reduced_wins
    /
    len(
        robustness_results
    )
)

In [ ]:
delta_summary = pd.Series({
    "mean_delta_f1":
        robustness_results[
            "delta_f1"
        ].mean(),

    "std_delta_f1":
        robustness_results[
            "delta_f1"
        ].std(),

    "median_delta_f1":
        robustness_results[
            "delta_f1"
        ].median(),

    "min_delta_f1":
        robustness_results[
            "delta_f1"
        ].min(),

    "max_delta_f1":
        robustness_results[
            "delta_f1"
        ].max(),

    "mean_delta_auc":
        robustness_results[
            "delta_auc"
        ].mean(),

    "mean_delta_mcc":
        robustness_results[
            "delta_mcc"
        ].mean()
})

delta_summary

In [ ]:
ROBUSTNESS_MARGIN = 0.005

within_margin = (
    robustness_results[
        "delta_f1"
    ]
    >
    -ROBUSTNESS_MARGIN
)

print(
    "Folds within -0.005 margin:",
    within_margin.sum(),
    "/",
    len(
        within_margin
    )
)

print(
    "Proportion within margin:",
    within_margin.mean()
)

In [ ]:
seed_summary = (
    robustness_results
    .groupby(
        "seed"
    )
    .agg(
        full_mean_f1=(
            "full_f1",
            "mean"
        ),

        reduced_mean_f1=(
            "reduced_f1",
            "mean"
        ),

        mean_delta_f1=(
            "delta_f1",
            "mean"
        ),

        full_mean_auc=(
            "full_auc",
            "mean"
        ),

        reduced_mean_auc=(
            "reduced_auc",
            "mean"
        ),

        mean_delta_auc=(
            "delta_auc",
            "mean"
        ),

        full_mean_mcc=(
            "full_mcc",
            "mean"
        ),

        reduced_mean_mcc=(
            "reduced_mcc",
            "mean"
        ),

        mean_delta_mcc=(
            "delta_mcc",
            "mean"
        )
    )
    .reset_index()
)

seed_summary

In [ ]:
seed_wins = (
    seed_summary[
        "mean_delta_f1"
    ] > 0
).sum()

seed_losses = (
    seed_summary[
        "mean_delta_f1"
    ] < 0
).sum()

print(
    "Reduced wins across seeds:",
    seed_wins
)

print(
    "Reduced losses across seeds:",
    seed_losses
)

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.hist(
    robustness_results[
        "delta_f1"
    ],
    bins=20,
    alpha=0.8
)

plt.axvline(
    0,
    linestyle="--",
    label="No difference"
)

plt.axvline(
    -0.005,
    linestyle=":",
    label="Tolerance = -0.005"
)

plt.xlabel(
    "ΔF1 = Reduced − Full"
)

plt.ylabel(
    "Number of Repeated Grouped-CV Folds"
)

plt.title(
    "Robustness of F1 Difference Across Repeated Grouped CV"
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.plot(
    seed_summary[
        "seed"
    ],
    seed_summary[
        "full_mean_f1"
    ],
    marker="o",
    label="Full 14-feature"
)

plt.plot(
    seed_summary[
        "seed"
    ],
    seed_summary[
        "reduced_mean_f1"
    ],
    marker="o",
    label="Reduced 8-feature"
)

plt.xlabel(
    "Random Seed"
)

plt.ylabel(
    "Mean 5-fold F1"
)

plt.title(
    "Robustness Across Grouped-CV Random Seeds"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.plot(
    seed_summary[
        "seed"
    ],
    seed_summary[
        "mean_delta_f1"
    ],
    marker="o"
)

plt.axhline(
    0,
    linestyle="--"
)

plt.axhline(
    -0.005,
    linestyle=":"
)

plt.xlabel(
    "Random Seed"
)

plt.ylabel(
    "Mean ΔF1"
)

plt.title(
    "Seed-Level Stability of Reduced vs Full Model"
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
rng = np.random.default_rng(
    42
)

seed_delta_values = (
    seed_summary[
        "mean_delta_f1"
    ].to_numpy()
)

bootstrap_seed_means = []

for _ in range(10000):

    sample = rng.choice(
        seed_delta_values,
        size=len(
            seed_delta_values
        ),
        replace=True
    )

    bootstrap_seed_means.append(
        sample.mean()
    )

bootstrap_seed_means = (
    np.asarray(
        bootstrap_seed_means
    )
)

In [ ]:
seed_delta_ci = (
    np.quantile(
        bootstrap_seed_means,
        0.025
    ),
    np.quantile(
        bootstrap_seed_means,
        0.975
    )
)

print(
    "Mean seed-level ΔF1:",
    seed_delta_values.mean()
)

print(
    "95% bootstrap CI:",
    seed_delta_ci
)

In [ ]:
full_seed_cv = (
    seed_summary[
        "full_mean_f1"
    ].std()
    /
    seed_summary[
        "full_mean_f1"
    ].mean()
)

reduced_seed_cv = (
    seed_summary[
        "reduced_mean_f1"
    ].std()
    /
    seed_summary[
        "reduced_mean_f1"
    ].mean()
)

print(
    "Full model seed-level coefficient of variation:",
    full_seed_cv
)

print(
    "Reduced model seed-level coefficient of variation:",
    reduced_seed_cv
)

In [ ]:
robustness_final_summary = pd.DataFrame([
    {
        "model":
            "Full_14",

        "mean_f1_50fold":
            robustness_results[
                "full_f1"
            ].mean(),

        "std_f1_50fold":
            robustness_results[
                "full_f1"
            ].std(),

        "mean_auc_50fold":
            robustness_results[
                "full_auc"
            ].mean(),

        "mean_mcc_50fold":
            robustness_results[
                "full_mcc"
            ].mean(),

        "seed_level_cv":
            full_seed_cv
    },

    {
        "model":
            "Reduced_8",

        "mean_f1_50fold":
            robustness_results[
                "reduced_f1"
            ].mean(),

        "std_f1_50fold":
            robustness_results[
                "reduced_f1"
            ].std(),

        "mean_auc_50fold":
            robustness_results[
                "reduced_auc"
            ].mean(),

        "mean_mcc_50fold":
            robustness_results[
                "reduced_mcc"
            ].mean(),

        "seed_level_cv":
            reduced_seed_cv
    }
])

robustness_final_summary

In [ ]:
print("=" * 75)
print("PHASE 14 — ROBUSTNESS SUMMARY")
print("=" * 75)

print(
    "Repeated grouped-CV runs:",
    len(
        robustness_results
    )
)

print()

print(
    "Full mean F1:",
    robustness_results[
        "full_f1"
    ].mean()
)

print(
    "Reduced mean F1:",
    robustness_results[
        "reduced_f1"
    ].mean()
)

print(
    "Mean ΔF1:",
    robustness_results[
        "delta_f1"
    ].mean()
)

print(
    "Median ΔF1:",
    robustness_results[
        "delta_f1"
    ].median()
)

print()

print(
    "Reduced fold wins:",
    reduced_wins
)

print(
    "Reduced fold losses:",
    reduced_losses
)

print(
    "Proportion within -0.005 margin:",
    within_margin.mean()
)

print()

print(
    "Mean seed-level ΔF1:",
    seed_delta_values.mean()
)

print(
    "95% seed-bootstrap CI:",
    seed_delta_ci
)

print()

print(
    "Reduced seed-level CV:",
    reduced_seed_cv
)

In [ ]:
robustness_results.to_csv(
    "phase14_repeated_grouped_cv_results.csv",
    index=False
)

seed_summary.to_csv(
    "phase14_seed_level_summary.csv",
    index=False
)

robustness_final_summary.to_csv(
    "phase14_robustness_summary.csv",
    index=False
)

pd.DataFrame({
    "bootstrap_seed_mean_delta_f1":
        bootstrap_seed_means
}).to_csv(
    "phase14_seed_bootstrap_delta_f1.csv",
    index=False
)

print(
    "Phase 14 robustness results saved."
)

In [ ]:
possible_vars = [
    "features_14",
    "features_12",
    "features_10",
    "features_8",
    "top14_features",
    "top12_features",
    "top10_features",
    "top8_features",
    "final_reduced_lexical_features"
]

for name in possible_vars:
    print(
        name,
        "FOUND" if name in globals() else "NOT FOUND"
    )

In [ ]:
possible_ranking_vars = [
    "feature_ranking",
    "ranked_features",
    "feature_importance_df",
    "permutation_importance_df",
    "importance_df",
    "feature_budget_results",
    "feature_budget_summary",
    "budget_results",
    "selected_features",
    "lexical_reduced"
]

for name in possible_ranking_vars:
    if name in globals():
        print(
            "\nFOUND:",
            name
        )
        obj = globals()[name]

        try:
            print(obj)
        except:
            print(type(obj))

In [ ]:
for name, obj in list(globals().items()):

    name_lower = name.lower()

    if (
        "feature" in name_lower
        or "importance" in name_lower
        or "rank" in name_lower
        or "budget" in name_lower
    ):
        if isinstance(
            obj,
            (
                list,
                dict,
                pd.DataFrame,
                pd.Series
            )
        ):
            print(
                name,
                type(obj),
                getattr(obj, "shape", "")
            )

In [ ]:
final_dev_ranking

In [ ]:
print(final_dev_ranking.columns)

In [ ]:
print(final_dev_ranking)
print("\nColumns:")
print(final_dev_ranking.columns.tolist())

In [ ]:
ranked_feature_list = (
    final_dev_ranking[
        "feature"
    ]
    .tolist()
)

print(
    "Total ranked features:",
    len(ranked_feature_list)
)

for i, f in enumerate(
    ranked_feature_list,
    start=1
):
    print(i, f)

In [ ]:
feature_budget_sets = {
    14: ranked_feature_list[:14],
    12: ranked_feature_list[:12],
    10: ranked_feature_list[:10],
    8: ranked_feature_list[:8]
}

In [ ]:
for budget, feats in feature_budget_sets.items():

    print("\n", "=" * 50)
    print("Feature budget:", budget)
    print("=" * 50)

    for i, f in enumerate(
        feats,
        start=1
    ):
        print(i, f)

In [ ]:
print(
    "Recovered Top-8:"
)

print(
    feature_budget_sets[8]
)

print()

print(
    "Previously frozen 8:"
)

print(
    final_reduced_lexical_features
)

print()

print(
    "Exact same order:",
    feature_budget_sets[8]
    ==
    final_reduced_lexical_features
)

print(
    "Same feature set:",
    set(
        feature_budget_sets[8]
    )
    ==
    set(
        final_reduced_lexical_features
    )
)

In [ ]:
final_dev_ranking

In [ ]:
# ============================================================
# Recover fixed feature-budget sets from Phase 12C ranking
# ============================================================

ranked_feature_list = (
    final_dev_ranking
    .sort_values("rank")
    ["feature"]
    .tolist()
)

feature_budget_sets = {
    14: ranked_feature_list[:14],
    12: ranked_feature_list[:12],
    10: ranked_feature_list[:10],
    8: ranked_feature_list[:8]
}

for budget, feats in feature_budget_sets.items():
    print("\n" + "=" * 55)
    print(f"{budget}-feature fixed set")
    print("=" * 55)

    for i, feature in enumerate(feats, start=1):
        print(i, feature)

In [ ]:
assert (
    feature_budget_sets[8]
    ==
    final_reduced_lexical_features
)

assert (
    feature_budget_sets[14]
    ==
    ranked_feature_list
)

print(
    "Feature-budget reconstruction verified."
)

In [ ]:
print(
    "feature_budget_sets exists:",
    "feature_budget_sets" in globals()
)

In [ ]:
feature_budget_sets

In [ ]:
ROBUSTNESS_SEEDS = [
    11,
    23,
    37,
    42,
    57,
    71,
    89,
    103,
    131,
    157
]

N_SPLITS = 5

F1_MARGIN = 0.005

In [ ]:
def evaluate_fixed_feature_budget_robustness(
    X,
    y,
    groups,
    feature_budget_sets,
    seeds,
    n_splits=5
):

    records = []

    budgets = sorted(
        feature_budget_sets.keys(),
        reverse=True
    )

    for seed in seeds:

        print(
            "\n"
            + "=" * 75
        )

        print(
            f"Seed {seed}"
        )

        print(
            "=" * 75
        )

        cv = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed
        )

        for fold, (
            tr_idx,
            va_idx
        ) in enumerate(
            cv.split(
                X,
                y,
                groups=groups
            ),
            start=1
        ):

            train_groups = set(
                groups.iloc[tr_idx]
            )

            val_groups = set(
                groups.iloc[va_idx]
            )

            overlap = len(
                train_groups
                &
                val_groups
            )

            if overlap != 0:
                raise RuntimeError(
                    f"Hostname leakage detected: "
                    f"seed={seed}, fold={fold}"
                )

            print(
                f"\nFold {fold}"
            )

            for budget in budgets:

                features = (
                    feature_budget_sets[
                        budget
                    ]
                )

                model = (
                    make_teacher_model()
                )

                model.fit(
                    X.iloc[
                        tr_idx
                    ][
                        features
                    ],
                    y.iloc[
                        tr_idx
                    ]
                )

                prob = (
                    model.predict_proba(
                        X.iloc[
                            va_idx
                        ][
                            features
                        ]
                    )[:, 1]
                )

                pred = (
                    prob >= 0.5
                ).astype(int)

                f1 = f1_score(
                    y.iloc[
                        va_idx
                    ],
                    pred
                )

                auc = roc_auc_score(
                    y.iloc[
                        va_idx
                    ],
                    prob
                )

                mcc = matthews_corrcoef(
                    y.iloc[
                        va_idx
                    ],
                    pred
                )

                precision = precision_score(
                    y.iloc[
                        va_idx
                    ],
                    pred,
                    zero_division=0
                )

                recall = recall_score(
                    y.iloc[
                        va_idx
                    ],
                    pred,
                    zero_division=0
                )

                records.append({

                    "seed":
                        seed,

                    "fold":
                        fold,

                    "feature_count":
                        budget,

                    "f1":
                        f1,

                    "auc":
                        auc,

                    "mcc":
                        mcc,

                    "precision":
                        precision,

                    "recall":
                        recall,

                    "hostname_overlap":
                        overlap
                })

                print(
                    f"{budget:>2} features | "
                    f"F1={f1:.4f} | "
                    f"AUC={auc:.4f}"
                )

    return pd.DataFrame(
        records
    )

In [ ]:
budget_robustness_results = (
    evaluate_fixed_feature_budget_robustness(
        X=X_dev.reset_index(drop=True),
        y=y_dev.reset_index(drop=True),
        groups=groups_dev.reset_index(drop=True),
        feature_budget_sets=feature_budget_sets,
        seeds=ROBUSTNESS_SEEDS,
        n_splits=N_SPLITS
    )
)

print(
    "Total evaluations:",
    len(
        budget_robustness_results
    )
)

budget_robustness_results.head()

In [ ]:
reference_14 = (
    budget_robustness_results[
        budget_robustness_results[
            "feature_count"
        ] == 14
    ][
        [
            "seed",
            "fold",
            "f1",
            "auc",
            "mcc"
        ]
    ]
    .rename(
        columns={
            "f1":
                "reference_f1",

            "auc":
                "reference_auc",

            "mcc":
                "reference_mcc"
        }
    )
)

In [ ]:
budget_robustness_results = (
    budget_robustness_results
    .merge(
        reference_14,
        on=[
            "seed",
            "fold"
        ],
        how="left"
    )
)

In [ ]:
budget_robustness_results[
    "delta_f1"
] = (
    budget_robustness_results[
        "f1"
    ]
    -
    budget_robustness_results[
        "reference_f1"
    ]
)

budget_robustness_results[
    "delta_auc"
] = (
    budget_robustness_results[
        "auc"
    ]
    -
    budget_robustness_results[
        "reference_auc"
    ]
)

budget_robustness_results[
    "delta_mcc"
] = (
    budget_robustness_results[
        "mcc"
    ]
    -
    budget_robustness_results[
        "reference_mcc"
    ]
)

In [ ]:
budget_summary = (
    budget_robustness_results
    .groupby(
        "feature_count"
    )
    .agg(
        mean_f1=(
            "f1",
            "mean"
        ),

        std_f1=(
            "f1",
            "std"
        ),

        mean_auc=(
            "auc",
            "mean"
        ),

        mean_mcc=(
            "mcc",
            "mean"
        ),

        mean_delta_f1=(
            "delta_f1",
            "mean"
        ),

        median_delta_f1=(
            "delta_f1",
            "median"
        ),

        min_delta_f1=(
            "delta_f1",
            "min"
        ),

        max_delta_f1=(
            "delta_f1",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        "feature_count",
        ascending=False
    )
)

budget_summary

In [ ]:
margin_summary = []

for budget in sorted(
    feature_budget_sets.keys(),
    reverse=True
):

    subset = (
        budget_robustness_results[
            budget_robustness_results[
                "feature_count"
            ] == budget
        ]
    )

    within_margin = (
        subset[
            "delta_f1"
        ]
        >
        -F1_MARGIN
    )

    margin_summary.append({

        "feature_count":
            budget,

        "folds_within_margin":
            int(
                within_margin.sum()
            ),

        "total_folds":
            len(
                subset
            ),

        "proportion_within_margin":
            within_margin.mean(),

        "mean_delta_f1":
            subset[
                "delta_f1"
            ].mean()
    })


margin_summary = pd.DataFrame(
    margin_summary
)

margin_summary

In [ ]:
budget_seed_summary = (
    budget_robustness_results
    .groupby(
        [
            "feature_count",
            "seed"
        ]
    )
    .agg(
        mean_f1=(
            "f1",
            "mean"
        ),

        mean_auc=(
            "auc",
            "mean"
        ),

        mean_mcc=(
            "mcc",
            "mean"
        ),

        mean_delta_f1=(
            "delta_f1",
            "mean"
        )
    )
    .reset_index()
)

budget_seed_summary.head()

In [ ]:
def bootstrap_seed_level_delta_ci(
    seed_delta_values,
    n_bootstrap=10000,
    random_state=42
):

    rng = np.random.default_rng(
        random_state
    )

    values = np.asarray(
        seed_delta_values
    )

    boot_means = []

    for _ in range(
        n_bootstrap
    ):

        sampled = rng.choice(
            values,
            size=len(
                values
            ),
            replace=True
        )

        boot_means.append(
            sampled.mean()
        )

    boot_means = np.asarray(
        boot_means
    )

    return {
        "mean_delta_f1":
            values.mean(),

        "ci_lower":
            np.quantile(
                boot_means,
                0.025
            ),

        "ci_upper":
            np.quantile(
                boot_means,
                0.975
            )
    }

In [ ]:
budget_ci_records = []

for budget in sorted(
    feature_budget_sets.keys(),
    reverse=True
):

    seed_values = (
        budget_seed_summary[
            budget_seed_summary[
                "feature_count"
            ] == budget
        ][
            "mean_delta_f1"
        ]
        .to_numpy()
    )

    ci_result = (
        bootstrap_seed_level_delta_ci(
            seed_values,
            n_bootstrap=10000,
            random_state=42
        )
    )

    budget_ci_records.append({

        "feature_count":
            budget,

        "mean_delta_f1":
            ci_result[
                "mean_delta_f1"
            ],

        "ci_lower":
            ci_result[
                "ci_lower"
            ],

        "ci_upper":
            ci_result[
                "ci_upper"
            ]
    })


budget_ci_summary = pd.DataFrame(
    budget_ci_records
)

budget_ci_summary

In [ ]:
budget_selection_table = (
    budget_summary[
        [
            "feature_count",
            "mean_f1",
            "std_f1",
            "mean_auc",
            "mean_mcc",
            "mean_delta_f1"
        ]
    ]
    .merge(
        margin_summary[
            [
                "feature_count",
                "proportion_within_margin"
            ]
        ],
        on="feature_count"
    )
    .merge(
        budget_ci_summary[
            [
                "feature_count",
                "ci_lower",
                "ci_upper"
            ]
        ],
        on="feature_count"
    )
)

In [ ]:
budget_selection_table[
    "mean_within_margin"
] = (
    budget_selection_table[
        "mean_delta_f1"
    ]
    >
    -F1_MARGIN
)

budget_selection_table[
    "ci_within_margin"
] = (
    budget_selection_table[
        "ci_lower"
    ]
    >
    -F1_MARGIN
)

budget_selection_table[
    "robust_pass"
] = (
    budget_selection_table[
        "mean_within_margin"
    ]
    &
    budget_selection_table[
        "ci_within_margin"
    ]
)

budget_selection_table

In [ ]:
passing_budgets = (
    budget_selection_table[
        budget_selection_table[
            "robust_pass"
        ]
    ]
    .sort_values(
        "feature_count",
        ascending=True
    )
)

In [ ]:
if len(
    passing_budgets
) > 0:

    selected_robust_budget = int(
        passing_budgets
        .iloc[0][
            "feature_count"
        ]
    )

else:

    selected_robust_budget = 14

In [ ]:
print(
    "Selected robust feature budget:",
    selected_robust_budget
)

In [ ]:
plot_summary = (
    budget_summary
    .sort_values(
        "feature_count"
    )
)

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    plot_summary[
        "feature_count"
    ],
    plot_summary[
        "mean_f1"
    ],
    marker="o"
)

plt.xlabel(
    "Number of Lexical Features"
)

plt.ylabel(
    "Mean F1 Across 50 Grouped-CV Runs"
)

plt.title(
    "Robustness-Guided Feature Budget Analysis"
)

plt.xticks(
    plot_summary[
        "feature_count"
    ]
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    plot_summary[
        "feature_count"
    ],
    plot_summary[
        "mean_delta_f1"
    ],
    marker="o"
)

plt.axhline(
    0,
    linestyle="--",
    label="Full-model reference"
)

plt.axhline(
    -F1_MARGIN,
    linestyle=":",
    label="Tolerance = -0.005"
)

plt.xlabel(
    "Number of Lexical Features"
)

plt.ylabel(
    "Mean ΔF1 vs 14-feature model"
)

plt.title(
    "Compression–Robustness Trade-off"
)

plt.xticks(
    plot_summary[
        "feature_count"
    ]
)

plt.legend()
plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
ci_plot = (
    budget_ci_summary
    .sort_values(
        "feature_count"
    )
)

yerr_lower = (
    ci_plot[
        "mean_delta_f1"
    ]
    -
    ci_plot[
        "ci_lower"
    ]
)

yerr_upper = (
    ci_plot[
        "ci_upper"
    ]
    -
    ci_plot[
        "mean_delta_f1"
    ]
)

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    ci_plot[
        "feature_count"
    ],
    ci_plot[
        "mean_delta_f1"
    ],
    yerr=[
        yerr_lower,
        yerr_upper
    ],
    marker="o",
    capsize=5
)

plt.axhline(
    0,
    linestyle="--"
)

plt.axhline(
    -F1_MARGIN,
    linestyle=":"
)

plt.xlabel(
    "Number of Lexical Features"
)

plt.ylabel(
    "Seed-Level Mean ΔF1 with 95% CI"
)

plt.title(
    "Robustness Confidence Intervals Across Feature Budgets"
)

plt.xticks(
    ci_plot[
        "feature_count"
    ]
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
print(
    "=" * 78
)

print(
    "PHASE 14B — ROBUSTNESS-GUIDED FEATURE BUDGET SUMMARY"
)

print(
    "=" * 78
)

print()

print(
    budget_selection_table[
        [
            "feature_count",
            "mean_f1",
            "mean_delta_f1",
            "proportion_within_margin",
            "ci_lower",
            "ci_upper",
            "robust_pass"
        ]
    ]
    .sort_values(
        "feature_count",
        ascending=False
    )
    .to_string(
        index=False
    )
)

print()

print(
    "Selected robust feature budget:",
    selected_robust_budget
)

print(
    "Feature reduction (%):",
    100
    *
    (
        1
        -
        selected_robust_budget
        /
        14
    )
)

In [ ]:
budget_robustness_results.to_csv(
    "phase14B_feature_budget_robustness_all_folds.csv",
    index=False
)

budget_seed_summary.to_csv(
    "phase14B_feature_budget_seed_summary.csv",
    index=False
)

budget_ci_summary.to_csv(
    "phase14B_feature_budget_bootstrap_ci.csv",
    index=False
)

budget_selection_table.to_csv(
    "phase14B_feature_budget_selection.csv",
    index=False
)

In [ ]:
final_robust_features = list(
    feature_budget_sets[
        selected_robust_budget
    ]
)

print(
    "Selected robust feature budget:",
    selected_robust_budget
)

print(
    "Final robust features:"
)

for i, feature in enumerate(
    final_robust_features,
    start=1
):
    print(i, feature)

In [ ]:
import json

with open(
    "phase14B_final_robust_features.json",
    "w"
) as f:

    json.dump(
        {
            "selected_feature_count":
                int(selected_robust_budget),

            "features":
                final_robust_features,

            "f1_margin":
                float(F1_MARGIN),

            "selection_rule":
                (
                    "Smallest fixed feature budget whose "
                    "mean delta F1 and lower 95% seed-bootstrap "
                    "CI both remain above -0.005."
                )
        },
        f,
        indent=4
    )

print(
    "Final robust feature configuration saved successfully."
)

In [ ]:
# ============================================================
# Phase 15 — Final 10-Feature Candidate Confirmation
# ============================================================

final_full_features = list(
    feature_budget_sets[14]
)

final_lightweight_features = list(
    final_robust_features
)

print(
    "Full feature count:",
    len(final_full_features)
)

print(
    "Final lightweight feature count:",
    len(final_lightweight_features)
)

print(
    "\nFinal lightweight features:"
)

for i, feature in enumerate(
    final_lightweight_features,
    start=1
):
    print(i, feature)

In [ ]:
def generate_final_paired_oof(
    X,
    y,
    groups,
    full_features,
    lightweight_features,
    n_splits=5,
    random_state=42
):

    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    records = []

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X,
            y,
            groups=groups
        ),
        start=1
    ):

        # Full model
        full_model = make_teacher_model()

        full_model.fit(
            X.iloc[tr_idx][full_features],
            y.iloc[tr_idx]
        )

        full_prob = (
            full_model.predict_proba(
                X.iloc[va_idx][full_features]
            )[:, 1]
        )

        full_pred = (
            full_prob >= 0.5
        ).astype(int)

        # Final 10-feature model
        light_model = make_teacher_model()

        light_model.fit(
            X.iloc[tr_idx][lightweight_features],
            y.iloc[tr_idx]
        )

        light_prob = (
            light_model.predict_proba(
                X.iloc[va_idx][lightweight_features]
            )[:, 1]
        )

        light_pred = (
            light_prob >= 0.5
        ).astype(int)

        # Leakage audit
        train_hosts = set(
            groups.iloc[tr_idx]
        )

        val_hosts = set(
            groups.iloc[va_idx]
        )

        overlap = len(
            train_hosts & val_hosts
        )

        print(
            f"Fold {fold} | "
            f"Hostname overlap={overlap}"
        )

        if overlap != 0:
            raise RuntimeError(
                "Hostname leakage detected."
            )

        for j, original_idx in enumerate(
            va_idx
        ):

            records.append({
                "dev_index":
                    int(original_idx),

                "fold":
                    fold,

                "hostname":
                    groups.iloc[
                        original_idx
                    ],

                "y_true":
                    int(
                        y.iloc[
                            original_idx
                        ]
                    ),

                "full_prob":
                    float(
                        full_prob[j]
                    ),

                "full_pred":
                    int(
                        full_pred[j]
                    ),

                "light_prob":
                    float(
                        light_prob[j]
                    ),

                "light_pred":
                    int(
                        light_pred[j]
                    )
            })

    return pd.DataFrame(
        records
    )

In [ ]:
final_paired_oof = (
    generate_final_paired_oof(
        X=X_dev.reset_index(drop=True),
        y=y_dev.reset_index(drop=True),
        groups=groups_dev.reset_index(drop=True),
        full_features=final_full_features,
        lightweight_features=final_lightweight_features,
        n_splits=5,
        random_state=42
    )
)

print(
    "OOF samples:",
    len(final_paired_oof)
)

print(
    "Unique samples:",
    final_paired_oof[
        "dev_index"
    ].nunique()
)

print(
    "Duplicate samples:",
    final_paired_oof[
        "dev_index"
    ].duplicated().sum()
)

In [ ]:
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    accuracy_score
)

In [ ]:
full_final_f1 = f1_score(
    final_paired_oof["y_true"],
    final_paired_oof["full_pred"]
)

light_final_f1 = f1_score(
    final_paired_oof["y_true"],
    final_paired_oof["light_pred"]
)

full_final_auc = roc_auc_score(
    final_paired_oof["y_true"],
    final_paired_oof["full_prob"]
)

light_final_auc = roc_auc_score(
    final_paired_oof["y_true"],
    final_paired_oof["light_prob"]
)

full_final_mcc = matthews_corrcoef(
    final_paired_oof["y_true"],
    final_paired_oof["full_pred"]
)

light_final_mcc = matthews_corrcoef(
    final_paired_oof["y_true"],
    final_paired_oof["light_pred"]
)

In [ ]:
print(
    "Full 14-feature F1:",
    full_final_f1
)

print(
    "Final 10-feature F1:",
    light_final_f1
)

print(
    "ΔF1:",
    light_final_f1
    -
    full_final_f1
)

print()

print(
    "Full AUC:",
    full_final_auc
)

print(
    "Final AUC:",
    light_final_auc
)

print(
    "ΔAUC:",
    light_final_auc
    -
    full_final_auc
)

print()

print(
    "Full MCC:",
    full_final_mcc
)

print(
    "Final MCC:",
    light_final_mcc
)

In [ ]:
from scipy.stats import binomtest

In [ ]:
full_correct = (
    final_paired_oof[
        "full_pred"
    ]
    ==
    final_paired_oof[
        "y_true"
    ]
)

light_correct = (
    final_paired_oof[
        "light_pred"
    ]
    ==
    final_paired_oof[
        "y_true"
    ]
)

full_only_correct = (
    full_correct
    &
    ~light_correct
).sum()

light_only_correct = (
    ~full_correct
    &
    light_correct
).sum()

discordant = (
    full_only_correct
    +
    light_only_correct
)

print(
    "Full-only correct:",
    full_only_correct
)

print(
    "10-feature-only correct:",
    light_only_correct
)

print(
    "Discordant:",
    discordant
)

In [ ]:
if discordant > 0:

    final_mcnemar = binomtest(
        k=min(
            full_only_correct,
            light_only_correct
        ),
        n=discordant,
        p=0.5,
        alternative="two-sided"
    )

    final_mcnemar_p = (
        final_mcnemar.pvalue
    )

else:
    final_mcnemar_p = 1.0


print(
    "Exact McNemar p-value:",
    final_mcnemar_p
)

In [ ]:
def final_metric_deltas(df):

    y_true = (
        df["y_true"]
        .to_numpy()
    )

    full_f1 = f1_score(
        y_true,
        df["full_pred"]
    )

    light_f1 = f1_score(
        y_true,
        df["light_pred"]
    )

    full_auc = roc_auc_score(
        y_true,
        df["full_prob"]
    )

    light_auc = roc_auc_score(
        y_true,
        df["light_prob"]
    )

    full_mcc = matthews_corrcoef(
        y_true,
        df["full_pred"]
    )

    light_mcc = matthews_corrcoef(
        y_true,
        df["light_pred"]
    )

    return {
        "full_f1":
            full_f1,

        "light_f1":
            light_f1,

        "delta_f1":
            light_f1 - full_f1,

        "full_auc":
            full_auc,

        "light_auc":
            light_auc,

        "delta_auc":
            light_auc - full_auc,

        "full_mcc":
            full_mcc,

        "light_mcc":
            light_mcc,

        "delta_mcc":
            light_mcc - full_mcc
    }

In [ ]:
def final_hostname_cluster_bootstrap(
    paired_df,
    n_bootstrap=5000,
    random_state=42
):

    rng = np.random.default_rng(
        random_state
    )

    host_groups = {
        host: df.copy()

        for host, df
        in paired_df.groupby(
            "hostname"
        )
    }

    hosts = np.array(
        list(
            host_groups.keys()
        ),
        dtype=object
    )

    records = []

    completed = 0

    while completed < n_bootstrap:

        sampled_hosts = rng.choice(
            hosts,
            size=len(hosts),
            replace=True
        )

        boot_df = pd.concat(
            [
                host_groups[h]
                for h in sampled_hosts
            ],
            ignore_index=True
        )

        if (
            boot_df[
                "y_true"
            ].nunique()
            <
            2
        ):
            continue

        records.append(
            final_metric_deltas(
                boot_df
            )
        )

        completed += 1

        if completed % 1000 == 0:
            print(
                "Completed:",
                completed
            )

    return pd.DataFrame(
        records
    )

In [ ]:
final_bootstrap = (
    final_hostname_cluster_bootstrap(
        final_paired_oof,
        n_bootstrap=5000,
        random_state=42
    )
)

In [ ]:
final_delta_f1_ci = (
    np.quantile(
        final_bootstrap[
            "delta_f1"
        ],
        0.025
    ),
    np.quantile(
        final_bootstrap[
            "delta_f1"
        ],
        0.975
    )
)

final_delta_auc_ci = (
    np.quantile(
        final_bootstrap[
            "delta_auc"
        ],
        0.025
    ),
    np.quantile(
        final_bootstrap[
            "delta_auc"
        ],
        0.975
    )
)

final_delta_mcc_ci = (
    np.quantile(
        final_bootstrap[
            "delta_mcc"
        ],
        0.025
    ),
    np.quantile(
        final_bootstrap[
            "delta_mcc"
        ],
        0.975
    )
)

In [ ]:
FINAL_NI_MARGIN = 0.005

final_noninferior = (
    final_delta_f1_ci[0]
    >
    -FINAL_NI_MARGIN
)

print(
    "Observed ΔF1:",
    light_final_f1
    -
    full_final_f1
)

print(
    "95% CI ΔF1:",
    final_delta_f1_ci
)

print(
    "NI margin:",
    FINAL_NI_MARGIN
)

print(
    "Formal non-inferiority:",
    final_noninferior
)

In [ ]:
# ============================================================
# Phase 16 — Final Lightweight Efficiency Evaluation
# ============================================================

import os
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
final_full_features = list(
    feature_budget_sets[14]
)

final_light_features = list(
    final_robust_features
)

print(
    "Full feature count:",
    len(final_full_features)
)

print(
    "Lightweight feature count:",
    len(final_light_features)
)

print(
    "\nFinal lightweight features:"
)

for i, feature in enumerate(
    final_light_features,
    start=1
):
    print(i, feature)

In [ ]:
final_full_model = make_teacher_model()

final_full_model.fit(
    X_dev[
        final_full_features
    ],
    y_dev
)


final_light_model = make_teacher_model()

final_light_model.fit(
    X_dev[
        final_light_features
    ],
    y_dev
)

print(
    "Final full and lightweight models trained."
)

In [ ]:
joblib.dump(
    final_full_model,
    "phase16_full14_teacher.pkl"
)

joblib.dump(
    final_light_model,
    "phase16_light10_teacher.pkl"
)

In [ ]:
full_model_size_kb = (
    os.path.getsize(
        "phase16_full14_teacher.pkl"
    )
    /
    1024
)

light_model_size_kb = (
    os.path.getsize(
        "phase16_light10_teacher.pkl"
    )
    /
    1024
)

model_size_change = (
    light_model_size_kb
    /
    full_model_size_kb
    -
    1
)

print(
    "Full model size (KB):",
    full_model_size_kb
)

print(
    "Lightweight model size (KB):",
    light_model_size_kb
)

print(
    "Relative model-size change (%):",
    100 * model_size_change
)

In [ ]:
def extract_final_10_features(
    url,
    hostname,
    domain
):

    url = str(url).strip()

    domain = (
        ""
        if pd.isna(domain)
        else str(domain).strip().lower()
    )

    (
        url,
        parsed,
        parsed_hostname,
        path,
        query,
        fragment
    ) = safe_parse_url(url)

    parsed_hostname = (
        ""
        if parsed_hostname is None
        else str(
            parsed_hostname
        ).lower()
    )

    url_lower = url.lower()

    # --------------------------------------------------------
    # num_subdomains
    # --------------------------------------------------------

    hostname_parts = [
        x
        for x
        in parsed_hostname.split(".")
        if x
    ]

    domain_parts = [
        x
        for x
        in domain.split(".")
        if x
    ]

    num_subdomains = max(
        len(hostname_parts)
        -
        len(domain_parts),
        0
    )

    # --------------------------------------------------------
    # suspicious_keyword_count
    # --------------------------------------------------------

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword
        in SUSPICIOUS_KEYWORDS
    )

    # --------------------------------------------------------
    # num_digits
    # --------------------------------------------------------

    num_digits = sum(
        char.isdigit()
        for char in url
    )

    # --------------------------------------------------------
    # path_length
    # --------------------------------------------------------

    path_length = len(path)

    # --------------------------------------------------------
    # hostname_length
    # --------------------------------------------------------

    hostname_length = len(
        parsed_hostname
    )

    # --------------------------------------------------------
    # url_length
    # --------------------------------------------------------

    url_length = len(url)

    # --------------------------------------------------------
    # num_dots
    # --------------------------------------------------------

    num_dots = url.count(".")

    # --------------------------------------------------------
    # special_char_ratio
    # --------------------------------------------------------

    num_special_chars = sum(
        not char.isalnum()
        for char in url
    )

    special_char_ratio = (
        num_special_chars
        /
        url_length
        if url_length > 0
        else 0.0
    )

    # --------------------------------------------------------
    # num_hyphens
    # --------------------------------------------------------

    num_hyphens = url.count("-")

    # --------------------------------------------------------
    # query_length
    # --------------------------------------------------------

    query_length = len(query)

    return {
        "num_subdomains":
            num_subdomains,

        "suspicious_keyword_count":
            suspicious_keyword_count,

        "num_digits":
            num_digits,

        "path_length":
            path_length,

        "hostname_length":
            hostname_length,

        "url_length":
            url_length,

        "num_dots":
            num_dots,

        "special_char_ratio":
            special_char_ratio,

        "num_hyphens":
            num_hyphens,

        "query_length":
            query_length
    }

In [ ]:
dev_meta_final = (
    df_all_features
    .iloc[train_idx]
    [
        [
            "url",
            "hostname",
            "domain"
        ]
    ]
    .reset_index(
        drop=True
    )
    .copy()
)

In [ ]:
consistency_records_10 = []

for idx in range(
    len(dev_meta_final)
):

    row = (
        dev_meta_final
        .iloc[idx]
    )

    extracted = (
        extract_final_10_features(
            row["url"],
            row["hostname"],
            row["domain"]
        )
    )

    for feature in (
        final_light_features
    ):

        stored_value = float(
            X_dev
            .iloc[idx][
                feature
            ]
        )

        extracted_value = float(
            extracted[
                feature
            ]
        )

        consistency_records_10.append({
            "sample":
                idx,

            "feature":
                feature,

            "stored_value":
                stored_value,

            "extracted_value":
                extracted_value,

            "difference":
                stored_value
                -
                extracted_value
        })


consistency_10_df = (
    pd.DataFrame(
        consistency_records_10
    )
)

mismatch_10_df = (
    consistency_10_df[
        consistency_10_df[
            "difference"
        ].abs()
        >
        1e-12
    ]
    .copy()
)

In [ ]:
print(
    "Maximum absolute difference:",
    consistency_10_df[
        "difference"
    ].abs().max()
)

print(
    "Mismatch count:",
    len(
        mismatch_10_df
    )
)

print(
    "Affected samples:",
    mismatch_10_df[
        "sample"
    ].nunique()
)

In [ ]:
extract_full_lexical_14_from_row

In [ ]:
print(
    "Full extractor available:",
    "extract_full_lexical_14_from_row"
    in globals()
)

In [ ]:
rng = np.random.default_rng(
    42
)

final_benchmark_idx = (
    rng.choice(
        len(dev_meta_final),
        size=min(
            2000,
            len(dev_meta_final)
        ),
        replace=False
    )
)

final_benchmark_meta = (
    dev_meta_final
    .iloc[
        final_benchmark_idx
    ]
    .reset_index(
        drop=True
    )
)

print(
    "Benchmark samples:",
    len(
        final_benchmark_meta
    )
)

In [ ]:
def measure_extractor_latency(
    meta_df,
    extractor,
    repeats=50
):

    records = list(
        meta_df[
            [
                "url",
                "hostname",
                "domain"
            ]
        ]
        .itertuples(
            index=False,
            name=None
        )
    )

    # warm-up
    for _ in range(3):

        for (
            url,
            hostname,
            domain
        ) in records[:100]:

            extractor(
                url,
                hostname,
                domain
            )

    times = []

    for _ in range(
        repeats
    ):

        start = (
            time.perf_counter()
        )

        for (
            url,
            hostname,
            domain
        ) in records:

            extractor(
                url,
                hostname,
                domain
            )

        end = (
            time.perf_counter()
        )

        times.append(
            end - start
        )

    times = np.asarray(
        times
    )

    return {
        "mean_ms_per_url":
            times.mean()
            *
            1000
            /
            len(records),

        "std_ms_per_url":
            times.std()
            *
            1000
            /
            len(records),

        "median_ms_per_url":
            np.median(
                times
            )
            *
            1000
            /
            len(records)
    }

In [ ]:
final_full_extraction_latency = (
    measure_extractor_latency(
        final_benchmark_meta,
        extract_full_lexical_14_from_row,
        repeats=50
    )
)

final_full_extraction_latency

In [ ]:
final_light_extraction_latency = (
    measure_extractor_latency(
        final_benchmark_meta,
        extract_final_10_features,
        repeats=50
    )
)

final_light_extraction_latency

In [ ]:
extraction_latency_reduction = (
    1
    -
    final_light_extraction_latency[
        "mean_ms_per_url"
    ]
    /
    final_full_extraction_latency[
        "mean_ms_per_url"
    ]
)

extraction_speedup = (
    final_full_extraction_latency[
        "mean_ms_per_url"
    ]
    /
    final_light_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Full extraction ms/URL:",
    final_full_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "10-feature extraction ms/URL:",
    final_light_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Extraction latency reduction (%):",
    100
    *
    extraction_latency_reduction
)

print(
    "Extraction speedup:",
    extraction_speedup,
    "x"
)

In [ ]:
prediction_idx = (
    final_benchmark_idx[
        :1000
    ]
)

X_latency_full = (
    X_dev.iloc[
        prediction_idx
    ][
        final_full_features
    ]
)

X_latency_light = (
    X_dev.iloc[
        prediction_idx
    ][
        final_light_features
    ]
)

In [ ]:
def measure_model_latency(
    model,
    X,
    repeats=200
):

    # warm-up
    for _ in range(10):
        model.predict_proba(
            X
        )

    times = []

    for _ in range(
        repeats
    ):

        start = (
            time.perf_counter()
        )

        model.predict_proba(
            X
        )

        end = (
            time.perf_counter()
        )

        times.append(
            end - start
        )

    times = np.asarray(
        times
    )

    return {
        "mean_ms_per_sample":
            times.mean()
            *
            1000
            /
            len(X),

        "std_ms_per_sample":
            times.std()
            *
            1000
            /
            len(X),

        "median_ms_per_sample":
            np.median(
                times
            )
            *
            1000
            /
            len(X)
    }

In [ ]:
full_prediction_latency = (
    measure_model_latency(
        final_full_model,
        X_latency_full,
        repeats=200
    )
)

light_prediction_latency = (
    measure_model_latency(
        final_light_model,
        X_latency_light,
        repeats=200
    )
)

In [ ]:
print(
    "Full inference ms/sample:",
    full_prediction_latency[
        "mean_ms_per_sample"
    ]
)

print(
    "10-feature inference ms/sample:",
    light_prediction_latency[
        "mean_ms_per_sample"
    ]
)

In [ ]:
full_total_latency = (
    final_full_extraction_latency[
        "mean_ms_per_url"
    ]
    +
    full_prediction_latency[
        "mean_ms_per_sample"
    ]
)

light_total_latency = (
    final_light_extraction_latency[
        "mean_ms_per_url"
    ]
    +
    light_prediction_latency[
        "mean_ms_per_sample"
    ]
)

total_latency_reduction = (
    1
    -
    light_total_latency
    /
    full_total_latency
)

total_speedup = (
    full_total_latency
    /
    light_total_latency
)

In [ ]:
print(
    "Full total latency (ms/sample):",
    full_total_latency
)

print(
    "10-feature total latency (ms/sample):",
    light_total_latency
)

print(
    "Total latency reduction (%):",
    100
    *
    total_latency_reduction
)

print(
    "End-to-end speedup:",
    total_speedup,
    "x"
)

In [ ]:
feature_reduction = (
    1
    -
    len(
        final_light_features
    )
    /
    len(
        final_full_features
    )
)

f1_retention = (
    light_final_f1
    /
    full_final_f1
)

In [ ]:
phase16_summary = pd.DataFrame([
    {
        "configuration":
            "Full_14",

        "feature_count":
            14,

        "OOF_F1":
            full_final_f1,

        "OOF_AUROC":
            full_final_auc,

        "OOF_MCC":
            full_final_mcc,

        "model_size_kb":
            full_model_size_kb,

        "feature_extraction_ms":
            final_full_extraction_latency[
                "mean_ms_per_url"
            ],

        "inference_ms":
            full_prediction_latency[
                "mean_ms_per_sample"
            ],

        "total_latency_ms":
            full_total_latency
    },

    {
        "configuration":
            "Final_Lightweight_10",

        "feature_count":
            10,

        "OOF_F1":
            light_final_f1,

        "OOF_AUROC":
            light_final_auc,

        "OOF_MCC":
            light_final_mcc,

        "model_size_kb":
            light_model_size_kb,

        "feature_extraction_ms":
            final_light_extraction_latency[
                "mean_ms_per_url"
            ],

        "inference_ms":
            light_prediction_latency[
                "mean_ms_per_sample"
            ],

        "total_latency_ms":
            light_total_latency
    }
])

phase16_summary

In [ ]:
print(
    "=" * 75
)

print(
    "PHASE 16 — FINAL LIGHTWEIGHT EFFICIENCY SUMMARY"
)

print(
    "=" * 75
)

print()

print(
    "Final configuration:"
)

print(
    "10 lexical features + Teacher XGB"
)

print()

print(
    "Feature count:",
    "14 -> 10"
)

print(
    "Feature reduction (%):",
    100
    *
    feature_reduction
)

print()

print(
    "Full OOF F1:",
    full_final_f1
)

print(
    "Lightweight OOF F1:",
    light_final_f1
)

print(
    "Observed ΔF1:",
    light_final_f1
    -
    full_final_f1
)

print(
    "F1 retention (%):",
    100
    *
    f1_retention
)

print(
    "95% CI ΔF1:",
    final_delta_f1_ci
)

print(
    "Formal non-inferiority:",
    final_noninferior
)

print()

print(
    "Full extraction latency:",
    final_full_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Lightweight extraction latency:",
    final_light_extraction_latency[
        "mean_ms_per_url"
    ]
)

print(
    "Extraction reduction (%):",
    100
    *
    extraction_latency_reduction
)

print()

print(
    "Full inference latency:",
    full_prediction_latency[
        "mean_ms_per_sample"
    ]
)

print(
    "Lightweight inference latency:",
    light_prediction_latency[
        "mean_ms_per_sample"
    ]
)

print()

print(
    "Full total latency:",
    full_total_latency
)

print(
    "Lightweight total latency:",
    light_total_latency
)

print(
    "Total latency reduction (%):",
    100
    *
    total_latency_reduction
)

print(
    "End-to-end speedup:",
    total_speedup,
    "x"
)

print()

print(
    "Full model size (KB):",
    full_model_size_kb
)

print(
    "Lightweight model size (KB):",
    light_model_size_kb
)

print(
    "Model-size change (%):",
    100
    *
    model_size_change
)

In [ ]:
plt.figure(
    figsize=(7, 5)
)

plt.scatter(
    full_total_latency,
    full_final_f1,
    s=100,
    label="Full 14-feature"
)

plt.scatter(
    light_total_latency,
    light_final_f1,
    s=100,
    label="Final 10-feature"
)

plt.annotate(
    "Full 14",
    (
        full_total_latency,
        full_final_f1
    ),
    xytext=(8, 6),
    textcoords="offset points"
)

plt.annotate(
    "Lightweight 10",
    (
        light_total_latency,
        light_final_f1
    ),
    xytext=(8, 6),
    textcoords="offset points"
)

plt.xlabel(
    "End-to-End Latency (ms/sample)"
)

plt.ylabel(
    "OOF F1"
)

plt.title(
    "Final Accuracy–Efficiency Trade-off"
)

plt.grid(
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
phase16_summary.to_csv(
    "phase16_final_lightweight_efficiency_summary.csv",
    index=False
)

consistency_10_df.to_csv(
    "phase16_10feature_consistency_check.csv",
    index=False
)

In [ ]:
joblib.dump(
    final_light_model,
    "FINAL_10feature_lightweight_xgb.pkl"
)

In [ ]:
with open(
    "FINAL_10feature_deployment_config.json",
    "w"
) as f:

    json.dump(
        {
            "model":
                "Teacher XGB",

            "feature_count":
                int(
                    len(
                        final_light_features
                    )
                ),

            "features":
                final_light_features,

            "feature_reduction":
                float(
                    feature_reduction
                ),

            "formal_noninferiority":
                bool(
                    final_noninferior
                ),

            "noninferiority_margin":
                0.005
        },
        f,
        indent=4
    )

In [ ]:
# ============================================================
# Phase 17 — External Validation
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

In [ ]:
final_external_model = joblib.load(
    "FINAL_10feature_lightweight_xgb.pkl"
)

In [ ]:
with open(
    "FINAL_10feature_deployment_config.json",
    "r"
) as f:

    deployment_config = json.load(f)

final_external_features = (
    deployment_config["features"]
)

print(
    "Frozen feature count:",
    len(final_external_features)
)

print(
    "Frozen features:"
)

for i, feature in enumerate(
    final_external_features,
    start=1
):
    print(i, feature)

In [ ]:
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nTest label distribution:")
print(y_test.value_counts())

print("\nFinal features:")
print(final_robust_features)

In [ ]:
# ============================================================
# Final 10-feature model
# Train on ALL development data only
# ============================================================

final_test_model = make_teacher_model()

final_test_model.fit(
    X_dev[final_robust_features],
    y_dev
)

print("Final 10-feature model trained on full development set.")

In [ ]:
# ============================================================
# Frozen test evaluation
# ============================================================

test_prob = final_test_model.predict_proba(
    X_test[final_robust_features]
)[:, 1]

# threshold remains frozen at 0.5
test_pred = (
    test_prob >= 0.5
).astype(int)

In [ ]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

final_test_f1 = f1_score(
    y_test,
    test_pred
)

final_test_precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

final_test_recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

final_test_accuracy = accuracy_score(
    y_test,
    test_pred
)

final_test_auc = roc_auc_score(
    y_test,
    test_prob
)

final_test_mcc = matthews_corrcoef(
    y_test,
    test_pred
)

final_test_cm = confusion_matrix(
    y_test,
    test_pred
)

In [ ]:
print("=" * 70)
print("FINAL HELD-OUT TEST RESULTS — 10-FEATURE MODEL")
print("=" * 70)

print("Samples:", len(y_test))
print("F1:", final_test_f1)
print("Precision:", final_test_precision)
print("Recall:", final_test_recall)
print("Accuracy:", final_test_accuracy)
print("AUROC:", final_test_auc)
print("MCC:", final_test_mcc)

print("\nConfusion Matrix:")
print(final_test_cm)

In [ ]:
print(
    classification_report(
        y_test,
        test_pred,
        target_names=[
            "Benign",
            "Phishing"
        ],
        digits=4
    )
)

In [ ]:
previous_full_test = {
    "F1": 0.8852272727272728,
    "Recall": 0.8674832962138085,
    "Precision": 0.9037122969837587,
    "AUROC": 0.9530228975526451,
    "MCC": 0.7733576742857327
}

In [ ]:
comparison_test = pd.DataFrame([
    {
        "Model": "Previous Full Lexical",
        "F1": previous_full_test["F1"],
        "Precision": previous_full_test["Precision"],
        "Recall": previous_full_test["Recall"],
        "AUROC": previous_full_test["AUROC"],
        "MCC": previous_full_test["MCC"]
    },
    {
        "Model": "Final 10-Feature",
        "F1": final_test_f1,
        "Precision": final_test_precision,
        "Recall": final_test_recall,
        "AUROC": final_test_auc,
        "MCC": final_test_mcc
    }
])

comparison_test

In [ ]:
print(
    "ΔF1 on held-out test:",
    final_test_f1
    -
    previous_full_test["F1"]
)

print(
    "ΔAUROC on held-out test:",
    final_test_auc
    -
    previous_full_test["AUROC"]
)

print(
    "ΔMCC on held-out test:",
    final_test_mcc
    -
    previous_full_test["MCC"]
)

In [ ]:
final_test_summary = pd.DataFrame([
    {
        "model": "Final_10Feature_XGB",
        "n_test": len(y_test),
        "f1": final_test_f1,
        "precision": final_test_precision,
        "recall": final_test_recall,
        "accuracy": final_test_accuracy,
        "auroc": final_test_auc,
        "mcc": final_test_mcc,
        "tn": int(final_test_cm[0, 0]),
        "fp": int(final_test_cm[0, 1]),
        "fn": int(final_test_cm[1, 0]),
        "tp": int(final_test_cm[1, 1])
    }
])

final_test_summary.to_csv(
    "phase17_final_heldout_test_summary.csv",
    index=False
)

comparison_test.to_csv(
    "phase17_full_vs_final10_test_comparison.csv",
    index=False
)

joblib.dump(
    final_test_model,
    "FINAL_10feature_XGB_for_deployment.pkl"
)

print("Final held-out test results saved.")

In [ ]:
# ============================================================
# Final held-out test paired bootstrap for ΔF1
# ============================================================

# We need predictions from BOTH models on the exact same test samples.

# Final 10-feature predictions already exist:
light_test_pred = test_pred.copy()

# Recreate / fit the full 14-feature reference model
final_full_test_model = make_teacher_model()

final_full_test_model.fit(
    X_dev[final_full_features],
    y_dev
)

full_test_prob = (
    final_full_test_model
    .predict_proba(
        X_test[final_full_features]
    )[:, 1]
)

full_test_pred = (
    full_test_prob >= 0.5
).astype(int)

# Sanity check
print("Test samples:", len(y_test))
print("Full predictions:", len(full_test_pred))
print("Light predictions:", len(light_test_pred))

In [ ]:
def paired_test_bootstrap_delta_f1(
    y_true,
    full_pred,
    light_pred,
    n_bootstrap=10000,
    random_state=42
):

    rng = np.random.default_rng(
        random_state
    )

    y_true = np.asarray(y_true)
    full_pred = np.asarray(full_pred)
    light_pred = np.asarray(light_pred)

    n = len(y_true)

    deltas = []

    for _ in range(n_bootstrap):

        idx = rng.choice(
            n,
            size=n,
            replace=True
        )

        y_b = y_true[idx]

        full_b = full_pred[idx]
        light_b = light_pred[idx]

        full_f1_b = f1_score(
            y_b,
            full_b
        )

        light_f1_b = f1_score(
            y_b,
            light_b
        )

        deltas.append(
            light_f1_b
            -
            full_f1_b
        )

    return np.asarray(deltas)

In [ ]:
test_delta_f1_bootstrap = (
    paired_test_bootstrap_delta_f1(
        y_true=y_test,
        full_pred=full_test_pred,
        light_pred=light_test_pred,
        n_bootstrap=10000,
        random_state=42
    )
)

In [ ]:
test_delta_f1_ci = (
    np.quantile(
        test_delta_f1_bootstrap,
        0.025
    ),
    np.quantile(
        test_delta_f1_bootstrap,
        0.975
    )
)

observed_test_delta_f1 = (
    final_test_f1
    -
    previous_full_test["F1"]
)

print(
    "Observed held-out ΔF1:",
    observed_test_delta_f1
)

print(
    "95% paired bootstrap CI:",
    test_delta_f1_ci
)

In [ ]:
TEST_NI_MARGIN = 0.005

test_noninferior = (
    test_delta_f1_ci[0]
    >
    -TEST_NI_MARGIN
)

print(
    "Held-out non-inferiority margin:",
    TEST_NI_MARGIN
)

print(
    "Held-out formal non-inferiority:",
    test_noninferior
)

In [ ]:
final_external_model = joblib.load(
    "FINAL_10feature_lightweight_xgb.pkl"
)

In [ ]:
# ============================================================
# Phase 17A — Restore Final Held-out Test Environment
# ============================================================

import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. Load full feature dataset
# ------------------------------------------------------------

df_all_features = pd.read_csv(
    "qr_full_feature_dataset_v2.csv"
)

print("Dataset shape:", df_all_features.shape)


# ------------------------------------------------------------
# 2. Recover final robust 10 features
# ------------------------------------------------------------

with open(
    "phase14B_final_robust_features.json",
    "r"
) as f:
    robust_config = json.load(f)

final_robust_features = robust_config["features"]

print("\nFinal robust features:")
for i, feature in enumerate(
    final_robust_features,
    start=1
):
    print(i, feature)


# ------------------------------------------------------------
# 3. Reproduce original hostname-grouped split
# ------------------------------------------------------------

y_all = (
    df_all_features["label"]
    .astype(int)
)

groups_all = (
    df_all_features["hostname"]
    .astype(str)
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df_all_features,
        y_all,
        groups=groups_all
    )
)


# ------------------------------------------------------------
# 4. Rebuild feature matrix
# ------------------------------------------------------------

metadata_cols = [
    "original_index",
    "url",
    "normalized_url",
    "hostname",
    "domain",
    "label"
]

feature_cols = [
    c
    for c in df_all_features.columns
    if c not in metadata_cols
]

X_all = (
    df_all_features[
        feature_cols
    ]
    .copy()
)

X_dev = (
    X_all
    .iloc[train_idx]
    .reset_index(drop=True)
)

X_test = (
    X_all
    .iloc[test_idx]
    .reset_index(drop=True)
)

y_dev = (
    y_all
    .iloc[train_idx]
    .reset_index(drop=True)
)

y_test = (
    y_all
    .iloc[test_idx]
    .reset_index(drop=True)
)

groups_dev = (
    groups_all
    .iloc[train_idx]
    .reset_index(drop=True)
)

groups_test = (
    groups_all
    .iloc[test_idx]
    .reset_index(drop=True)
)


print("\nDevelopment samples:", len(X_dev))
print("Test samples:", len(X_test))

print(
    "Hostname overlap:",
    len(
        set(groups_dev)
        &
        set(groups_test)
    )
)

print("\nTest labels:")
print(
    y_test.value_counts()
)

In [ ]:
# ============================================================
# Phase 17B — Load Frozen Final Model
# ============================================================

final_test_model = joblib.load(
    "FINAL_10feature_XGB_for_deployment.pkl"
)

print(
    "Final 10-feature model loaded successfully."
)

In [ ]:
# ============================================================
# Phase 17C — Frozen Held-out Test Prediction
# ============================================================

test_prob = (
    final_test_model
    .predict_proba(
        X_test[
            final_robust_features
        ]
    )[:, 1]
)

test_pred = (
    test_prob >= 0.5
).astype(int)

print(
    "Predictions generated:",
    len(test_pred)
)

In [ ]:
# ============================================================
# Phase 17D — Final Held-out Metrics
# ============================================================

final_test_f1 = f1_score(
    y_test,
    test_pred
)

final_test_precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

final_test_recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

final_test_accuracy = accuracy_score(
    y_test,
    test_pred
)

final_test_auc = roc_auc_score(
    y_test,
    test_prob
)

final_test_mcc = matthews_corrcoef(
    y_test,
    test_pred
)

final_test_cm = confusion_matrix(
    y_test,
    test_pred
)

print(
    "=" * 70
)

print(
    "PHASE 17 — FINAL HELD-OUT TEST RESULTS"
)

print(
    "=" * 70
)

print(
    "Samples:",
    len(y_test)
)

print(
    "F1:",
    final_test_f1
)

print(
    "Precision:",
    final_test_precision
)

print(
    "Recall:",
    final_test_recall
)

print(
    "Accuracy:",
    final_test_accuracy
)

print(
    "AUROC:",
    final_test_auc
)

print(
    "MCC:",
    final_test_mcc
)

print(
    "\nConfusion Matrix:"
)

print(
    final_test_cm
)

In [ ]:
print(
    classification_report(
        y_test,
        test_pred,
        target_names=[
            "Benign",
            "Phishing"
        ],
        digits=4
    )
)

In [ ]:
previous_full_test = {
    "F1": 0.8852272727272728,
    "Precision": 0.9037122969837587,
    "Recall": 0.8674832962138085,
    "AUROC": 0.9530228975526451,
    "MCC": 0.7733576742857327
}

In [ ]:
comparison_test = pd.DataFrame([
    {
        "Model":
            "Full Lexical Reference",

        "F1":
            previous_full_test["F1"],

        "Precision":
            previous_full_test["Precision"],

        "Recall":
            previous_full_test["Recall"],

        "AUROC":
            previous_full_test["AUROC"],

        "MCC":
            previous_full_test["MCC"]
    },

    {
        "Model":
            "Final 10-Feature",

        "F1":
            final_test_f1,

        "Precision":
            final_test_precision,

        "Recall":
            final_test_recall,

        "AUROC":
            final_test_auc,

        "MCC":
            final_test_mcc
    }
])

comparison_test

In [ ]:
delta_test_f1 = (
    final_test_f1
    -
    previous_full_test["F1"]
)

delta_test_auc = (
    final_test_auc
    -
    previous_full_test["AUROC"]
)

delta_test_mcc = (
    final_test_mcc
    -
    previous_full_test["MCC"]
)

print(
    "ΔF1:",
    delta_test_f1
)

print(
    "ΔAUROC:",
    delta_test_auc
)

print(
    "ΔMCC:",
    delta_test_mcc
)

In [ ]:
# ============================================================
# Phase 17H — Paired Bootstrap for Held-out ΔF1
# ============================================================

# Need full-reference predictions on exactly the same test set

final_full_features = list(
    feature_budget_sets[14]
) if "feature_budget_sets" in globals() else [
    "num_subdomains",
    "suspicious_keyword_count",
    "num_digits",
    "path_length",
    "hostname_length",
    "url_length",
    "num_dots",
    "special_char_ratio",
    "num_hyphens",
    "query_length",
    "num_special_chars",
    "has_fragment",
    "contains_at",
    "contains_port"
]

In [ ]:
final_full_test_model = (
    make_teacher_model()
)

final_full_test_model.fit(
    X_dev[
        final_full_features
    ],
    y_dev
)

In [ ]:
full_test_prob = (
    final_full_test_model
    .predict_proba(
        X_test[
            final_full_features
        ]
    )[:, 1]
)

full_test_pred = (
    full_test_prob >= 0.5
).astype(int)

In [ ]:
def paired_test_bootstrap_delta_f1(
    y_true,
    full_pred,
    light_pred,
    n_bootstrap=10000,
    random_state=42
):

    rng = np.random.default_rng(
        random_state
    )

    y_true = np.asarray(
        y_true
    )

    full_pred = np.asarray(
        full_pred
    )

    light_pred = np.asarray(
        light_pred
    )

    n = len(
        y_true
    )

    deltas = []

    for _ in range(
        n_bootstrap
    ):

        idx = rng.choice(
            n,
            size=n,
            replace=True
        )

        y_b = y_true[idx]

        full_b = (
            full_pred[idx]
        )

        light_b = (
            light_pred[idx]
        )

        full_f1_b = f1_score(
            y_b,
            full_b
        )

        light_f1_b = f1_score(
            y_b,
            light_b
        )

        deltas.append(
            light_f1_b
            -
            full_f1_b
        )

    return np.asarray(
        deltas
    )

In [ ]:
test_delta_f1_bootstrap = (
    paired_test_bootstrap_delta_f1(
        y_true=y_test,
        full_pred=full_test_pred,
        light_pred=test_pred,
        n_bootstrap=10000,
        random_state=42
    )
)

In [ ]:
test_delta_f1_ci = (
    np.quantile(
        test_delta_f1_bootstrap,
        0.025
    ),
    np.quantile(
        test_delta_f1_bootstrap,
        0.975
    )
)

TEST_NI_MARGIN = 0.005

test_noninferior = (
    test_delta_f1_ci[0]
    >
    -TEST_NI_MARGIN
)

print(
    "Observed held-out ΔF1:",
    delta_test_f1
)

print(
    "95% paired bootstrap CI:",
    test_delta_f1_ci
)

print(
    "NI margin:",
    TEST_NI_MARGIN
)

print(
    "Held-out formal non-inferiority:",
    test_noninferior
)

In [ ]:
final_test_summary = pd.DataFrame([
    {
        "model":
            "Final_10Feature_XGB",

        "n_test":
            len(y_test),

        "f1":
            final_test_f1,

        "precision":
            final_test_precision,

        "recall":
            final_test_recall,

        "accuracy":
            final_test_accuracy,

        "auroc":
            final_test_auc,

        "mcc":
            final_test_mcc,

        "delta_f1_vs_full":
            delta_test_f1,

        "delta_auc_vs_full":
            delta_test_auc,

        "delta_mcc_vs_full":
            delta_test_mcc,

        "delta_f1_ci_lower":
            test_delta_f1_ci[0],

        "delta_f1_ci_upper":
            test_delta_f1_ci[1],

        "noninferiority_margin":
            TEST_NI_MARGIN,

        "formal_noninferiority":
            test_noninferior,

        "tn":
            int(
                final_test_cm[0, 0]
            ),

        "fp":
            int(
                final_test_cm[0, 1]
            ),

        "fn":
            int(
                final_test_cm[1, 0]
            ),

        "tp":
            int(
                final_test_cm[1, 1]
            )
    }
])

final_test_summary

In [ ]:
final_test_summary.to_csv(
    "phase17_final_heldout_test_summary.csv",
    index=False
)

comparison_test.to_csv(
    "phase17_full_vs_final10_test_comparison.csv",
    index=False
)

pd.DataFrame({
    "delta_f1_bootstrap":
        test_delta_f1_bootstrap
}).to_csv(
    "phase17_test_delta_f1_bootstrap.csv",
    index=False
)

print(
    "Phase 17 results saved successfully."
)

In [ ]:
# ============================================================
# Phase 18A — Permutation Importance
# ============================================================

from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    final_test_model,
    X_test[final_robust_features],
    y_test,
    scoring="f1",
    n_repeats=20,
    random_state=42,
    n_jobs=1
)

phase18_importance = (
    pd.DataFrame({
        "feature": final_robust_features,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std
    })
    .sort_values(
        "importance_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

phase18_importance

In [ ]:
import matplotlib.pyplot as plt

plot_df = (
    phase18_importance
    .sort_values(
        "importance_mean",
        ascending=True
    )
)

plt.figure(
    figsize=(8, 5)
)

plt.barh(
    plot_df["feature"],
    plot_df["importance_mean"],
    xerr=plot_df["importance_std"]
)

plt.xlabel(
    "Mean decrease in held-out F1 after permutation"
)

plt.ylabel(
    "Feature"
)

plt.title(
    "Permutation Importance of Final 10-Feature Model"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Phase 18B — Error Analysis Table
# ============================================================

test_meta = (
    df_all_features
    .iloc[test_idx]
    .reset_index(drop=True)
    .copy()
)

error_df = (
    test_meta[
        [
            "url",
            "normalized_url",
            "hostname",
            "domain",
            "label"
        ]
    ]
    .copy()
)

error_df[
    "predicted_label"
] = test_pred

error_df[
    "predicted_probability"
] = test_prob

error_df[
    "correct"
] = (
    error_df["label"]
    ==
    error_df["predicted_label"]
)

error_df[
    "error_type"
] = "Correct"

error_df.loc[
    (
        (error_df["label"] == 0)
        &
        (error_df["predicted_label"] == 1)
    ),
    "error_type"
] = "False Positive"

error_df.loc[
    (
        (error_df["label"] == 1)
        &
        (error_df["predicted_label"] == 0)
    ),
    "error_type"
] = "False Negative"

error_df[
    "confidence"
] = np.where(
    error_df["predicted_label"] == 1,
    error_df["predicted_probability"],
    1 - error_df["predicted_probability"]
)

for feature in final_robust_features:
    error_df[feature] = (
        X_test[feature]
        .reset_index(drop=True)
    )

print(
    error_df["error_type"]
    .value_counts()
)

In [ ]:
# ============================================================
# Phase 18C — FP / FN subsets
# ============================================================

fp_df = (
    error_df[
        error_df["error_type"]
        ==
        "False Positive"
    ]
    .copy()
)

fn_df = (
    error_df[
        error_df["error_type"]
        ==
        "False Negative"
    ]
    .copy()
)

print(
    "False Positives:",
    len(fp_df)
)

print(
    "False Negatives:",
    len(fn_df)
)

In [ ]:
feature_error_summary = pd.DataFrame({
    "Benign_correct_mean": (
        error_df[
            (error_df["label"] == 0)
            &
            (error_df["correct"])
        ][final_robust_features]
        .mean()
    ),

    "False_positive_mean": (
        fp_df[
            final_robust_features
        ]
        .mean()
    ),

    "Phishing_correct_mean": (
        error_df[
            (error_df["label"] == 1)
            &
            (error_df["correct"])
        ][final_robust_features]
        .mean()
    ),

    "False_negative_mean": (
        fn_df[
            final_robust_features
        ]
        .mean()
    )
})

feature_error_summary

In [ ]:
# ============================================================
# Phase 18D — High-confidence Error Analysis
# ============================================================

high_conf_errors = (
    error_df[
        error_df["correct"] == False
    ]
    .sort_values(
        "confidence",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Total errors:",
    len(high_conf_errors)
)

high_conf_errors[
    [
        "url",
        "label",
        "predicted_label",
        "predicted_probability",
        "confidence",
        "error_type"
    ]
].head(20)

In [ ]:
high_conf_fp = (
    fp_df
    .sort_values(
        "predicted_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Top high-confidence False Positives:"
)

high_conf_fp[
    [
        "url",
        "hostname",
        "predicted_probability",
        "confidence"
    ]
    +
    final_robust_features
].head(15)

In [ ]:
high_conf_fn = (
    fn_df
    .sort_values(
        "predicted_probability",
        ascending=True
    )
    .reset_index(drop=True)
)

print(
    "Top high-confidence False Negatives:"
)

high_conf_fn[
    [
        "url",
        "hostname",
        "predicted_probability",
        "confidence"
    ]
    +
    final_robust_features
].head(15)

In [ ]:
# ============================================================
# Phase 18E — Confidence Distribution of Errors
# ============================================================

error_conf_summary = (
    error_df
    .groupby(
        "error_type"
    )
    ["confidence"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max"
        ]
    )
)

error_conf_summary

In [ ]:
plt.figure(
    figsize=(8, 5)
)

for error_type in [
    "Correct",
    "False Positive",
    "False Negative"
]:

    values = (
        error_df.loc[
            error_df["error_type"]
            ==
            error_type,
            "confidence"
        ]
    )

    plt.hist(
        values,
        bins=20,
        alpha=0.5,
        label=error_type,
        density=True
    )

plt.xlabel(
    "Prediction Confidence"
)

plt.ylabel(
    "Density"
)

plt.title(
    "Confidence Distribution of Correct and Incorrect Predictions"
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
confidence_thresholds = [
    0.70,
    0.80,
    0.90,
    0.95
]

confidence_error_records = []

for threshold in confidence_thresholds:

    subset = error_df[
        error_df["confidence"]
        >=
        threshold
    ]

    errors = subset[
        subset["correct"]
        ==
        False
    ]

    confidence_error_records.append({
        "confidence_threshold":
            threshold,

        "samples":
            len(subset),

        "errors":
            len(errors),

        "false_positives":
            (
                errors["error_type"]
                ==
                "False Positive"
            ).sum(),

        "false_negatives":
            (
                errors["error_type"]
                ==
                "False Negative"
            ).sum(),

        "error_rate":
            (
                len(errors)
                /
                len(subset)
                if len(subset) > 0
                else np.nan
            )
    })

confidence_error_summary = (
    pd.DataFrame(
        confidence_error_records
    )
)

confidence_error_summary

In [ ]:
# ============================================================
# Phase 18G — Error-profile Relative Differences
# ============================================================

error_profile = pd.DataFrame(
    index=final_robust_features
)

error_profile[
    "FP_vs_correct_benign"
] = (
    feature_error_summary[
        "False_positive_mean"
    ]
    -
    feature_error_summary[
        "Benign_correct_mean"
    ]
)

error_profile[
    "FN_vs_correct_phishing"
] = (
    feature_error_summary[
        "False_negative_mean"
    ]
    -
    feature_error_summary[
        "Phishing_correct_mean"
    ]
)

error_profile[
    "FP_relative_change"
] = (
    error_profile[
        "FP_vs_correct_benign"
    ]
    /
    feature_error_summary[
        "Benign_correct_mean"
    ]
    .replace(0, np.nan)
)

error_profile[
    "FN_relative_change"
] = (
    error_profile[
        "FN_vs_correct_phishing"
    ]
    /
    feature_error_summary[
        "Phishing_correct_mean"
    ]
    .replace(0, np.nan)
)

error_profile

In [ ]:
error_profile[
    [
        "FN_vs_correct_phishing",
        "FN_relative_change"
    ]
].sort_values(
    "FN_relative_change"
)

In [ ]:
phase18_importance.to_csv(
    "phase18_final10_permutation_importance.csv",
    index=False
)

feature_error_summary.to_csv(
    "phase18_error_feature_summary.csv"
)

error_profile.to_csv(
    "phase18_error_profile_differences.csv"
)

high_conf_fp.to_csv(
    "phase18_high_confidence_false_positives.csv",
    index=False
)

high_conf_fn.to_csv(
    "phase18_high_confidence_false_negatives.csv",
    index=False
)

confidence_error_summary.to_csv(
    "phase18_confidence_error_summary.csv",
    index=False
)

error_df.to_csv(
    "phase18_complete_heldout_error_analysis.csv",
    index=False
)

print(
    "Phase 18 interpretability and error-analysis outputs saved."
)

In [ ]:
phase18_final_summary = pd.DataFrame([
    {
        "metric": "Total held-out samples",
        "value": len(error_df)
    },
    {
        "metric": "Correct predictions",
        "value": int((error_df["correct"] == True).sum())
    },
    {
        "metric": "False positives",
        "value": len(fp_df)
    },
    {
        "metric": "False negatives",
        "value": len(fn_df)
    },
    {
        "metric": "Mean confidence - correct",
        "value": error_df.loc[
            error_df["correct"] == True,
            "confidence"
        ].mean()
    },
    {
        "metric": "Mean confidence - FP",
        "value": fp_df["confidence"].mean()
    },
    {
        "metric": "Mean confidence - FN",
        "value": fn_df["confidence"].mean()
    },
    {
        "metric": "Error rate at confidence >= 0.90",
        "value": confidence_error_summary.loc[
            confidence_error_summary["confidence_threshold"] == 0.90,
            "error_rate"
        ].iloc[0]
    },
    {
        "metric": "Error rate at confidence >= 0.95",
        "value": confidence_error_summary.loc[
            confidence_error_summary["confidence_threshold"] == 0.95,
            "error_rate"
        ].iloc[0]
    }
])

phase18_final_summary

In [ ]:
phase18_final_summary.to_csv(
    "phase18_final_interpretability_summary.csv",
    index=False
)

print("Phase 18 completed.")

In [ ]:
# ============================================================
# Phase 19A — Load Final Deployment Model
# ============================================================

import re
import json
import joblib
import numpy as np
import pandas as pd

from urllib.parse import urlsplit

FINAL_MODEL_PATH = "FINAL_10feature_XGB_for_deployment.pkl"
FINAL_CONFIG_PATH = "phase14B_final_robust_features.json"

final_deployment_model = joblib.load(
    FINAL_MODEL_PATH
)

with open(
    FINAL_CONFIG_PATH,
    "r"
) as f:
    deployment_config = json.load(f)

final_deployment_features = deployment_config["features"]

print("Final deployment model loaded.")
print()
print("Feature count:", len(final_deployment_features))

for i, feature in enumerate(
    final_deployment_features,
    start=1
):
    print(i, feature)

In [ ]:
try:
    import tldextract

    domain_extractor = tldextract.TLDExtract(
        suffix_list_urls=None
    )

    print("tldextract ready.")

except ImportError:

    domain_extractor = None

    print(
        "tldextract is not installed."
    )

In [ ]:
# ============================================================
# Phase 19C — Final deployment parser
# ============================================================

def parse_deployment_url(url):

    raw_url = str(url).strip()

    # Reuse the exact training-time normalization function
    normalized_url = normalize_url(
        raw_url
    )

    # Raw hostname: keep www. because training hostname_length did
    try:
        raw_parsed = urlsplit(
            ensure_scheme(raw_url)
        )

        raw_hostname = (
            raw_parsed.hostname or ""
        ).lower()

    except Exception:
        raw_hostname = ""

    # Path/query: use normalized URL
    try:
        normalized_parsed = urlsplit(
            normalized_url
        )

        normalized_path = (
            normalized_parsed.path or ""
        )

        normalized_query = (
            normalized_parsed.query or ""
        )

    except Exception:
        normalized_path = ""
        normalized_query = ""

    return {
        "raw_url": raw_url,
        "normalized_url": normalized_url,
        "raw_hostname": raw_hostname,
        "path": normalized_path,
        "query": normalized_query
    }

In [ ]:
def get_registered_domain(hostname):

    hostname = str(
        hostname
    ).lower().strip()

    if not hostname:
        return ""

    if domain_extractor is None:
        return hostname

    ext = domain_extractor(
        hostname
    )

    if not ext.domain:
        return hostname

    if ext.suffix:

        return (
            ext.domain
            + "."
            + ext.suffix
        )

    return ext.domain


def count_subdomains(
    raw_hostname
):

    raw_hostname = str(
        raw_hostname
    ).lower().strip(".")

    if not raw_hostname:
        return 0

    registered_domain = (
        get_registered_domain(
            raw_hostname
        )
    )

    host_parts = [
        x
        for x in raw_hostname.split(".")
        if x
    ]

    domain_parts = [
        x
        for x in registered_domain.split(".")
        if x
    ]

    num_subdomains = (
        len(host_parts)
        -
        len(domain_parts)
    )

    return max(
        num_subdomains,
        0
    )

In [ ]:
print(
    count_subdomains(
        "www.bbc.co.uk"
    )
)

In [ ]:
# ============================================================
# Phase 19E — Final deployment feature extractor
# ============================================================

def extract_deployment_features(url):

    parsed = parse_deployment_url(
        url
    )

    raw_url = parsed["raw_url"]
    raw_hostname = parsed["raw_hostname"]
    path = parsed["path"]
    query = parsed["query"]

    url_lower = raw_url.lower()

    url_length = len(
        raw_url
    )

    # Verified against training representation
    hostname_length = len(
        raw_hostname
    )

    # Verified against training representation
    path_length = len(
        path
    )

    query_length = len(
        query
    )

    num_digits = sum(
        c.isdigit()
        for c in raw_url
    )

    num_dots = raw_url.count(
        "."
    )

    num_hyphens = raw_url.count(
        "-"
    )

    num_special_chars = sum(
        not c.isalnum()
        for c in raw_url
    )

    special_char_ratio = (
        num_special_chars / url_length
        if url_length > 0
        else 0.0
    )

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    num_subdomains = count_subdomains(
        raw_hostname
    )

    return {
        "num_subdomains": num_subdomains,
        "suspicious_keyword_count": suspicious_keyword_count,
        "num_digits": num_digits,
        "path_length": path_length,
        "hostname_length": hostname_length,
        "url_length": url_length,
        "num_dots": num_dots,
        "special_char_ratio": special_char_ratio,
        "num_hyphens": num_hyphens,
        "query_length": query_length
    }

In [ ]:
test_feature_example = (
    extract_deployment_features(
        "www.bbc.co.uk/lottery/"
    )
)

test_feature_example

In [ ]:
training_feature_data = pd.read_csv(
    "qr_full_feature_dataset_v2.csv"
)

print(
    training_feature_data.shape
)

In [ ]:
# ============================================================
# Phase 19F — Full-dataset consistency audit
# ============================================================

consistency_sample = (
    training_feature_data
    .reset_index(drop=True)
    .copy()
)

consistency_records = []

for i, row in consistency_sample.iterrows():

    new_features = (
        extract_deployment_features(
            row["url"]
        )
    )

    for feature in final_deployment_features:

        old_value = float(
            row[feature]
        )

        new_value = float(
            new_features[feature]
        )

        consistency_records.append({
            "sample": i,
            "feature": feature,
            "old_value": old_value,
            "new_value": new_value,
            "difference":
                old_value - new_value
        })

deployment_consistency_df = pd.DataFrame(
    consistency_records
)

mismatch_df = (
    deployment_consistency_df[
        deployment_consistency_df[
            "difference"
        ].abs() > 1e-12
    ]
    .copy()
)

print(
    "Samples checked:",
    len(consistency_sample)
)

print(
    "Feature values checked:",
    len(deployment_consistency_df)
)

print(
    "Maximum absolute difference:",
    deployment_consistency_df[
        "difference"
    ].abs().max()
)

print(
    "Mismatch count:",
    len(mismatch_df)
)

if len(mismatch_df) > 0:
    print("\nMismatch features:")
    print(
        mismatch_df[
            "feature"
        ].value_counts()
    )
else:
    print(
        "\nSUCCESS: deployment extractor exactly reproduces training features."
    )

In [ ]:
# ============================================================
# Diagnose remaining hostname/subdomain mismatches
# ============================================================

remaining_mismatch = (
    mismatch_df[
        mismatch_df["feature"].isin(
            [
                "hostname_length",
                "num_subdomains"
            ]
        )
    ]
    .copy()
)

remaining_mismatch

In [ ]:
affected_samples = (
    remaining_mismatch["sample"]
    .unique()
)

print(
    "Affected samples:",
    affected_samples
)

In [ ]:
remaining_diag = []

for idx in affected_samples:

    row = (
        training_feature_data
        .iloc[int(idx)]
    )

    raw_url = str(
        row["url"]
    )

    normalized_url = str(
        row["normalized_url"]
    )

    raw_parsed = urlsplit(
        ensure_scheme(raw_url)
    )

    norm_parsed = urlsplit(
        normalized_url
    )

    raw_hostname = (
        raw_parsed.hostname or ""
    )

    normalized_hostname = (
        norm_parsed.hostname or ""
    )

    remaining_diag.append({
        "sample":
            idx,

        "url":
            raw_url,

        "normalized_url":
            normalized_url,

        "stored_hostname":
            row["hostname"],

        "stored_hostname_length":
            row["hostname_length"],

        "stored_num_subdomains":
            row["num_subdomains"],

        "raw_hostname":
            raw_hostname,

        "raw_hostname_length":
            len(raw_hostname),

        "normalized_hostname":
            normalized_hostname,

        "normalized_hostname_length":
            len(normalized_hostname),

        "deployment_num_subdomains":
            count_subdomains(
                raw_hostname
            )
    })

remaining_diag_df = pd.DataFrame(
    remaining_diag
)

remaining_diag_df

In [ ]:
for _, row in remaining_diag_df.iterrows():

    print("=" * 80)

    print(
        "Sample:",
        row["sample"]
    )

    print(
        "URL:",
        row["url"]
    )

    print(
        "Normalized URL:",
        row["normalized_url"]
    )

    print()

    print(
        "Stored hostname:",
        row["stored_hostname"]
    )

    print(
        "Stored hostname length:",
        row["stored_hostname_length"]
    )

    print(
        "Stored num_subdomains:",
        row["stored_num_subdomains"]
    )

    print()

    print(
        "Raw hostname:",
        row["raw_hostname"]
    )

    print(
        "Raw hostname length:",
        row["raw_hostname_length"]
    )

    print()

    print(
        "Normalized hostname:",
        row["normalized_hostname"]
    )

    print(
        "Normalized hostname length:",
        row["normalized_hostname_length"]
    )

    print(
        "Deployment num_subdomains:",
        row["deployment_num_subdomains"]
    )

In [ ]:
# ============================================================
# Inspect the 5 remaining special samples
# ============================================================

remaining_diag = []

for idx in affected_samples:

    row = training_feature_data.iloc[int(idx)]

    raw_url = str(row["url"])
    normalized_url = str(row["normalized_url"])

    raw_parsed = urlsplit(
        ensure_scheme(raw_url)
    )

    normalized_parsed = urlsplit(
        normalized_url
    )

    raw_hostname = (
        raw_parsed.hostname or ""
    )

    normalized_hostname = (
        normalized_parsed.hostname or ""
    )

    stored_hostname = str(
        row["hostname"]
    )

    remaining_diag.append({
        "sample":
            int(idx),

        "url":
            raw_url,

        "normalized_url":
            normalized_url,

        "stored_hostname":
            stored_hostname,

        "stored_hostname_length":
            int(row["hostname_length"]),

        "stored_num_subdomains":
            int(row["num_subdomains"]),

        "raw_hostname":
            raw_hostname,

        "raw_hostname_length":
            len(raw_hostname),

        "normalized_hostname":
            normalized_hostname,

        "normalized_hostname_length":
            len(normalized_hostname),

        "raw_subdomains":
            count_subdomains(
                raw_hostname
            ),

        "normalized_subdomains":
            count_subdomains(
                normalized_hostname
            ),

        "stored_hostname_len_check":
            len(stored_hostname)
    })

remaining_diag_df = pd.DataFrame(
    remaining_diag
)

remaining_diag_df

In [ ]:
for _, row in remaining_diag_df.iterrows():

    print("=" * 100)

    print("Sample:", row["sample"])

    print("\nURL:")
    print(row["url"])

    print("\nNormalized URL:")
    print(row["normalized_url"])

    print("\nStored hostname:")
    print(row["stored_hostname"])

    print(
        "Stored hostname length:",
        row["stored_hostname_length"]
    )

    print(
        "len(stored hostname):",
        row["stored_hostname_len_check"]
    )

    print(
        "Stored num_subdomains:",
        row["stored_num_subdomains"]
    )

    print("\nRaw parsed hostname:")
    print(row["raw_hostname"])

    print(
        "Raw hostname length:",
        row["raw_hostname_length"]
    )

    print(
        "Raw subdomains:",
        row["raw_subdomains"]
    )

    print("\nNormalized parsed hostname:")
    print(row["normalized_hostname"])

    print(
        "Normalized hostname length:",
        row["normalized_hostname_length"]
    )

    print(
        "Normalized subdomains:",
        row["normalized_subdomains"]
    )

    print()

In [ ]:
# ============================================================
# Phase 19A — Imports and load final deployment artifacts
# ============================================================

import re
import json
import joblib
import numpy as np
import pandas as pd

from urllib.parse import urlsplit

FINAL_MODEL_PATH = "FINAL_10feature_XGB_for_deployment.pkl"
FINAL_CONFIG_PATH = "phase14B_final_robust_features.json"
DATA_PATH = "qr_full_feature_dataset_v2.csv"

# Load model
final_deployment_model = joblib.load(
    FINAL_MODEL_PATH
)

# Load selected feature configuration
with open(
    FINAL_CONFIG_PATH,
    "r"
) as f:
    deployment_config = json.load(f)

final_deployment_features = (
    deployment_config["features"]
)

# Load feature dataset
training_feature_data = pd.read_csv(
    DATA_PATH
)

print("Final deployment model loaded.")
print("Dataset shape:", training_feature_data.shape)

print("\nFinal deployment features:")

for i, feature in enumerate(
    final_deployment_features,
    start=1
):
    print(i, feature)

In [ ]:
# ============================================================
# Phase 19B — Suspicious lexical keyword list
# ============================================================

SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]

In [ ]:
# ============================================================
# Phase 19C — Robust scheme handling
# ============================================================

def ensure_scheme(url):

    url = str(url).strip()

    if not url:
        return ""

    # Only detect a scheme if it appears at the BEGINNING
    # of the URL.
    has_real_scheme = re.match(
        r'^[A-Za-z][A-Za-z0-9+.\-]*://',
        url
    )

    if has_real_scheme:
        return url

    return "http://" + url

In [ ]:
scheme_test_urls = [
    "google.com/test",
    "https://google.com/test",
    "example.com/path?redirect=file:///C:/test",
    r"example.com/path?u=xhxxp\://fake.com/test"
]

for url in scheme_test_urls:

    parsed = urlsplit(
        ensure_scheme(url)
    )

    print(
        url,
        "->",
        parsed.hostname
    )

In [ ]:
# ============================================================
# Phase 19D — Verify training-time URL normalizer
# ============================================================

assert "normalize_url" in globals(), (
    "normalize_url is not defined. "
    "Run the original normalization-function cell first."
)

print(
    "normalize_url is available."
)

In [ ]:
# ============================================================
# Phase 19E — Domain and subdomain utilities
# ============================================================

try:
    import tldextract

    domain_extractor = tldextract.TLDExtract(
        suffix_list_urls=None
    )

except Exception:
    domain_extractor = None


def get_registered_domain(hostname):

    hostname = str(
        hostname
    ).lower().strip(".")

    if not hostname:
        return ""

    if domain_extractor is None:
        return hostname

    ext = domain_extractor(
        hostname
    )

    if not ext.domain:
        return hostname

    if ext.suffix:
        return (
            ext.domain
            + "."
            + ext.suffix
        )

    return ext.domain


def count_subdomains(hostname):

    hostname = str(
        hostname
    ).lower().strip(".")

    if not hostname:
        return 0

    registered_domain = (
        get_registered_domain(
            hostname
        )
    )

    hostname_parts = [
        x
        for x in hostname.split(".")
        if x
    ]

    domain_parts = [
        x
        for x in registered_domain.split(".")
        if x
    ]

    return max(
        len(hostname_parts)
        -
        len(domain_parts),
        0
    )

In [ ]:
# ============================================================
# Phase 19F — Final deployment URL parser
# ============================================================

def parse_deployment_url(url):

    raw_url = str(url).strip()

    # Reuse EXACT training-time normalization
    normalized_url = normalize_url(
        raw_url
    )

    # --------------------------------------------------------
    # Raw hostname
    # --------------------------------------------------------

    try:

        raw_parsed = urlsplit(
            ensure_scheme(raw_url)
        )

        raw_hostname = (
            raw_parsed.hostname
            or ""
        ).lower()

    except Exception:

        raw_hostname = ""

    # --------------------------------------------------------
    # Normalized path and query
    # --------------------------------------------------------

    try:

        normalized_parsed = urlsplit(
            normalized_url
        )

        normalized_path = (
            normalized_parsed.path
            or ""
        )

        normalized_query = (
            normalized_parsed.query
            or ""
        )

    except Exception:

        normalized_path = ""
        normalized_query = ""

    return {
        "raw_url":
            raw_url,

        "normalized_url":
            normalized_url,

        "raw_hostname":
            raw_hostname,

        "path":
            normalized_path,

        "query":
            normalized_query
    }

In [ ]:
# ============================================================
# Phase 19G — Final deployment feature extractor
# ============================================================

def extract_deployment_features(url):

    parsed = parse_deployment_url(
        url
    )

    raw_url = parsed[
        "raw_url"
    ]

    raw_hostname = parsed[
        "raw_hostname"
    ]

    path = parsed[
        "path"
    ]

    query = parsed[
        "query"
    ]

    url_lower = raw_url.lower()

    # --------------------------------------------------------
    # Final verified feature definitions
    # --------------------------------------------------------

    url_length = len(
        raw_url
    )

    # Verified:
    # training-time feature keeps "www."
    hostname_length = len(
        raw_hostname
    )

    # Verified:
    # path comes from normalized_url
    path_length = len(
        path
    )

    query_length = len(
        query
    )

    num_digits = sum(
        c.isdigit()
        for c in raw_url
    )

    num_dots = raw_url.count(
        "."
    )

    num_hyphens = raw_url.count(
        "-"
    )

    num_special_chars = sum(
        not c.isalnum()
        for c in raw_url
    )

    special_char_ratio = (
        num_special_chars
        /
        url_length
        if url_length > 0
        else 0.0
    )

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    num_subdomains = (
        count_subdomains(
            raw_hostname
        )
    )

    return {
        "num_subdomains":
            num_subdomains,

        "suspicious_keyword_count":
            suspicious_keyword_count,

        "num_digits":
            num_digits,

        "path_length":
            path_length,

        "hostname_length":
            hostname_length,

        "url_length":
            url_length,

        "num_dots":
            num_dots,

        "special_char_ratio":
            special_char_ratio,

        "num_hyphens":
            num_hyphens,

        "query_length":
            query_length
    }

In [ ]:
# ============================================================
# Phase 19H — Verify the five previously problematic samples
# ============================================================

problem_samples = [
    1318,
    2097,
    5575,
    5577,
    5773
]

problem_check_records = []

for idx in problem_samples:

    row = (
        training_feature_data
        .iloc[idx]
    )

    new_features = (
        extract_deployment_features(
            row["url"]
        )
    )

    problem_check_records.append({
        "sample":
            idx,

        "url":
            row["url"],

        "old_hostname_length":
            row["hostname_length"],

        "new_hostname_length":
            new_features[
                "hostname_length"
            ],

        "old_num_subdomains":
            row["num_subdomains"],

        "new_num_subdomains":
            new_features[
                "num_subdomains"
            ],

        "old_path_length":
            row["path_length"],

        "new_path_length":
            new_features[
                "path_length"
            ]
    })

problem_check_df = pd.DataFrame(
    problem_check_records
)

problem_check_df

In [ ]:
print(
    "Hostname mismatches:",
    (
        problem_check_df[
            "old_hostname_length"
        ]
        !=
        problem_check_df[
            "new_hostname_length"
        ]
    ).sum()
)

print(
    "Subdomain mismatches:",
    (
        problem_check_df[
            "old_num_subdomains"
        ]
        !=
        problem_check_df[
            "new_num_subdomains"
        ]
    ).sum()
)

print(
    "Path mismatches:",
    (
        problem_check_df[
            "old_path_length"
        ]
        !=
        problem_check_df[
            "new_path_length"
        ]
    ).sum()
)

In [ ]:
# ============================================================
# Phase 19I — Full-dataset deployment consistency audit
# ============================================================

consistency_records = []

for i, row in (
    training_feature_data
    .reset_index(drop=True)
    .iterrows()
):

    new_features = (
        extract_deployment_features(
            row["url"]
        )
    )

    for feature in final_deployment_features:

        old_value = float(
            row[feature]
        )

        new_value = float(
            new_features[
                feature
            ]
        )

        consistency_records.append({
            "sample":
                i,

            "feature":
                feature,

            "old_value":
                old_value,

            "new_value":
                new_value,

            "difference":
                old_value
                -
                new_value
        })


deployment_consistency_df = (
    pd.DataFrame(
        consistency_records
    )
)


mismatch_df = (
    deployment_consistency_df[
        deployment_consistency_df[
            "difference"
        ]
        .abs()
        >
        1e-12
    ]
    .copy()
)


max_difference = (
    deployment_consistency_df[
        "difference"
    ]
    .abs()
    .max()
)


print(
    "Samples checked:",
    len(training_feature_data)
)

print(
    "Feature values checked:",
    len(deployment_consistency_df)
)

print(
    "Maximum absolute difference:",
    max_difference
)

print(
    "Mismatch count:",
    len(mismatch_df)
)


if len(mismatch_df) > 0:

    print(
        "\nMismatch features:"
    )

    print(
        mismatch_df[
            "feature"
        ]
        .value_counts()
    )

else:

    print(
        "\nSUCCESS: deployment extractor exactly reproduces training features."
    )

In [ ]:
# ============================================================
# Phase 19J — Single URL prediction function
# ============================================================

def predict_url(
    url,
    threshold=0.5
):

    feature_dict = (
        extract_deployment_features(
            url
        )
    )

    X_input = pd.DataFrame(
        [
            feature_dict
        ]
    )

    X_input = X_input[
        final_deployment_features
    ]

    phishing_probability = (
        final_deployment_model
        .predict_proba(
            X_input
        )[0, 1]
    )

    prediction = int(
        phishing_probability
        >=
        threshold
    )

    return {
        "url":
            url,

        "prediction":
            prediction,

        "label":
            (
                "PHISHING"
                if prediction == 1
                else "BENIGN"
            ),

        "phishing_probability":
            float(
                phishing_probability
            ),

        "features":
            feature_dict
    }

In [ ]:
# ============================================================
# Phase 19K — Human-readable lexical explanation
# ============================================================

def generate_basic_explanation(
    feature_dict
):

    explanations = []

    if (
        feature_dict[
            "suspicious_keyword_count"
        ]
        >
        0
    ):
        explanations.append(
            "Suspicious lexical keywords were detected."
        )

    if (
        feature_dict[
            "num_subdomains"
        ]
        >=
        2
    ):
        explanations.append(
            "The URL contains multiple subdomain levels."
        )

    if (
        feature_dict[
            "url_length"
        ]
        >=
        75
    ):
        explanations.append(
            "The URL is relatively long."
        )

    if (
        feature_dict[
            "num_digits"
        ]
        >=
        5
    ):
        explanations.append(
            "The URL contains several numeric characters."
        )

    if (
        feature_dict[
            "query_length"
        ]
        >=
        15
    ):
        explanations.append(
            "The URL contains a relatively long query string."
        )

    if (
        feature_dict[
            "path_length"
        ]
        >=
        30
    ):
        explanations.append(
            "The URL contains a relatively long path."
        )

    if not explanations:

        explanations.append(
            "No strong manually defined lexical warning pattern was observed."
        )

    return explanations

In [ ]:
# ============================================================
# Phase 19L — Final notebook prototype
# ============================================================

def phishing_detection_system(
    url
):

    result = predict_url(
        url
    )

    probability = (
        result[
            "phishing_probability"
        ]
    )

    explanations = (
        generate_basic_explanation(
            result[
                "features"
            ]
        )
    )

    print(
        "=" * 72
    )

    print(
        "LIGHTWEIGHT QR-PHISHING URL DETECTION SYSTEM"
    )

    print(
        "=" * 72
    )

    print(
        "\nInput URL:"
    )

    print(
        result["url"]
    )

    print(
        "\nPrediction:",
        result["label"]
    )

    print(
        "Phishing probability:",
        f"{probability:.4f}"
    )

    print(
        "Decision threshold:",
        "0.5000"
    )

    print(
        "\nLexical observations:"
    )

    for item in explanations:

        print(
            "-",
            item
        )

    feature_table = pd.DataFrame({
        "Feature":
            list(
                result[
                    "features"
                ].keys()
            ),

        "Value":
            list(
                result[
                    "features"
                ].values()
            )
    })

    return feature_table

In [ ]:
phishing_detection_system(
    "https://www.google.com/"
)

In [ ]:
phishing_detection_system(
    "http://secure-login-account-example.com/verify/account?id=123456"
)

In [ ]:
def run_prototype():

    print("=" * 72)
    print("LIGHTWEIGHT QR-PHISHING URL DETECTION SYSTEM")
    print("=" * 72)

    url = input("\nEnter a URL: ").strip()

    if not url:
        print("No URL entered.")
        return

    feature_table = phishing_detection_system(url)

    display(feature_table)

In [ ]:
run_prototype()

In [ ]:
# ============================================================
# Phase 19N-1 — QR Prototype Imports
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd

from PIL import Image
from IPython.display import display

In [ ]:
print("OpenCV version:", cv2.__version__)

In [ ]:
# ============================================================
# Phase 19N-2 — QR Code Decoder
# ============================================================

def decode_qr_image(image_path):

    if not image_path:
        return {
            "success": False,
            "decoded_text": None,
            "message": "No image was selected."
        }

    if not os.path.exists(image_path):
        return {
            "success": False,
            "decoded_text": None,
            "message": "The selected image does not exist."
        }

    image = cv2.imread(
        image_path
    )

    if image is None:
        return {
            "success": False,
            "decoded_text": None,
            "message": "The image could not be loaded."
        }

    detector = cv2.QRCodeDetector()

    decoded_text, points, straight_qrcode = (
        detector.detectAndDecode(
            image
        )
    )

    decoded_text = str(
        decoded_text
    ).strip()

    if not decoded_text:

        return {
            "success": False,
            "decoded_text": None,
            "message":
                "A decodable QR code could not be detected."
        }

    return {
        "success": True,
        "decoded_text": decoded_text,
        "message":
            "QR code decoded successfully."
    }

In [ ]:
import qrcode

test_url = "https://www.google.com/"

qr = qrcode.QRCode(
    version=None,
    error_correction=qrcode.constants.ERROR_CORRECT_M,
    box_size=10,
    border=4
)

qr.add_data(test_url)
qr.make(fit=True)

img = qr.make_image(
    fill_color="black",
    back_color="white"
)

img.save("test_qr_valid.png")

print("Saved: test_qr_valid.png")
print("Encoded URL:", test_url)

In [ ]:
qr_test = decode_qr_image(
    "test_qr_valid.png"
)

qr_test

In [ ]:
# ============================================================
# Complete QR phishing detection pipeline
# ============================================================

def analyze_qr_image(
    image_path,
    threshold=0.5
):

    qr_result = decode_qr_image(
        image_path
    )

    if not qr_result["success"]:

        return {
            "success": False,
            "message": qr_result["message"]
        }

    decoded_url = qr_result[
        "decoded_text"
    ]

    feature_dict = (
        extract_deployment_features(
            decoded_url
        )
    )

    X_input = pd.DataFrame(
        [feature_dict]
    )

    X_input = X_input[
        final_deployment_features
    ]

    phishing_probability = (
        final_deployment_model
        .predict_proba(
            X_input
        )[0, 1]
    )

    predicted_label = int(
        phishing_probability >= threshold
    )

    prediction = (
        "PHISHING"
        if predicted_label == 1
        else "BENIGN"
    )

    explanations = (
        generate_basic_explanation(
            feature_dict
        )
    )

    return {
        "success": True,
        "image_path": image_path,
        "decoded_url": decoded_url,
        "prediction": prediction,
        "predicted_label": predicted_label,
        "phishing_probability": float(
            phishing_probability
        ),
        "threshold": threshold,
        "features": feature_dict,
        "explanations": explanations
    }

In [ ]:
# ============================================================
# Display QR detection result
# ============================================================

def display_qr_detection_result(
    result
):

    if not result["success"]:

        print("Detection failed:")
        print(result["message"])

        return

    print("=" * 72)
    print("LIGHTWEIGHT QR-PHISHING DETECTION SYSTEM")
    print("=" * 72)

    print("\nDecoded URL:")
    print(result["decoded_url"])

    print("\nPrediction:")
    print(result["prediction"])

    print(
        "\nPhishing probability:",
        f'{result["phishing_probability"]:.4f}'
    )

    print(
        "Decision threshold:",
        f'{result["threshold"]:.4f}'
    )

    print("\nLexical observations:")

    for item in result["explanations"]:
        print("-", item)

    feature_table = pd.DataFrame({
        "Feature":
            list(
                result["features"].keys()
            ),

        "Value":
            list(
                result["features"].values()
            )
    })

    print("\nExtracted Features:")

    display(feature_table)

In [ ]:
result = analyze_qr_image(
    "test_qr_valid.png"
)

display_qr_detection_result(
    result
)

In [ ]:
# ============================================================
# Sanity check — verify label interpretation
# ============================================================

check_samples = pd.concat([
    training_feature_data[
        training_feature_data["label"] == 0
    ].sample(
        5,
        random_state=42
    ),

    training_feature_data[
        training_feature_data["label"] == 1
    ].sample(
        5,
        random_state=42
    )
]).reset_index(drop=True)

records = []

for _, row in check_samples.iterrows():

    result = predict_url(
        row["url"]
    )

    records.append({
        "url":
            row["url"],

        "true_label":
            row["label"],

        "predicted_label":
            result["prediction"],

        "phishing_probability":
            result["phishing_probability"]
    })

sanity_df = pd.DataFrame(
    records
)

sanity_df

In [ ]:
print("Remaining mismatches:", len(mismatch_df))

if len(mismatch_df) == 0:
    print("Deployment feature extraction is fully consistent.")
else:
    print(mismatch_df["feature"].value_counts())

In [ ]:
# ============================================================
# Diagnose the final two num_subdomains mismatches
# ============================================================

final_subdomain_mismatch = (
    mismatch_df[
        mismatch_df["feature"] == "num_subdomains"
    ]
    .copy()
)

final_subdomain_mismatch

In [ ]:
final_subdomain_diag = []

for idx in final_subdomain_mismatch["sample"].unique():

    row = training_feature_data.iloc[int(idx)]

    raw_url = str(row["url"])
    normalized_url = str(row["normalized_url"])

    raw_parsed = urlsplit(
        ensure_scheme(raw_url)
    )

    norm_parsed = urlsplit(
        normalized_url
    )

    raw_hostname = (
        raw_parsed.hostname or ""
    ).lower()

    normalized_hostname = (
        norm_parsed.hostname or ""
    ).lower()

    final_subdomain_diag.append({
        "sample": int(idx),
        "url": raw_url,
        "normalized_url": normalized_url,
        "stored_hostname": row["hostname"],
        "stored_domain": row["domain"],
        "stored_num_subdomains": int(row["num_subdomains"]),
        "raw_hostname": raw_hostname,
        "normalized_hostname": normalized_hostname,
        "current_registered_domain":
            get_registered_domain(raw_hostname),
        "current_num_subdomains":
            count_subdomains(raw_hostname)
    })

final_subdomain_diag_df = pd.DataFrame(
    final_subdomain_diag
)

final_subdomain_diag_df

In [ ]:
for _, row in final_subdomain_diag_df.iterrows():

    print("=" * 90)

    print("Sample:", row["sample"])

    print("\nURL:")
    print(row["url"])

    print("\nStored hostname:")
    print(row["stored_hostname"])

    print("Stored domain:")
    print(row["stored_domain"])

    print(
        "Stored num_subdomains:",
        row["stored_num_subdomains"]
    )

    print("\nRaw hostname:")
    print(row["raw_hostname"])

    print(
        "Current registered domain:",
        row["current_registered_domain"]
    )

    print(
        "Current num_subdomains:",
        row["current_num_subdomains"]
    )

In [ ]:
print(
    get_registered_domain(
        "www.fpl.fs.fed.us"
    )
)

print(
    count_subdomains(
        "www.fpl.fs.fed.us"
    )
)

In [ ]:
for idx in [1318, 5577]:

    row = training_feature_data.iloc[idx]

    new_features = extract_deployment_features(
        row["url"]
    )

    print("=" * 60)
    print("Sample:", idx)

    print(
        "Stored:",
        row["num_subdomains"]
    )

    print(
        "Deployment:",
        new_features["num_subdomains"]
    )

In [ ]:
# ============================================================
# FINAL FIX — training-compatible domain parsing
# ============================================================

def get_registered_domain(hostname):

    hostname = str(hostname).lower().strip(".")

    if not hostname:
        return ""

    # Compatibility with training-time domain definition
    if hostname == "fed.us" or hostname.endswith(".fed.us"):
        return "fed.us"

    # Normal cases
    if domain_extractor is None:
        return hostname

    ext = domain_extractor(hostname)

    if not ext.domain:
        return hostname

    if ext.suffix:
        return ext.domain + "." + ext.suffix

    return ext.domain


def count_subdomains(hostname):

    hostname = str(hostname).lower().strip(".")

    if not hostname:
        return 0

    registered_domain = get_registered_domain(
        hostname
    )

    hostname_parts = [
        x for x in hostname.split(".") if x
    ]

    domain_parts = [
        x for x in registered_domain.split(".") if x
    ]

    return max(
        len(hostname_parts)
        -
        len(domain_parts),
        0
    )


print(
    "Registered domain:",
    get_registered_domain(
        "www.fpl.fs.fed.us"
    )
)

print(
    "Subdomains:",
    count_subdomains(
        "www.fpl.fs.fed.us"
    )
)

In [ ]:
for idx in [1318, 5577]:

    row = training_feature_data.iloc[idx]

    new_features = extract_deployment_features(
        row["url"]
    )

    print("=" * 50)
    print("Sample:", idx)
    print(
        "Stored:",
        row["num_subdomains"]
    )
    print(
        "Deployment:",
        new_features["num_subdomains"]
    )

In [ ]:
# ============================================================
# Phase 19I — Full-dataset deployment consistency audit
# Run this AFTER the final domain/subdomain fix
# ============================================================

consistency_records = []

for i, row in (
    training_feature_data
    .reset_index(drop=True)
    .iterrows()
):

    new_features = (
        extract_deployment_features(
            row["url"]
        )
    )

    for feature in final_deployment_features:

        old_value = float(
            row[feature]
        )

        new_value = float(
            new_features[feature]
        )

        consistency_records.append({
            "sample":
                i,

            "feature":
                feature,

            "old_value":
                old_value,

            "new_value":
                new_value,

            "difference":
                old_value - new_value
        })


deployment_consistency_df = pd.DataFrame(
    consistency_records
)


mismatch_df = (
    deployment_consistency_df[
        deployment_consistency_df[
            "difference"
        ].abs() > 1e-12
    ]
    .copy()
)


max_difference = (
    deployment_consistency_df[
        "difference"
    ]
    .abs()
    .max()
)


print("=" * 72)
print("PHASE 19I — FULL DEPLOYMENT CONSISTENCY AUDIT")
print("=" * 72)

print(
    "Samples checked:",
    len(training_feature_data)
)

print(
    "Feature values checked:",
    len(deployment_consistency_df)
)

print(
    "Maximum absolute difference:",
    max_difference
)

print(
    "Mismatch count:",
    len(mismatch_df)
)


if len(mismatch_df) > 0:

    print("\nMismatch features:")

    print(
        mismatch_df[
            "feature"
        ]
        .value_counts()
    )

else:

    print(
        "\nSUCCESS: deployment extractor exactly reproduces training features."
    )

In [ ]:
print(
    "Remaining mismatches:",
    len(mismatch_df)
)

In [ ]:
# ============================================================
# Phase 19J — Native QR Image Selector
# ============================================================

import tkinter as tk
from tkinter import filedialog

def select_qr_image():

    root = tk.Tk()
    root.withdraw()

    root.attributes(
        "-topmost",
        True
    )

    file_path = filedialog.askopenfilename(
        title="Select QR Code Image",
        filetypes=[
            (
                "Image files",
                "*.png *.jpg *.jpeg *.bmp *.webp"
            ),
            (
                "All files",
                "*.*"
            )
        ]
    )

    root.destroy()

    return file_path

In [ ]:
# ============================================================
# Phase 19K — Final QR Phishing Detection Prototype
# ============================================================

from PIL import Image
from IPython.display import display

def run_qr_phishing_detector():

    print("=" * 72)
    print("LIGHTWEIGHT QR-PHISHING DETECTION SYSTEM")
    print("=" * 72)

    print(
        "\nPlease select a QR-code image..."
    )

    # --------------------------------------------------------
    # Step 1 — Select image
    # --------------------------------------------------------

    image_path = select_qr_image()

    if not image_path:

        print(
            "\nNo image selected."
        )

        return None

    print(
        "\nSelected image:"
    )

    print(
        image_path
    )

    # --------------------------------------------------------
    # Step 2 — Display QR image
    # --------------------------------------------------------

    try:

        image = Image.open(
            image_path
        )

        display(
            image
        )

    except Exception as e:

        print(
            "Image preview unavailable:",
            e
        )

    # --------------------------------------------------------
    # Step 3 — Decode + classify
    # --------------------------------------------------------

    result = analyze_qr_image(
        image_path
    )

    # --------------------------------------------------------
    # Step 4 — Display prediction
    # --------------------------------------------------------

    display_qr_detection_result(
        result
    )

    return result

In [ ]:
result = run_qr_phishing_detector()

In [ ]:
# ============================================================
# Explain one prediction using XGBoost feature contributions
# ============================================================

import xgboost as xgb
import pandas as pd
import numpy as np

test_url = "https://www.google.com/"

# Extract exactly the same deployment features
feature_dict = extract_deployment_features(
    test_url
)

X_explain = pd.DataFrame(
    [feature_dict]
)[final_deployment_features]

# Get underlying booster
booster = final_deployment_model.get_booster()

dmatrix = xgb.DMatrix(
    X_explain,
    feature_names=final_deployment_features
)

# SHAP-like additive contributions from XGBoost
contributions = booster.predict(
    dmatrix,
    pred_contribs=True
)[0]

# Last value = bias/base value
feature_contribs = contributions[:-1]
bias = contributions[-1]

explanation_df = pd.DataFrame({
    "feature":
        final_deployment_features,

    "value":
        X_explain.iloc[0].values,

    "contribution":
        feature_contribs
})

explanation_df["direction"] = np.where(
    explanation_df["contribution"] > 0,
    "Toward phishing",
    "Toward benign"
)

explanation_df["abs_contribution"] = (
    explanation_df["contribution"].abs()
)

explanation_df = (
    explanation_df
    .sort_values(
        "abs_contribution",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Model probability:",
    final_deployment_model.predict_proba(
        X_explain
    )[0, 1]
)

print(
    "Base contribution:",
    bias
)

explanation_df

In [ ]:
# ============================================================
# Diagnose path_length = 1
# ============================================================

path1 = training_feature_data[
    training_feature_data["path_length"] == 1
]

print("Samples with path_length = 1:", len(path1))

print("\nClass distribution:")
print(
    path1["label"].value_counts(
        normalize=False
    )
)

print("\nClass proportions:")
print(
    path1["label"].value_counts(
        normalize=True
    )
)

In [ ]:
path_stats = (
    training_feature_data
    .groupby("path_length")
    .agg(
        samples=("label", "size"),
        phishing_rate=("label", "mean")
    )
    .reset_index()
)

path_stats[
    path_stats["samples"] >= 20
].head(30)

In [ ]:
# ============================================================
# Short-path bias analysis
# ============================================================

short_path_summary = []

for threshold in [0, 1, 2, 3, 5, 10]:

    subset = training_feature_data[
        training_feature_data["path_length"] <= threshold
    ]

    if len(subset) == 0:
        continue

    counts = (
        subset["label"]
        .value_counts()
    )

    benign = int(
        counts.get(0, 0)
    )

    phishing = int(
        counts.get(1, 0)
    )

    short_path_summary.append({
        "path_length_max":
            threshold,

        "samples":
            len(subset),

        "benign":
            benign,

        "phishing":
            phishing,

        "phishing_rate":
            subset["label"].mean()
    })

short_path_summary = pd.DataFrame(
    short_path_summary
)

short_path_summary

In [ ]:
# ============================================================
# Save short-path dataset-bias analysis
# ============================================================

short_path_summary.to_csv(
    "phase19_short_path_bias_analysis.csv",
    index=False
)

print("Saved: phase19_short_path_bias_analysis.csv")

In [ ]:
# ============================================================
# Phase 20A — XGBoost local prediction explanation
# ============================================================

import xgboost as xgb
import numpy as np
import pandas as pd


def explain_xgb_prediction(
    feature_dict,
    top_n=5
):

    X_explain = pd.DataFrame(
        [feature_dict]
    )[final_deployment_features]

    booster = (
        final_deployment_model
        .get_booster()
    )

    dmatrix = xgb.DMatrix(
        X_explain,
        feature_names=final_deployment_features
    )

    contributions = booster.predict(
        dmatrix,
        pred_contribs=True
    )[0]

    # Last value is bias
    feature_contributions = (
        contributions[:-1]
    )

    explanation_df = pd.DataFrame({
        "Feature":
            final_deployment_features,

        "Value":
            X_explain.iloc[0].values,

        "Contribution":
            feature_contributions
    })

    explanation_df[
        "Direction"
    ] = np.where(
        explanation_df[
            "Contribution"
        ] > 0,
        "Toward phishing",
        "Toward benign"
    )

    explanation_df[
        "Absolute Contribution"
    ] = (
        explanation_df[
            "Contribution"
        ].abs()
    )

    explanation_df = (
        explanation_df
        .sort_values(
            "Absolute Contribution",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    return explanation_df

In [ ]:
# ============================================================
# Phase 20B — Improved final result display
# ============================================================

from IPython.display import display


def display_qr_detection_result(
    result
):

    if not result["success"]:

        print("=" * 72)
        print("QR-PHISHING DETECTION SYSTEM")
        print("=" * 72)

        print("\nQR decoding failed:")
        print(result["message"])

        return

    probability = result[
        "phishing_probability"
    ]

    prediction = result[
        "prediction"
    ]

    print("=" * 72)
    print("LIGHTWEIGHT QR-PHISHING DETECTION SYSTEM")
    print("=" * 72)

    print("\nDecoded URL:")
    print(
        result["decoded_url"]
    )

    print("\nPrediction:")
    print(
        prediction
    )

    print(
        "\nPhishing probability:",
        f"{probability:.4f}"
    )

    print(
        "Decision threshold:",
        f'{result["threshold"]:.4f}'
    )

    # --------------------------------------------------------
    # Real model explanation
    # --------------------------------------------------------

    explanation_df = (
        explain_xgb_prediction(
            result["features"],
            top_n=5
        )
    )

    print(
        "\nTop model feature contributions:"
    )

    display(
        explanation_df[
            [
                "Feature",
                "Value",
                "Contribution",
                "Direction"
            ]
        ]
    )

    # --------------------------------------------------------
    # Full feature table
    # --------------------------------------------------------

    feature_table = pd.DataFrame({
        "Feature":
            list(
                result[
                    "features"
                ].keys()
            ),

        "Value":
            list(
                result[
                    "features"
                ].values()
            )
    })

    print(
        "\nExtracted lexical features:"
    )

    display(
        feature_table
    )

    print(
        "\nNote:"
    )

    print(
        "This prototype uses lexical URL features only. "
        "It does not use domain reputation, website content, "
        "or external threat intelligence."
    )

In [ ]:
result = run_qr_phishing_detector()

# Phase 21 — External Validation on Independent URL Dataset

In [ ]:
# ============================================================
# Phase 21A — Load External Dataset
# ============================================================

import pandas as pd
import numpy as np

external_df = pd.read_csv(
    "C:\\Users\\LENOVO\\Desktop\\FYP\\archive\\balanced_urls.csv"
)

print(
    "External dataset shape:",
    external_df.shape
)

print(
    "\nColumns:",
    external_df.columns.tolist()
)

print(
    "\nLabel counts:"
)

print(
    external_df["result"]
    .value_counts()
    .sort_index()
)

external_df.head()

In [ ]:
# ============================================================
# Phase 21B — Basic External Dataset Audit
# ============================================================

print(
    "Missing URLs:",
    external_df["url"].isna().sum()
)

print(
    "Missing labels:",
    external_df["result"].isna().sum()
)

print(
    "Duplicate URLs:",
    external_df["url"].duplicated().sum()
)

print(
    "Unique URLs:",
    external_df["url"].nunique()
)

print(
    "\nClass proportions:"
)

print(
    external_df["result"]
    .value_counts(
        normalize=True
    )
    .sort_index()
)

In [ ]:
# ============================================================
# Phase 21C — Extract External Hostnames
# ============================================================

from urllib.parse import urlsplit


def get_external_hostname(url):

    try:

        parsed = urlsplit(
            ensure_scheme(
                str(url)
            )
        )

        return (
            parsed.hostname
            or ""
        ).lower()

    except Exception:

        return ""


external_df["hostname"] = (
    external_df["url"]
    .apply(
        get_external_hostname
    )
)

print(
    "Empty hostnames:",
    (
        external_df["hostname"]
        == ""
    ).sum()
)

print(
    "Unique external hostnames:",
    external_df["hostname"].nunique()
)

In [ ]:
# ============================================================
# Phase 21D — Original Dataset Hostnames
# ============================================================

original_urls = set(
    training_feature_data[
        "url"
    ].astype(str)
)

original_hostnames = set(
    training_feature_data[
        "hostname"
    ]
    .astype(str)
    .str.lower()
)

print(
    "Original unique URLs:",
    len(original_urls)
)

print(
    "Original unique hostnames:",
    len(original_hostnames)
)

In [ ]:
# ============================================================
# Phase 21E — Exact URL Overlap Audit
# ============================================================

external_df[
    "url_overlap"
] = (
    external_df["url"]
    .astype(str)
    .isin(
        original_urls
    )
)

url_overlap_count = (
    external_df[
        "url_overlap"
    ]
    .sum()
)

print(
    "Exact URL overlap:",
    url_overlap_count
)

print(
    "Exact URL overlap rate:",
    url_overlap_count
    /
    len(external_df)
)

In [ ]:
# ============================================================
# Phase 21F — Hostname Overlap Audit
# ============================================================

external_df[
    "hostname_overlap"
] = (
    external_df[
        "hostname"
    ]
    .isin(
        original_hostnames
    )
)

hostname_overlap_count = (
    external_df[
        "hostname_overlap"
    ]
    .sum()
)

print(
    "Hostname-overlap samples:",
    hostname_overlap_count
)

print(
    "Hostname-overlap rate:",
    hostname_overlap_count
    /
    len(external_df)
)

print(
    "Unique overlapping hostnames:",
    external_df.loc[
        external_df[
            "hostname_overlap"
        ],
        "hostname"
    ].nunique()
)

In [ ]:
# ============================================================
# Phase 21G — Strict Unseen-Host External Dataset
# ============================================================

external_unseen = (
    external_df[
        (~external_df["hostname_overlap"])
        &
        (external_df["hostname"] != "")
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Unseen-host external samples:",
    len(external_unseen)
)

print(
    "\nClass counts:"
)

print(
    external_unseen[
        "result"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nClass proportions:"
)

print(
    external_unseen[
        "result"
    ]
    .value_counts(
        normalize=True
    )
    .sort_index()
)

In [ ]:
# ============================================================
# Phase 21H — Balanced External Evaluation Sample
# ============================================================

EXTERNAL_PER_CLASS = 10000
RANDOM_STATE = 42

external_benign = (
    external_unseen[
        external_unseen[
            "result"
        ] == 0
    ]
    .sample(
        n=min(
            EXTERNAL_PER_CLASS,
            (
                external_unseen[
                    "result"
                ] == 0
            ).sum()
        ),
        random_state=RANDOM_STATE
    )
)

external_malicious = (
    external_unseen[
        external_unseen[
            "result"
        ] == 1
    ]
    .sample(
        n=min(
            EXTERNAL_PER_CLASS,
            (
                external_unseen[
                    "result"
                ] == 1
            ).sum()
        ),
        random_state=RANDOM_STATE
    )
)

external_eval = (
    pd.concat(
        [
            external_benign,
            external_malicious
        ],
        ignore_index=True
    )
    .sample(
        frac=1,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

print(
    "External evaluation size:",
    len(external_eval)
)

print(
    external_eval[
        "result"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# Phase 21I — External Feature Extraction
# ============================================================

external_feature_records = []

for i, url in enumerate(
    external_eval["url"]
):

    features = (
        extract_deployment_features(
            url
        )
    )

    external_feature_records.append(
        features
    )

    if (
        (i + 1)
        % 2000
        == 0
    ):

        print(
            "Processed:",
            i + 1,
            "/",
            len(external_eval)
        )


X_external = pd.DataFrame(
    external_feature_records
)

X_external = X_external[
    final_deployment_features
]

y_external = (
    external_eval[
        "result"
    ]
    .astype(int)
    .to_numpy()
)

print(
    "\nExternal feature matrix:",
    X_external.shape
)

print(
    "Missing feature values:",
    X_external.isna().sum().sum()
)

X_external.head()

In [ ]:
# ============================================================
# Phase 21J — Frozen Model External Prediction
# ============================================================

external_prob = (
    final_deployment_model
    .predict_proba(
        X_external
    )[:, 1]
)

external_pred = (
    external_prob
    >= 0.5
).astype(int)

print(
    "External predictions completed."
)

In [ ]:
# ============================================================
# Phase 21K — External Validation Metrics
# ============================================================

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix
)


external_f1 = f1_score(
    y_external,
    external_pred
)

external_precision = (
    precision_score(
        y_external,
        external_pred
    )
)

external_recall = (
    recall_score(
        y_external,
        external_pred
    )
)

external_accuracy = (
    accuracy_score(
        y_external,
        external_pred
    )
)

external_auc = (
    roc_auc_score(
        y_external,
        external_prob
    )
)

external_mcc = (
    matthews_corrcoef(
        y_external,
        external_pred
    )
)

external_cm = confusion_matrix(
    y_external,
    external_pred
)

print("=" * 72)
print("PHASE 21 — EXTERNAL VALIDATION RESULTS")
print("=" * 72)

print(
    "Samples:",
    len(y_external)
)

print(
    "F1:",
    external_f1
)

print(
    "Precision:",
    external_precision
)

print(
    "Recall:",
    external_recall
)

print(
    "Accuracy:",
    external_accuracy
)

print(
    "AUROC:",
    external_auc
)

print(
    "MCC:",
    external_mcc
)

print(
    "\nConfusion matrix:"
)

print(
    external_cm
)

In [ ]:
# ============================================================
# Phase 21L — Save External Predictions
# ============================================================

external_prediction_df = (
    external_eval[
        [
            "url",
            "hostname",
            "label",
            "result"
        ]
    ]
    .copy()
)

external_prediction_df[
    "predicted_label"
] = external_pred

external_prediction_df[
    "phishing_probability"
] = external_prob

external_prediction_df[
    "correct"
] = (
    external_prediction_df[
        "result"
    ]
    ==
    external_prediction_df[
        "predicted_label"
    ]
)

external_prediction_df.to_csv(
    "phase21_external_predictions.csv",
    index=False
)

print(
    "Saved: phase21_external_predictions.csv"
)

In [ ]:
# ============================================================
# Phase 21M — Internal vs External Comparison
# ============================================================

comparison_df = pd.DataFrame([
    {
        "dataset":
            "Internal held-out",

        "n":
            1777,

        "f1":
            0.881087,

        "precision":
            0.896313,

        "recall":
            0.866370,

        "auroc":
            0.950949,

        "mcc":
            0.764140
    },

    {
        "dataset":
            "External unseen-host",

        "n":
            len(y_external),

        "f1":
            external_f1,

        "precision":
            external_precision,

        "recall":
            external_recall,

        "auroc":
            external_auc,

        "mcc":
            external_mcc
    }
])

comparison_df

In [ ]:
comparison_df.to_csv(
    "phase21_internal_external_comparison.csv",
    index=False
)

In [ ]:
# ============================================================
# Phase 21N — Generalization Gap
# ============================================================

internal_f1 = 0.881087
internal_auc = 0.950949
internal_mcc = 0.764140

print(
    "ΔF1 external - internal:",
    external_f1
    -
    internal_f1
)

print(
    "ΔAUROC external - internal:",
    external_auc
    -
    internal_auc
)

print(
    "ΔMCC external - internal:",
    external_mcc
    -
    internal_mcc
)

In [ ]:
overlap_summary = pd.DataFrame([{
    "external_total": len(external_df),
    "unseen_host_samples": len(external_unseen),
    "unseen_host_rate": len(external_unseen) / len(external_df),
    "benign_unseen": (external_unseen["result"] == 0).sum(),
    "malicious_unseen": (external_unseen["result"] == 1).sum()
}])

overlap_summary.to_csv(
    "phase21_external_overlap_summary.csv",
    index=False
)

overlap_summary

In [ ]:
# ============================================================
# Phase 21O — Bootstrap CI for External Validation
# ============================================================

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    matthews_corrcoef
)

import numpy as np
import pandas as pd


N_BOOTSTRAP = 5000
RANDOM_STATE = 42

rng = np.random.default_rng(
    RANDOM_STATE
)

bootstrap_records = []

n = len(y_external)

for b in range(N_BOOTSTRAP):

    idx = rng.integers(
        0,
        n,
        size=n
    )

    y_b = y_external[idx]

    pred_b = external_pred[idx]

    prob_b = external_prob[idx]

    # Rare safeguard
    if len(np.unique(y_b)) < 2:
        continue

    bootstrap_records.append({
        "f1":
            f1_score(
                y_b,
                pred_b
            ),

        "auroc":
            roc_auc_score(
                y_b,
                prob_b
            ),

        "mcc":
            matthews_corrcoef(
                y_b,
                pred_b
            )
    })

    if (b + 1) % 1000 == 0:
        print(
            "Completed:",
            b + 1
        )


external_bootstrap = pd.DataFrame(
    bootstrap_records
)


def percentile_ci(series):

    return (
        np.percentile(
            series,
            2.5
        ),
        np.percentile(
            series,
            97.5
        )
    )


f1_ci = percentile_ci(
    external_bootstrap["f1"]
)

auc_ci = percentile_ci(
    external_bootstrap["auroc"]
)

mcc_ci = percentile_ci(
    external_bootstrap["mcc"]
)


print(
    "External F1:",
    external_f1
)

print(
    "95% CI:",
    f1_ci
)

print()

print(
    "External AUROC:",
    external_auc
)

print(
    "95% CI:",
    auc_ci
)

print()

print(
    "External MCC:",
    external_mcc
)

print(
    "95% CI:",
    mcc_ci
)

In [ ]:
# ============================================================
# Phase 21P — Internal vs External Feature Distribution
# ============================================================

feature_shift_records = []

for feature in final_deployment_features:

    internal_values = (
        training_feature_data[
            feature
        ]
    )

    external_values = (
        X_external[
            feature
        ]
    )

    feature_shift_records.append({
        "feature":
            feature,

        "internal_mean":
            internal_values.mean(),

        "external_mean":
            external_values.mean(),

        "internal_median":
            internal_values.median(),

        "external_median":
            external_values.median(),

        "internal_std":
            internal_values.std(),

        "external_std":
            external_values.std()
    })


feature_shift_df = pd.DataFrame(
    feature_shift_records
)

feature_shift_df

In [ ]:
# ============================================================
# Phase 21Q — Save final external-validation analyses
# ============================================================

feature_shift_df.to_csv(
    "phase21_external_feature_shift.csv",
    index=False
)

external_bootstrap.to_csv(
    "phase21_external_bootstrap_results.csv",
    index=False
)

external_ci_summary = pd.DataFrame([
    {
        "metric": "F1",
        "estimate": external_f1,
        "ci_lower": f1_ci[0],
        "ci_upper": f1_ci[1]
    },
    {
        "metric": "AUROC",
        "estimate": external_auc,
        "ci_lower": auc_ci[0],
        "ci_upper": auc_ci[1]
    },
    {
        "metric": "MCC",
        "estimate": external_mcc,
        "ci_lower": mcc_ci[0],
        "ci_upper": mcc_ci[1]
    }
])

external_ci_summary.to_csv(
    "phase21_external_ci_summary.csv",
    index=False
)

external_ci_summary